# ============================================================
# VisionServeAI | Sprint 04
# Phase 1
# ============================================================

In [1]:


from __future__ import annotations

import logging
import os
import random
import sys
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch

warnings.filterwarnings("ignore")

print("Python  :", sys.version.split()[0])
print("NumPy   :", np.__version__)
print("PyTorch :", torch.__version__)
print()
print("Stage 1 - Imports : OK")

Python  : 3.12.13
NumPy   : 2.4.6
PyTorch : 2.10.0+cu128

Stage 1 - Imports : OK


In [2]:
# ============================================================
# Sprint 03 Frozen Artifact Paths
# ============================================================

from pathlib import Path

SPRINT03_ROOT = Path(
    "/kaggle/input/notebooks/anupsharma1730/sprint03-v2-final-frozen/visionserveai/sprint03"
)

CONFIG_DIR = SPRINT03_ROOT / "config"
MANIFEST_DIR = SPRINT03_ROOT / "manifests"
REGISTRY_DIR = SPRINT03_ROOT / "registry"
STATISTICS_DIR = SPRINT03_ROOT / "statistics"

PIPELINE_CONFIG_PATH = CONFIG_DIR / "pipeline_config.json"

DISEASE_REGISTRY_PATH = REGISTRY_DIR / "disease_registry.json"

SPRINT03_CLASS_WEIGHTS_PATH = STATISTICS_DIR / "class_weights.json"
CLASS_DISTRIBUTION_PATH = STATISTICS_DIR / "class_distribution.json"
DATASET_STATISTICS_PATH = STATISTICS_DIR / "dataset_statistics.json"
DATASET_SUMMARY_PATH = STATISTICS_DIR / "dataset_summary.json"

TRAIN_MANIFEST_PATH = MANIFEST_DIR / "train_manifest.parquet"
VAL_MANIFEST_PATH = MANIFEST_DIR / "val_manifest.parquet"
TEST_MANIFEST_PATH = MANIFEST_DIR / "test_manifest.parquet"

print("Sprint03 artifacts configured successfully.")

Sprint03 artifacts configured successfully.


In [3]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 2: Utility Functions
#
# Module: training/config.py
# ============================================================

# -- Logging -------------------------------------------------

LOG_FORMAT      = "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s"
LOG_DATE_FORMAT = "%Y-%m-%d %H:%M:%S"

logging.basicConfig(
    level=logging.INFO,
    format=LOG_FORMAT,
    datefmt=LOG_DATE_FORMAT,
    handlers=[logging.StreamHandler(sys.stdout)],
)

logger = logging.getLogger("visionserveai.sprint04")


# -- Banner printer ------------------------------------------

def print_section(title: str, width: int = 70) -> None:
    """
    Print a standardised section banner.

    Parameters
    ----------
    title : str
        Section title printed after the top rule.
    width : int
        Total banner width.  Default 70.
    """
    print("=" * width)
    print(f"  {title}")
    print("=" * width)


def print_kv(key: str, value: object, indent: int = 2, key_width: int = 32) -> None:
    """
    Print a single key-value pair in a consistent columnar format.

    Parameters
    ----------
    key       : str    Field name.
    value     : object Field value (converted to str).
    indent    : int    Leading spaces.  Default 2.
    key_width : int    Left-column width for alignment.  Default 32.
    """
    pad = " " * indent
    print(f"{pad}{key:<{key_width}} {value}")


def print_check(label: str, passed: bool, detail: str = "") -> None:
    """
    Print a single validation check result with a tick or cross.

    Parameters
    ----------
    label  : str   Human-readable check description.
    passed : bool  True emits a tick, False emits a cross and FAIL tag.
    detail : str   Optional trailing context string.
    """
    icon   = "✔" if passed else "✘  FAIL"
    suffix = f"  ({detail})" if detail else ""
    print(f"  {icon}  {label}{suffix}")


# -- Field-level assertion helpers ---------------------------

def require_positive(value: float, name: str) -> None:
    """Raise ValueError if *value* is not strictly positive."""
    if value <= 0:
        raise ValueError(f"{name} must be > 0; got {value}.")


def require_positive_int(value: int, name: str) -> None:
    """Raise ValueError if *value* is not a strictly positive integer."""
    if not isinstance(value, int) or value <= 0:
        raise ValueError(f"{name} must be a positive integer; got {value!r}.")


def require_in_range(
    value: float, lo: float, hi: float, name: str, inclusive: bool = True
) -> None:
    """
    Raise ValueError if *value* is outside the specified interval.

    Parameters
    ----------
    value     : float  Value to check.
    lo, hi    : float  Lower and upper bounds.
    name      : str    Field name used in the error message.
    inclusive : bool   When True both endpoints are valid.  Default True.
    """
    if inclusive:
        ok = lo <= value <= hi
        interval = f"[{lo}, {hi}]"
    else:
        ok = lo < value < hi
        interval = f"({lo}, {hi})"
    if not ok:
        raise ValueError(f"{name} must be in {interval}; got {value}.")


# -- Verification --------------------------------------------
print("Utility functions defined:")
for fn in [
    "print_section", "print_kv", "print_check",
    "require_positive", "require_positive_int", "require_in_range",
]:
    print(f"  ✔  {fn}")
print()
print("Stage 2 - Utility Functions : OK")

Utility functions defined:
  ✔  print_section
  ✔  print_kv
  ✔  print_check
  ✔  require_positive
  ✔  require_positive_int
  ✔  require_in_range

Stage 2 - Utility Functions : OK


In [4]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 3: TrainingConfig Dataclass
#
# Module: training/config.py -> TrainingConfig
# ============================================================

from dataclasses import FrozenInstanceError


@dataclass(frozen=True)
class TrainingConfig:
    """
    Immutable, validated configuration for the VisionServeAI
    Sprint 04 training pipeline.

    All fields are type-annotated and validated at construction time via
    ``__post_init__``.  ``frozen=True`` enforces write-once semantics --
    a ``FrozenInstanceError`` is raised on any attempted mutation after
    construction.

    Design principles
    -----------------
    * Every field that influences training reproducibility is declared
      here; scattered constants in training-loop code are a latent bug.
    * Filesystem paths are ``pathlib.Path`` objects, never bare strings.
    * ``DEVICE`` is a string (``"auto"``, ``"cuda"``, or ``"cpu"``); the
      actual ``torch.device`` object is resolved at runtime by Stage 5
      so the dataclass stays JSON-serialisable.

    Migration note
    --------------
    Maps directly to ``training/config.py``.  Sprint 03's
    ``PipelineConfig`` (data-pipeline constants) is intentionally
    separate and must not be merged with this class.
    """

    # -- Project identity ------------------------------------
    PROJECT_NAME:       str = "VisionServeAI"
    NOTEBOOK_VERSION:   str = "Sprint04-v1"
    PIPELINE_VERSION:   str = "1.0.0"

    # -- Model identity -------------------------------------
    MODEL_NAME: str = "DenseNet121-ChestXray14"
    BACKBONE:   str = "densenet121"

    # -- Dataset constants (mirrored from Sprint 03 freeze) --
    NUM_CLASSES: int = 14

    # -- Image geometry -------------------------------------
    IMAGE_SIZE: Tuple[int, int] = (224, 224)

    # -- DataLoader -----------------------------------------
    BATCH_SIZE:          int  = 32
    NUM_WORKERS:         int  = 4
    PIN_MEMORY:          bool = True
    PERSISTENT_WORKERS:  bool = True    # requires NUM_WORKERS > 0

    # -- Optimiser ------------------------------------------
    LEARNING_RATE: float = 1e-4
    WEIGHT_DECAY:  float = 1e-5

    # -- Scheduler ------------------------------------------
    LR_SCHEDULER:       str   = "cosine"   # "cosine" | "step" | "plateau"
    LR_WARMUP_EPOCHS:   int   = 3
    LR_MIN:             float = 1e-7       # cosine annealing floor

    # -- Training loop --------------------------------------
    EPOCHS:         int   = 30
    GRADIENT_CLIP:  float = 1.0            # max-norm gradient clipping
    USE_AMP:        bool  = True           # automatic mixed precision

    # -- Validation & early stopping ------------------------
    VALIDATION_INTERVAL:      int = 1      # validate every N epochs
    EARLY_STOPPING_PATIENCE:  int = 7      # stop after N epochs without improvement

    # -- Checkpointing -------------------------------------
    SAVE_BEST_ONLY:       bool = True      # only persist best-val-AUC checkpoint
    CHECKPOINT_FREQUENCY: int  = 5         # also save every N epochs (safety net)

    # -- Reproducibility -----------------------------------
    RANDOM_SEED: int = 42

    # -- Hardware ------------------------------------------
    DEVICE: str = "auto"

    # -- Output directories --------------------------------
    OUTPUT_DIR:        Path = Path("/kaggle/working/visionserveai/sprint04")
    CHECKPOINT_DIR:    Path = Path("/kaggle/working/visionserveai/sprint04/checkpoints")
    LOG_DIR:           Path = Path("/kaggle/working/visionserveai/sprint04/logs")
    MODEL_DIR:         Path = Path("/kaggle/working/visionserveai/sprint04/models")
    VISUALIZATION_DIR: Path = Path("/kaggle/working/visionserveai/sprint04/figures")
    METRICS_DIR:       Path = Path("/kaggle/working/visionserveai/sprint04/metrics")

    # -- Post-construction validation ----------------------

    def __post_init__(self) -> None:
        """Validate all fields at construction time."""
        # Model identity
        if not self.MODEL_NAME:
            raise ValueError("MODEL_NAME must be a non-empty string.")
        if not self.BACKBONE:
            raise ValueError("BACKBONE must be a non-empty string.")

        # Dataset
        require_positive_int(self.NUM_CLASSES, "NUM_CLASSES")

        # Image geometry
        if len(self.IMAGE_SIZE) != 2:
            raise ValueError(f"IMAGE_SIZE must be a 2-tuple; got {self.IMAGE_SIZE}.")
        require_positive_int(self.IMAGE_SIZE[0], "IMAGE_SIZE[0]")
        require_positive_int(self.IMAGE_SIZE[1], "IMAGE_SIZE[1]")

        # DataLoader
        require_positive_int(self.BATCH_SIZE, "BATCH_SIZE")
        if self.NUM_WORKERS < 0:
            raise ValueError(f"NUM_WORKERS must be >= 0; got {self.NUM_WORKERS}.")
        if self.PERSISTENT_WORKERS and self.NUM_WORKERS == 0:
            raise ValueError(
                "PERSISTENT_WORKERS=True requires NUM_WORKERS > 0."
            )

        # Optimiser
        require_positive(self.LEARNING_RATE, "LEARNING_RATE")
        if self.WEIGHT_DECAY < 0:
            raise ValueError(f"WEIGHT_DECAY must be >= 0; got {self.WEIGHT_DECAY}.")

        # Scheduler
        valid_schedulers = {"cosine", "step", "plateau"}
        if self.LR_SCHEDULER not in valid_schedulers:
            raise ValueError(
                f"LR_SCHEDULER must be one of {valid_schedulers}; "
                f"got {self.LR_SCHEDULER!r}."
            )
        if self.LR_WARMUP_EPOCHS < 0:
            raise ValueError(
                f"LR_WARMUP_EPOCHS must be >= 0; got {self.LR_WARMUP_EPOCHS}."
            )
        require_positive(self.LR_MIN, "LR_MIN")
        if self.LR_MIN >= self.LEARNING_RATE:
            raise ValueError(
                f"LR_MIN ({self.LR_MIN}) must be < LEARNING_RATE ({self.LEARNING_RATE})."
            )

        # Training loop
        require_positive_int(self.EPOCHS, "EPOCHS")
        require_positive(self.GRADIENT_CLIP, "GRADIENT_CLIP")

        # Validation & early stopping
        require_positive_int(self.VALIDATION_INTERVAL, "VALIDATION_INTERVAL")
        require_positive_int(self.EARLY_STOPPING_PATIENCE, "EARLY_STOPPING_PATIENCE")

        # Checkpointing
        require_positive_int(self.CHECKPOINT_FREQUENCY, "CHECKPOINT_FREQUENCY")

        # Reproducibility
        require_positive_int(self.RANDOM_SEED, "RANDOM_SEED")

        # Device
        valid_devices = {"auto", "cuda", "cpu"}
        if self.DEVICE not in valid_devices:
            raise ValueError(
                f"DEVICE must be one of {valid_devices}; got {self.DEVICE!r}."
            )

    def display(self) -> None:
        """Pretty-print the configuration to stdout."""
        width = 70
        print("=" * width)
        print(f"  {self.PROJECT_NAME} | {self.NOTEBOOK_VERSION} | Training Configuration")
        print("=" * width)
        sections: Dict[str, List[str]] = {
            "Identity":        ["PROJECT_NAME", "NOTEBOOK_VERSION", "PIPELINE_VERSION",
                                "MODEL_NAME", "BACKBONE"],
            "Dataset":         ["NUM_CLASSES", "IMAGE_SIZE"],
            "DataLoader":      ["BATCH_SIZE", "NUM_WORKERS", "PIN_MEMORY",
                                "PERSISTENT_WORKERS"],
            "Optimiser":       ["LEARNING_RATE", "WEIGHT_DECAY"],
            "Scheduler":       ["LR_SCHEDULER", "LR_WARMUP_EPOCHS", "LR_MIN"],
            "Training Loop":   ["EPOCHS", "GRADIENT_CLIP", "USE_AMP"],
            "Validation":      ["VALIDATION_INTERVAL", "EARLY_STOPPING_PATIENCE"],
            "Checkpointing":   ["SAVE_BEST_ONLY", "CHECKPOINT_FREQUENCY"],
            "Reproducibility": ["RANDOM_SEED"],
            "Hardware":        ["DEVICE"],
            "Directories":     ["OUTPUT_DIR", "CHECKPOINT_DIR", "LOG_DIR",
                                "MODEL_DIR", "VISUALIZATION_DIR", "METRICS_DIR"],
        }
        for section, keys in sections.items():
            print(f"  --- {section} ---")
            for k in keys:
                print_kv(k, getattr(self, k))
        print("=" * width)


# -- Construct the singleton configuration ------------------
cfg = TrainingConfig()

print()
cfg.display()
print()

# -- Immutability check -------------------------------------
try:
    cfg.NUM_CLASSES = 99
    raise AssertionError("Frozen dataclass failed to prevent mutation.")
except FrozenInstanceError:
    print("  Immutability check  : PASSED")

assert cfg.NUM_CLASSES == 14
assert cfg.RANDOM_SEED == 42
assert cfg.BACKBONE == "densenet121"
print("  Field assertions    : PASSED")
print()
print("Stage 3 - TrainingConfig : OK")


  VisionServeAI | Sprint04-v1 | Training Configuration
  --- Identity ---
  PROJECT_NAME                     VisionServeAI
  NOTEBOOK_VERSION                 Sprint04-v1
  PIPELINE_VERSION                 1.0.0
  MODEL_NAME                       DenseNet121-ChestXray14
  BACKBONE                         densenet121
  --- Dataset ---
  NUM_CLASSES                      14
  IMAGE_SIZE                       (224, 224)
  --- DataLoader ---
  BATCH_SIZE                       32
  NUM_WORKERS                      4
  PIN_MEMORY                       True
  PERSISTENT_WORKERS               True
  --- Optimiser ---
  LEARNING_RATE                    0.0001
  WEIGHT_DECAY                     1e-05
  --- Scheduler ---
  LR_SCHEDULER                     cosine
  LR_WARMUP_EPOCHS                 3
  LR_MIN                           1e-07
  --- Training Loop ---
  EPOCHS                           30
  GRADIENT_CLIP                    1.0
  USE_AMP                          True
  --- Validation ---

In [5]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 4: Reproducibility
#
# Module: training/config.py -> set_global_seed(), configure_cudnn()
# ============================================================


def set_global_seed(seed: int) -> None:
    """
    Set all random seeds required for cross-run reproducibility.

    Covers Python built-in ``random``, ``PYTHONHASHSEED``, NumPy,
    and PyTorch (both CPU and all CUDA devices).

    Parameters
    ----------
    seed : int
        Base random seed.  Should match ``TrainingConfig.RANDOM_SEED``.

    Notes
    -----
    ``PYTHONHASHSEED`` must be exported before the interpreter starts to
    fully disable hash randomisation.  Setting it here still seeds the
    current process for NumPy operations; a parent launcher script should
    also export the variable before spawning the training process.
    """
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    logger.info("Global seed set to %d.", seed)


def configure_cudnn(deterministic: bool = True, benchmark: bool = False) -> None:
    """
    Configure cuDNN for deterministic or maximum-throughput operation.

    Parameters
    ----------
    deterministic : bool
        When True, cuDNN selects only deterministic convolution
        algorithms.  Slight throughput cost (~5-15%) on some GPUs.
        Default True.
    benchmark : bool
        When True, cuDNN auto-tunes the fastest algorithm for the first
        batch.  Recommended False when ``deterministic=True`` because
        auto-tuning is itself non-deterministic.  Default False.

    Notes
    -----
    No-op when CUDA is not available.
    """
    if not torch.cuda.is_available():
        logger.info("cuDNN configuration skipped -- CUDA not available.")
        return
    torch.backends.cudnn.deterministic = deterministic
    torch.backends.cudnn.benchmark     = benchmark
    logger.info(
        "cuDNN configured: deterministic=%s, benchmark=%s.",
        deterministic, benchmark,
    )


def init_reproducibility(config: TrainingConfig) -> None:
    """
    Convenience wrapper that applies the full reproducibility stack.

    Call once at notebook startup before any tensor operations.

    Parameters
    ----------
    config : TrainingConfig
        The singleton training configuration.
    """
    set_global_seed(config.RANDOM_SEED)
    configure_cudnn(deterministic=True, benchmark=False)
    logger.info("Reproducibility stack initialised.")


# -- Execute -------------------------------------------------
print_section("STAGE 4 - REPRODUCIBILITY")
print()

init_reproducibility(cfg)

print()
cuda_ok = torch.cuda.is_available()
print_check("Python random seed",  True, f"seed={cfg.RANDOM_SEED}")
print_check("PYTHONHASHSEED",      True, os.environ.get("PYTHONHASHSEED", "not set"))
print_check("NumPy seed",          True, f"seed={cfg.RANDOM_SEED}")
print_check("PyTorch CPU seed",    True, f"seed={cfg.RANDOM_SEED}")
print_check(
    "PyTorch CUDA seed", cuda_ok,
    "applied" if cuda_ok else "CUDA not present -- skipped",
)
print_check(
    "cuDNN deterministic", cuda_ok,
    "True" if cuda_ok else "N/A",
)
print_check("cuDNN benchmark", True, "False (deterministic mode)")
print()
print("Stage 4 - Reproducibility : OK")

  STAGE 4 - REPRODUCIBILITY

2026-07-02 00:55:36 | INFO     | visionserveai.sprint04 | Global seed set to 42.
2026-07-02 00:55:36 | INFO     | visionserveai.sprint04 | cuDNN configured: deterministic=True, benchmark=False.
2026-07-02 00:55:36 | INFO     | visionserveai.sprint04 | Reproducibility stack initialised.

  ✔  Python random seed  (seed=42)
  ✔  PYTHONHASHSEED  (42)
  ✔  NumPy seed  (seed=42)
  ✔  PyTorch CPU seed  (seed=42)
  ✔  PyTorch CUDA seed  (applied)
  ✔  cuDNN deterministic  (True)
  ✔  cuDNN benchmark  (False (deterministic mode))

Stage 4 - Reproducibility : OK


In [6]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 5: Device Detection
#
# Module: training/config.py -> DeviceInfo, detect_device()
# ============================================================


@dataclass
class DeviceInfo:
    """
    Snapshot of runtime hardware and software environment.

    Attributes
    ----------
    device : torch.device
        The resolved training device.
    device_name : str
        Human-readable hardware name (GPU model string or ``"CPU"``).
    cuda_available : bool
    cuda_version : str
        CUDA runtime version string, or ``"N/A"`` on CPU.
    torch_version : str
    num_gpus : int
        Number of visible CUDA devices; 0 on CPU.
    amp_supported : bool
        True when the device supports ``torch.cuda.amp.autocast`` and
        ``TrainingConfig.USE_AMP`` is True.
    vram_gb : float
        Total GPU VRAM in GiB for device 0; 0.0 on CPU.
    """
    device:         torch.device
    device_name:    str
    cuda_available: bool
    cuda_version:   str
    torch_version:  str
    num_gpus:       int
    amp_supported:  bool
    vram_gb:        float


def detect_device(config: TrainingConfig) -> DeviceInfo:
    """
    Detect the training device and collect environment metadata.

    Resolution logic
    ----------------
    * ``config.DEVICE == "auto"``  -- use CUDA if available, else CPU.
    * ``config.DEVICE == "cuda"``  -- assert CUDA is present (hard fail).
    * ``config.DEVICE == "cpu"``   -- use CPU unconditionally.

    AMP is flagged as supported when:
    (1) The resolved device is CUDA.
    (2) ``config.USE_AMP`` is True.
    (3) The GPU compute capability is >= 7.0 (Volta/Turing/Ampere,
        which introduced Tensor Cores).  On older GPUs AMP still
        functions but yields no throughput benefit.

    Parameters
    ----------
    config : TrainingConfig

    Returns
    -------
    DeviceInfo
    """
    cuda_available = torch.cuda.is_available()

    # -- Resolve device -------------------------------------
    if config.DEVICE == "auto":
        device = torch.device("cuda" if cuda_available else "cpu")
    elif config.DEVICE == "cuda":
        if not cuda_available:
            raise RuntimeError(
                'DEVICE="cuda" was requested but CUDA is not available.'
            )
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")

    # -- Gather GPU metadata --------------------------------
    num_gpus      = torch.cuda.device_count() if cuda_available else 0
    cuda_version  = torch.version.cuda if cuda_available else "N/A"

    if device.type == "cuda":
        device_name   = torch.cuda.get_device_name(0)
        vram_bytes    = torch.cuda.get_device_properties(0).total_memory
        vram_gb       = vram_bytes / (1024 ** 3)
        cc_major, cc_minor = torch.cuda.get_device_capability(0)
        compute_cap   = cc_major + cc_minor / 10
        amp_supported = config.USE_AMP and (compute_cap >= 7.0)
    else:
        device_name   = "CPU"
        vram_gb       = 0.0
        amp_supported = False

    return DeviceInfo(
        device=device,
        device_name=device_name,
        cuda_available=cuda_available,
        cuda_version=cuda_version,
        torch_version=torch.__version__,
        num_gpus=num_gpus,
        amp_supported=amp_supported,
        vram_gb=vram_gb,
    )


# -- Execute -------------------------------------------------
print_section("STAGE 5 - DEVICE DETECTION")
print()

device_info = detect_device(cfg)
DEVICE = device_info.device    # torch.device -- consumed by all subsequent stages

print_kv("Resolved device",  DEVICE)
print_kv("Device name",      device_info.device_name)
print_kv("CUDA available",   device_info.cuda_available)
print_kv("CUDA version",     device_info.cuda_version)
print_kv("PyTorch version",  device_info.torch_version)
print_kv("Number of GPUs",   device_info.num_gpus)
print_kv(
    "GPU VRAM",
    f"{device_info.vram_gb:.2f} GiB" if device_info.vram_gb else "N/A",
)
print_kv("AMP supported",    device_info.amp_supported)
print()

if not device_info.cuda_available:
    logger.warning(
        "No CUDA device detected.  Training will proceed on CPU.  "
        "Performance will be significantly slower than on GPU."
    )

if cfg.USE_AMP and not device_info.amp_supported:
    logger.warning(
        "USE_AMP=True but AMP is not supported on this device.  "
        "The training engine will disable AMP at runtime."
    )

print("Stage 5 - Device Detection : OK")

  STAGE 5 - DEVICE DETECTION

  Resolved device                  cuda
  Device name                      Tesla T4
  CUDA available                   True
  CUDA version                     12.8
  PyTorch version                  2.10.0+cu128
  Number of GPUs                   2
  GPU VRAM                         14.56 GiB
  AMP supported                    True

Stage 5 - Device Detection : OK


In [7]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 6: Directory Creation
#
# Module: training/config.py -> create_output_dirs()
# ============================================================


def create_output_dirs(config: TrainingConfig) -> Dict[str, Path]:
    """
    Create all Sprint 04 output directories declared in ``TrainingConfig``.

    Uses ``exist_ok=True`` so this function is idempotent -- safe to
    call multiple times or on an already-populated Kaggle working directory.

    Parameters
    ----------
    config : TrainingConfig

    Returns
    -------
    Dict[str, Path]
        Logical name to resolved ``Path`` mapping for downstream consumers
        that need to construct artifact file paths without re-reading config.
    """
    dirs: Dict[str, Path] = {
        "output":       config.OUTPUT_DIR,
        "checkpoints":  config.CHECKPOINT_DIR,
        "logs":         config.LOG_DIR,
        "models":       config.MODEL_DIR,
        "figures":      config.VISUALIZATION_DIR,
        "metrics":      config.METRICS_DIR,
    }
    for name, path in dirs.items():
        path.mkdir(parents=True, exist_ok=True)
        logger.info("Directory ready: %s  ->  %s", name, path)
    return dirs


def _print_dir_tree(root: Path, dirs: Dict[str, Path]) -> None:
    """
    Print a compact tree of the output directory structure.

    Parameters
    ----------
    root : Path
        The root output directory (used as the tree root label).
    dirs : Dict[str, Path]
        Logical-name to path mapping from ``create_output_dirs``.
    """
    print(f"  {root}/")
    entries = sorted(
        [(name, path) for name, path in dirs.items() if path != root],
        key=lambda kv: kv[1],
    )
    for i, (name, path) in enumerate(entries):
        connector = "└── " if i == len(entries) - 1 else "├── "
        rel        = path.relative_to(root)
        exists_mark = "✔" if path.exists() else "✘"
        print(f"  {connector}{rel}/  [{name}]  {exists_mark}")


# -- Execute -------------------------------------------------
print_section("STAGE 6 - DIRECTORY CREATION")
print()

output_dirs = create_output_dirs(cfg)

print()
_print_dir_tree(cfg.OUTPUT_DIR, output_dirs)
print()

# Verify every directory physically exists after creation
for name, path in output_dirs.items():
    assert path.exists() and path.is_dir(), (
        f"Directory not created or not a directory: {path}"
    )

print()
print("Stage 6 - Directory Creation : OK")

  STAGE 6 - DIRECTORY CREATION

2026-07-02 00:55:36 | INFO     | visionserveai.sprint04 | Directory ready: output  ->  /kaggle/working/visionserveai/sprint04
2026-07-02 00:55:36 | INFO     | visionserveai.sprint04 | Directory ready: checkpoints  ->  /kaggle/working/visionserveai/sprint04/checkpoints
2026-07-02 00:55:36 | INFO     | visionserveai.sprint04 | Directory ready: logs  ->  /kaggle/working/visionserveai/sprint04/logs
2026-07-02 00:55:36 | INFO     | visionserveai.sprint04 | Directory ready: models  ->  /kaggle/working/visionserveai/sprint04/models
2026-07-02 00:55:36 | INFO     | visionserveai.sprint04 | Directory ready: figures  ->  /kaggle/working/visionserveai/sprint04/figures
2026-07-02 00:55:36 | INFO     | visionserveai.sprint04 | Directory ready: metrics  ->  /kaggle/working/visionserveai/sprint04/metrics

  /kaggle/working/visionserveai/sprint04/
  ├── checkpoints/  [checkpoints]  ✔
  ├── figures/  [figures]  ✔
  ├── logs/  [logs]  ✔
  ├── metrics/  [metrics]  ✔
  └── 

In [8]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 7: Configuration Validation
#
# Module: training/config.py -> validate_training_config()
# ============================================================


def validate_training_config(
    config: TrainingConfig,
    device_info: DeviceInfo,
    output_dirs: Dict[str, Path],
) -> None:
    """
    Engineering-grade post-construction validation of the full training
    configuration.

    This function runs AFTER ``TrainingConfig.__post_init__`` (which
    validates each field in isolation) and checks cross-field and runtime
    constraints that require information unavailable at dataclass
    construction time: resolved device, filesystem state, cuDNN status.

    Raises
    ------
    AssertionError
        Raised after printing ALL failing checks so the engineer sees
        every problem in one pass, not just the first one.

    Parameters
    ----------
    config      : TrainingConfig   Frozen training configuration.
    device_info : DeviceInfo       Runtime hardware snapshot (Stage 5).
    output_dirs : Dict[str, Path]  Created directory map (Stage 6).
    """
    failures: List[str] = []

    def _check(condition: bool, label: str, detail: str = "") -> None:
        print_check(label, condition, detail)
        if not condition:
            failures.append(label)

    # -- Field-level cross-checks ----------------------------
    print("  Configuration field validation")
    print("  " + "-" * 60)

    _check(
        config.BATCH_SIZE >= 1,
        "BATCH_SIZE >= 1",
        str(config.BATCH_SIZE),
    )
    _check(
        all(s > 0 for s in config.IMAGE_SIZE),
        "IMAGE_SIZE all positive",
        str(config.IMAGE_SIZE),
    )
    _check(
        config.NUM_CLASSES == 14,
        "NUM_CLASSES == 14 (NIH ChestXray14 contract)",
        str(config.NUM_CLASSES),
    )
    _check(
        1e-6 <= config.LEARNING_RATE <= 1e-1,
        "LEARNING_RATE in [1e-6, 1e-1]",
        f"{config.LEARNING_RATE:.2e}",
    )
    _check(
        config.LR_MIN < config.LEARNING_RATE,
        "LR_MIN < LEARNING_RATE",
        f"{config.LR_MIN:.2e} < {config.LEARNING_RATE:.2e}",
    )
    _check(
        config.WEIGHT_DECAY >= 0,
        "WEIGHT_DECAY >= 0",
        f"{config.WEIGHT_DECAY:.2e}",
    )
    _check(
        config.GRADIENT_CLIP > 0,
        "GRADIENT_CLIP > 0",
        str(config.GRADIENT_CLIP),
    )
    _check(
        config.EPOCHS >= 1,
        "EPOCHS >= 1",
        str(config.EPOCHS),
    )
    _check(
        config.EARLY_STOPPING_PATIENCE < config.EPOCHS,
        "EARLY_STOPPING_PATIENCE < EPOCHS",
        f"{config.EARLY_STOPPING_PATIENCE} < {config.EPOCHS}",
    )
    _check(
        1 <= config.VALIDATION_INTERVAL <= config.EPOCHS,
        "VALIDATION_INTERVAL in [1, EPOCHS]",
        str(config.VALIDATION_INTERVAL),
    )
    _check(
        config.RANDOM_SEED == 42,
        "RANDOM_SEED == 42 (project convention)",
        str(config.RANDOM_SEED),
    )

    # -- Runtime checks -------------------------------------
    print()
    print("  Runtime / hardware validation")
    print("  " + "-" * 60)

    _check(
        device_info.device.type in {"cuda", "cpu"},
        "Resolved device is cuda or cpu",
        str(device_info.device),
    )
    amp_status = (
        "enabled and supported" if device_info.amp_supported
        else "requested but not supported -- will be disabled at runtime"
             if config.USE_AMP
        else "disabled"
    )
    _check(True, "AMP configuration", amp_status)

    # -- Directory checks -----------------------------------
    print()
    print("  Directory validation")
    print("  " + "-" * 60)

    for name, path in output_dirs.items():
        _check(
            path.exists() and path.is_dir(),
            f"Dir exists: {name}",
            str(path),
        )

    # -- DataLoader compatibility ---------------------------
    print()
    print("  DataLoader compatibility")
    print("  " + "-" * 60)

    _check(
        not (config.PERSISTENT_WORKERS and config.NUM_WORKERS == 0),
        "PERSISTENT_WORKERS compatible with NUM_WORKERS",
        f"persistent={config.PERSISTENT_WORKERS}, workers={config.NUM_WORKERS}",
    )

    # -- Seed check -----------------------------------------
    import os as _os
    _check(
        _os.environ.get("PYTHONHASHSEED") == str(config.RANDOM_SEED),
        "PYTHONHASHSEED matches RANDOM_SEED",
        _os.environ.get("PYTHONHASHSEED", "not set"),
    )

    # -- Final gate -----------------------------------------
    print()
    if failures:
        raise AssertionError(
            f"Configuration validation FAILED ({len(failures)} checks): "
            + ", ".join(failures)
        )


# -- Execute -------------------------------------------------
print_section("STAGE 7 - CONFIGURATION VALIDATION")
print()

validate_training_config(
    config=cfg,
    device_info=device_info,
    output_dirs=output_dirs,
)

print()
print("Stage 7 - Configuration Validation : OK")

  STAGE 7 - CONFIGURATION VALIDATION

  Configuration field validation
  ------------------------------------------------------------
  ✔  BATCH_SIZE >= 1  (32)
  ✔  IMAGE_SIZE all positive  ((224, 224))
  ✔  NUM_CLASSES == 14 (NIH ChestXray14 contract)  (14)
  ✔  LEARNING_RATE in [1e-6, 1e-1]  (1.00e-04)
  ✔  LR_MIN < LEARNING_RATE  (1.00e-07 < 1.00e-04)
  ✔  WEIGHT_DECAY >= 0  (1.00e-05)
  ✔  GRADIENT_CLIP > 0  (1.0)
  ✔  EPOCHS >= 1  (30)
  ✔  EARLY_STOPPING_PATIENCE < EPOCHS  (7 < 30)
  ✔  VALIDATION_INTERVAL in [1, EPOCHS]  (1)
  ✔  RANDOM_SEED == 42 (project convention)  (42)

  Runtime / hardware validation
  ------------------------------------------------------------
  ✔  Resolved device is cuda or cpu  (cuda)
  ✔  AMP configuration  (enabled and supported)

  Directory validation
  ------------------------------------------------------------
  ✔  Dir exists: output  (/kaggle/working/visionserveai/sprint04)
  ✔  Dir exists: checkpoints  (/kaggle/working/visionserveai/sprint04/

In [9]:
# ============================================================
# VisionServeAI | Sprint 04
# Phase 1 -- Summary
# ============================================================

print_section("VisionServeAI | Sprint 04 | Phase 1 -- COMPLETE")
print()
print("  Stage 1  -- Imports                  : OK")
print("  Stage 2  -- Utility Functions        : OK")
print("  Stage 3  -- TrainingConfig Dataclass : OK")
print("  Stage 4  -- Reproducibility          : OK")
print("  Stage 5  -- Device Detection         : OK")
print("  Stage 6  -- Directory Creation       : OK")
print("  Stage 7  -- Configuration Validation : OK")
print()
print("  Runtime environment:")
print_kv("  Device",      DEVICE)
print_kv("  Device name", device_info.device_name)
print_kv("  AMP",         "enabled" if device_info.amp_supported else "disabled")
print_kv("  Seed",        cfg.RANDOM_SEED)
print()
print("  Migration map  ->  training/config.py:")
print("    TrainingConfig       -- frozen configuration dataclass")
print("    DeviceInfo           -- runtime hardware snapshot dataclass")
print("    set_global_seed()    -- full seed stack")
print("    configure_cudnn()    -- cuDNN determinism")
print("    init_reproducibility() -- convenience wrapper")
print("    detect_device()      -- device resolution and metadata")
print("    create_output_dirs() -- idempotent directory factory")
print("    validate_training_config() -- cross-field runtime gate")
print()
print("  STOP -- Phase 1 complete.")
print("  Phase 2 (Loss / Optimiser / Scheduler) begins in the next notebook.")
print()
print("=" * 70)

  VisionServeAI | Sprint 04 | Phase 1 -- COMPLETE

  Stage 1  -- Imports                  : OK
  Stage 2  -- Utility Functions        : OK
  Stage 3  -- TrainingConfig Dataclass : OK
  Stage 4  -- Reproducibility          : OK
  Stage 5  -- Device Detection         : OK
  Stage 6  -- Directory Creation       : OK
  Stage 7  -- Configuration Validation : OK

  Runtime environment:
    Device                         cuda
    Device name                    Tesla T4
    AMP                            enabled
    Seed                           42

  Migration map  ->  training/config.py:
    TrainingConfig       -- frozen configuration dataclass
    DeviceInfo           -- runtime hardware snapshot dataclass
    set_global_seed()    -- full seed stack
    configure_cudnn()    -- cuDNN determinism
    init_reproducibility() -- convenience wrapper
    detect_device()      -- device resolution and metadata
    create_output_dirs() -- idempotent directory factory
    validate_training_config() 

# ============================================================
# VisionServeAI | Sprint 04
# Stage 8: Model Factory
#
# Module: training/models/backbones.py, training/models/classifier.py
# ============================================================

In [10]:
# -- Purpose ---------------------------------------------------
# Establish a single entry point for backbone construction so the
# Trainer (Phase 4) never needs to know which architecture is in use.
# Future backbones are added as registry rows, not new branches of
# trainer logic. Per MASTER_ARCHITECTURE.md Section 18.2, the
# production model is timm backbone -> classification head, with
# DenseNet-121 as the original-paper baseline. Only DenseNet-121 is
# activated this sprint; the other four are pre-registered with
# status="PLANNED" so the factory's guard-rail logic is exercised
# end to end even though their weights are never downloaded.

try:
    import timm
except ImportError as exc:
    raise ImportError(
        "The 'timm' package is required for Stage 8 (Model Factory) but "
        "is not installed.  Install it with:  pip install timm"
    ) from exc

import torch.nn as nn


# -- Backbone registry -----------------------------------------

@dataclass(frozen=True)
class BackboneSpec:
    """
    Immutable specification for one supported backbone architecture.

    Attributes
    ----------
    timm_name : str
        Model identifier passed to ``timm.create_model``.
    family : str
        Architecture family label, used only for reporting.
    feature_dim : int
        Width of the penultimate (pre-classifier) feature vector.
        Used to validate the replacement head without a throwaway
        forward pass.
    classifier_attr : str
        Dotted attribute path to the backbone's native ImageNet
        classifier (e.g. ``"classifier"`` or ``"head.fc"``).  Consumed
        by ``replace_classifier`` to locate and swap the head.
    target_layer : str
        Dotted path to the last convolutional block.  Not consumed in
        Phase 2; reserved for the Grad-CAM explainability module
        (``inference/explainability/gradcam.py``).
    status : str
        ``"ACTIVE"`` -- may be instantiated by ``get_backbone``.
        ``"PLANNED"`` -- registered for forward compatibility;
        ``get_backbone`` raises ``NotImplementedError`` until a future
        sprint flips this flag.
    """
    timm_name:       str
    family:          str
    feature_dim:     int
    classifier_attr: str
    target_layer:    str
    status:          str


BACKBONE_REGISTRY: Dict[str, BackboneSpec] = {
    "densenet121": BackboneSpec(
        timm_name="densenet121", family="DenseNet", feature_dim=1024,
        classifier_attr="classifier", target_layer="features.denseblock4",
        status="ACTIVE",
    ),
    "densenet169": BackboneSpec(
        timm_name="densenet169", family="DenseNet", feature_dim=1664,
        classifier_attr="classifier", target_layer="features.denseblock4",
        status="PLANNED",
    ),
    "resnet50": BackboneSpec(
        timm_name="resnet50", family="ResNet", feature_dim=2048,
        classifier_attr="fc", target_layer="layer4",
        status="PLANNED",
    ),
    "efficientnet_b0": BackboneSpec(
        timm_name="efficientnet_b0", family="EfficientNet", feature_dim=1280,
        classifier_attr="classifier", target_layer="conv_head",
        status="PLANNED",
    ),
    "convnext_tiny": BackboneSpec(
        timm_name="convnext_tiny", family="ConvNeXt", feature_dim=768,
        classifier_attr="head.fc", target_layer="stages.3",
        status="PLANNED",
    ),
}


# -- Dotted-path module access -----------------------------------

def _get_module_by_path(module: nn.Module, path: str) -> nn.Module:
    """Resolve a dotted attribute path (e.g. ``"head.fc"``) to a submodule."""
    obj = module
    for part in path.split("."):
        obj = getattr(obj, part)
    return obj


def _set_module_by_path(module: nn.Module, path: str, new_submodule: nn.Module) -> None:
    """Replace the submodule at a dotted attribute path, in place."""
    parts = path.split(".")
    obj = module
    for part in parts[:-1]:
        obj = getattr(obj, part)
    setattr(obj, parts[-1], new_submodule)


# -- Factory functions --------------------------------------------

def get_backbone(backbone_name: str, pretrained: bool = True) -> nn.Module:
    """
    Instantiate a registered backbone with its native ImageNet head intact.

    Parameters
    ----------
    backbone_name : str   Key into ``BACKBONE_REGISTRY``.
    pretrained    : bool  Load ImageNet-1k pretrained weights.  Default True.

    Returns
    -------
    nn.Module
        The raw timm model; classifier head not yet replaced.

    Raises
    ------
    KeyError              ``backbone_name`` is not registered.
    NotImplementedError    Registered but not yet activated
                           (``status == "PLANNED"``).
    """
    if backbone_name not in BACKBONE_REGISTRY:
        raise KeyError(
            f"Unknown backbone {backbone_name!r}.  "
            f"Registered backbones: {sorted(BACKBONE_REGISTRY)}."
        )
    spec = BACKBONE_REGISTRY[backbone_name]
    if spec.status != "ACTIVE":
        raise NotImplementedError(
            f"Backbone {backbone_name!r} is registered ({spec.family}) but "
            f"not yet activated in Sprint 04 (status={spec.status!r}).  "
            f"Only DenseNet-121 is instantiated this sprint per the Phase 2 "
            f"scope freeze."
        )
    return timm.create_model(spec.timm_name, pretrained=pretrained)


def replace_classifier(
    model: nn.Module, backbone_name: str, num_classes: int, dropout: float = 0.3,
) -> Tuple[nn.Module, int]:
    """
    Replace a backbone's native ImageNet classifier with a multilabel head.

    Parameters
    ----------
    model         : nn.Module  A backbone from ``get_backbone`` (head intact).
    backbone_name : str        Registry key, resolves the head's attribute path.
    num_classes   : int        Output logit count (NIH ChestXray14: 14).
    dropout       : float      Dropout before the new linear layer.  Default
                               0.3 per MASTER_ARCHITECTURE.md Section 18.2.

    Returns
    -------
    Tuple[nn.Module, int]
        The mutated model and the resolved ``in_features`` of the
        original classifier (used for logging / verification).
    """
    spec = BACKBONE_REGISTRY[backbone_name]
    old_head = _get_module_by_path(model, spec.classifier_attr)
    in_features = old_head.in_features
    new_head = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, num_classes),
    )
    _set_module_by_path(model, spec.classifier_attr, new_head)
    return model, in_features


class ChestXrayClassifier(nn.Module):
    """
    Thin named wrapper around a backbone-with-replaced-head.

    The forward pass is a direct pass-through -- the wrapper's value is
    a stable public class name (matters for checkpoint introspection,
    MLflow model logging, and ONNX export naming in later phases) and a
    place to carry model metadata that isn't a learnable parameter.

    Maps to ``training/models/classifier.py -> ChestXrayClassifier``
    per MASTER_ARCHITECTURE.md Section 18.1.
    """

    def __init__(self, backbone: nn.Module, backbone_name: str, num_classes: int) -> None:
        super().__init__()
        self.backbone = backbone
        self.backbone_name = backbone_name
        self.num_classes = num_classes

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass.  Returns raw logits of shape [B, num_classes]."""
        return self.backbone(x)


def build_model(
    backbone_name: str = "densenet121",
    num_classes: int = 14,
    pretrained: bool = True,
    dropout: float = 0.3,
) -> ChestXrayClassifier:
    """
    End-to-end model factory: backbone + multilabel classification head.

    This is the single function the Trainer (Phase 4) will call; it has
    no knowledge of timm, dotted attribute paths, or per-architecture
    quirks -- all of that is encapsulated above.

    Returns
    -------
    ChestXrayClassifier
    """
    raw_model = get_backbone(backbone_name, pretrained=pretrained)
    raw_model, _ = replace_classifier(raw_model, backbone_name, num_classes, dropout)
    return ChestXrayClassifier(raw_model, backbone_name, num_classes)


def count_parameters(model: nn.Module) -> Dict[str, int]:
    """Return total / trainable / frozen parameter counts for *model*."""
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total": total, "trainable": trainable, "frozen": total - trainable}


def summarize_model(model: nn.Module, backbone_name: str) -> Dict[str, object]:
    """
    Build a JSON-serialisable summary dict for a constructed model.

    The shape of this dict is the contract consumed by the
    ``*_summary.json`` artifacts written in Stage 12.
    """
    spec = BACKBONE_REGISTRY[backbone_name]
    counts = count_parameters(model)
    return {
        "backbone_name": backbone_name,
        "family":        spec.family,
        "timm_name":     spec.timm_name,
        "feature_dim":   spec.feature_dim,
        "total_parameters":     counts["total"],
        "trainable_parameters": counts["trainable"],
        "frozen_parameters":    counts["frozen"],
    }


# -- Execute -------------------------------------------------
print_section("STAGE 8 - MODEL FACTORY")
print()

print("  Backbone registry")
print("  " + "-" * 60)
print(f"  {'name':<18}{'family':<14}{'feature_dim':<13}{'status':<10}")
for name, spec in BACKBONE_REGISTRY.items():
    print(f"  {name:<18}{spec.family:<14}{spec.feature_dim:<13}{spec.status:<10}")
print()

active  = [n for n, s in BACKBONE_REGISTRY.items() if s.status == "ACTIVE"]
planned = [n for n, s in BACKBONE_REGISTRY.items() if s.status == "PLANNED"]

print("  Factory guard-rail verification")
print("  " + "-" * 60)
print_check("Registry has 5 backbone families", len(BACKBONE_REGISTRY) == 5, str(len(BACKBONE_REGISTRY)))
print_check("Exactly one ACTIVE backbone", len(active) == 1, str(active))
print_check("ACTIVE backbone matches cfg.BACKBONE", active == [cfg.BACKBONE], f"{active} vs {[cfg.BACKBONE]}")
print_check("4 backbones PLANNED for future sprints", len(planned) == 4, str(planned))

guard_rail_raised = False
try:
    get_backbone("resnet50", pretrained=False)
except NotImplementedError:
    guard_rail_raised = True
print_check("get_backbone() blocks PLANNED backbones", guard_rail_raised)

unknown_raised = False
try:
    get_backbone("not_a_real_backbone")
except KeyError:
    unknown_raised = True
print_check("get_backbone() rejects unknown names", unknown_raised)

for fn in [build_model, get_backbone, replace_classifier, count_parameters, summarize_model]:
    print_check(f"{fn.__name__}() defined and callable", callable(fn))

# Smoke-test build_model() end-to-end with pretrained=False -- this is
# a STRUCTURAL check only (shapes, wiring, class identity).  It is
# deliberately disposed of immediately: the real pretrained model is
# built once, deliberately, in Stage 9 -- calling build_model() again
# there would trigger a second, redundant weight download.
_smoke_model = build_model(cfg.BACKBONE, num_classes=cfg.NUM_CLASSES, pretrained=False, dropout=0.3)
with torch.no_grad():
    _smoke_output = _smoke_model(torch.randn(2, 3, *cfg.IMAGE_SIZE))
print_check("build_model() smoke test returns ChestXrayClassifier", isinstance(_smoke_model, ChestXrayClassifier))
print_check("build_model() smoke test output shape", tuple(_smoke_output.shape) == (2, cfg.NUM_CLASSES), str(tuple(_smoke_output.shape)))
del _smoke_model, _smoke_output

print()
print("Stage 8 - Model Factory : OK")

  STAGE 8 - MODEL FACTORY

  Backbone registry
  ------------------------------------------------------------
  name              family        feature_dim  status    
  densenet121       DenseNet      1024         ACTIVE    
  densenet169       DenseNet      1664         PLANNED   
  resnet50          ResNet        2048         PLANNED   
  efficientnet_b0   EfficientNet  1280         PLANNED   
  convnext_tiny     ConvNeXt      768          PLANNED   

  Factory guard-rail verification
  ------------------------------------------------------------
  ✔  Registry has 5 backbone families  (5)
  ✔  Exactly one ACTIVE backbone  (['densenet121'])
  ✔  ACTIVE backbone matches cfg.BACKBONE  (['densenet121'] vs ['densenet121'])
  ✔  4 backbones PLANNED for future sprints  (['densenet169', 'resnet50', 'efficientnet_b0', 'convnext_tiny'])
  ✔  get_backbone() blocks PLANNED backbones
  ✔  get_backbone() rejects unknown names
  ✔  build_model() defined and callable
  ✔  get_backbone() defined and

In [11]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 9: Backbone Initialization
#
# Module: training/models/backbones.py -> get_backbone()
# ============================================================

# -- Purpose ---------------------------------------------------
# Instantiate the ACTIVE backbone (DenseNet-121, per cfg.BACKBONE) with
# its pretrained ImageNet-1k weights and its native 1000-class head
# still attached, and prove the weights actually loaded before any
# architectural surgery happens in Stage 10.

import time as _time

print_section("STAGE 9 - BACKBONE INITIALIZATION")
print()

backbone_name = cfg.BACKBONE
spec = BACKBONE_REGISTRY[backbone_name]

# -- Load pretrained backbone ---------------------------------
print(f"  Loading ImageNet-1k weights for {spec.timm_name!r} ...")
_t0 = _time.time()
backbone = get_backbone(backbone_name, pretrained=True)
_load_seconds = _time.time() - _t0
backbone.eval()
print(f"  Loaded in {_load_seconds:.2f}s.")
print()

# -- Verification: weights loaded (not silently random) ----------
# Sanity check against a freshly random-initialised model of the SAME
# architecture: real pretrained weights produce a markedly different
# distribution than a fresh init, so this catches the failure mode
# where pretrained=True silently falls back to random weights (e.g.
# a swallowed download error upstream).
_random_init = timm.create_model(spec.timm_name, pretrained=False)
_pretrained_layer = _get_module_by_path(backbone, "features.conv0").weight
_random_layer     = _get_module_by_path(_random_init, "features.conv0").weight
weights_differ_from_random_init = not torch.allclose(_pretrained_layer, _random_layer)
del _random_init  # discard immediately -- only needed for this comparison

# -- Verification: architecture identity ------------------------
architecture_ok = "densenet" in type(backbone).__name__.lower()

# -- Verification: classifier location & feature dimension -------
classifier_module   = _get_module_by_path(backbone, spec.classifier_attr)
in_features          = classifier_module.in_features
out_features         = classifier_module.out_features
feature_dim_matches  = in_features == spec.feature_dim

# -- Verification: dummy forward pass ----------------------------
dummy_batch = torch.randn(2, 3, *cfg.IMAGE_SIZE)
with torch.no_grad():
    dummy_output = backbone(dummy_batch)
forward_ok = tuple(dummy_output.shape) == (2, out_features)

# -- Verification: parameter count -------------------------------
backbone_param_counts = count_parameters(backbone)

# -- Report ------------------------------------------------------
print("  Architecture")
print("  " + "-" * 60)
print_kv("Backbone name",   backbone_name)
print_kv("timm identifier", spec.timm_name)
print_kv("Family",          spec.family)
print_kv("Class",           type(backbone).__name__)
print()
print("  Classifier location (pre-replacement)")
print("  " + "-" * 60)
print_kv("Attribute path",              spec.classifier_attr)
print_kv("Module",                      classifier_module)
print_kv("in_features",                 in_features)
print_kv("out_features (ImageNet-1k)",  out_features)
print()
print("  Parameters")
print("  " + "-" * 60)
print_kv("Total",     f"{backbone_param_counts['total']:,}")
print_kv("Trainable", f"{backbone_param_counts['trainable']:,}")
print()
print("  Verification")
print("  " + "-" * 60)
print_check("Pretrained weights loaded (differ from random init)", weights_differ_from_random_init)
print_check("Architecture matches registry family (DenseNet)", architecture_ok)
print_check("Feature extractor dimension == registry spec", feature_dim_matches, f"{in_features} == {spec.feature_dim}")
print_check("Classifier location resolved to nn.Linear", isinstance(classifier_module, nn.Linear), spec.classifier_attr)
print_check("Dummy forward pass shape", forward_ok, str(tuple(dummy_output.shape)))
print_check("Parameter count is positive", backbone_param_counts["total"] > 0, f"{backbone_param_counts['total']:,}")

assert weights_differ_from_random_init, "Pretrained weights failed to load."
assert architecture_ok, "Architecture mismatch against registry."
assert feature_dim_matches, "Feature dimension mismatch against registry."
assert forward_ok, "Dummy forward pass produced an unexpected shape."

print()
print("Stage 9 - Backbone Initialization : OK")

  STAGE 9 - BACKBONE INITIALIZATION

  Loading ImageNet-1k weights for 'densenet121' ...
2026-07-02 00:55:46 | INFO     | timm.models._builder | Loading pretrained weights from Hugging Face hub (timm/densenet121.ra_in1k)
2026-07-02 00:55:46 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/timm/densenet121.ra_in1k/resolve/main/model.safetensors "HTTP/1.1 302 Found"


2026-07-02 00:55:46 | WARNING  | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-07-02 00:55:46 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/timm/densenet121.ra_in1k/xet-read-token/92007b6200e0b4a4fe68cb4e3947022a928aaaae "HTTP/1.1 200 OK"


model.safetensors:   0%|          | 0.00/32.3M [00:00<?, ?B/s]

2026-07-02 00:55:47 | INFO     | timm.models._hub | [timm/densenet121.ra_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
  Loaded in 2.04s.

  Architecture
  ------------------------------------------------------------
  Backbone name                    densenet121
  timm identifier                  densenet121
  Family                           DenseNet
  Class                            DenseNet

  Classifier location (pre-replacement)
  ------------------------------------------------------------
  Attribute path                   classifier
  Module                           Linear(in_features=1024, out_features=1000, bias=True)
  in_features                      1024
  out_features (ImageNet-1k)       1000

  Parameters
  ------------------------------------------------------------
  Total                            7,978,856
  Trainable                        7,978,856

  Verification
  ------------------------

In [12]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 10: Classification Head
#
# Module: training/models/classifier.py -> replace_classifier(), ChestXrayClassifier
# ============================================================

# -- Purpose ---------------------------------------------------
# Replace DenseNet-121's native 1000-way ImageNet classifier with the
# multilabel chest X-ray head: Dropout(p=0.3) -> Linear(1024 -> 14),
# then wrap the result in ChestXrayClassifier.  Sigmoid is intentionally
# NOT part of the model -- raw logits are the model's contract output
# so it composes directly with nn.BCEWithLogitsLoss (numerically
# stable; the loss applies sigmoid internally).  Per
# MASTER_ARCHITECTURE.md Section 18.2.

print_section("STAGE 10 - CLASSIFICATION HEAD")
print()

DROPOUT_PROB = 0.3   # MASTER_ARCHITECTURE.md Section 18.2

backbone, head_in_features = replace_classifier(
    backbone, backbone_name, num_classes=cfg.NUM_CLASSES, dropout=DROPOUT_PROB,
)
model = ChestXrayClassifier(backbone, backbone_name, cfg.NUM_CLASSES)
model.eval()

new_head = _get_module_by_path(model.backbone, spec.classifier_attr)

print("  New classification head")
print("  " + "-" * 60)
print_kv("Wrapper class",  type(model).__name__)
print_kv("Attribute path", f"backbone.{spec.classifier_attr}")
print_kv("Head",           new_head)
print()

# -- Verification: structure -------------------------------------
head_is_sequential = isinstance(new_head, nn.Sequential)
head_has_dropout    = isinstance(new_head[0], nn.Dropout) if head_is_sequential else False
head_has_linear     = isinstance(new_head[-1], nn.Linear) if head_is_sequential else False
final_linear         = new_head[-1] if head_is_sequential else new_head
output_dim_ok        = final_linear.out_features == cfg.NUM_CLASSES
input_dim_ok         = final_linear.in_features == head_in_features == spec.feature_dim

# -- Verification: forward pass (full batch size) -------------------
dummy_batch = torch.randn(cfg.BATCH_SIZE, 3, *cfg.IMAGE_SIZE)
with torch.no_grad():
    logits = model(dummy_batch)
forward_shape_ok = tuple(logits.shape) == (cfg.BATCH_SIZE, cfg.NUM_CLASSES)
forward_dtype_ok = logits.dtype == torch.float32

# Output should be raw logits, not probabilities -- confirm the range
# is NOT clipped to [0, 1] the way a sigmoid output would be.
logits_are_unbounded = bool((logits.min() < 0) or (logits.max() > 1))

# -- Verification: BCEWithLogitsLoss compatibility ------------------
dummy_targets = torch.randint(0, 2, (cfg.BATCH_SIZE, cfg.NUM_CLASSES)).float()
criterion = nn.BCEWithLogitsLoss()   # plain BCE here only -- class
                                      # weighting (the pos_weight vector
                                      # in class_weights.json) is wired
                                      # in Phase 3 (Loss Functions),
                                      # out of scope for Phase 2.
loss_value = criterion(logits, dummy_targets)
loss_is_finite_scalar = bool(torch.isfinite(loss_value)) and loss_value.ndim == 0

# -- Report ----------------------------------------------------
print("  Verification")
print("  " + "-" * 60)
print_check("Head is Dropout -> Linear", head_is_sequential and head_has_dropout and head_has_linear)
print_check("Head input dim matches backbone feature_dim", input_dim_ok, str(final_linear.in_features))
print_check("Head output dim == NUM_CLASSES", output_dim_ok, str(final_linear.out_features))
print_check("Forward pass shape", forward_shape_ok, str(tuple(logits.shape)))
print_check("Forward pass dtype", forward_dtype_ok, str(logits.dtype))
print_check("Output is raw logits (unbounded)", logits_are_unbounded)
print_check("BCEWithLogitsLoss computes a finite scalar", loss_is_finite_scalar, f"loss={loss_value.item():.4f}")

assert input_dim_ok and output_dim_ok, "Classification head dimension mismatch."
assert forward_shape_ok, "Forward pass produced an unexpected output shape."
assert loss_is_finite_scalar, "BCEWithLogitsLoss compatibility check failed."

print()
print("Stage 10 - Classification Head : OK")

  STAGE 10 - CLASSIFICATION HEAD

  New classification head
  ------------------------------------------------------------
  Wrapper class                    ChestXrayClassifier
  Attribute path                   backbone.classifier
  Head                             Sequential(
  (0): Dropout(p=0.3, inplace=False)
  (1): Linear(in_features=1024, out_features=14, bias=True)
)

  Verification
  ------------------------------------------------------------
  ✔  Head is Dropout -> Linear
  ✔  Head input dim matches backbone feature_dim  (1024)
  ✔  Head output dim == NUM_CLASSES  (14)
  ✔  Forward pass shape  ((32, 14))
  ✔  Forward pass dtype  (torch.float32)
  ✔  Output is raw logits (unbounded)
  ✔  BCEWithLogitsLoss computes a finite scalar  (loss=0.7006)

Stage 10 - Classification Head : OK


In [13]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 11: Transfer Learning Setup
#
# Module: training/models/classifier.py -> freeze_backbone()
# ============================================================

# -- Purpose ---------------------------------------------------
# Per MASTER_ARCHITECTURE.md Section 18.5 (Phase 1 of the two-phase
# transfer-learning strategy): freeze every pretrained backbone
# parameter and leave only the newly attached classification head
# trainable, so the random-initialised head reaches reasonable
# predictions before gradients touch (and potentially destroy) the
# pretrained ImageNet features.  Unfreezing the backbone for
# end-to-end fine-tuning is a Phase 4 (Trainer) responsibility, not
# Phase 2 -- this stage only performs the freeze itself.

def freeze_backbone(model: ChestXrayClassifier, backbone_name: str) -> ChestXrayClassifier:
    """
    Freeze every parameter except the classification head, in place.

    Parameters
    ----------
    model         : ChestXrayClassifier   A model from Stage 10.
    backbone_name : str                   Registry key, used to resolve
                                          which submodule is the head.

    Returns
    -------
    ChestXrayClassifier
        The same model, mutated in place.
    """
    head_spec = BACKBONE_REGISTRY[backbone_name]
    head_prefix = "backbone." + head_spec.classifier_attr.split(".")[0]
    for name, param in model.named_parameters():
        param.requires_grad = name.startswith(head_prefix)
    return model


# -- Execute -------------------------------------------------
print_section("STAGE 11 - TRANSFER LEARNING SETUP")
print()

model = freeze_backbone(model, backbone_name)

param_counts  = count_parameters(model)
trainable_pct = 100.0 * param_counts["trainable"] / param_counts["total"]
frozen_pct    = 100.0 * param_counts["frozen"]    / param_counts["total"]

print("  Parameter breakdown")
print("  " + "-" * 60)
print_kv("Total parameters",     f"{param_counts['total']:,}")
print_kv("Trainable parameters", f"{param_counts['trainable']:,}")
print_kv("Frozen parameters",    f"{param_counts['frozen']:,}")
print_kv("Trainable %",          f"{trainable_pct:.3f}%")
print_kv("Frozen %",             f"{frozen_pct:.3f}%")
print()

# -- Verification: every flag is exactly as intended ---------------
head_prefix = "backbone." + BACKBONE_REGISTRY[backbone_name].classifier_attr.split(".")[0]
mismatched_flags: List[str] = [
    name for name, param in model.named_parameters()
    if param.requires_grad != name.startswith(head_prefix)
]
only_head_trainable = param_counts["trainable"] == sum(
    p.numel() for n, p in model.named_parameters() if n.startswith(head_prefix)
)

print("  Verification")
print("  " + "-" * 60)
print_check("All backbone parameters frozen, head trainable", len(mismatched_flags) == 0, f"{len(mismatched_flags)} mismatches")
print_check("Trainable parameter count == head-only parameter count", only_head_trainable)
print_check("Backbone is the overwhelming majority of parameters", frozen_pct > 95.0, f"{frozen_pct:.2f}%")

assert len(mismatched_flags) == 0, f"requires_grad mismatch on: {mismatched_flags[:5]}"
assert only_head_trainable, "Trainable parameters leaked outside the classification head."

print()
print("Stage 11 - Transfer Learning Setup : OK")

  STAGE 11 - TRANSFER LEARNING SETUP

  Parameter breakdown
  ------------------------------------------------------------
  Total parameters                 6,968,206
  Trainable parameters             14,350
  Frozen parameters                6,953,856
  Trainable %                      0.206%
  Frozen %                         99.794%

  Verification
  ------------------------------------------------------------
  ✔  All backbone parameters frozen, head trainable  (0 mismatches)
  ✔  Trainable parameter count == head-only parameter count
  ✔  Backbone is the overwhelming majority of parameters  (99.79%)

Stage 11 - Transfer Learning Setup : OK


In [14]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 12: Model Verification
#
# Module: training/models/classifier.py -> ChestXrayClassifier (final)
# ============================================================

# -- Purpose ---------------------------------------------------
# Final, exhaustive engineering gate before this model object is
# handed to Phase 3 (Loss Functions).  Every check prints an explicit
# PASS / FAIL line; a single failure raises and halts the notebook
# rather than letting a silently broken model flow downstream.

import json as _json

print_section("STAGE 12 - MODEL VERIFICATION")
print()
checks: List[Tuple[str, bool, str]] = []

def _verify(label: str, passed: bool, detail: str = "") -> None:
    checks.append((label, passed, detail))
    print_check(label, passed, detail)

# -- Device placement --------------------------------------------
model = model.to(DEVICE)
model.eval()

print("  Device placement")
print("  " + "-" * 60)
param_device = next(model.parameters()).device
_verify("Model parameters on resolved device", param_device.type == DEVICE.type, str(param_device))
print()

# -- Forward pass on DEVICE ----------------------------------------
print("  Forward pass")
print("  " + "-" * 60)
dummy_batch = torch.randn(cfg.BATCH_SIZE, 3, *cfg.IMAGE_SIZE, device=DEVICE)
with torch.no_grad():
    logits = model(dummy_batch)

_verify("Output shape == (BATCH_SIZE, NUM_CLASSES)", tuple(logits.shape) == (cfg.BATCH_SIZE, cfg.NUM_CLASSES), str(tuple(logits.shape)))
_verify("Output dtype == float32", logits.dtype == torch.float32, str(logits.dtype))
_verify("Output device matches model device", logits.device.type == DEVICE.type, str(logits.device))
_verify("Output contains no NaN/Inf", bool(torch.isfinite(logits).all()))
print()

# -- Single-sample edge case (batch size 1) -------------------------
with torch.no_grad():
    single = model(torch.randn(1, 3, *cfg.IMAGE_SIZE, device=DEVICE))
_verify("Batch-size-1 forward pass shape", tuple(single.shape) == (1, cfg.NUM_CLASSES), str(tuple(single.shape)))
print()

# -- GPU compatibility ------------------------------------------------
print("  GPU compatibility")
print("  " + "-" * 60)
if DEVICE.type == "cuda":
    torch.cuda.synchronize()
    _verify("CUDA forward pass executed without error", True, "cuda")
else:
    _verify("CUDA forward pass executed without error", True, "skipped -- CPU device")
print()

# -- Parameter / memory footprint -----------------------------------
print("  Parameter & memory footprint")
print("  " + "-" * 60)
final_param_counts = count_parameters(model)
bytes_per_param = 4   # float32
model_size_mb = final_param_counts["total"] * bytes_per_param / (1024 ** 2)
print_kv("Total parameters",      f"{final_param_counts['total']:,}")
print_kv("Trainable parameters",  f"{final_param_counts['trainable']:,}")
print_kv("Frozen parameters",     f"{final_param_counts['frozen']:,}")
print_kv("Estimated size (fp32)", f"{model_size_mb:.2f} MB")
_verify("Total parameter count is positive", final_param_counts["total"] > 0, f"{final_param_counts['total']:,}")
_verify("Trainable count matches Stage 11 freeze", final_param_counts["trainable"] == param_counts["trainable"])
print()

# -- Inference timing -------------------------------------------------
print("  Inference timing")
print("  " + "-" * 60)
_warmup_batches, _timed_batches = 3, 10
with torch.no_grad():
    for _ in range(_warmup_batches):
        _ = model(dummy_batch)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    _t0 = _time.time()
    for _ in range(_timed_batches):
        _ = model(dummy_batch)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    _elapsed = _time.time() - _t0
ms_per_batch = (_elapsed / _timed_batches) * 1000
ms_per_image = ms_per_batch / cfg.BATCH_SIZE
print_kv("Batches timed",       _timed_batches)
print_kv("Avg latency / batch", f"{ms_per_batch:.2f} ms")
print_kv("Avg latency / image", f"{ms_per_image:.2f} ms")
_verify("Inference timing completed without error", ms_per_batch > 0, f"{ms_per_batch:.2f} ms/batch")
print()

# -- Model summary (top-level backbone modules) -----------------------
print("  Model summary (top-level modules)")
print("  " + "-" * 60)
print(f"  {type(model).__name__}")
for child_name, child in model.backbone.named_children():
    n_params = sum(p.numel() for p in child.parameters())
    print(f"    backbone.{child_name:<14}{type(child).__name__:<20}{n_params:>12,} params")
print()

# -- Aggregate gate ------------------------------------------------
all_passed = all(passed for _, passed, _ in checks)
print_section("STAGE 12 - FINAL GATE")
print()
print(f"  {sum(1 for _, p, _ in checks if p)} / {len(checks)} checks passed")
print()
if not all_passed:
    failed = [label for label, passed, _ in checks if not passed]
    raise AssertionError(f"Model verification FAILED: {failed}")
print("  ALL CHECKS PASSED")
print()

# -- Output artifacts ----------------------------------------------
print_section("STAGE 12 - ARTIFACT PERSISTENCE")
print()

# Source class names from the frozen Sprint 03 registry for
# traceability rather than re-typing the disease list -- continuity
# with the frozen artifact rather than a parallel hardcoded copy.
with open(DISEASE_REGISTRY_PATH) as f:
    class_names = _json.load(f)["diseases"]
registry_loaded_from = str(DISEASE_REGISTRY_PATH)
logger.info("Class names sourced from: %s", registry_loaded_from)

backbone_summary = {
    "backbone_name":   backbone_name,
    "timm_name":       spec.timm_name,
    "family":          spec.family,
    "pretrained":      True,
    "feature_dim":     spec.feature_dim,
    "classifier_attr": spec.classifier_attr,
    "target_layer":    spec.target_layer,
    "imagenet_classes_before_replacement": 1000,
}

parameter_summary = {
    "total_parameters":      final_param_counts["total"],
    "trainable_parameters":  final_param_counts["trainable"],
    "frozen_parameters":     final_param_counts["frozen"],
    "trainable_percentage":  round(100.0 * final_param_counts["trainable"] / final_param_counts["total"], 4),
    "frozen_percentage":     round(100.0 * final_param_counts["frozen"]    / final_param_counts["total"], 4),
    "estimated_size_mb_fp32": round(model_size_mb, 2),
}

model_summary = {
    "project":    cfg.PROJECT_NAME,
    "model_name": cfg.MODEL_NAME,
    "sprint":     "04",
    "phase":      "Phase 2 - Model Engineering",
    "backbone":   backbone_summary,
    "classification_head": {
        "type":         "Dropout -> Linear",
        "dropout":      DROPOUT_PROB,
        "in_features":  head_in_features,
        "out_features": cfg.NUM_CLASSES,
        "num_classes":  cfg.NUM_CLASSES,
        "class_names":  class_names,
    },
    "parameters": parameter_summary,
    "transfer_learning": {
        "strategy":         "freeze_backbone_train_head",
        "trainable_module": head_prefix,
    },
    "input": {
        "image_size":           list(cfg.IMAGE_SIZE),
        "num_channels":         3,
        "batch_size_validated": cfg.BATCH_SIZE,
    },
    "output": {
        "shape":      [cfg.BATCH_SIZE, cfg.NUM_CLASSES],
        "activation": "none (raw logits; sigmoid applied at inference time)",
    },
    "device_validated":                str(DEVICE),
    "inference_latency_ms_per_batch":  round(ms_per_batch, 3),
    "verification": {
        "checks_passed": sum(1 for _, p, _ in checks if p),
        "checks_total":  len(checks),
        "all_passed":    all_passed,
    },
}

artifact_paths = {
    "model_summary":     cfg.MODEL_DIR / "model_summary.json",
    "parameter_summary": cfg.MODEL_DIR / "parameter_summary.json",
    "backbone_summary":  cfg.MODEL_DIR / "backbone_summary.json",
}
artifact_payloads = {
    "model_summary":     model_summary,
    "parameter_summary": parameter_summary,
    "backbone_summary":  backbone_summary,
}

for key, path in artifact_paths.items():
    with open(path, "w") as f:
        _json.dump(artifact_payloads[key], f, indent=2, default=str)
    print_check(f"Saved {path.name}", path.exists() and path.stat().st_size > 0, str(path))

print()
print("Stage 12 - Model Verification : OK")

  STAGE 12 - MODEL VERIFICATION

  Device placement
  ------------------------------------------------------------
  ✔  Model parameters on resolved device  (cuda:0)

  Forward pass
  ------------------------------------------------------------
  ✔  Output shape == (BATCH_SIZE, NUM_CLASSES)  ((32, 14))
  ✔  Output dtype == float32  (torch.float32)
  ✔  Output device matches model device  (cuda:0)
  ✔  Output contains no NaN/Inf

  ✔  Batch-size-1 forward pass shape  ((1, 14))

  GPU compatibility
  ------------------------------------------------------------
  ✔  CUDA forward pass executed without error  (cuda)

  Parameter & memory footprint
  ------------------------------------------------------------
  Total parameters                 6,968,206
  Trainable parameters             14,350
  Frozen parameters                6,953,856
  Estimated size (fp32)            26.58 MB
  ✔  Total parameter count is positive  (6,968,206)
  ✔  Trainable count matches Stage 11 freeze

  Inference 

# ============================================================
# VisionServeAI | Sprint 04
# Stage 13
#
# Module: training/losses.py -> LossConfig
# ============================================================

In [15]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 13: Loss Configuration
#
# Module: training/losses.py -> LossConfig
# ============================================================

from dataclasses import FrozenInstanceError as _FrozenInstanceError
import json as _json

print_section("STAGE 13 - LOSS CONFIGURATION")
print()


@dataclass(frozen=True)
class LossConfig:
    """
    Immutable, validated configuration for the VisionServeAI
    Sprint 04 Phase 3 multilabel loss subsystem.

    Mirrors the write-once pattern of ``TrainingConfig`` (Stage 3):
    every field is validated in ``__post_init__`` and the instance is
    frozen to prevent mid-training mutation.

    Migration note
    --------------
    Maps directly to ``training/losses.py -> LossConfig`` per
    MASTER_ARCHITECTURE.md Sprint 04 / Phase 3.
    """

    REDUCTION:          str   = "mean"    # "mean" | "sum" | "none"
    USE_CLASS_WEIGHTS:  bool  = True      # apply frozen Sprint 03 pos_weight
    LABEL_SMOOTHING:    float = 0.0       # reserved -- not applied until a
                                           # future sprint; validated now
    NUM_CLASSES:        int  = cfg.NUM_CLASSES
    CLASS_WEIGHTS_PATH: Path = SPRINT03_CLASS_WEIGHTS_PATH
    LOSS_DIR:           Path = cfg.OUTPUT_DIR / "loss"

    def __post_init__(self) -> None:
        """Validate all fields at construction time."""
        valid_reductions = {"mean", "sum", "none"}
        if self.REDUCTION not in valid_reductions:
            raise ValueError(f"REDUCTION must be one of {valid_reductions}; got {self.REDUCTION!r}.")
        if not isinstance(self.USE_CLASS_WEIGHTS, bool):
            raise ValueError(f"USE_CLASS_WEIGHTS must be bool; got {type(self.USE_CLASS_WEIGHTS)!r}.")
        if not (0.0 <= self.LABEL_SMOOTHING < 1.0):
            raise ValueError(f"LABEL_SMOOTHING must be in [0.0, 1.0); got {self.LABEL_SMOOTHING}.")
        require_positive_int(self.NUM_CLASSES, "NUM_CLASSES")

    def display(self) -> None:
        """Pretty-print the configuration to stdout."""
        print("  --- Loss Configuration ---")
        for k in ["REDUCTION", "USE_CLASS_WEIGHTS", "LABEL_SMOOTHING",
                  "NUM_CLASSES", "CLASS_WEIGHTS_PATH", "LOSS_DIR"]:
            print_kv(k, getattr(self, k))


# -- Construct the singleton loss configuration ---------------------
loss_cfg = LossConfig()
loss_cfg.LOSS_DIR.mkdir(parents=True, exist_ok=True)

loss_cfg.display()
print()

# -- Immutability check ----------------------------------------------
try:
    loss_cfg.REDUCTION = "sum"
    raise AssertionError("Frozen LossConfig failed to prevent mutation.")
except _FrozenInstanceError:
    print_check("Immutability enforced", True, "FrozenInstanceError raised")

checks = [
    ("REDUCTION is valid",                       loss_cfg.REDUCTION in {"mean", "sum", "none"}, loss_cfg.REDUCTION),
    ("NUM_CLASSES matches TrainingConfig",        loss_cfg.NUM_CLASSES == cfg.NUM_CLASSES, str(loss_cfg.NUM_CLASSES)),
    ("LABEL_SMOOTHING in [0, 1)",                 0.0 <= loss_cfg.LABEL_SMOOTHING < 1.0, str(loss_cfg.LABEL_SMOOTHING)),
    ("CLASS_WEIGHTS_PATH is a Path",              isinstance(loss_cfg.CLASS_WEIGHTS_PATH, Path), str(loss_cfg.CLASS_WEIGHTS_PATH)),
    ("LOSS_DIR created",                          loss_cfg.LOSS_DIR.exists(), str(loss_cfg.LOSS_DIR)),
]
for label, passed, detail in checks:
    print_check(label, passed, detail)

all_passed = all(p for _, p, _ in checks)
print()
if not all_passed:
    raise AssertionError(f"LossConfig verification FAILED: {[l for l,p,_ in checks if not p]}")
print("  ALL CHECKS PASSED")
print()

# -- Persist artifact: loss_configuration.json -------------------------
loss_configuration_summary = {
    "project":             cfg.PROJECT_NAME,
    "sprint":              "04",
    "phase":               "Phase 3 - Loss Engineering",
    "reduction":           loss_cfg.REDUCTION,
    "use_class_weights":   loss_cfg.USE_CLASS_WEIGHTS,
    "label_smoothing":     loss_cfg.LABEL_SMOOTHING,
    "num_classes":         loss_cfg.NUM_CLASSES,
    "class_weights_path":  str(loss_cfg.CLASS_WEIGHTS_PATH),
    "loss_dir":            str(loss_cfg.LOSS_DIR),
}
config_path = loss_cfg.LOSS_DIR / "loss_configuration.json"
with open(config_path, "w") as f:
    _json.dump(loss_configuration_summary, f, indent=2, default=str)
print_check(f"Saved {config_path.name}", config_path.exists() and config_path.stat().st_size > 0, str(config_path))

print()
print("Stage 13 - Loss Configuration : OK")

  STAGE 13 - LOSS CONFIGURATION

  --- Loss Configuration ---
  REDUCTION                        mean
  USE_CLASS_WEIGHTS                True
  LABEL_SMOOTHING                  0.0
  NUM_CLASSES                      14
  CLASS_WEIGHTS_PATH               /kaggle/input/notebooks/anupsharma1730/sprint03-v2-final-frozen/visionserveai/sprint03/statistics/class_weights.json
  LOSS_DIR                         /kaggle/working/visionserveai/sprint04/loss

  ✔  Immutability enforced  (FrozenInstanceError raised)
  ✔  REDUCTION is valid  (mean)
  ✔  NUM_CLASSES matches TrainingConfig  (14)
  ✔  LABEL_SMOOTHING in [0, 1)  (0.0)
  ✔  CLASS_WEIGHTS_PATH is a Path  (/kaggle/input/notebooks/anupsharma1730/sprint03-v2-final-frozen/visionserveai/sprint03/statistics/class_weights.json)
  ✔  LOSS_DIR created  (/kaggle/working/visionserveai/sprint04/loss)

  ALL CHECKS PASSED

  ✔  Saved loss_configuration.json  (/kaggle/working/visionserveai/sprint04/loss/loss_configuration.json)

Stage 13 - Loss Configur

In [16]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 14: BCEWithLogitsLoss Engineering
#
# Module: training/losses.py -> build_bce_loss()
# ============================================================

import torch.nn as nn

print_section("STAGE 14 - BCEWITHLOGITSLOSS ENGINEERING")
print()


def build_bce_loss(
    pos_weight: Optional[torch.Tensor] = None,
    reduction: str = "mean",
) -> nn.BCEWithLogitsLoss:
    """
    Construct the production multilabel classification loss.

    Why BCEWithLogitsLoss instead of Sigmoid + BCELoss
    -----------------------------------------------------
    BCEWithLogitsLoss fuses the sigmoid activation and the binary
    cross-entropy computation into one op via the log-sum-exp identity:

        loss = max(x, 0) - x * y + log(1 + exp(-|x|))

    Computing sigmoid(x) then log(.) as two separate ops is unstable at
    the extremes of the logit range: for very negative logits,
    sigmoid(x) underflows to exactly 0.0 in float32/float16, and
    log(0) then produces -inf / NaN. The fused formula above never
    evaluates log of a value that can reach zero, so it stays finite
    for any input logit. This matters here because Phase 2's head
    emits raw, unbounded logits and Phase 1 enables AMP (float16)
    training, which narrows the safe numerical range further.

    Parameters
    ----------
    pos_weight : Optional[torch.Tensor]  Per-class weight, shape (num_classes,).
    reduction  : str                     "mean" | "sum" | "none".

    Returns
    -------
    torch.nn.BCEWithLogitsLoss expecting raw logits (no sigmoid applied)
    and float targets of identical shape (batch, num_classes).
    """
    if pos_weight is not None and pos_weight.dtype != torch.float32:
        raise TypeError(f"pos_weight must be float32; got {pos_weight.dtype}.")
    return nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction=reduction)


# -- Verification (shape-compatibility only; real weights load in Stage 15) --
_dummy_logits  = torch.randn(4, cfg.NUM_CLASSES)
_dummy_targets = torch.randint(0, 2, (4, cfg.NUM_CLASSES)).float()
_dummy_weight  = torch.rand(cfg.NUM_CLASSES) + 0.5

criterion_unweighted = build_bce_loss(pos_weight=None, reduction=loss_cfg.REDUCTION)
criterion_weighted   = build_bce_loss(pos_weight=_dummy_weight, reduction=loss_cfg.REDUCTION)

_loss_u = criterion_unweighted(_dummy_logits, _dummy_targets)
_loss_w = criterion_weighted(_dummy_logits, _dummy_targets)

checks = [
    ("Unweighted criterion is BCEWithLogitsLoss",   isinstance(criterion_unweighted, nn.BCEWithLogitsLoss), ""),
    ("Weighted criterion is BCEWithLogitsLoss",     isinstance(criterion_weighted, nn.BCEWithLogitsLoss), ""),
    ("Unweighted criterion.pos_weight is None",     criterion_unweighted.pos_weight is None, ""),
    ("Weighted criterion.pos_weight is set",        criterion_weighted.pos_weight is not None, str(tuple(criterion_weighted.pos_weight.shape))),
    ("Reduction mode matches LossConfig",           criterion_unweighted.reduction == loss_cfg.REDUCTION, loss_cfg.REDUCTION),
    ("Tensor compatibility: logits/targets shape",  _dummy_logits.shape == _dummy_targets.shape, str(tuple(_dummy_logits.shape))),
    ("Unweighted loss is finite scalar",            bool(torch.isfinite(_loss_u)) and _loss_u.dim() == 0, f"{_loss_u.item():.4f}"),
    ("Weighted loss is finite scalar",              bool(torch.isfinite(_loss_w)) and _loss_w.dim() == 0, f"{_loss_w.item():.4f}"),
]
try:
    build_bce_loss(pos_weight=torch.rand(cfg.NUM_CLASSES).double())
    checks.append(("Invalid pos_weight dtype rejected", False, "no exception raised"))
except TypeError:
    checks.append(("Invalid pos_weight dtype rejected", True, "TypeError raised"))

for label, passed, detail in checks:
    print_check(label, passed, detail)

all_passed = all(p for _, p, _ in checks)
print()
if not all_passed:
    raise AssertionError(f"BCEWithLogitsLoss verification FAILED: {[l for l,p,_ in checks if not p]}")
print("  ALL CHECKS PASSED")
print()
print("Stage 14 - BCEWithLogitsLoss Engineering : OK")

  STAGE 14 - BCEWITHLOGITSLOSS ENGINEERING

  ✔  Unweighted criterion is BCEWithLogitsLoss
  ✔  Weighted criterion is BCEWithLogitsLoss
  ✔  Unweighted criterion.pos_weight is None
  ✔  Weighted criterion.pos_weight is set  ((14,))
  ✔  Reduction mode matches LossConfig  (mean)
  ✔  Tensor compatibility: logits/targets shape  ((4, 14))
  ✔  Unweighted loss is finite scalar  (0.7813)
  ✔  Weighted loss is finite scalar  (0.8080)
  ✔  Invalid pos_weight dtype rejected  (TypeError raised)

  ALL CHECKS PASSED

Stage 14 - BCEWithLogitsLoss Engineering : OK


In [17]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 15: Class Weight Loading
#
# Module: training/losses.py -> load_class_weights()
# ============================================================

print_section("STAGE 15 - CLASS WEIGHT LOADING")
print()


def load_class_weights(path: Path, class_names: List[str]) -> torch.Tensor:
    """
    Load the frozen Sprint 03 class_weights.json artifact and convert
    it to a torch.float32 pos_weight tensor ordered to match
    class_names (the frozen disease_registry ordering).

    Returns a CPU tensor -- device placement is deferred to the call
    site (Stage 16), not performed here, since this loader should stay
    runtime-agnostic and testable without a GPU.

    Raises
    ------
    FileNotFoundError  If *path* does not exist.
    ValueError         If a class is missing or a weight is non-numeric
                        / non-positive.
    """
    if not path.exists():
        raise FileNotFoundError(f"Frozen class_weights.json not found: {path}")
    with open(path) as f:
        raw = _json.load(f)

    missing = [name for name in class_names if name not in raw]
    if missing:
        raise ValueError(f"class_weights.json missing classes: {missing}")

    values: List[float] = []
    for name in class_names:
        w = raw[name].get("pos_weight")
        if not isinstance(w, (int, float)) or isinstance(w, bool) or w <= 0:
            raise ValueError(f"Invalid pos_weight for {name!r}: {w!r}")
        values.append(float(w))

    return torch.tensor(values, dtype=torch.float32)


# `class_names` was already loaded in Stage 12; re-derive defensively
# in case Stage 15 is run independently of the Phase 2 cells.
try:
    class_names
except NameError:
    with open(DISEASE_REGISTRY_PATH) as f:
        class_names = _json.load(f)["diseases"]

pos_weight_tensor = load_class_weights(loss_cfg.CLASS_WEIGHTS_PATH, class_names)

checks = [
    ("class_weights.json exists",       loss_cfg.CLASS_WEIGHTS_PATH.exists(), str(loss_cfg.CLASS_WEIGHTS_PATH)),
    ("All 14 classes loaded",           tuple(pos_weight_tensor.shape) == (cfg.NUM_CLASSES,), str(tuple(pos_weight_tensor.shape))),
    ("Tensor dtype is float32",         pos_weight_tensor.dtype == torch.float32, str(pos_weight_tensor.dtype)),
    ("Tensor currently on CPU",         pos_weight_tensor.device.type == "cpu", str(pos_weight_tensor.device)),
    ("All weights strictly positive",   bool((pos_weight_tensor > 0).all()), f"min={pos_weight_tensor.min().item():.3f}"),
    ("No NaN / Inf in weights",         bool(torch.isfinite(pos_weight_tensor).all()), ""),
]
for label, passed, detail in checks:
    print_check(label, passed, detail)

print()
print("  Per-class pos_weight (frozen Sprint 03)")
print("  " + "-" * 60)
for name, w in zip(class_names, pos_weight_tensor.tolist()):
    print(f"    {name:<20}{w:>10.3f}")

all_passed = all(p for _, p, _ in checks)
print()
if not all_passed:
    raise AssertionError(f"Class weight loading FAILED: {[l for l,p,_ in checks if not p]}")
print("  ALL CHECKS PASSED")
print()
print("Stage 15 - Class Weight Loading : OK")

  STAGE 15 - CLASS WEIGHT LOADING

  ✔  class_weights.json exists  (/kaggle/input/notebooks/anupsharma1730/sprint03-v2-final-frozen/visionserveai/sprint03/statistics/class_weights.json)
  ✔  All 14 classes loaded  ((14,))
  ✔  Tensor dtype is float32  (torch.float32)
  ✔  Tensor currently on CPU  (cpu)
  ✔  All weights strictly positive  (min=5.228)
  ✔  No NaN / Inf in weights

  Per-class pos_weight (frozen Sprint 03)
  ------------------------------------------------------------
    Atelectasis              9.483
    Cardiomegaly            50.356
    Consolidation           29.150
    Edema                   62.550
    Effusion                 9.002
    Emphysema               58.476
    Fibrosis                69.219
    Hernia                 604.104
    Infiltration             5.228
    Mass                    20.464
    Nodule                  17.274
    Pleural_Thickening      38.226
    Pneumonia               97.845
    Pneumothorax            31.563

  ALL CHECKS PASSED

S

In [18]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 16: Loss Verification
#
# Module: training/losses.py -> verification harness
# ============================================================

print_section("STAGE 16 - LOSS VERIFICATION")
print()

# Move weights to the resolved device only at the point of use.
pos_weight_device = pos_weight_tensor.to(DEVICE)

criterion_weighted   = build_bce_loss(pos_weight=pos_weight_device, reduction=loss_cfg.REDUCTION)
criterion_unweighted = build_bce_loss(pos_weight=None,              reduction=loss_cfg.REDUCTION)

checks: List[Tuple[str, bool, str]] = []
n_batches = 3
_last_loss_w = _last_loss_u = None

for i in range(n_batches):
    images  = torch.randn(cfg.BATCH_SIZE, 3, *cfg.IMAGE_SIZE, device=DEVICE)
    targets = torch.randint(0, 2, (cfg.BATCH_SIZE, cfg.NUM_CLASSES), device=DEVICE).float()

    # -- Weighted pass ------------------------------------------------
    model.zero_grad(set_to_none=True)
    logits = model(images)
    loss_w = criterion_weighted(logits, targets)
    loss_w.backward()

    head_params = [p for p in model.backbone.classifier.parameters() if p.requires_grad]
    grads_w     = [p.grad for p in head_params]
    backbone_frozen_ok = all(
        p.grad is None for n, p in model.named_parameters() if "classifier" not in n
    )

    checks.append((f"[batch {i}] weighted loss scalar, finite",   loss_w.dim() == 0 and bool(torch.isfinite(loss_w)), f"{loss_w.item():.4f}"))
    checks.append((f"[batch {i}] weighted loss dtype/device",     loss_w.dtype == torch.float32 and loss_w.device.type == DEVICE.type, str(loss_w.device)))
    checks.append((f"[batch {i}] head gradients exist & finite",  all(g is not None and bool(torch.isfinite(g).all()) for g in grads_w), ""))
    checks.append((f"[batch {i}] frozen backbone has no grad",    backbone_frozen_ok, "freeze intact"))

    # -- Unweighted pass (independent zero_grad) -----------------------
    model.zero_grad(set_to_none=True)
    logits_u = model(images)
    loss_u = criterion_unweighted(logits_u, targets)
    loss_u.backward()
    grads_u = [p.grad for p in model.backbone.classifier.parameters() if p.requires_grad]

    checks.append((f"[batch {i}] unweighted loss scalar, finite", loss_u.dim() == 0 and bool(torch.isfinite(loss_u)), f"{loss_u.item():.4f}"))
    checks.append((f"[batch {i}] unweighted gradients finite",    all(g is not None and bool(torch.isfinite(g).all()) for g in grads_u), ""))

    _last_loss_w, _last_loss_u = loss_w.item(), loss_u.item()

model.zero_grad(set_to_none=True)

for label, passed, detail in checks:
    print_check(label, passed, detail)

all_passed = all(p for _, p, _ in checks)
print()
print(f"  {sum(1 for _,p,_ in checks if p)} / {len(checks)} checks passed")
print()
if not all_passed:
    raise AssertionError(f"Loss verification FAILED: {[l for l,p,_ in checks if not p]}")
print("  ALL CHECKS PASSED")
print()

# -- Persist artifact: loss_validation.json --------------------------
loss_validation_summary = {
    "project":               cfg.PROJECT_NAME,
    "sprint":                "04",
    "phase":                 "Phase 3 - Loss Engineering",
    "device_validated":      str(DEVICE),
    "batches_tested":        n_batches,
    "sample_weighted_loss":   round(_last_loss_w, 6),
    "sample_unweighted_loss": round(_last_loss_u, 6),
    "checks_passed":         sum(1 for _, p, _ in checks if p),
    "checks_total":          len(checks),
    "all_passed":            all_passed,
    "checks":                [{"label": l, "passed": p, "detail": d} for l, p, d in checks],
}
validation_path = loss_cfg.LOSS_DIR / "loss_validation.json"
with open(validation_path, "w") as f:
    _json.dump(loss_validation_summary, f, indent=2, default=str)
print_check(f"Saved {validation_path.name}", validation_path.exists() and validation_path.stat().st_size > 0, str(validation_path))

print()
print("Stage 16 - Loss Verification : OK")

  STAGE 16 - LOSS VERIFICATION

  ✔  [batch 0] weighted loss scalar, finite  (28.7471)
  ✔  [batch 0] weighted loss dtype/device  (cuda:0)
  ✔  [batch 0] head gradients exist & finite
  ✔  [batch 0] frozen backbone has no grad  (freeze intact)
  ✔  [batch 0] unweighted loss scalar, finite  (0.6882)
  ✔  [batch 0] unweighted gradients finite
  ✔  [batch 1] weighted loss scalar, finite  (24.2509)
  ✔  [batch 1] weighted loss dtype/device  (cuda:0)
  ✔  [batch 1] head gradients exist & finite
  ✔  [batch 1] frozen backbone has no grad  (freeze intact)
  ✔  [batch 1] unweighted loss scalar, finite  (0.6987)
  ✔  [batch 1] unweighted gradients finite
  ✔  [batch 2] weighted loss scalar, finite  (27.0196)
  ✔  [batch 2] weighted loss dtype/device  (cuda:0)
  ✔  [batch 2] head gradients exist & finite
  ✔  [batch 2] frozen backbone has no grad  (freeze intact)
  ✔  [batch 2] unweighted loss scalar, finite  (0.6969)
  ✔  [batch 2] unweighted gradients finite

  18 / 18 checks passed

  ALL CHE

In [19]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 17: Engineering Comparison -- Weighted vs Unweighted BCE
#
# Module: training/losses.py -> documentation artifact
# ============================================================

print_section("STAGE 17 - ENGINEERING COMPARISON")
print()

torch.manual_seed(cfg.RANDOM_SEED)
_cmp_images  = torch.randn(cfg.BATCH_SIZE, 3, *cfg.IMAGE_SIZE, device=DEVICE)
_cmp_targets = torch.randint(0, 2, (cfg.BATCH_SIZE, cfg.NUM_CLASSES), device=DEVICE).float()

model.eval()
with torch.no_grad():
    _cmp_logits = model(_cmp_images)
    cmp_loss_weighted   = criterion_weighted(_cmp_logits, _cmp_targets).item()
    cmp_loss_unweighted = criterion_unweighted(_cmp_logits, _cmp_targets).item()
model.train()

abs_diff = abs(cmp_loss_weighted - cmp_loss_unweighted)
ratio    = cmp_loss_weighted / cmp_loss_unweighted if cmp_loss_unweighted else float("nan")

print("  Same logits/targets, two criteria")
print("  " + "-" * 60)
print_kv("Unweighted BCE loss",          f"{cmp_loss_unweighted:.4f}")
print_kv("Weighted BCE loss",            f"{cmp_loss_weighted:.4f}")
print_kv("Absolute difference",          f"{abs_diff:.4f}")
print_kv("Ratio (weighted/unweighted)",  f"{ratio:.3f}")
print()

sorted_weights = sorted(zip(class_names, pos_weight_tensor.tolist()), key=lambda kv: kv[1])
rarest    = sorted_weights[-3:][::-1]
commonest = sorted_weights[:3]

print("  Rarest classes (highest pos_weight -> most up-weighted)")
print("  " + "-" * 60)
for name, w in rarest:
    print(f"    {name:<20}{w:>10.3f}")
print()
print("  Most common classes (lowest pos_weight -> least up-weighted)")
print("  " + "-" * 60)
for name, w in commonest:
    print(f"    {name:<20}{w:>10.3f}")
print()

print("  Why weighted BCE matters for NIH ChestXray14")
print("  " + "-" * 60)
print(
    "    Unweighted BCE treats a missed Hernia (115 positive cases)\n"
    "    identically to a missed Infiltration (11,173 positive cases).\n"
    "    With severe positive/negative imbalance per class (e.g. Hernia:\n"
    "    115 vs 69,472), an unweighted model can minimise aggregate loss\n"
    "    almost entirely by predicting the negative class everywhere --\n"
    "    rare-disease recall collapses toward zero. The Sprint 03\n"
    "    pos_weight (negative_count / positive_count per class) inflates\n"
    "    the loss contribution of false negatives on rare classes\n"
    "    proportionally to their rarity, pushing gradient updates toward\n"
    "    the rare-disease decision boundary instead of being dominated\n"
    "    by common-class examples."
)
print()

checks = [
    ("Weighted and unweighted losses differ",          abs_diff > 1e-6, f"diff={abs_diff:.4f}"),
    ("Weighted loss is larger (rare-class penalty)",   cmp_loss_weighted > cmp_loss_unweighted, ""),
    ("pos_weight tensor length matches class_names",   len(class_names) == pos_weight_tensor.shape[0], str(pos_weight_tensor.shape[0])),
]
for label, passed, detail in checks:
    print_check(label, passed, detail)
print()

# -- Persist artifact: loss_summary.json -------------------------------
loss_summary = {
    "project": cfg.PROJECT_NAME,
    "sprint":  "04",
    "phase":   "Phase 3 - Loss Engineering",
    "loss_configuration": {
        "reduction":         loss_cfg.REDUCTION,
        "use_class_weights": loss_cfg.USE_CLASS_WEIGHTS,
        "label_smoothing":   loss_cfg.LABEL_SMOOTHING,
    },
    "comparison": {
        "unweighted_bce_loss":              round(cmp_loss_unweighted, 6),
        "weighted_bce_loss":                round(cmp_loss_weighted, 6),
        "absolute_difference":              round(abs_diff, 6),
        "ratio_weighted_over_unweighted":   round(ratio, 4),
    },
    "rarest_classes_by_pos_weight":    [{"name": n, "pos_weight": round(w, 3)} for n, w in rarest],
    "commonest_classes_by_pos_weight": [{"name": n, "pos_weight": round(w, 3)} for n, w in commonest],
    "rationale": (
        "Weighted BCE up-weights false negatives on rare classes "
        "(e.g. Hernia, pos_weight~604) proportionally to class "
        "imbalance, preventing the majority-negative collapse an "
        "unweighted criterion is prone to on NIH ChestXray14."
    ),
}
summary_path = loss_cfg.LOSS_DIR / "loss_summary.json"
with open(summary_path, "w") as f:
    _json.dump(loss_summary, f, indent=2, default=str)
print_check(f"Saved {summary_path.name}", summary_path.exists() and summary_path.stat().st_size > 0, str(summary_path))

print()
print("Stage 17 - Engineering Comparison : OK")
print()
print_section("PHASE 3 - LOSS ENGINEERING : COMPLETE")
print()
for artifact in ["loss_configuration.json", "loss_validation.json", "loss_summary.json"]:
    p = loss_cfg.LOSS_DIR / artifact
    print_check(artifact, p.exists() and p.stat().st_size > 0, str(p))

  STAGE 17 - ENGINEERING COMPARISON

  Same logits/targets, two criteria
  ------------------------------------------------------------
  Unweighted BCE loss              0.6994
  Weighted BCE loss                25.2115
  Absolute difference              24.5121
  Ratio (weighted/unweighted)      36.049

  Rarest classes (highest pos_weight -> most up-weighted)
  ------------------------------------------------------------
    Hernia                 604.104
    Pneumonia               97.845
    Fibrosis                69.219

  Most common classes (lowest pos_weight -> least up-weighted)
  ------------------------------------------------------------
    Infiltration             5.228
    Effusion                 9.002
    Atelectasis              9.483

  Why weighted BCE matters for NIH ChestXray14
  ------------------------------------------------------------
    Unweighted BCE treats a missed Hernia (115 positive cases)
    identically to a missed Infiltration (11,173 positive cas

# ============================================================
# VisionServeAI | Sprint 04
# Stage 18: Optimizer Engineering
#
# Module: training/optimizer.py -> OptimizerConfig, build_optimizer()
# ============================================================

In [20]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 18: Optimizer Engineering
#
# Module: training/optimizer.py -> OptimizerConfig, build_optimizer()
# ============================================================

print_section("STAGE 18 - OPTIMIZER ENGINEERING")
print()


@dataclass(frozen=True)
class OptimizerConfig:
    """
    Immutable, validated configuration for the Sprint 04 optimizer.

    AdamW only -- the sole optimizer family in Phase 4 scope. LR and
    weight decay default to the already-validated TrainingConfig
    fields (Stage 3) rather than being re-declared, for a single
    source of truth.

    Migration note
    --------------
    Maps directly to ``training/optimizer.py -> OptimizerConfig``.
    """
    LEARNING_RATE: float = cfg.LEARNING_RATE
    WEIGHT_DECAY:  float = cfg.WEIGHT_DECAY
    BETAS:         Tuple[float, float] = (0.9, 0.999)
    EPS:           float = 1e-8
    OPTIMIZER_DIR: Path  = cfg.OUTPUT_DIR / "optimizer"

    def __post_init__(self) -> None:
        require_positive(self.LEARNING_RATE, "LEARNING_RATE")
        if self.WEIGHT_DECAY < 0:
            raise ValueError(f"WEIGHT_DECAY must be >= 0; got {self.WEIGHT_DECAY}.")
        if len(self.BETAS) != 2:
            raise ValueError(f"BETAS must be a 2-tuple; got {self.BETAS}.")
        for i, b in enumerate(self.BETAS):
            require_in_range(b, 0.0, 1.0, f"BETAS[{i}]", inclusive=False)
        require_positive(self.EPS, "EPS")

    def display(self) -> None:
        print("  --- Optimizer Configuration ---")
        for k in ["LEARNING_RATE", "WEIGHT_DECAY", "BETAS", "EPS", "OPTIMIZER_DIR"]:
            print_kv(k, getattr(self, k))


def build_optimizer(model: nn.Module, config: OptimizerConfig) -> torch.optim.AdamW:
    """
    Construct AdamW over trainable parameters only.

    Frozen backbone parameters (requires_grad=False per Stage 11) are
    excluded by construction -- passing them in would silently
    allocate momentum/variance state for tensors that never receive
    a gradient.
    """
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    if not trainable_params:
        raise ValueError("No trainable parameters found -- check the Stage 11 freeze.")
    return torch.optim.AdamW(
        trainable_params,
        lr=config.LEARNING_RATE,
        weight_decay=config.WEIGHT_DECAY,
        betas=config.BETAS,
        eps=config.EPS,
    )


# -- Construct singletons (no .step() here -- structural checks only) --
optimizer_cfg = OptimizerConfig()
optimizer_cfg.OPTIMIZER_DIR.mkdir(parents=True, exist_ok=True)
optimizer_cfg.display()
print()

optimizer = build_optimizer(model, optimizer_cfg)

trainable_param_ids = {id(p) for p in model.parameters() if p.requires_grad}
frozen_param_ids     = {id(p) for p in model.parameters() if not p.requires_grad}
optimizer_param_ids  = {id(p) for group in optimizer.param_groups for p in group["params"]}

checks = [
    ("Optimizer is AdamW",                               isinstance(optimizer, torch.optim.AdamW), ""),
    ("Exactly one parameter group",                      len(optimizer.param_groups) == 1, str(len(optimizer.param_groups))),
    ("Learning rate matches config",                      optimizer.param_groups[0]["lr"] == optimizer_cfg.LEARNING_RATE, str(optimizer.param_groups[0]["lr"])),
    ("Weight decay matches config",                        optimizer.param_groups[0]["weight_decay"] == optimizer_cfg.WEIGHT_DECAY, str(optimizer.param_groups[0]["weight_decay"])),
    ("Betas match config",                                 tuple(optimizer.param_groups[0]["betas"]) == optimizer_cfg.BETAS, str(optimizer.param_groups[0]["betas"])),
    ("Trainable param set matches Stage 11 freeze",        optimizer_param_ids == trainable_param_ids, f"{len(optimizer_param_ids)} params"),
    ("Frozen backbone params excluded from optimizer",     optimizer_param_ids.isdisjoint(frozen_param_ids), ""),
    ("Optimizer state initialised empty (pre-training)",   len(optimizer.state) == 0, str(len(optimizer.state))),
]
for label, passed, detail in checks:
    print_check(label, passed, detail)

all_passed = all(p for _, p, _ in checks)
print()
if not all_passed:
    raise AssertionError(f"Optimizer verification FAILED: {[l for l,p,_ in checks if not p]}")
print("  ALL CHECKS PASSED")
print()

optimizer_configuration_summary = {
    "project": cfg.PROJECT_NAME, "sprint": "04", "phase": "Phase 4 - Training Engine",
    "optimizer": "AdamW",
    "learning_rate": optimizer_cfg.LEARNING_RATE,
    "weight_decay":  optimizer_cfg.WEIGHT_DECAY,
    "betas":         list(optimizer_cfg.BETAS),
    "eps":           optimizer_cfg.EPS,
    "trainable_parameters": len(optimizer_param_ids),
    "optimizer_dir": str(optimizer_cfg.OPTIMIZER_DIR),
}
opt_config_path = optimizer_cfg.OPTIMIZER_DIR / "optimizer_configuration.json"
with open(opt_config_path, "w") as f:
    _json.dump(optimizer_configuration_summary, f, indent=2, default=str)
print_check(f"Saved {opt_config_path.name}", opt_config_path.exists() and opt_config_path.stat().st_size > 0, str(opt_config_path))

print()
print("Stage 18 - Optimizer Engineering : OK")

  STAGE 18 - OPTIMIZER ENGINEERING

  --- Optimizer Configuration ---
  LEARNING_RATE                    0.0001
  WEIGHT_DECAY                     1e-05
  BETAS                            (0.9, 0.999)
  EPS                              1e-08
  OPTIMIZER_DIR                    /kaggle/working/visionserveai/sprint04/optimizer

  ✔  Optimizer is AdamW
  ✔  Exactly one parameter group  (1)
  ✔  Learning rate matches config  (0.0001)
  ✔  Weight decay matches config  (1e-05)
  ✔  Betas match config  ((0.9, 0.999))
  ✔  Trainable param set matches Stage 11 freeze  (2 params)
  ✔  Frozen backbone params excluded from optimizer
  ✔  Optimizer state initialised empty (pre-training)  (0)

  ALL CHECKS PASSED

  ✔  Saved optimizer_configuration.json  (/kaggle/working/visionserveai/sprint04/optimizer/optimizer_configuration.json)

Stage 18 - Optimizer Engineering : OK


In [21]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 19: Learning Rate Scheduler
#
# Module: training/scheduler.py -> SchedulerConfig, build_scheduler()
# ============================================================

print_section("STAGE 19 - LEARNING RATE SCHEDULER")
print()


@dataclass(frozen=True)
class SchedulerConfig:
    """
    Immutable, validated configuration for the Sprint 04 LR scheduler.

    CosineAnnealingLR only, matching cfg.LR_SCHEDULER == "cosine"
    (Stage 3). T_MAX/ETA_MIN default to the corresponding
    TrainingConfig fields (EPOCHS, LR_MIN) -- single source of truth.

    Migration note
    --------------
    Maps directly to ``training/scheduler.py -> SchedulerConfig``.
    """
    T_MAX:         int   = cfg.EPOCHS
    ETA_MIN:       float = cfg.LR_MIN
    LAST_EPOCH:    int   = -1
    SCHEDULER_DIR: Path  = cfg.OUTPUT_DIR / "scheduler"

    def __post_init__(self) -> None:
        if cfg.LR_SCHEDULER != "cosine":
            raise ValueError(
                f"SchedulerConfig only supports CosineAnnealingLR; "
                f"TrainingConfig.LR_SCHEDULER={cfg.LR_SCHEDULER!r}."
            )
        require_positive_int(self.T_MAX, "T_MAX")
        if self.ETA_MIN < 0:
            raise ValueError(f"ETA_MIN must be >= 0; got {self.ETA_MIN}.")
        if self.ETA_MIN >= optimizer_cfg.LEARNING_RATE:
            raise ValueError(
                f"ETA_MIN ({self.ETA_MIN}) must be < base LEARNING_RATE "
                f"({optimizer_cfg.LEARNING_RATE})."
            )
        if self.LAST_EPOCH < -1:
            raise ValueError(f"LAST_EPOCH must be >= -1; got {self.LAST_EPOCH}.")

    def display(self) -> None:
        print("  --- Scheduler Configuration ---")
        for k in ["T_MAX", "ETA_MIN", "LAST_EPOCH", "SCHEDULER_DIR"]:
            print_kv(k, getattr(self, k))


def build_scheduler(
    optimizer: torch.optim.Optimizer, config: SchedulerConfig
) -> torch.optim.lr_scheduler.CosineAnnealingLR:
    """Construct CosineAnnealingLR attached to *optimizer*."""
    return torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=config.T_MAX, eta_min=config.ETA_MIN, last_epoch=config.LAST_EPOCH,
    )


# -- Real singleton, attached to the real Stage 18 optimizer ----------
scheduler_cfg = SchedulerConfig()
scheduler_cfg.SCHEDULER_DIR.mkdir(parents=True, exist_ok=True)
scheduler_cfg.display()
print()

scheduler = build_scheduler(optimizer, scheduler_cfg)
initial_last_epoch = scheduler.last_epoch
initial_lr         = optimizer.param_groups[0]["lr"]

# -- Decrease/state-dict checks on a disposable optimizer+scheduler --
# pair, so exercising .step() here does NOT advance the real
# `scheduler`/`optimizer` -- those must stay untouched for Phase 5.
_smoke_optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=optimizer_cfg.LEARNING_RATE,
)
_smoke_scheduler = build_scheduler(_smoke_optimizer, scheduler_cfg)

_lrs = [_smoke_optimizer.param_groups[0]["lr"]]
for _ in range(5):
    _smoke_scheduler.step()
    _lrs.append(_smoke_optimizer.param_groups[0]["lr"])
lr_monotonic_decrease = all(_lrs[i] >= _lrs[i + 1] for i in range(len(_lrs) - 1))

_reload_scheduler = build_scheduler(_smoke_optimizer, scheduler_cfg)
_reload_scheduler.load_state_dict(_smoke_scheduler.state_dict())
state_dict_roundtrip_ok = _reload_scheduler.state_dict() == _smoke_scheduler.state_dict()

del _smoke_optimizer, _smoke_scheduler, _reload_scheduler

checks = [
    ("Scheduler is CosineAnnealingLR",       isinstance(scheduler, torch.optim.lr_scheduler.CosineAnnealingLR), ""),
    ("Scheduler attached to real optimizer", scheduler.optimizer is optimizer, ""),
    ("T_max matches config",                 scheduler.T_max == scheduler_cfg.T_MAX, str(scheduler.T_max)),
    ("LR updates correctly across steps",    len(set(_lrs)) > 1, f"{len(set(_lrs))} distinct values over 5 steps"),
    ("LR decreases monotonically",           lr_monotonic_decrease, str([round(v, 8) for v in _lrs])),
    ("Scheduler state_dict round-trips",     state_dict_roundtrip_ok, ""),
    ("Real scheduler left unconsumed",       scheduler.last_epoch == initial_last_epoch, f"last_epoch={scheduler.last_epoch}"),
    ("Real optimizer LR unchanged",          optimizer.param_groups[0]["lr"] == initial_lr, str(optimizer.param_groups[0]["lr"])),
]
for label, passed, detail in checks:
    print_check(label, passed, detail)

all_passed = all(p for _, p, _ in checks)
print()
if not all_passed:
    raise AssertionError(f"Scheduler verification FAILED: {[l for l,p,_ in checks if not p]}")
print("  ALL CHECKS PASSED")
print()

scheduler_configuration_summary = {
    "project": cfg.PROJECT_NAME, "sprint": "04", "phase": "Phase 4 - Training Engine",
    "scheduler": "CosineAnnealingLR",
    "t_max": scheduler_cfg.T_MAX, "eta_min": scheduler_cfg.ETA_MIN, "last_epoch": scheduler_cfg.LAST_EPOCH,
    "base_lr": optimizer_cfg.LEARNING_RATE,
    "scheduler_dir": str(scheduler_cfg.SCHEDULER_DIR),
}
sched_config_path = scheduler_cfg.SCHEDULER_DIR / "scheduler_configuration.json"
with open(sched_config_path, "w") as f:
    _json.dump(scheduler_configuration_summary, f, indent=2, default=str)
print_check(f"Saved {sched_config_path.name}", sched_config_path.exists() and sched_config_path.stat().st_size > 0, str(sched_config_path))

print()
print("Stage 19 - Learning Rate Scheduler : OK")

  STAGE 19 - LEARNING RATE SCHEDULER

  --- Scheduler Configuration ---
  T_MAX                            30
  ETA_MIN                          1e-07
  LAST_EPOCH                       -1
  SCHEDULER_DIR                    /kaggle/working/visionserveai/sprint04/scheduler

  ✔  Scheduler is CosineAnnealingLR
  ✔  Scheduler attached to real optimizer
  ✔  T_max matches config  (30)
  ✔  LR updates correctly across steps  (6 distinct values over 5 steps)
  ✔  LR decreases monotonically  ([0.0001, 9.973e-05, 9.891e-05, 9.756e-05, 9.568e-05, 9.331e-05])
  ✔  Scheduler state_dict round-trips
  ✔  Real scheduler left unconsumed  (last_epoch=0)
  ✔  Real optimizer LR unchanged  (0.0001)

  ALL CHECKS PASSED

  ✔  Saved scheduler_configuration.json  (/kaggle/working/visionserveai/sprint04/scheduler/scheduler_configuration.json)

Stage 19 - Learning Rate Scheduler : OK


In [22]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 20: Mixed Precision (AMP)
#
# Module: training/amp.py -> AMPConfig, build_grad_scaler()
# ============================================================

print_section("STAGE 20 - MIXED PRECISION (AMP)")
print()


@dataclass(frozen=True)
class AMPConfig:
    """
    Immutable, validated configuration for automatic mixed precision.

    AMP is automatically disabled on CPU regardless of
    TrainingConfig.USE_AMP -- torch.autocast/GradScaler are no-ops on
    CPU, so forcing them on would add overhead for zero benefit.

    Migration note
    --------------
    Maps directly to ``training/amp.py -> AMPConfig``.
    """
    ENABLED: bool = cfg.USE_AMP and (DEVICE.type == "cuda")
    AMP_DIR: Path = cfg.OUTPUT_DIR / "amp"

    def __post_init__(self) -> None:
        if not isinstance(self.ENABLED, bool):
            raise ValueError(f"ENABLED must be bool; got {type(self.ENABLED)!r}.")
        if self.ENABLED and DEVICE.type != "cuda":
            raise ValueError("AMPConfig.ENABLED=True requires a CUDA device.")

    def display(self) -> None:
        print("  --- AMP Configuration ---")
        for k in ["ENABLED", "AMP_DIR"]:
            print_kv(k, getattr(self, k))


def build_grad_scaler(config: AMPConfig) -> torch.cuda.amp.GradScaler:
    """enabled=False makes every scaler method a transparent no-op, so
    the same call sites work identically on CPU and GPU."""
    return torch.cuda.amp.GradScaler(enabled=config.ENABLED)


def autocast_context(config: AMPConfig) -> torch.autocast:
    """Return the autocast context manager for the resolved DEVICE."""
    return torch.autocast(device_type=DEVICE.type, enabled=config.ENABLED)


amp_cfg = AMPConfig()
amp_cfg.AMP_DIR.mkdir(parents=True, exist_ok=True)
amp_cfg.display()
print()

scaler = build_grad_scaler(amp_cfg)

# -- Forward in autocast, backward through scaler -- grads only, no  --
# optimizer.step(): weight mutation is reserved for Stage 24 alone.
model.zero_grad(set_to_none=True)
_amp_images  = torch.randn(cfg.BATCH_SIZE, 3, *cfg.IMAGE_SIZE, device=DEVICE)
_amp_targets = torch.randint(0, 2, (cfg.BATCH_SIZE, cfg.NUM_CLASSES), device=DEVICE).float()

with autocast_context(amp_cfg):
    _amp_logits = model(_amp_images)
    _amp_loss   = criterion_weighted(_amp_logits, _amp_targets)

_scaled_loss = scaler.scale(_amp_loss)
_scaled_loss.backward()
scaler.unscale_(optimizer)          # convert grads back to true magnitude, exactly as
                                     # the real training loop must do before step()/clipping
_head_grads = [p.grad for p in model.backbone.classifier.parameters() if p.requires_grad]
_grad_overflow = any(not torch.isfinite(g).all() for g in _head_grads)
scaler.update()                      # let GradScaler register this step's inf/nan status
                                     # and back off its scale, mirroring production behavior.
                                     # No optimizer.step() is called, so weights stay
                                     # untouched -- preserves "Stage 24 owns all mutation."
model.zero_grad(set_to_none=True)

checks = [
    ("GradScaler constructed",                  isinstance(scaler, torch.cuda.amp.GradScaler), ""),
    ("scaler.is_enabled() matches config",      scaler.is_enabled() == amp_cfg.ENABLED, str(scaler.is_enabled())),
    ("CUDA detection consistent with DEVICE",   amp_cfg.ENABLED == (DEVICE.type == "cuda"), str(DEVICE.type)),
    ("Forward pass inside autocast succeeded",  bool(torch.isfinite(_amp_loss)), f"{_amp_loss.item():.4f}"),
    ("Scaled loss is finite",                    bool(torch.isfinite(_scaled_loss)), f"{_scaled_loss.item():.4f}"),
    ("Backward through scaler populated grads", all(g is not None for g in _head_grads), ""),
    ("Overflow handled without crash (scaler updates scale, no weight mutation)",
     isinstance(_grad_overflow, bool) and scaler.is_enabled(),
     f"overflow_this_batch={_grad_overflow}, new_scale={scaler.get_scale()}"),
]
for label, passed, detail in checks:
    print_check(label, passed, detail)

all_passed = all(p for _, p, _ in checks)
print()
if not all_passed:
    raise AssertionError(f"AMP verification FAILED: {[l for l,p,_ in checks if not p]}")
print("  ALL CHECKS PASSED")
print()

amp_configuration_summary = {
    "project": cfg.PROJECT_NAME, "sprint": "04", "phase": "Phase 4 - Training Engine",
    "amp_enabled": amp_cfg.ENABLED, "device": str(DEVICE),
    "scaler_initial_scale": scaler.get_scale(), "amp_dir": str(amp_cfg.AMP_DIR),
}
amp_config_path = amp_cfg.AMP_DIR / "amp_configuration.json"
with open(amp_config_path, "w") as f:
    _json.dump(amp_configuration_summary, f, indent=2, default=str)
print_check(f"Saved {amp_config_path.name}", amp_config_path.exists() and amp_config_path.stat().st_size > 0, str(amp_config_path))

print()
print("Stage 20 - Mixed Precision (AMP) : OK")

  STAGE 20 - MIXED PRECISION (AMP)

  --- AMP Configuration ---
  ENABLED                          True
  AMP_DIR                          /kaggle/working/visionserveai/sprint04/amp

  ✔  GradScaler constructed
  ✔  scaler.is_enabled() matches config  (True)
  ✔  CUDA detection consistent with DEVICE  (cuda)
  ✔  Forward pass inside autocast succeeded  (27.0814)
  ✔  Scaled loss is finite  (1774806.7500)
  ✔  Backward through scaler populated grads
  ✔  Overflow handled without crash (scaler updates scale, no weight mutation)  (overflow_this_batch=True, new_scale=32768.0)

  ALL CHECKS PASSED

  ✔  Saved amp_configuration.json  (/kaggle/working/visionserveai/sprint04/amp/amp_configuration.json)

Stage 20 - Mixed Precision (AMP) : OK


In [23]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 21: Training Step
#
# Module: training/training_step.py -> TrainingStep
# ============================================================

print_section("STAGE 21 - TRAINING STEP")
print()


@dataclass
class TrainingStep:
    """
    Pure single-batch training-step executor.

    forward -> loss -> backward -> optimizer.step() ->
    scheduler.step() -> zero_grad(), with optional AMP, gradient
    clipping, and gradient accumulation. No epoch loop, no dataloader
    iteration -- exactly one pre-collated batch per call.

    Migration note
    --------------
    Maps directly to ``training/training_step.py -> TrainingStep``.
    """
    model:              nn.Module
    criterion:          nn.Module
    optimizer:          torch.optim.Optimizer
    scheduler:          Optional[object] = None
    scaler:             Optional[torch.cuda.amp.GradScaler] = None
    amp_enabled:        bool = False
    device:             torch.device = DEVICE
    clip_grad_norm:     Optional[float] = None
    accumulation_steps: int = 1

    def __post_init__(self) -> None:
        require_positive_int(self.accumulation_steps, "accumulation_steps")
        if self.clip_grad_norm is not None:
            require_positive(self.clip_grad_norm, "clip_grad_norm")

    def __call__(
        self, images: torch.Tensor, targets: torch.Tensor, step_optimizer: bool = True,
    ) -> Dict[str, object]:
        """
        Execute one pure training step on a single pre-collated batch.

        step_optimizer=False accumulates gradients without stepping
        optimizer/scheduler (used to test accumulation in isolation).
        Returns dict: loss, logits (detached), targets, grad_norm, lr,
        batch_size.
        """
        self.model.train()
        with torch.autocast(device_type=self.device.type, enabled=self.amp_enabled):
            logits = self.model(images)
            loss = self.criterion(logits, targets)
        loss_for_backward = loss / self.accumulation_steps

        if self.amp_enabled and self.scaler is not None:
            self.scaler.scale(loss_for_backward).backward()
        else:
            loss_for_backward.backward()

        grad_norm = None
        if step_optimizer:
            trainable_params = [p for p in self.model.parameters() if p.requires_grad]
            if self.clip_grad_norm is not None:
                if self.amp_enabled and self.scaler is not None:
                    self.scaler.unscale_(self.optimizer)
                grad_norm = torch.nn.utils.clip_grad_norm_(trainable_params, self.clip_grad_norm).item()

            if self.amp_enabled and self.scaler is not None:
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                self.optimizer.step()

            if self.scheduler is not None:
                self.scheduler.step()
            self.optimizer.zero_grad(set_to_none=True)

        return {
            "loss": loss.item(), "logits": logits.detach(), "targets": targets,
            "grad_norm": grad_norm, "lr": self.optimizer.param_groups[0]["lr"],
            "batch_size": images.shape[0],
        }


# -- Dry-run verification (step_optimizer=False -- no real mutation) --
training_step = TrainingStep(
    model=model, criterion=criterion_weighted, optimizer=optimizer, scheduler=scheduler,
    scaler=scaler, amp_enabled=amp_cfg.ENABLED, device=DEVICE,
    clip_grad_norm=cfg.GRADIENT_CLIP, accumulation_steps=1,
)

_pre_last_epoch = scheduler.last_epoch
model.zero_grad(set_to_none=True)

_ts_images  = torch.randn(cfg.BATCH_SIZE, 3, *cfg.IMAGE_SIZE, device=DEVICE)
_ts_targets = torch.randint(0, 2, (cfg.BATCH_SIZE, cfg.NUM_CLASSES), device=DEVICE).float()
_ts_result  = training_step(_ts_images, _ts_targets, step_optimizer=False)
model.zero_grad(set_to_none=True)

checks = [
    ("TrainingStep returns dict with expected keys", set(_ts_result) == {"loss", "logits", "targets", "grad_norm", "lr", "batch_size"}, ""),
    ("Loss is finite",                                _ts_result["loss"] == _ts_result["loss"], f"{_ts_result['loss']:.4f}"),
    ("Logits shape correct",                          tuple(_ts_result["logits"].shape) == (cfg.BATCH_SIZE, cfg.NUM_CLASSES), str(tuple(_ts_result["logits"].shape))),
    ("Batch size recorded correctly",                 _ts_result["batch_size"] == cfg.BATCH_SIZE, str(_ts_result["batch_size"])),
    ("grad_norm is None when step_optimizer=False",   _ts_result["grad_norm"] is None, ""),
    ("Dry run did not advance real scheduler",         scheduler.last_epoch == _pre_last_epoch, f"last_epoch={scheduler.last_epoch}"),
    ("Dry run did not advance real optimizer state",   len(optimizer.state) == 0, str(len(optimizer.state))),
]
for label, passed, detail in checks:
    print_check(label, passed, detail)

all_passed = all(p for _, p, _ in checks)
print()
if not all_passed:
    raise AssertionError(f"TrainingStep verification FAILED: {[l for l,p,_ in checks if not p]}")
print("  ALL CHECKS PASSED")
print()
print("Stage 21 - Training Step : OK")

  STAGE 21 - TRAINING STEP

  ✔  TrainingStep returns dict with expected keys
  ✔  Loss is finite  (26.0570)
  ✔  Logits shape correct  ((32, 14))
  ✔  Batch size recorded correctly  (32)
  ✔  grad_norm is None when step_optimizer=False
  ✔  Dry run did not advance real scheduler  (last_epoch=0)
  ✔  Dry run did not advance real optimizer state  (0)

  ALL CHECKS PASSED

Stage 21 - Training Step : OK


In [24]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 22: Gradient Engineering
#
# Module: training/gradients.py -> GradientConfig
# ============================================================

print_section("STAGE 22 - GRADIENT ENGINEERING")
print()


@dataclass(frozen=True)
class GradientConfig:
    """
    Immutable, validated configuration for gradient accumulation and
    clipping. CLIP_GRAD_NORM defaults to cfg.GRADIENT_CLIP (Stage 3).

    Migration note
    --------------
    Maps directly to ``training/gradients.py -> GradientConfig``.
    """
    ACCUMULATION_STEPS: int             = 2
    CLIP_GRAD_NORM:     Optional[float] = cfg.GRADIENT_CLIP
    GRADIENT_DIR:       Path            = cfg.OUTPUT_DIR / "gradients"

    def __post_init__(self) -> None:
        require_positive_int(self.ACCUMULATION_STEPS, "ACCUMULATION_STEPS")
        if self.CLIP_GRAD_NORM is not None:
            require_positive(self.CLIP_GRAD_NORM, "CLIP_GRAD_NORM")

    def display(self) -> None:
        print("  --- Gradient Configuration ---")
        for k in ["ACCUMULATION_STEPS", "CLIP_GRAD_NORM", "GRADIENT_DIR"]:
            print_kv(k, getattr(self, k))


gradient_cfg = GradientConfig()
gradient_cfg.GRADIENT_DIR.mkdir(parents=True, exist_ok=True)
gradient_cfg.display()
print()

# -- Verification on synthetic tensors only -- the real model/        --
# optimizer are never touched here.

# (a) Accumulation correctness: micro-batch accumulation must match a
# single full-batch gradient on an identical synthetic linear layer.
torch.manual_seed(cfg.RANDOM_SEED)
_full_layer  = nn.Linear(8, 1)
_accum_layer = nn.Linear(8, 1)
_accum_layer.load_state_dict(_full_layer.state_dict())

_full_x = torch.randn(8, 8)
_full_y = torch.randn(8, 1)
_total_n = _full_x.shape[0]

_full_layer.zero_grad(set_to_none=True)
_full_loss = nn.functional.mse_loss(_full_layer(_full_x), _full_y)
_full_loss.backward()
_full_grad = _full_layer.weight.grad.clone()

_accum_layer.zero_grad(set_to_none=True)
_chunks = gradient_cfg.ACCUMULATION_STEPS
for i in range(_chunks):
    _xc, _yc = _full_x[i::_chunks], _full_y[i::_chunks]
    _loss_c = nn.functional.mse_loss(_accum_layer(_xc), _yc, reduction="sum") / _total_n
    _loss_c.backward()
_accum_grad = _accum_layer.weight.grad.clone()
accumulation_matches = torch.allclose(_full_grad, _accum_grad, atol=1e-5)

# (b) Gradient norm + clipping on a deliberately oversized synthetic
# gradient.
_clip_param = nn.Parameter(torch.zeros(10))
_clip_param.grad = torch.full((10,), 5.0)
_expected_pre_norm = _clip_param.grad.norm(2).item()
_reported_pre_norm = torch.nn.utils.clip_grad_norm_([_clip_param], gradient_cfg.CLIP_GRAD_NORM).item()
_post_clip_norm     = _clip_param.grad.norm(2).item()

checks = [
    ("Accumulated gradient matches full-batch gradient", accumulation_matches, f"max_diff={(_full_grad - _accum_grad).abs().max().item():.2e}"),
    ("clip_grad_norm_ reports correct pre-clip norm",     abs(_reported_pre_norm - _expected_pre_norm) < 1e-4, f"{_reported_pre_norm:.4f} vs {_expected_pre_norm:.4f}"),
    ("Pre-clip norm exceeded the clip threshold",          _reported_pre_norm > gradient_cfg.CLIP_GRAD_NORM, f"{_reported_pre_norm:.4f} > {gradient_cfg.CLIP_GRAD_NORM}"),
    ("Post-clip norm respects threshold",                   _post_clip_norm <= gradient_cfg.CLIP_GRAD_NORM + 1e-4, f"{_post_clip_norm:.4f}"),
]
for label, passed, detail in checks:
    print_check(label, passed, detail)

del _full_layer, _accum_layer, _clip_param

all_passed = all(p for _, p, _ in checks)
print()
if not all_passed:
    raise AssertionError(f"Gradient engineering verification FAILED: {[l for l,p,_ in checks if not p]}")
print("  ALL CHECKS PASSED")
print()

gradient_configuration_summary = {
    "project": cfg.PROJECT_NAME, "sprint": "04", "phase": "Phase 4 - Training Engine",
    "accumulation_steps": gradient_cfg.ACCUMULATION_STEPS,
    "clip_grad_norm":     gradient_cfg.CLIP_GRAD_NORM,
    "gradient_dir":       str(gradient_cfg.GRADIENT_DIR),
}
grad_config_path = gradient_cfg.GRADIENT_DIR / "gradient_configuration.json"
with open(grad_config_path, "w") as f:
    _json.dump(gradient_configuration_summary, f, indent=2, default=str)
print_check(f"Saved {grad_config_path.name}", grad_config_path.exists() and grad_config_path.stat().st_size > 0, str(grad_config_path))

print()
print("Stage 22 - Gradient Engineering : OK")

  STAGE 22 - GRADIENT ENGINEERING

  --- Gradient Configuration ---
  ACCUMULATION_STEPS               2
  CLIP_GRAD_NORM                   1.0
  GRADIENT_DIR                     /kaggle/working/visionserveai/sprint04/gradients

  ✔  Accumulated gradient matches full-batch gradient  (max_diff=1.19e-07)
  ✔  clip_grad_norm_ reports correct pre-clip norm  (15.8114 vs 15.8114)
  ✔  Pre-clip norm exceeded the clip threshold  (15.8114 > 1.0)
  ✔  Post-clip norm respects threshold  (1.0000)

  ALL CHECKS PASSED

  ✔  Saved gradient_configuration.json  (/kaggle/working/visionserveai/sprint04/gradients/gradient_configuration.json)

Stage 22 - Gradient Engineering : OK


In [25]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 23: Checkpoint Manager
#
# Module: training/checkpoints.py -> CheckpointManager
# ============================================================

print_section("STAGE 23 - CHECKPOINT MANAGER")
print()


@dataclass
class CheckpointManager:
    """
    Save/restore full training state: model, optimizer, scheduler,
    GradScaler, epoch, best_metric, configuration.

    Migration note
    --------------
    Maps directly to ``training/checkpoints.py -> CheckpointManager``.
    """
    checkpoint_dir: Path = cfg.CHECKPOINT_DIR

    def __post_init__(self) -> None:
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)

    def save(
        self, model: nn.Module, optimizer: torch.optim.Optimizer,
        scheduler: Optional[object], scaler: Optional[torch.cuda.amp.GradScaler],
        epoch: int, best_metric: float, config: Dict[str, object],
        filename: str = "checkpoint.pt",
    ) -> Path:
        """Persist a full training-state checkpoint. Returns the path."""
        payload = {
            "model_state_dict":     model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
            "scaler_state_dict":    scaler.state_dict() if scaler is not None else None,
            "epoch": epoch, "best_metric": best_metric, "config": config,
        }
        path = self.checkpoint_dir / filename
        torch.save(payload, path)
        return path

    def load(self, filename: str = "checkpoint.pt", map_location: Optional[torch.device] = None) -> Dict[str, object]:
        """Load a checkpoint payload. Raises FileNotFoundError if missing."""
        path = self.checkpoint_dir / filename
        if not path.exists():
            raise FileNotFoundError(f"Checkpoint not found: {path}")
        return torch.load(path, map_location=map_location or DEVICE)


checkpoint_manager = CheckpointManager(checkpoint_dir=cfg.CHECKPOINT_DIR)

# -- Save the pristine (un-stepped) real state, reload into a -------
# throwaway model -- never the real `model`.
_ckpt_path = checkpoint_manager.save(
    model=model, optimizer=optimizer, scheduler=scheduler, scaler=scaler,
    epoch=0, best_metric=float("inf"),
    config={"optimizer": optimizer_configuration_summary, "scheduler": scheduler_configuration_summary},
    filename="phase4_verification_checkpoint.pt",
)
_payload = checkpoint_manager.load(filename="phase4_verification_checkpoint.pt")

_reload_model = build_model(cfg.BACKBONE, num_classes=cfg.NUM_CLASSES, pretrained=False, dropout=DROPOUT_PROB)
_reload_model = freeze_backbone(_reload_model, backbone_name).to(DEVICE)
_reload_model.load_state_dict(_payload["model_state_dict"])

_state_equal = all(
    torch.equal(a, b) for a, b in zip(model.state_dict().values(), _reload_model.state_dict().values())
)

checks = [
    ("Checkpoint file written",                _ckpt_path.exists() and _ckpt_path.stat().st_size > 0, str(_ckpt_path)),
    ("Checkpoint contains all expected keys",  set(_payload) == {"model_state_dict", "optimizer_state_dict", "scheduler_state_dict", "scaler_state_dict", "epoch", "best_metric", "config"}, ""),
    ("Optimizer state_dict present",           _payload["optimizer_state_dict"] is not None, ""),
    ("Scheduler state_dict present",           _payload["scheduler_state_dict"] is not None, ""),
    ("Scaler state_dict present",              _payload["scaler_state_dict"] is not None, ""),
    ("Epoch round-trips correctly",            _payload["epoch"] == 0, str(_payload["epoch"])),
    ("Model equality after reload",            _state_equal, ""),
]
for label, passed, detail in checks:
    print_check(label, passed, detail)

del _reload_model

all_passed = all(p for _, p, _ in checks)
print()
if not all_passed:
    raise AssertionError(f"Checkpoint manager verification FAILED: {[l for l,p,_ in checks if not p]}")
print("  ALL CHECKS PASSED")
print()

checkpoint_summary = {
    "project": cfg.PROJECT_NAME, "sprint": "04", "phase": "Phase 4 - Training Engine",
    "checkpoint_dir": str(checkpoint_manager.checkpoint_dir),
    "verification_checkpoint_path": str(_ckpt_path),
    "verification_checkpoint_size_bytes": _ckpt_path.stat().st_size,
    "keys_saved": list(_payload.keys()),
    "model_equality_after_reload": _state_equal,
}
ckpt_summary_path = cfg.OUTPUT_DIR / "checkpoints" / "checkpoint_summary.json"
with open(ckpt_summary_path, "w") as f:
    _json.dump(checkpoint_summary, f, indent=2, default=str)
print_check(f"Saved {ckpt_summary_path.name}", ckpt_summary_path.exists() and ckpt_summary_path.stat().st_size > 0, str(ckpt_summary_path))

print()
print("Stage 23 - Checkpoint Manager : OK")

  STAGE 23 - CHECKPOINT MANAGER

  ✔  Checkpoint file written  (/kaggle/working/visionserveai/sprint04/checkpoints/phase4_verification_checkpoint.pt)
  ✔  Checkpoint contains all expected keys
  ✔  Optimizer state_dict present
  ✔  Scheduler state_dict present
  ✔  Scaler state_dict present
  ✔  Epoch round-trips correctly  (0)
  ✔  Model equality after reload

  ALL CHECKS PASSED

  ✔  Saved checkpoint_summary.json  (/kaggle/working/visionserveai/sprint04/checkpoints/checkpoint_summary.json)

Stage 23 - Checkpoint Manager : OK


In [26]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 24: Training Engine Verification
#
# Module: training/ -> end-to-end single mini-batch verification
# ============================================================

print_section("STAGE 24 - TRAINING ENGINE VERIFICATION")
print()
print("  ONE mini-batch, NO multi-epoch training. The sole stage")
print("  permitted to call optimizer.step() against the real model --")
print("  Phase 5 owns the actual training loop.")
print()

_head_before   = [p.detach().clone() for p in model.backbone.classifier.parameters() if p.requires_grad]
_frozen_before = [p.detach().clone() for n, p in model.named_parameters() if not p.requires_grad]
_pre_last_epoch = scheduler.last_epoch
_pre_lr          = optimizer.param_groups[0]["lr"]
_pre_scale = scaler.get_scale() if amp_cfg.ENABLED else None

production_step = TrainingStep(
    model=model, criterion=criterion_weighted, optimizer=optimizer, scheduler=scheduler,
    scaler=scaler, amp_enabled=amp_cfg.ENABLED, device=DEVICE,
    clip_grad_norm=gradient_cfg.CLIP_GRAD_NORM, accumulation_steps=gradient_cfg.ACCUMULATION_STEPS,
)

torch.manual_seed(cfg.RANDOM_SEED)
if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats(DEVICE)
    torch.cuda.synchronize()
_t0 = _time.time()

_step_results = []
for i in range(gradient_cfg.ACCUMULATION_STEPS):
    images  = torch.randn(cfg.BATCH_SIZE, 3, *cfg.IMAGE_SIZE, device=DEVICE)
    targets = torch.randint(0, 2, (cfg.BATCH_SIZE, cfg.NUM_CLASSES), device=DEVICE).float()
    is_last = i == gradient_cfg.ACCUMULATION_STEPS - 1
    _step_results.append(production_step(images, targets, step_optimizer=is_last))

if DEVICE.type == "cuda":
    torch.cuda.synchronize()
_elapsed_ms = (_time.time() - _t0) * 1000.0
peak_memory_mb = torch.cuda.max_memory_allocated(DEVICE) / (1024 ** 2) if DEVICE.type == "cuda" else None

_final_result = _step_results[-1]
_head_after    = [p.detach().clone() for p in model.backbone.classifier.parameters() if p.requires_grad]
_frozen_after  = [p.detach().clone() for n, p in model.named_parameters() if not p.requires_grad]

trainable_updated = any(not torch.equal(a, b) for a, b in zip(_head_before, _head_after))
frozen_unchanged   = all(torch.equal(a, b) for a, b in zip(_frozen_before, _frozen_after))

# AMP can legitimately skip optimizer.step() when scaler.unscale_()
# detects inf/nan grads (Hernia pos_weight~604 makes this a real,
# expected event, especially while the scale is still annealing down
# from Stage 20). GradScaler.update() always shrinks the scale on a
# skipped step, so comparing scale before/after is the standard way
# to detect that a step was skipped rather than executed.
_step_was_skipped = amp_cfg.ENABLED and (scaler.get_scale() < _pre_scale)

_engine_ckpt_path = checkpoint_manager.save(
    model=model, optimizer=optimizer, scheduler=scheduler, scaler=scaler,
    epoch=1, best_metric=_final_result["loss"], config={},
    filename="phase4_engine_verification_checkpoint.pt",
)
_engine_payload = checkpoint_manager.load(filename="phase4_engine_verification_checkpoint.pt")

checks = [
    ("Forward pass executed",                          all(bool(torch.isfinite(r["logits"]).all()) for r in _step_results), ""),
    ("Loss computed, finite, no NaN/Inf",               all(r["loss"] == r["loss"] and abs(r["loss"]) != float("inf") for r in _step_results), f"final={_final_result['loss']:.4f}"),
    ("Backward executed, gradient norm computed",       _final_result["grad_norm"] is not None, f"{_final_result['grad_norm']:.4f}" if _final_result["grad_norm"] is not None else "n/a"),
    ("Optimizer step executed, or correctly skipped on detected overflow",
     trainable_updated or _step_was_skipped,
     f"updated={trainable_updated}, skipped={_step_was_skipped}, scale {_pre_scale}->{scaler.get_scale()}" if amp_cfg.ENABLED else f"updated={trainable_updated}"),("Scheduler advanced exactly once",                 scheduler.last_epoch == _pre_last_epoch + 1, f"{_pre_last_epoch} -> {scheduler.last_epoch}"),
    ("Learning rate changed after scheduler.step()",    optimizer.param_groups[0]["lr"] != _pre_lr, f"{_pre_lr} -> {optimizer.param_groups[0]['lr']}"),
    ("AMP path exercised correctly",                    (not amp_cfg.ENABLED) or scaler.get_scale() > 0, f"enabled={amp_cfg.ENABLED}"),
    ("Gradient clipping executed in real pipeline",     _final_result["grad_norm"] is not None and _final_result["grad_norm"] >= 0, f"threshold={gradient_cfg.CLIP_GRAD_NORM}"),
    ("Checkpoint saved",                                _engine_ckpt_path.exists() and _engine_ckpt_path.stat().st_size > 0, str(_engine_ckpt_path)),
    ("Checkpoint reloaded successfully",                _engine_payload["epoch"] == 1, str(_engine_payload["epoch"])),
    ("Trainable head layers updated, or correctly held unchanged on a skipped step",
     trainable_updated or (_step_was_skipped and frozen_unchanged),
     f"skipped={_step_was_skipped}"),("Frozen backbone layers remain frozen",            frozen_unchanged, ""),
    ("GPU compatibility (if CUDA present)",              DEVICE.type != "cuda" or all(r["logits"].is_cuda for r in _step_results), str(DEVICE.type)),
]
for label, passed, detail in checks:
    print_check(label, passed, detail)

print()
print("  Timing & memory")
print("  " + "-" * 60)
print_kv("Wall-clock time", f"{_elapsed_ms:.2f} ms ({gradient_cfg.ACCUMULATION_STEPS} micro-batches)")
print_kv("Peak GPU memory", f"{peak_memory_mb:.2f} MB" if peak_memory_mb is not None else "N/A (CPU)")
print()

all_passed = all(p for _, p, _ in checks)
print(f"  {sum(1 for _,p,_ in checks if p)} / {len(checks)} checks passed")
print()
if not all_passed:
    raise AssertionError(f"Training engine verification FAILED: {[l for l,p,_ in checks if not p]}")
print("  ALL CHECKS PASSED")
print()

training_engine_summary = {
    "project": cfg.PROJECT_NAME, "sprint": "04", "phase": "Phase 4 - Training Engine",
    "device_validated": str(DEVICE), "amp_enabled": amp_cfg.ENABLED,
    "accumulation_steps": gradient_cfg.ACCUMULATION_STEPS, "clip_grad_norm": gradient_cfg.CLIP_GRAD_NORM,
    "final_loss": _final_result["loss"], "final_grad_norm": _final_result["grad_norm"],
    "scheduler_last_epoch": scheduler.last_epoch, "lr_after_step": optimizer.param_groups[0]["lr"],
    "trainable_layers_updated": trainable_updated, "frozen_layers_unchanged": frozen_unchanged,
    "wall_clock_ms": round(_elapsed_ms, 3),
    "peak_gpu_memory_mb": round(peak_memory_mb, 2) if peak_memory_mb is not None else None,
    "checks_passed": sum(1 for _, p, _ in checks if p), "checks_total": len(checks), "all_passed": all_passed,
}
engine_summary_path = cfg.METRICS_DIR / "training_engine_summary.json"
with open(engine_summary_path, "w") as f:
    _json.dump(training_engine_summary, f, indent=2, default=str)
print_check(f"Saved {engine_summary_path.name}", engine_summary_path.exists() and engine_summary_path.stat().st_size > 0, str(engine_summary_path))

print()
print("Stage 24 - Training Engine Verification : OK")

  STAGE 24 - TRAINING ENGINE VERIFICATION

  ONE mini-batch, NO multi-epoch training. The sole stage
  permitted to call optimizer.step() against the real model --
  Phase 5 owns the actual training loop.

  ✔  Forward pass executed
  ✔  Loss computed, finite, no NaN/Inf  (final=27.3287)
  ✔  Backward executed, gradient norm computed  (inf)
  ✔  Optimizer step executed, or correctly skipped on detected overflow  (updated=False, skipped=True, scale 32768.0->16384.0)
  ✔  Scheduler advanced exactly once  (0 -> 1)
  ✔  Learning rate changed after scheduler.step()  (0.0001 -> 9.972636867364526e-05)
  ✔  AMP path exercised correctly  (enabled=True)
  ✔  Gradient clipping executed in real pipeline  (threshold=1.0)
  ✔  Checkpoint saved  (/kaggle/working/visionserveai/sprint04/checkpoints/phase4_engine_verification_checkpoint.pt)
  ✔  Checkpoint reloaded successfully  (1)
  ✔  Trainable head layers updated, or correctly held unchanged on a skipped step  (skipped=True)
  ✔  Frozen backbone lay

In [27]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 25: Artifact Persistence -- Phase 4 Master Summary
# ============================================================

print_section("STAGE 25 - ARTIFACT PERSISTENCE")
print()

phase4_artifacts = {
    "optimizer_configuration.json": optimizer_cfg.OPTIMIZER_DIR / "optimizer_configuration.json",
    "scheduler_configuration.json": scheduler_cfg.SCHEDULER_DIR / "scheduler_configuration.json",
    "gradient_configuration.json":  gradient_cfg.GRADIENT_DIR / "gradient_configuration.json",
    "amp_configuration.json":       amp_cfg.AMP_DIR / "amp_configuration.json",
    "checkpoint_summary.json":      cfg.OUTPUT_DIR / "checkpoints" / "checkpoint_summary.json",
    "training_engine_summary.json": cfg.METRICS_DIR / "training_engine_summary.json",
}

checks = [(name, path.exists() and path.stat().st_size > 0, str(path)) for name, path in phase4_artifacts.items()]
for label, passed, detail in checks:
    print_check(label, passed, detail)

all_passed = all(p for _, p, _ in checks)
print()
if not all_passed:
    raise AssertionError(f"Phase 4 artifact persistence FAILED: {[l for l,p,_ in checks if not p]}")
print("  ALL ARTIFACTS PRESENT")
print()

training_phase4_summary = {
    "project": cfg.PROJECT_NAME, "sprint": "04", "phase": "Phase 4 - Training Engine",
    "components": {
        "optimizer":     {"type": "AdamW", "learning_rate": optimizer_cfg.LEARNING_RATE, "weight_decay": optimizer_cfg.WEIGHT_DECAY},
        "scheduler":     {"type": "CosineAnnealingLR", "t_max": scheduler_cfg.T_MAX, "eta_min": scheduler_cfg.ETA_MIN},
        "amp":           {"enabled": amp_cfg.ENABLED, "device": str(DEVICE)},
        "gradients":     {"accumulation_steps": gradient_cfg.ACCUMULATION_STEPS, "clip_grad_norm": gradient_cfg.CLIP_GRAD_NORM},
        "checkpointing": {"checkpoint_dir": str(checkpoint_manager.checkpoint_dir)},
    },
    "training_engine_verification": {
        "trainable_layers_updated": training_engine_summary["trainable_layers_updated"],
        "frozen_layers_unchanged":  training_engine_summary["frozen_layers_unchanged"],
        "final_loss":               training_engine_summary["final_loss"],
        "wall_clock_ms":            training_engine_summary["wall_clock_ms"],
    },
    "artifacts": {name: str(path) for name, path in phase4_artifacts.items()},
    "all_checks_passed": all_passed,
}
phase4_summary_path = cfg.OUTPUT_DIR / "training_phase4_summary.json"
with open(phase4_summary_path, "w") as f:
    _json.dump(training_phase4_summary, f, indent=2, default=str)
print_check(f"Saved {phase4_summary_path.name}", phase4_summary_path.exists() and phase4_summary_path.stat().st_size > 0, str(phase4_summary_path))

print()
print("Stage 25 - Artifact Persistence : OK")
print()
print_section("PHASE 4 - TRAINING ENGINE : COMPLETE")
print()
print("  Frozen: OptimizerConfig, SchedulerConfig, AMPConfig,")
print("  GradientConfig, TrainingStep, CheckpointManager.")
print("  Ready for Phase 5 (epoch-level training loop).")

  STAGE 25 - ARTIFACT PERSISTENCE

  ✔  optimizer_configuration.json  (/kaggle/working/visionserveai/sprint04/optimizer/optimizer_configuration.json)
  ✔  scheduler_configuration.json  (/kaggle/working/visionserveai/sprint04/scheduler/scheduler_configuration.json)
  ✔  gradient_configuration.json  (/kaggle/working/visionserveai/sprint04/gradients/gradient_configuration.json)
  ✔  amp_configuration.json  (/kaggle/working/visionserveai/sprint04/amp/amp_configuration.json)
  ✔  checkpoint_summary.json  (/kaggle/working/visionserveai/sprint04/checkpoints/checkpoint_summary.json)
  ✔  training_engine_summary.json  (/kaggle/working/visionserveai/sprint04/metrics/training_engine_summary.json)

  ALL ARTIFACTS PRESENT

  ✔  Saved training_phase4_summary.json  (/kaggle/working/visionserveai/sprint04/training_phase4_summary.json)

Stage 25 - Artifact Persistence : OK

  PHASE 4 - TRAINING ENGINE : COMPLETE

  Frozen: OptimizerConfig, SchedulerConfig, AMPConfig,
  GradientConfig, TrainingStep, Ch

# ============================================================
# VisionServeAI | Sprint 04
# Stage 26: Epoch Engine
#
# Module: training/engine.py -> EpochResult, train_one_epoch()
# ============================================================

In [28]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 26: Epoch Engine
#
# Module: training/engine.py -> EpochResult, train_one_epoch()
# ============================================================

# -- Purpose ---------------------------------------------------
# Drive TrainingStep (Stage 21) across every batch in one epoch.
# This stage owns NO per-batch math of its own -- forward, loss,
# backward, AMP, clipping, optimizer.step(), and scheduler.step()
# all remain inside TrainingStep, exactly as Phase 4 froze them.
# train_one_epoch() is purely an iteration / accumulation /
# timing / logging layer on top of that primitive, per
# MASTER_ARCHITECTURE.md Section 18.1 (Epoch Loop -> Forward Pass
# -> Loss Compute -> Backward Pass -> Optimizer Step).

import time as _time
from dataclasses import dataclass, field

print_section("STAGE 26 - EPOCH ENGINE")
print()


@dataclass
class EpochResult:
    """
    Structured outcome of one full training epoch.

    Attributes
    ----------
    epoch : int
        1-indexed epoch number this result describes.
    average_loss : float
        Mean per-batch loss over the epoch (loss accumulated as
        ``loss * batch_size`` and divided by ``num_samples`` --
        sample-weighted, not batch-weighted, so a final partial
        batch does not skew the average).
    num_batches : int
        Number of optimizer-stepping batches processed.
    num_samples : int
        Total number of individual examples processed.
    epoch_time_seconds : float
        Wall-clock duration of the full epoch.
    average_batch_time_seconds : float
        Mean wall-clock time per batch (forward + backward + step).
    samples_per_second : float
        Throughput: ``num_samples / epoch_time_seconds``.
    final_lr : float
        Learning rate read from the optimizer after the last
        scheduler step in this epoch.
    grad_norms : List[float]
        Per-batch gradient norm (pre-clip-threshold value reported
        by ``clip_grad_norm_``), one entry per stepped batch.
    mean_grad_norm : float
        Mean of ``grad_norms``; 0.0 if no batches recorded a norm
        (e.g. ``clip_grad_norm=None``).
    loss_is_finite : bool
        True iff every per-batch loss value was finite (no NaN/Inf)
        -- the headline epoch-level sanity flag consumed by the
        Trainer (Stage 30) and by EarlyStopping (Stage 29).

    Migration note
    --------------
    Maps directly to ``training/engine.py -> EpochResult``.
    """
    epoch:                       int
    average_loss:                float
    num_batches:                 int
    num_samples:                 int
    epoch_time_seconds:          float
    average_batch_time_seconds:  float
    samples_per_second:          float
    final_lr:                    float
    grad_norms:                  List[float] = field(default_factory=list)
    mean_grad_norm:               float = 0.0
    loss_is_finite:               bool  = True

    def to_dict(self) -> Dict[str, object]:
        """JSON-serialisable representation (grad_norms rounded for readability)."""
        return {
            "epoch":                       self.epoch,
            "average_loss":                round(self.average_loss, 6),
            "num_batches":                 self.num_batches,
            "num_samples":                 self.num_samples,
            "epoch_time_seconds":          round(self.epoch_time_seconds, 4),
            "average_batch_time_seconds":  round(self.average_batch_time_seconds, 6),
            "samples_per_second":          round(self.samples_per_second, 2),
            "final_lr":                    self.final_lr,
            "mean_grad_norm":              round(self.mean_grad_norm, 6),
            "loss_is_finite":              self.loss_is_finite,
        }


def train_one_epoch(
    training_step: "TrainingStep",
    dataloader: torch.utils.data.DataLoader,
    epoch: int,
    log_interval: int = 50,
) -> EpochResult:
    """
    Run exactly one training epoch over *dataloader*.

    Pure with respect to its own scope: this function holds no model,
    optimizer, scheduler, or criterion state of its own -- all of that
    lives inside the supplied ``training_step`` (Stage 21). Calling
    this function mutates ``training_step.model`` /
    ``training_step.optimizer`` / ``training_step.scheduler`` exactly
    once per batch, identically to calling ``training_step(...)``
    directly in a loop -- this function adds accumulation, timing,
    and structured reporting on top.

    Responsibilities
    ----------------
    * Forward / loss / backward / optimizer step / scheduler step
      -- delegated entirely to ``training_step.__call__``.
    * Loss accumulation        -- sample-weighted running sum.
    * Gradient-norm collection -- one entry per stepped batch.
    * Batch timing             -- wall-clock per batch and per epoch.
    * Running averages         -- logged every ``log_interval`` batches.
    * Structured return        -- ``EpochResult``.

    Parameters
    ----------
    training_step : TrainingStep
        A constructed Stage 21 ``TrainingStep`` bound to the real
        ``model``, ``criterion``, ``optimizer``, ``scheduler``,
        ``scaler``. ``accumulation_steps`` on the instance is
        respected: a batch only "steps" the optimizer on its final
        accumulation micro-step, mirroring Stage 24 production usage.
    dataloader : torch.utils.data.DataLoader
        Yields ``(images, targets)`` pairs already on CPU; this
        function performs the ``.to(device, non_blocking=True)``
        transfer so callers never need a device-aware DataLoader.
    epoch : int
        1-indexed epoch number, stamped onto the returned
        ``EpochResult`` and used in log lines.
    log_interval : int
        Print a running-average log line every N stepped batches.
        Default 50.  Must be a positive integer.

    Returns
    -------
    EpochResult

    Raises
    ------
    ValueError
        If *dataloader* yields zero batches (empty epoch), or if
        ``log_interval`` is not a positive integer.
    """
    require_positive_int(log_interval, "log_interval")

    model = training_step.model
    device = training_step.device
    accumulation_steps = training_step.accumulation_steps

    model.train()

    running_loss_sum: float = 0.0
    running_sample_count: int = 0
    grad_norms: List[float] = []
    batch_times: List[float] = []
    num_optimizer_steps: int = 0
    loss_is_finite = True

    epoch_start = _time.time()

    micro_step_in_accum = 0
    num_batches_in_loader = len(dataloader)
    if num_batches_in_loader == 0:
        raise ValueError(
            "train_one_epoch() received an empty dataloader -- "
            "zero batches to train on."
        )

    for batch_idx, (images, targets) in enumerate(dataloader):
        batch_start = _time.time()

        images  = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        micro_step_in_accum += 1
        is_last_micro_step = (
            micro_step_in_accum == accumulation_steps
            or batch_idx == num_batches_in_loader - 1
        )

        result = training_step(images, targets, step_optimizer=is_last_micro_step)

        if is_last_micro_step:
            micro_step_in_accum = 0
            num_optimizer_steps += 1
            if result["grad_norm"] is not None:
                grad_norms.append(result["grad_norm"])

        batch_loss = result["loss"]
        batch_size = result["batch_size"]

        if batch_loss != batch_loss or abs(batch_loss) == float("inf"):
            loss_is_finite = False
            logger.warning(
                "Non-finite loss encountered at epoch %d batch %d: %s",
                epoch, batch_idx, batch_loss,
            )
        else:
            running_loss_sum += batch_loss * batch_size
        running_sample_count += batch_size

        if device.type == "cuda":
            torch.cuda.synchronize()
        batch_times.append(_time.time() - batch_start)

        if is_last_micro_step and (num_optimizer_steps % log_interval == 0):
            running_avg = (
                running_loss_sum / running_sample_count
                if running_sample_count > 0 else float("nan")
            )
            logger.info(
                "Epoch %d | step %d/%d | loss=%.4f | running_avg_loss=%.4f | lr=%.2e",
                epoch, num_optimizer_steps,
                (num_batches_in_loader + accumulation_steps - 1) // accumulation_steps,
                batch_loss, running_avg, result["lr"],
            )

    epoch_time_seconds = _time.time() - epoch_start
    average_loss = (
        running_loss_sum / running_sample_count
        if running_sample_count > 0 else float("nan")
    )
    average_batch_time = (
        sum(batch_times) / len(batch_times) if batch_times else 0.0
    )
    samples_per_second = (
        running_sample_count / epoch_time_seconds if epoch_time_seconds > 0 else 0.0
    )
    mean_grad_norm = sum(grad_norms) / len(grad_norms) if grad_norms else 0.0
    final_lr = training_step.optimizer.param_groups[0]["lr"]

    return EpochResult(
        epoch=epoch,
        average_loss=average_loss,
        num_batches=num_optimizer_steps,
        num_samples=running_sample_count,
        epoch_time_seconds=epoch_time_seconds,
        average_batch_time_seconds=average_batch_time,
        samples_per_second=samples_per_second,
        final_lr=final_lr,
        grad_norms=grad_norms,
        mean_grad_norm=mean_grad_norm,
        loss_is_finite=loss_is_finite,
    )


# -- Verification ----------------------------------------------------
# A tiny synthetic in-memory dataset/dataloader, structurally
# identical to a real one (yields (images, targets) CPU tensors of
# the correct shape/dtype), used ONLY to prove train_one_epoch()'s
# iteration/accumulation/timing logic end-to-end. This does NOT
# touch the frozen Sprint03 manifests -- that wiring is Stage 29.5.
# The real model/optimizer/scheduler/scaler ARE exercised here (this
# stage performs real weight updates), so a disposable clone of the
# Phase 4 components is used, leaving the Phase 4 singletons
# untouched for Stage 27 onward, exactly as Stage 19/20/21 already
# established the "disposable smoke-test object" pattern.

class _SyntheticEpochDataset(torch.utils.data.Dataset):
    """Synthetic structural stand-in for the Stage 29.5 manifest dataset."""

    def __init__(self, num_samples: int, image_size: Tuple[int, int], num_classes: int) -> None:
        self.num_samples = num_samples
        self.image_size = image_size
        self.num_classes = num_classes

    def __len__(self) -> int:
        return self.num_samples

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        torch.manual_seed(cfg.RANDOM_SEED + idx)
        image  = torch.randn(3, *self.image_size)
        target = torch.randint(0, 2, (self.num_classes,)).float()
        return image, target


_smoke_model     = build_model(cfg.BACKBONE, num_classes=cfg.NUM_CLASSES, pretrained=False, dropout=DROPOUT_PROB)
_smoke_model     = freeze_backbone(_smoke_model, backbone_name).to(DEVICE)
_smoke_optimizer = build_optimizer(_smoke_model, optimizer_cfg)
_smoke_scheduler = build_scheduler(_smoke_optimizer, scheduler_cfg)
_smoke_scaler    = build_grad_scaler(amp_cfg)

_smoke_training_step = TrainingStep(
    model=_smoke_model, criterion=criterion_weighted, optimizer=_smoke_optimizer,
    scheduler=_smoke_scheduler, scaler=_smoke_scaler, amp_enabled=amp_cfg.ENABLED,
    device=DEVICE, clip_grad_norm=gradient_cfg.CLIP_GRAD_NORM,
    accumulation_steps=gradient_cfg.ACCUMULATION_STEPS,
)

_smoke_dataset = _SyntheticEpochDataset(
    num_samples=cfg.BATCH_SIZE * 5, image_size=cfg.IMAGE_SIZE, num_classes=cfg.NUM_CLASSES,
)
_smoke_loader = torch.utils.data.DataLoader(
    _smoke_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=0,
)

_real_pre_last_epoch = scheduler.last_epoch   # captured for the "real Phase 4 singletons
                                               # stay untouched" check below
_head_before = [p.detach().clone() for p in _smoke_model.backbone.classifier.parameters() if p.requires_grad]
_pre_last_epoch = _smoke_scheduler.last_epoch
_pre_scale = _smoke_scaler.get_scale() if amp_cfg.ENABLED else None

epoch_result = train_one_epoch(_smoke_training_step, _smoke_loader, epoch=1, log_interval=2)

_post_scale = _smoke_scaler.get_scale() if amp_cfg.ENABLED else None
_any_step_skipped = amp_cfg.ENABLED and (_post_scale < _pre_scale)

_head_after = [p.detach().clone() for p in _smoke_model.backbone.classifier.parameters() if p.requires_grad]
_weights_updated = any(not torch.equal(a, b) for a, b in zip(_head_before, _head_after))

expected_optimizer_steps = (len(_smoke_loader) + gradient_cfg.ACCUMULATION_STEPS - 1) // gradient_cfg.ACCUMULATION_STEPS

checks = [
    ("EpochResult returned",                        isinstance(epoch_result, EpochResult), ""),
    ("epoch field matches call argument",            epoch_result.epoch == 1, str(epoch_result.epoch)),
    ("num_samples == len(dataset)",                  epoch_result.num_samples == len(_smoke_dataset), f"{epoch_result.num_samples} == {len(_smoke_dataset)}"),
    ("num_batches == expected optimizer steps",      epoch_result.num_batches == expected_optimizer_steps, f"{epoch_result.num_batches} == {expected_optimizer_steps}"),
    ("average_loss is finite",                       epoch_result.average_loss == epoch_result.average_loss and abs(epoch_result.average_loss) != float("inf"), f"{epoch_result.average_loss:.4f}"),
    ("loss_is_finite flag is True",                  epoch_result.loss_is_finite, ""),
    ("epoch_time_seconds > 0",                        epoch_result.epoch_time_seconds > 0, f"{epoch_result.epoch_time_seconds:.4f}s"),
    ("samples_per_second > 0",                         epoch_result.samples_per_second > 0, f"{epoch_result.samples_per_second:.2f}"),
    ("Optimizer updated weights, or all steps correctly skipped on detected overflow",
     _weights_updated or _any_step_skipped,
     f"updated={_weights_updated}, scale {_pre_scale}->{_post_scale}" if amp_cfg.ENABLED
     else f"updated={_weights_updated}"),("Scheduler advanced once per optimizer step",     _smoke_scheduler.last_epoch == _pre_last_epoch + expected_optimizer_steps, f"{_pre_last_epoch} -> {_smoke_scheduler.last_epoch}"),
    ("final_lr matches live optimizer LR",             epoch_result.final_lr == _smoke_optimizer.param_groups[0]["lr"], ""),
    ("Real Phase 4 model object untouched (different id)", _smoke_model is not model, ""),
    ("Real Phase 4 scheduler untouched",                scheduler.last_epoch == _real_pre_last_epoch, f"last_epoch={scheduler.last_epoch}"),
    ("EpochResult.to_dict() is JSON-serialisable",      bool(_json.dumps(epoch_result.to_dict())), ""),
]
for label, passed, detail in checks:
    print_check(label, passed, detail)

del _smoke_model, _smoke_optimizer, _smoke_scheduler, _smoke_scaler, _smoke_training_step
del _smoke_dataset, _smoke_loader, _head_before, _head_after

all_passed = all(p for _, p, _ in checks)
print()
if not all_passed:
    raise AssertionError(f"Epoch engine verification FAILED: {[l for l,p,_ in checks if not p]}")
print("  ALL CHECKS PASSED")
print()
print("Stage 26 - Epoch Engine : OK")


  STAGE 26 - EPOCH ENGINE

2026-07-02 00:56:10 | INFO     | visionserveai.sprint04 | Epoch 1 | step 2/3 | loss=22.8389 | running_avg_loss=24.2540 | lr=9.89e-05
  ✔  EpochResult returned
  ✔  epoch field matches call argument  (1)
  ✔  num_samples == len(dataset)  (160 == 160)
  ✔  num_batches == expected optimizer steps  (3 == 3)
  ✔  average_loss is finite  (24.7149)
  ✔  loss_is_finite flag is True
  ✔  epoch_time_seconds > 0  (0.5063s)
  ✔  samples_per_second > 0  (315.99)
  ✔  Optimizer updated weights, or all steps correctly skipped on detected overflow  (updated=False, scale 65536.0->8192.0)
  ✔  Scheduler advanced once per optimizer step  (0 -> 3)
  ✔  final_lr matches live optimizer LR
  ✔  Real Phase 4 model object untouched (different id)
  ✔  Real Phase 4 scheduler untouched  (last_epoch=1)
  ✔  EpochResult.to_dict() is JSON-serialisable

  ALL CHECKS PASSED

Stage 26 - Epoch Engine : OK


In [29]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 27: Validation Engine
#
# Module: training/validation.py -> ValidationResult, validate_one_epoch()
# ============================================================

# -- Purpose ---------------------------------------------------
# Run the model in pure inference mode over a validation/test
# DataLoader: no optimizer, no scheduler, no gradient computation,
# eval() mode (BatchNorm/Dropout frozen). Collects raw logits,
# sigmoid probabilities, and targets for every sample so Stage 28
# (Metrics Engine) can compute AUROC / precision / recall / F1 without
# this function knowing anything about sklearn. Per
# MASTER_ARCHITECTURE.md Section 18.1 (Validation Loop -> Metric
# Compute).

import time as _time

print_section("STAGE 27 - VALIDATION ENGINE")
print()


@dataclass
class ValidationResult:
    """
    Structured outcome of one full validation pass.

    Attributes
    ----------
    average_loss : float
        Sample-weighted mean validation loss.
    predictions : torch.Tensor
        Binary predictions, shape (N, num_classes), produced by
        thresholding ``probabilities`` at ``threshold`` (default 0.5
        in ``validate_one_epoch``; Stage 28 may re-threshold the raw
        ``probabilities`` tensor without re-running inference).
    probabilities : torch.Tensor
        Sigmoid-activated probabilities, shape (N, num_classes),
        dtype float32, values in [0, 1].
    targets : torch.Tensor
        Ground-truth multilabel targets, shape (N, num_classes),
        dtype float32, values in {0, 1}.
    num_samples : int
        Total number of validation examples processed.
    num_batches : int
        Number of batches processed.
    validation_time_seconds : float
        Wall-clock duration of the full validation pass.
    loss_is_finite : bool
        True iff every per-batch validation loss was finite.

    Notes
    -----
    All three tensors (``predictions``, ``probabilities``,
    ``targets``) are returned on CPU regardless of the training
    device -- sklearn (Stage 28) requires NumPy-convertible CPU
    tensors, and keeping large concatenated validation-set tensors
    on GPU for the duration of metrics computation is wasteful VRAM
    that production inference will need.

    Migration note
    --------------
    Maps directly to ``training/validation.py -> ValidationResult``.
    """
    average_loss:             float
    predictions:              torch.Tensor
    probabilities:            torch.Tensor
    targets:                  torch.Tensor
    num_samples:               int
    num_batches:                int
    validation_time_seconds:    float
    loss_is_finite:              bool = True

    def to_summary_dict(self) -> Dict[str, object]:
        """JSON-serialisable summary (tensors excluded -- shape/dtype only)."""
        return {
            "average_loss":             round(self.average_loss, 6),
            "num_samples":              self.num_samples,
            "num_batches":              self.num_batches,
            "validation_time_seconds":  round(self.validation_time_seconds, 4),
            "loss_is_finite":           self.loss_is_finite,
            "predictions_shape":        list(self.predictions.shape),
            "probabilities_shape":      list(self.probabilities.shape),
            "targets_shape":            list(self.targets.shape),
        }


@torch.no_grad()
def validate_one_epoch(
    model: nn.Module,
    criterion: nn.Module,
    dataloader: torch.utils.data.DataLoader,
    device: torch.device,
    amp_enabled: bool = False,
    threshold: float = 0.5,
) -> ValidationResult:
    """
    Run one full no-gradient validation pass over *dataloader*.

    Guarantees
    ----------
    * ``model.eval()`` is set on entry and the model's PRIOR mode
      (train/eval) is restored on exit -- so calling this mid-training
      never leaves the model accidentally stuck in eval mode for the
      next training epoch.
    * Wrapped in ``@torch.no_grad()`` -- no autograd graph is ever
      built; this is stronger than merely "not calling backward()".
    * No optimizer, no scheduler, no ``.step()`` of any kind, no
      ``zero_grad()`` -- this function does not import or touch
      ``TrainingStep`` at all.

    Parameters
    ----------
    model       : nn.Module                 The model under evaluation.
    criterion   : nn.Module                  Loss function (e.g. ``criterion_weighted``).
    dataloader  : torch.utils.data.DataLoader  Validation or test loader.
    device      : torch.device                Device to run inference on.
    amp_enabled : bool                        Use autocast for the forward pass.
                                              Default False -- validation is not
                                              throughput-critical the way training
                                              is, and full precision gives the
                                              most trustworthy metrics; pass True
                                              to match training-time numerics if
                                              required.
    threshold   : float                       Sigmoid-probability cutoff for
                                              binarising predictions.  Default 0.5.
                                              Must be in (0, 1).

    Returns
    -------
    ValidationResult

    Raises
    ------
    ValueError
        If *dataloader* yields zero batches, or ``threshold`` is not
        in the open interval (0, 1).
    """
    require_in_range(threshold, 0.0, 1.0, "threshold", inclusive=False)

    num_batches_in_loader = len(dataloader)
    if num_batches_in_loader == 0:
        raise ValueError(
            "validate_one_epoch() received an empty dataloader -- "
            "zero batches to validate on."
        )

    was_training = model.training
    model.eval()

    running_loss_sum: float = 0.0
    running_sample_count: int = 0
    loss_is_finite = True

    all_probabilities: List[torch.Tensor] = []
    all_targets: List[torch.Tensor] = []

    validation_start = _time.time()

    for batch_idx, (images, targets) in enumerate(dataloader):
        images  = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with torch.autocast(device_type=device.type, enabled=amp_enabled):
            logits = model(images)
            loss = criterion(logits, targets)

        loss_value = loss.item()
        batch_size = images.shape[0]

        if loss_value != loss_value or abs(loss_value) == float("inf"):
            loss_is_finite = False
            logger.warning(
                "Non-finite validation loss at batch %d: %s", batch_idx, loss_value,
            )
        else:
            running_loss_sum += loss_value * batch_size
        running_sample_count += batch_size

        probabilities = torch.sigmoid(logits.float())
        all_probabilities.append(probabilities.detach().cpu())
        all_targets.append(targets.detach().cpu())

    validation_time_seconds = _time.time() - validation_start

    probabilities_full = torch.cat(all_probabilities, dim=0)
    targets_full        = torch.cat(all_targets, dim=0)
    predictions_full     = (probabilities_full >= threshold).float()

    average_loss = (
        running_loss_sum / running_sample_count
        if running_sample_count > 0 else float("nan")
    )

    if was_training:
        model.train()

    return ValidationResult(
        average_loss=average_loss,
        predictions=predictions_full,
        probabilities=probabilities_full,
        targets=targets_full,
        num_samples=running_sample_count,
        num_batches=num_batches_in_loader,
        validation_time_seconds=validation_time_seconds,
        loss_is_finite=loss_is_finite,
    )


# -- Verification ----------------------------------------------------
# Reuses the Stage 26 synthetic-dataset pattern. A disposable model
# clone is used (never the real `model`) so this stage cannot
# accidentally mutate Phase 2/4 state -- but since validate_one_epoch
# performs NO gradient updates by construction, the disposable clone
# here exists purely to test the train<->eval mode restoration
# contract without risking the real model's BatchNorm/Dropout state.

_val_model = build_model(cfg.BACKBONE, num_classes=cfg.NUM_CLASSES, pretrained=False, dropout=DROPOUT_PROB)
_val_model = freeze_backbone(_val_model, backbone_name).to(DEVICE)

_val_dataset = _SyntheticEpochDataset(
    num_samples=cfg.BATCH_SIZE * 4, image_size=cfg.IMAGE_SIZE, num_classes=cfg.NUM_CLASSES,
)
_val_loader = torch.utils.data.DataLoader(
    _val_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=0,
)

# -- (a) Mode-restoration contract: started in train(), must end in train() --
_val_model.train()
_mode_before = _val_model.training
_ = validate_one_epoch(_val_model, criterion_weighted, _val_loader, DEVICE, amp_enabled=amp_cfg.ENABLED)
_mode_after_from_train = _val_model.training

# -- (b) Mode-restoration contract: started in eval(), must end in eval() --
_val_model.eval()
val_result = validate_one_epoch(_val_model, criterion_weighted, _val_loader, DEVICE, amp_enabled=amp_cfg.ENABLED)
_mode_after_from_eval = _val_model.training

# -- (c) No-gradient guarantee: parameters must be byte-identical pre/post --
_params_before = [p.detach().clone() for p in _val_model.parameters()]
_ = validate_one_epoch(_val_model, criterion_weighted, _val_loader, DEVICE, amp_enabled=amp_cfg.ENABLED)
_params_after = [p.detach().clone() for p in _val_model.parameters()]
_no_weight_change = all(torch.equal(a, b) for a, b in zip(_params_before, _params_after))

# -- (d) requires_grad is never set False/True by this function (untouched) --
_grad_flags_before = [p.requires_grad for p in _val_model.parameters()]
_ = validate_one_epoch(_val_model, criterion_weighted, _val_loader, DEVICE)
_grad_flags_after = [p.requires_grad for p in _val_model.parameters()]
_grad_flags_unchanged = _grad_flags_before == _grad_flags_after

expected_n = len(_val_dataset)

checks = [
    ("ValidationResult returned",                    isinstance(val_result, ValidationResult), ""),
    ("num_samples == len(dataset)",                  val_result.num_samples == expected_n, f"{val_result.num_samples} == {expected_n}"),
    ("num_batches == len(loader)",                    val_result.num_batches == len(_val_loader), f"{val_result.num_batches} == {len(_val_loader)}"),
    ("average_loss is finite",                        val_result.average_loss == val_result.average_loss and abs(val_result.average_loss) != float("inf"), f"{val_result.average_loss:.4f}"),
    ("loss_is_finite flag is True",                    val_result.loss_is_finite, ""),
    ("predictions shape correct",                      tuple(val_result.predictions.shape) == (expected_n, cfg.NUM_CLASSES), str(tuple(val_result.predictions.shape))),
    ("probabilities shape correct",                    tuple(val_result.probabilities.shape) == (expected_n, cfg.NUM_CLASSES), str(tuple(val_result.probabilities.shape))),
    ("targets shape correct",                          tuple(val_result.targets.shape) == (expected_n, cfg.NUM_CLASSES), str(tuple(val_result.targets.shape))),
    ("probabilities bounded in [0, 1]",                bool((val_result.probabilities >= 0).all() and (val_result.probabilities <= 1).all()), f"min={val_result.probabilities.min().item():.4f}, max={val_result.probabilities.max().item():.4f}"),
    ("predictions are binary {0, 1}",                  bool(((val_result.predictions == 0) | (val_result.predictions == 1)).all()), ""),
    ("targets are binary {0, 1}",                      bool(((val_result.targets == 0) | (val_result.targets == 1)).all()), ""),
    ("Output tensors are on CPU",                      val_result.predictions.device.type == "cpu" and val_result.probabilities.device.type == "cpu" and val_result.targets.device.type == "cpu", ""),
    ("Model left in train() if it started in train()", _mode_after_from_train is True, str(_mode_after_from_train)),
    ("Model left in eval() if it started in eval()",   _mode_after_from_eval is False, str(_mode_after_from_eval)),
    ("No weight mutation across a validation pass",     _no_weight_change, ""),
    ("requires_grad flags never touched",                _grad_flags_unchanged, ""),
    ("Threshold rejects out-of-range value",             True, "see exception check below"),
]
for label, passed, detail in checks:
    print_check(label, passed, detail)

threshold_raised = False
try:
    validate_one_epoch(_val_model, criterion_weighted, _val_loader, DEVICE, threshold=1.5)
except ValueError:
    threshold_raised = True
print_check("validate_one_epoch() rejects threshold outside (0, 1)", threshold_raised)

empty_loader_raised = False
try:
    _empty_dataset = _SyntheticEpochDataset(num_samples=0, image_size=cfg.IMAGE_SIZE, num_classes=cfg.NUM_CLASSES)
    _empty_loader = torch.utils.data.DataLoader(_empty_dataset, batch_size=cfg.BATCH_SIZE)
    validate_one_epoch(_val_model, criterion_weighted, _empty_loader, DEVICE)
except ValueError:
    empty_loader_raised = True
print_check("validate_one_epoch() rejects an empty dataloader", empty_loader_raised)

all_checks_2 = checks + [
    ("validate_one_epoch() rejects threshold outside (0, 1)", threshold_raised, ""),
    ("validate_one_epoch() rejects an empty dataloader",       empty_loader_raised, ""),
]

del _val_model, _val_dataset, _val_loader, _params_before, _params_after
del _grad_flags_before, _grad_flags_after

all_passed = all(p for _, p, _ in all_checks_2)
print()
if not all_passed:
    raise AssertionError(f"Validation engine verification FAILED: {[l for l,p,_ in all_checks_2 if not p]}")
print("  ALL CHECKS PASSED")
print()
print("Stage 27 - Validation Engine : OK")


  STAGE 27 - VALIDATION ENGINE

  ✔  ValidationResult returned
  ✔  num_samples == len(dataset)  (128 == 128)
  ✔  num_batches == len(loader)  (4 == 4)
  ✔  average_loss is finite  (38.5232)
  ✔  loss_is_finite flag is True
  ✔  predictions shape correct  ((128, 14))
  ✔  probabilities shape correct  ((128, 14))
  ✔  targets shape correct  ((128, 14))
  ✔  probabilities bounded in [0, 1]  (min=0.0730, max=0.8093)
  ✔  predictions are binary {0, 1}
  ✔  targets are binary {0, 1}
  ✔  Output tensors are on CPU
  ✔  Model left in train() if it started in train()  (True)
  ✔  Model left in eval() if it started in eval()  (False)
  ✔  No weight mutation across a validation pass
  ✔  requires_grad flags never touched
  ✔  Threshold rejects out-of-range value  (see exception check below)
  ✔  validate_one_epoch() rejects threshold outside (0, 1)
  ✔  validate_one_epoch() rejects an empty dataloader

  ALL CHECKS PASSED

Stage 27 - Validation Engine : OK


In [30]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 28: Metrics Engine
#
# Module: training/metrics.py -> MetricsResult, compute_metrics()
# ============================================================

# -- Purpose ---------------------------------------------------
# Convert the raw probabilities/predictions/targets produced by
# Stage 27 (Validation Engine) into the production metric suite:
# macro/micro/per-class AUROC, precision, recall, F1, accuracy.
# This stage owns zero forward-pass / inference logic of its own --
# it is a pure NumPy/sklearn transform over already-computed tensors.
# Per MASTER_ARCHITECTURE.md Section 18.1 (Metric Compute -> AUC, F1)
# and Section 8 (Success Metrics).

from sklearn.metrics import (
    roc_auc_score,
    precision_recall_fscore_support,
    accuracy_score,
)

print_section("STAGE 28 - METRICS ENGINE")
print()


@dataclass
class MetricsResult:
    """
    Production multilabel classification metric suite for one
    validation/test pass.

    Attributes
    ----------
    macro_auroc : float
        Unweighted mean of per-class AUROC, over classes for which
        AUROC is defined (see ``undefined_auroc_classes``).
    micro_auroc : float
        AUROC computed over the flattened (sample, class) prediction
        stream -- weights every individual label decision equally
        rather than every class equally, so common classes
        (Infiltration, ~11k positives) dominate more than rare ones
        (Hernia, ~115 positives). Reported alongside macro_auroc
        rather than instead of it, since the two answer different
        questions about an imbalanced multilabel problem.
    per_class_auroc : Dict[str, float]
        Class name -> AUROC.  ``float("nan")`` for any class with
        only one label value present in this pass (AUROC undefined;
        see ``undefined_auroc_classes``).
    undefined_auroc_classes : List[str]
        Class names excluded from ``macro_auroc`` because the
        validation pass contained only positive or only negative
        examples for that class -- expected on small engineering
        verification slices (Stage 30) for the rarest classes
        (Hernia: ~0.1% prevalence) and should shrink to empty on a
        full validation set.
    precision_macro, recall_macro, f1_macro : float
        Unweighted mean of per-class precision/recall/F1 at the
        binarisation ``threshold`` used to produce ``predictions``.
    precision_per_class, recall_per_class, f1_per_class : Dict[str, float]
        Class name -> metric value.
    accuracy_exact_match : float
        Fraction of samples where ALL 14 predicted labels exactly
        match all 14 ground-truth labels -- the strict multilabel
        "subset accuracy". Expected to be low / near-zero on a
        14-way multilabel problem even for a well-performing model;
        reported for completeness, not as a primary metric (macro
        AUROC is the primary metric per Section 8).
    accuracy_per_class : Dict[str, float]
        Per-class binary accuracy ((TP+TN)/N) -- more informative
        than exact-match for an imbalanced multilabel setting.
    average_loss : float
        Carried through unchanged from the ``ValidationResult`` that
        produced this metrics computation, so a single artifact
        captures both the loss and the metric suite for one epoch.
    num_samples, threshold : as supplied.

    Migration note
    --------------
    Maps directly to ``training/metrics.py -> MetricsResult``.
    """
    macro_auroc:               float
    micro_auroc:                float
    per_class_auroc:             Dict[str, float]
    undefined_auroc_classes:      List[str]
    precision_macro:               float
    recall_macro:                   float
    f1_macro:                        float
    precision_per_class:              Dict[str, float]
    recall_per_class:                  Dict[str, float]
    f1_per_class:                        Dict[str, float]
    accuracy_exact_match:                 float
    accuracy_per_class:                    Dict[str, float]
    average_loss:                           float
    num_samples:                              int
    threshold:                                 float

    def to_dict(self) -> Dict[str, object]:
        """JSON-serialisable representation, all floats rounded to 6dp."""
        def _round_map(d: Dict[str, float]) -> Dict[str, float]:
            return {k: (round(v, 6) if v == v else None) for k, v in d.items()}

        return {
            "macro_auroc":               round(self.macro_auroc, 6) if self.macro_auroc == self.macro_auroc else None,
            "micro_auroc":                round(self.micro_auroc, 6) if self.micro_auroc == self.micro_auroc else None,
            "per_class_auroc":            _round_map(self.per_class_auroc),
            "undefined_auroc_classes":    self.undefined_auroc_classes,
            "precision_macro":            round(self.precision_macro, 6),
            "recall_macro":                round(self.recall_macro, 6),
            "f1_macro":                     round(self.f1_macro, 6),
            "precision_per_class":          _round_map(self.precision_per_class),
            "recall_per_class":              _round_map(self.recall_per_class),
            "f1_per_class":                   _round_map(self.f1_per_class),
            "accuracy_exact_match":            round(self.accuracy_exact_match, 6),
            "accuracy_per_class":                _round_map(self.accuracy_per_class),
            "average_loss":                       round(self.average_loss, 6),
            "num_samples":                          self.num_samples,
            "threshold":                              self.threshold,
        }


def compute_metrics(
    validation_result: ValidationResult,
    class_names: List[str],
    threshold: Optional[float] = None,
) -> MetricsResult:
    """
    Compute the full production metric suite from a ``ValidationResult``.

    Parameters
    ----------
    validation_result : ValidationResult
        Output of Stage 27 ``validate_one_epoch()``.
    class_names : List[str]
        Ordered class names (the frozen Sprint 03 disease_registry
        ordering); length must equal
        ``validation_result.targets.shape[1]``.
    threshold : Optional[float]
        Re-binarisation threshold. ``None`` (default) reuses
        ``validation_result.predictions`` as-is (the threshold
        already applied in Stage 27). Supplying a float re-thresholds
        ``validation_result.probabilities`` without re-running
        inference -- useful for threshold sweeps that Stage 30 does
        not perform but a future sprint's calibration stage will.

    Returns
    -------
    MetricsResult

    Raises
    ------
    ValueError
        If ``class_names`` length does not match the target tensor's
        class dimension, or if ``threshold`` (when supplied) is not
        in the open interval (0, 1).
    """
    num_classes = validation_result.targets.shape[1]
    if len(class_names) != num_classes:
        raise ValueError(
            f"class_names has {len(class_names)} entries but targets "
            f"tensor has {num_classes} classes."
        )

    if threshold is None:
        predictions_np = validation_result.predictions.numpy()
        resolved_threshold = 0.5  # Stage 27's hardcoded default; recorded for
                                   # the artifact even when not re-applied here.
    else:
        require_in_range(threshold, 0.0, 1.0, "threshold", inclusive=False)
        predictions_np = (validation_result.probabilities.numpy() >= threshold).astype("float32")
        resolved_threshold = threshold

    probabilities_np = validation_result.probabilities.numpy()
    targets_np         = validation_result.targets.numpy()

    # -- Per-class AUROC --------------------------------------------
    per_class_auroc: Dict[str, float] = {}
    undefined_auroc_classes: List[str] = []
    for i, name in enumerate(class_names):
        col_targets = targets_np[:, i]
        if len(np.unique(col_targets)) < 2:
            per_class_auroc[name] = float("nan")
            undefined_auroc_classes.append(name)
            continue
        per_class_auroc[name] = float(
            roc_auc_score(col_targets, probabilities_np[:, i])
        )

    defined_aurocs = [v for v in per_class_auroc.values() if v == v]
    macro_auroc = float(np.mean(defined_aurocs)) if defined_aurocs else float("nan")

    # -- Micro AUROC (flattened) -------------------------------------
    flat_targets       = targets_np.ravel()
    flat_probabilities = probabilities_np.ravel()
    if len(np.unique(flat_targets)) < 2:
        micro_auroc = float("nan")
        logger.warning("Micro AUROC undefined -- only one class present across all flattened labels.")
    else:
        micro_auroc = float(roc_auc_score(flat_targets, flat_probabilities))

    # -- Precision / Recall / F1 (per-class + macro) ------------------
    precision_arr, recall_arr, f1_arr, _support = precision_recall_fscore_support(
        targets_np, predictions_np, average=None, zero_division=0,
    )
    precision_per_class = {name: float(v) for name, v in zip(class_names, precision_arr)}
    recall_per_class     = {name: float(v) for name, v in zip(class_names, recall_arr)}
    f1_per_class           = {name: float(v) for name, v in zip(class_names, f1_arr)}

    precision_macro = float(np.mean(precision_arr))
    recall_macro     = float(np.mean(recall_arr))
    f1_macro           = float(np.mean(f1_arr))

    # -- Accuracy -------------------------------------------------------
    accuracy_exact_match = float(accuracy_score(targets_np, predictions_np))
    accuracy_per_class = {
        name: float((predictions_np[:, i] == targets_np[:, i]).mean())
        for i, name in enumerate(class_names)
    }

    return MetricsResult(
        macro_auroc=macro_auroc,
        micro_auroc=micro_auroc,
        per_class_auroc=per_class_auroc,
        undefined_auroc_classes=undefined_auroc_classes,
        precision_macro=precision_macro,
        recall_macro=recall_macro,
        f1_macro=f1_macro,
        precision_per_class=precision_per_class,
        recall_per_class=recall_per_class,
        f1_per_class=f1_per_class,
        accuracy_exact_match=accuracy_exact_match,
        accuracy_per_class=accuracy_per_class,
        average_loss=validation_result.average_loss,
        num_samples=validation_result.num_samples,
        threshold=resolved_threshold,
    )


# -- Verification ----------------------------------------------------
# Two synthetic ValidationResult fixtures:
#   (a) a "good model" fixture -- probabilities strongly correlated
#       with targets, to prove AUROC/F1 land near their ceiling
#       rather than merely "some float came back".
#   (b) a "random model" fixture -- uncorrelated probabilities, to
#       prove AUROC lands near the 0.5 floor (and isn't silently
#       always 1.0 from a computation bug).
# Both also include one rare class set to ALL-NEGATIVE, deliberately
# reproducing the Stage 30 small-sample-size edge case so the
# undefined_auroc_classes path is exercised here, not discovered
# for the first time during the real Trainer run.

torch.manual_seed(cfg.RANDOM_SEED)
_n_samples = 200

_synthetic_targets = torch.randint(0, 2, (_n_samples, cfg.NUM_CLASSES)).float()
_synthetic_targets[:, -1] = 0.0   # force last class (Pneumothorax) all-negative

# (a) Good model: probabilities pushed toward the true label.
_good_probs = torch.clamp(
    _synthetic_targets * 0.85 + (1 - _synthetic_targets) * 0.15
    + torch.randn(_n_samples, cfg.NUM_CLASSES) * 0.05,
    0.0, 1.0,
)
_good_preds = (_good_probs >= 0.5).float()
_good_val_result = ValidationResult(
    average_loss=0.1234, predictions=_good_preds, probabilities=_good_probs,
    targets=_synthetic_targets, num_samples=_n_samples, num_batches=1,
    validation_time_seconds=0.01, loss_is_finite=True,
)

# (b) Random model: probabilities uncorrelated with targets.
torch.manual_seed(cfg.RANDOM_SEED + 1)
_random_probs = torch.rand(_n_samples, cfg.NUM_CLASSES)
_random_preds = (_random_probs >= 0.5).float()
_random_val_result = ValidationResult(
    average_loss=0.6931, predictions=_random_preds, probabilities=_random_probs,
    targets=_synthetic_targets, num_samples=_n_samples, num_batches=1,
    validation_time_seconds=0.01, loss_is_finite=True,
)

good_metrics   = compute_metrics(_good_val_result, class_names)
random_metrics = compute_metrics(_random_val_result, class_names)

checks = [
    ("MetricsResult returned",                            isinstance(good_metrics, MetricsResult), ""),
    ("Good-model macro AUROC near ceiling (> 0.9)",        good_metrics.macro_auroc > 0.9, f"{good_metrics.macro_auroc:.4f}"),
    ("Random-model macro AUROC near floor (0.35-0.65)",    0.35 <= random_metrics.macro_auroc <= 0.65, f"{random_metrics.macro_auroc:.4f}"),
    ("Good model clearly outperforms random model",         good_metrics.macro_auroc > random_metrics.macro_auroc, f"{good_metrics.macro_auroc:.4f} > {random_metrics.macro_auroc:.4f}"),
    ("Forced all-negative class flagged as undefined",       "Pneumothorax" in good_metrics.undefined_auroc_classes, str(good_metrics.undefined_auroc_classes)),
    ("Undefined class excluded from per_class_auroc=nan",    good_metrics.per_class_auroc["Pneumothorax"] != good_metrics.per_class_auroc["Pneumothorax"], ""),
    ("macro_auroc itself is finite despite one undefined",   good_metrics.macro_auroc == good_metrics.macro_auroc, f"{good_metrics.macro_auroc:.4f}"),
    ("per_class_auroc has all 14 classes",                    len(good_metrics.per_class_auroc) == cfg.NUM_CLASSES, str(len(good_metrics.per_class_auroc))),
    ("micro_auroc is finite",                                  good_metrics.micro_auroc == good_metrics.micro_auroc, f"{good_metrics.micro_auroc:.4f}"),
    ("precision_macro in [0, 1]",                               0.0 <= good_metrics.precision_macro <= 1.0, f"{good_metrics.precision_macro:.4f}"),
    ("recall_macro in [0, 1]",                                   0.0 <= good_metrics.recall_macro <= 1.0, f"{good_metrics.recall_macro:.4f}"),
    ("f1_macro in [0, 1]",                                         0.0 <= good_metrics.f1_macro <= 1.0, f"{good_metrics.f1_macro:.4f}"),
    ("Good model F1 > random model F1",                            good_metrics.f1_macro > random_metrics.f1_macro, f"{good_metrics.f1_macro:.4f} > {random_metrics.f1_macro:.4f}"),
    ("accuracy_exact_match in [0, 1]",                              0.0 <= good_metrics.accuracy_exact_match <= 1.0, f"{good_metrics.accuracy_exact_match:.4f}"),
    ("accuracy_per_class has all 14 classes",                       len(good_metrics.accuracy_per_class) == cfg.NUM_CLASSES, ""),
    ("precision_per_class/recall_per_class/f1_per_class complete",  all(len(d) == cfg.NUM_CLASSES for d in [good_metrics.precision_per_class, good_metrics.recall_per_class, good_metrics.f1_per_class]), ""),
    ("average_loss carried through from ValidationResult",          good_metrics.average_loss == 0.1234, str(good_metrics.average_loss)),
    ("num_samples carried through from ValidationResult",            good_metrics.num_samples == _n_samples, str(good_metrics.num_samples)),
    ("MetricsResult.to_dict() is JSON-serialisable",                  bool(_json.dumps(good_metrics.to_dict())), ""),
]
for label, passed, detail in checks:
    print_check(label, passed, detail)

class_count_mismatch_raised = False
try:
    compute_metrics(_good_val_result, class_names[:-1])
except ValueError:
    class_count_mismatch_raised = True
print_check("compute_metrics() rejects class_names/target shape mismatch", class_count_mismatch_raised)

bad_threshold_raised = False
try:
    compute_metrics(_good_val_result, class_names, threshold=1.5)
except ValueError:
    bad_threshold_raised = True
print_check("compute_metrics() rejects threshold outside (0, 1)", bad_threshold_raised)

all_checks_3 = checks + [
    ("compute_metrics() rejects class_names/target shape mismatch", class_count_mismatch_raised, ""),
    ("compute_metrics() rejects threshold outside (0, 1)",            bad_threshold_raised, ""),
]

all_passed = all(p for _, p, _ in all_checks_3)
print()
if not all_passed:
    raise AssertionError(f"Metrics engine verification FAILED: {[l for l,p,_ in all_checks_3 if not p]}")
print("  ALL CHECKS PASSED")
print()

# -- Persist artifact: metrics_summary.json --------------------------
metrics_summary_artifact = {
    "project": cfg.PROJECT_NAME, "sprint": "04", "phase": "Phase 5 - Training Loop",
    "engineering_verification": {
        "good_model_fixture":   good_metrics.to_dict(),
        "random_model_fixture": random_metrics.to_dict(),
    },
    "class_names": class_names,
}
metrics_summary_path = cfg.METRICS_DIR / "metrics_summary.json"
with open(metrics_summary_path, "w") as f:
    _json.dump(metrics_summary_artifact, f, indent=2, default=str)
print_check(f"Saved {metrics_summary_path.name}", metrics_summary_path.exists() and metrics_summary_path.stat().st_size > 0, str(metrics_summary_path))

print()
print("Stage 28 - Metrics Engine : OK")


2026-07-02 00:56:12 | INFO     | numexpr.utils | NumExpr defaulting to 4 threads.
  STAGE 28 - METRICS ENGINE

  ✔  MetricsResult returned
  ✔  Good-model macro AUROC near ceiling (> 0.9)  (1.0000)
  ✔  Random-model macro AUROC near floor (0.35-0.65)  (0.5066)
  ✔  Good model clearly outperforms random model  (1.0000 > 0.5066)
  ✔  Forced all-negative class flagged as undefined  (['Pneumothorax'])
  ✔  Undefined class excluded from per_class_auroc=nan
  ✔  macro_auroc itself is finite despite one undefined  (1.0000)
  ✔  per_class_auroc has all 14 classes  (14)
  ✔  micro_auroc is finite  (1.0000)
  ✔  precision_macro in [0, 1]  (0.9286)
  ✔  recall_macro in [0, 1]  (0.9286)
  ✔  f1_macro in [0, 1]  (0.9286)
  ✔  Good model F1 > random model F1  (0.9286 > 0.4697)
  ✔  accuracy_exact_match in [0, 1]  (1.0000)
  ✔  accuracy_per_class has all 14 classes
  ✔  precision_per_class/recall_per_class/f1_per_class complete
  ✔  average_loss carried through from ValidationResult  (0.1234)
  ✔  nu

In [31]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 29: Early Stopping
#
# Module: training/early_stopping.py -> EarlyStopping
# ============================================================

# -- Purpose ---------------------------------------------------
# Reusable, framework-agnostic early-stopping tracker. Watches a
# single scalar metric (validation loss by convention, lower-is-
# better) across epochs and signals when the Trainer (Stage 30)
# should halt. Owns no model/optimizer state itself -- "save_best"
# is a boolean signal consumed by the Trainer, which already has
# CheckpointManager (Stage 23) for the actual save call. Per
# MASTER_ARCHITECTURE.md Section 18.1 / Section 6.2 (FR-T4: early
# stopping, LR scheduling, gradient clipping).

print_section("STAGE 29 - EARLY STOPPING")
print()


class EarlyStopping:
    """
    Track a monitored metric across epochs and signal when to stop.

    Convention: LOWER is better (designed around validation loss).
    To monitor a higher-is-better metric (e.g. macro AUROC), negate
    it at the call site: ``early_stopping.step(-val_macro_auroc)``.

    Parameters
    ----------
    patience : int
        Number of consecutive non-improving epochs to tolerate
        before ``should_stop`` becomes True. Must be a positive
        integer.
    min_delta : float
        Minimum decrease required to count as an improvement.
        A new value must satisfy ``value < best_loss - min_delta``.
        Must be >= 0.

    Attributes
    ----------
    best_loss : float
        Best (lowest) monitored value seen so far.
        ``float("inf")`` before the first ``step()`` call.
    counter : int
        Consecutive non-improving epochs since the last improvement.
    should_stop : bool
        True once ``counter >= patience``.
    save_best : bool
        True only on the call to ``step()`` that recorded a new best
        value -- the Trainer's single source of truth for "save a
        checkpoint this epoch", consumed alongside
        ``CheckpointManager.save()`` (Stage 23).

    Migration note
    --------------
    Maps directly to ``training/early_stopping.py -> EarlyStopping``.
    """

    def __init__(self, patience: int = 7, min_delta: float = 0.0) -> None:
        require_positive_int(patience, "patience")
        if min_delta < 0:
            raise ValueError(f"min_delta must be >= 0; got {min_delta}.")

        self.patience  = patience
        self.min_delta = min_delta

        self.best_loss:   float = float("inf")
        self.counter:      int  = 0
        self.should_stop:   bool = False
        self.save_best:       bool = False
        self.best_epoch:       Optional[int] = None

    def step(self, value: float, epoch: Optional[int] = None) -> bool:
        """
        Register one epoch's monitored value.

        Parameters
        ----------
        value : float
            The monitored metric for this epoch (lower is better).
        epoch : Optional[int]
            Epoch number, recorded on improvement for reporting.
            Purely informational -- not used in any stopping logic.

        Returns
        -------
        bool
            The updated ``should_stop`` flag (also available as
            ``self.should_stop`` afterward).

        Raises
        ------
        ValueError
            If ``value`` is NaN -- a NaN monitored value is a
            training-pipeline bug (see ``EpochResult.loss_is_finite``
            / ``ValidationResult.loss_is_finite``), not a legitimate
            "no improvement" epoch, and silently absorbing it here
            would let early stopping fire on garbage.
        """
        if value != value:
            raise ValueError(
                "EarlyStopping.step() received NaN -- check "
                "loss_is_finite on the upstream EpochResult / "
                "ValidationResult before calling step()."
            )

        if value < self.best_loss - self.min_delta:
            self.best_loss  = value
            self.counter     = 0
            self.save_best     = True
            self.best_epoch      = epoch
        else:
            self.counter += 1
            self.save_best = False

        self.should_stop = self.counter >= self.patience
        return self.should_stop

    def reset(self) -> None:
        """Reset all mutable tracking state to its initial (construction-time) values."""
        self.best_loss   = float("inf")
        self.counter      = 0
        self.should_stop   = False
        self.save_best        = False
        self.best_epoch         = None

    def state_dict(self) -> Dict[str, object]:
        """Serialise tracking state for checkpoint persistence / resume."""
        return {
            "patience":     self.patience,
            "min_delta":    self.min_delta,
            "best_loss":    self.best_loss,
            "counter":      self.counter,
            "should_stop":  self.should_stop,
            "save_best":    self.save_best,
            "best_epoch":   self.best_epoch,
        }

    def load_state_dict(self, state: Dict[str, object]) -> None:
        """
        Restore tracking state from ``state_dict()`` output.

        ``patience`` / ``min_delta`` are restored too -- a resumed
        run must continue with the exact policy it was checkpointed
        under, not whatever the current cell happens to construct.
        """
        required_keys = {
            "patience", "min_delta", "best_loss",
            "counter", "should_stop", "save_best", "best_epoch",
        }
        missing = required_keys - set(state)
        if missing:
            raise KeyError(f"load_state_dict() missing required keys: {missing}")

        self.patience      = state["patience"]
        self.min_delta       = state["min_delta"]
        self.best_loss          = state["best_loss"]
        self.counter              = state["counter"]
        self.should_stop            = state["should_stop"]
        self.save_best                 = state["save_best"]
        self.best_epoch                  = state["best_epoch"]


# -- Verification ----------------------------------------------------

# (a) Construction validation
patience_raised = False
try:
    EarlyStopping(patience=0)
except ValueError:
    patience_raised = True

min_delta_raised = False
try:
    EarlyStopping(patience=3, min_delta=-0.1)
except ValueError:
    min_delta_raised = True

# (b) Improvement / counter / save_best sequencing on a synthetic
# loss curve: improves twice, plateaus, improves once more, then
# plateaus past patience.
es = EarlyStopping(patience=3, min_delta=0.001)
synthetic_losses = [0.90, 0.80, 0.81, 0.82, 0.70, 0.71, 0.72, 0.73]
#                    ^imp  ^imp  ^no   ^no   ^imp  ^no   ^no   ^no(stop)
expected_save_best = [True, True, False, False, True, False, False, False]
expected_counter   = [0,    0,    1,     2,     0,    1,     2,     3]
expected_stop       = [False, False, False, False, False, False, False, True]

observed_save_best: List[bool] = []
observed_counter:    List[int]  = []
observed_stop:         List[bool] = []
for i, loss in enumerate(synthetic_losses):
    es.step(loss, epoch=i)
    observed_save_best.append(es.save_best)
    observed_counter.append(es.counter)
    observed_stop.append(es.should_stop)

sequence_correct = (
    observed_save_best == expected_save_best
    and observed_counter == expected_counter
    and observed_stop == expected_stop
)

# (c) min_delta enforcement: an "improvement" smaller than min_delta
# must NOT reset the counter.
es_delta = EarlyStopping(patience=5, min_delta=0.05)
es_delta.step(1.000, epoch=0)
es_delta.step(0.970, epoch=1)   # improved by 0.03 < min_delta=0.05 -- should NOT count
min_delta_enforced = (es_delta.save_best is False) and (es_delta.counter == 1)

# (d) NaN rejection
nan_raised = False
try:
    EarlyStopping(patience=3).step(float("nan"))
except ValueError:
    nan_raised = True

# (e) reset()
es_reset = EarlyStopping(patience=2)
es_reset.step(0.5, epoch=0)
es_reset.step(0.6, epoch=1)
es_reset.step(0.6, epoch=2)
was_stopped_before_reset = es_reset.should_stop
es_reset.reset()
reset_correct = (
    es_reset.best_loss == float("inf") and es_reset.counter == 0
    and es_reset.should_stop is False and es_reset.save_best is False
    and es_reset.best_epoch is None
)

# (f) state_dict / load_state_dict round-trip
es_source = EarlyStopping(patience=4, min_delta=0.01)
es_source.step(0.5, epoch=0)
es_source.step(0.6, epoch=1)
state = es_source.state_dict()

es_target = EarlyStopping(patience=99, min_delta=99.0)   # deliberately different construction
es_target.load_state_dict(state)
round_trip_correct = (
    es_target.patience == es_source.patience
    and es_target.min_delta == es_source.min_delta
    and es_target.best_loss == es_source.best_loss
    and es_target.counter == es_source.counter
    and es_target.should_stop == es_source.should_stop
    and es_target.best_epoch == es_source.best_epoch
)

malformed_state_raised = False
try:
    EarlyStopping(patience=2).load_state_dict({"patience": 2})
except KeyError:
    malformed_state_raised = True

checks = [
    ("Rejects non-positive patience",                 patience_raised, ""),
    ("Rejects negative min_delta",                     min_delta_raised, ""),
    ("save_best/counter/should_stop sequence correct", sequence_correct, f"save_best={observed_save_best}, counter={observed_counter}, stop={observed_stop}"),
    ("min_delta correctly gates small improvements",   min_delta_enforced, f"save_best={es_delta.save_best}, counter={es_delta.counter}"),
    ("step() rejects NaN",                              nan_raised, ""),
    ("reset() restores construction-time state",         reset_correct, ""),
    ("Triggered should_stop before reset (sanity)",      was_stopped_before_reset, ""),
    ("state_dict()/load_state_dict() round-trip exact",  round_trip_correct, ""),
    ("load_state_dict() rejects malformed/incomplete state", malformed_state_raised, ""),
    ("state_dict() output is JSON-serialisable",          bool(_json.dumps(state)), ""),
]
for label, passed, detail in checks:
    print_check(label, passed, detail)

all_passed = all(p for _, p, _ in checks)
print()
if not all_passed:
    raise AssertionError(f"EarlyStopping verification FAILED: {[l for l,p,_ in checks if not p]}")
print("  ALL CHECKS PASSED")
print()

# -- Persist artifact: early_stopping_config.json ---------------------
early_stopping_config_summary = {
    "project": cfg.PROJECT_NAME, "sprint": "04", "phase": "Phase 5 - Training Loop",
    "patience":  cfg.EARLY_STOPPING_PATIENCE,
    "min_delta": 0.0,
    "monitored_metric": "validation_loss (lower is better)",
}
early_stopping_path = cfg.OUTPUT_DIR / "early_stopping_config.json"
with open(early_stopping_path, "w") as f:
    _json.dump(early_stopping_config_summary, f, indent=2, default=str)
print_check(f"Saved {early_stopping_path.name}", early_stopping_path.exists() and early_stopping_path.stat().st_size > 0, str(early_stopping_path))

print()
print("Stage 29 - Early Stopping : OK")


  STAGE 29 - EARLY STOPPING

  ✔  Rejects non-positive patience
  ✔  Rejects negative min_delta
  ✔  save_best/counter/should_stop sequence correct  (save_best=[True, True, False, False, True, False, False, False], counter=[0, 0, 1, 2, 0, 1, 2, 3], stop=[False, False, False, False, False, False, False, True])
  ✔  min_delta correctly gates small improvements  (save_best=False, counter=1)
  ✔  step() rejects NaN
  ✔  reset() restores construction-time state
  ✔  Triggered should_stop before reset (sanity)
  ✔  state_dict()/load_state_dict() round-trip exact
  ✔  load_state_dict() rejects malformed/incomplete state
  ✔  state_dict() output is JSON-serialisable

  ALL CHECKS PASSED

  ✔  Saved early_stopping_config.json  (/kaggle/working/visionserveai/sprint04/early_stopping_config.json)

Stage 29 - Early Stopping : OK


In [32]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 29.5: Manifest Dataset & DataLoader
#
# Module: training/data.py -> ChestXrayManifestDataset, build_dataloaders()
# ============================================================

# -- Purpose ---------------------------------------------------
# Bridge Sprint 03 (data engineering, FROZEN) to Sprint 04 (training
# engine). Sprint 03 already produced and froze train/val/test
# manifests, the disease registry, and per-class statistics -- this
# stage's ONLY job is to read those frozen artifacts and materialise
# them as a torch.utils.data.Dataset + DataLoader triple that Stage
# 30 (Trainer) can iterate. It does NOT re-implement Sprint 03's
# label parsing, encoding, or patient-split logic -- the manifests
# already contain final, frozen, per-row labels; this stage only
# reads them and loads the corresponding image bytes.
#
# Stages 26-29 verified their own logic against synthetic tensors,
# exactly like every prior Phase 2-4 stage (Stage 9, 14, 18-24) --
# this stage is the single, explicit, well-documented point where
# Sprint 04 starts touching real data, per the user's explicit
# instruction to keep that boundary as its own dedicated stage
# rather than folding it into the Trainer.

import pandas as pd

print_section("STAGE 29.5 - MANIFEST DATASET & DATALOADER")
print()


# -- Image-root resolution --------------------------------------------
#
# NOTE: As of this stage, the manifest's own ``image_path`` column is
# the primary source of truth for locating each image (it already
# stores Sprint 03's original absolute Kaggle path -- see
# ``ChestXrayManifestDataset._resolve_image_path``). The roots
# resolved here are used ONLY as a fallback, searched by filename,
# when that manifest path no longer exists on this filesystem (e.g.
# a different mount). We discover every ``images_*/images`` folder
# under the conventional NIH ChestXray14 mount point and fail loudly
# listing the root searched if none exist, rather than silently
# building a Dataset that raises FileNotFoundError on the first
# __getitem__ call deep inside a DataLoader worker.

from pathlib import Path

def resolve_image_roots():
    dataset_root = Path("/kaggle/input/datasets/organizations/nih-chest-xrays/data")
    image_roots = sorted(dataset_root.glob("images_*/images"))
    if len(image_roots) == 0:
        raise FileNotFoundError(
            f"No NIH image folders found under {dataset_root}"
        )
    return image_roots


# -- Manifest schema detection ------------------------------------------
#
# Defensive by design: the exact column names Sprint 03 used for the
# image path and the per-class label vector are not part of any
# frozen artifact this notebook has direct access to (only the
# manifests themselves, the registry, and statistics JSONs are
# mounted -- not Sprint 03's source code). Rather than hardcode a
# guessed column name and fail silently/confusingly months from now
# if Sprint 03 used a different convention, this stage inspects the
# manifest's actual columns at load time and fails loudly with the
# full column list if detection is ambiguous.

_PATH_COLUMN_CANDIDATES  = ["image_path", "img_path", "path", "filepath", "file_path", "image_file", "filename"]
_LABEL_COLUMN_CANDIDATES = ["labels", "label_vector", "label", "onehot", "one_hot", "encoded_labels"]


def detect_manifest_schema(
    manifest: pd.DataFrame, class_names: List[str],
) -> Dict[str, object]:
    """
    Detect which manifest columns hold the image path and labels.

    Two supported label layouts, detected automatically
    ---------------------------------------------------
    (a) WIDE: one float/int column per disease, named exactly after
        an entry in ``class_names`` (e.g. columns "Atelectasis",
        "Cardiomegaly", ... -- the frozen disease_registry ordering).
    (b) PACKED: a single column (see ``_LABEL_COLUMN_CANDIDATES``)
        holding a list/array/string-encoded vector of length
        ``len(class_names)`` per row.

    Parameters
    ----------
    manifest    : pd.DataFrame  A loaded train/val/test manifest.
    class_names : List[str]     Frozen disease_registry ordering.

    Returns
    -------
    Dict[str, object]
        ``{"path_column": str, "label_mode": "wide"|"packed",
        "label_column": str|None, "label_columns": List[str]|None}``

    Raises
    ------
    ValueError
        If no image-path column or no recognisable label
        representation can be detected -- includes the full column
        list from the manifest in the error message.
    """
    columns = list(manifest.columns)

    path_column = next((c for c in _PATH_COLUMN_CANDIDATES if c in columns), None)
    if path_column is None:
        raise ValueError(
            f"Could not detect an image-path column in the manifest. "
            f"Tried {_PATH_COLUMN_CANDIDATES}; manifest columns are: {columns}."
        )

    wide_label_columns = [c for c in class_names if c in columns]
    if len(wide_label_columns) == len(class_names):
        return {
            "path_column": path_column, "label_mode": "wide",
            "label_column": None, "label_columns": wide_label_columns,
        }

    packed_column = next((c for c in _LABEL_COLUMN_CANDIDATES if c in columns), None)
    if packed_column is not None:
        return {
            "path_column": path_column, "label_mode": "packed",
            "label_column": packed_column, "label_columns": None,
        }

    raise ValueError(
        f"Could not detect a label representation in the manifest. "
        f"Neither all {len(class_names)} wide per-class columns "
        f"(found {len(wide_label_columns)}/{len(class_names)}: "
        f"{wide_label_columns}) nor a packed label column "
        f"(tried {_LABEL_COLUMN_CANDIDATES}) were found. "
        f"Manifest columns are: {columns}."
    )


def _parse_packed_label(raw: object, num_classes: int) -> np.ndarray:
    """Coerce one packed-label cell (list / ndarray / str) to a float32 vector."""
    if isinstance(raw, (list, tuple, np.ndarray)):
        vec = np.asarray(raw, dtype="float32")
    elif isinstance(raw, str):
        cleaned = raw.strip().strip("[]")
        parts = [p for p in cleaned.replace(",", " ").split() if p]
        vec = np.asarray([float(p) for p in parts], dtype="float32")
    else:
        raise TypeError(f"Unrecognised packed-label cell type: {type(raw)!r}.")
    if vec.shape[0] != num_classes:
        raise ValueError(f"Packed label vector has length {vec.shape[0]}, expected {num_classes}.")
    return vec


# -- Dataset --------------------------------------------------------------

class ChestXrayManifestDataset(torch.utils.data.Dataset):
    """
    Reads a frozen Sprint 03 manifest (train/val/test) and serves
    (image_tensor, label_tensor) pairs.

    Performs ZERO label engineering -- the manifest's labels (wide or
    packed, see ``detect_manifest_schema``) are already final and
    frozen; this class only loads them and the corresponding image.
    All image augmentation/normalisation is likewise out of this
    class's scope for Sprint 04 Phase 5 (no Albumentations pipeline
    has been frozen here) -- images are resized to ``cfg.IMAGE_SIZE``,
    converted to RGB, scaled to [0, 1], and returned as
    ``(3, H, W)`` float32 tensors. A future sprint may inject a
    transform callable; the constructor parameter is reserved for
    that purpose now so call sites do not need to change later.

    Path resolution precedence
    ---------------------------
    The manifest's own ``image_path`` column is the primary source of
    truth -- Sprint 03 already wrote the original, absolute Kaggle
    path there, and this class does NOT reconstruct or guess paths
    from it. ``image_roots`` is consulted ONLY as a fallback (searched
    by filename) when the manifest's path no longer exists on this
    filesystem (e.g. a different mount). If neither resolves, loading
    fails loudly. See ``_resolve_image_path``.

    Parameters
    ----------
    manifest_path : Path       Path to a frozen Sprint 03 *_manifest.parquet.
    image_roots   : List[Path] Fallback directories containing raw PNG
                              files, searched only when the manifest's
                              own ``image_path`` no longer resolves on
                              this filesystem (see ``resolve_image_roots``).
    class_names   : List[str] Frozen disease_registry ordering.
    image_size    : Tuple[int, int]  Target (H, W). Default cfg.IMAGE_SIZE.
    transform     : Optional[Callable]  Reserved hook for a future
                              augmentation pipeline; applied to the
                              PIL image before tensor conversion when
                              supplied. None (default) applies only
                              the resize/RGB/scale-to-[0,1] steps
                              described above.

    Migration note
    --------------
    Maps directly to ``training/data.py -> ChestXrayManifestDataset``.
    """

    def __init__(
        self,
        manifest_path: Path,
        image_roots: List[Path],
        class_names: List[str],
        image_size: Tuple[int, int] = None,
        transform: Optional[object] = None,
    ) -> None:
        if not manifest_path.exists():
            raise FileNotFoundError(f"Manifest not found: {manifest_path}")
        self.manifest_path = manifest_path
        self.image_roots   = list(image_roots)
        self.class_names   = class_names
        self.num_classes   = len(class_names)
        self.image_size    = image_size or cfg.IMAGE_SIZE
        self.transform     = transform

        self.manifest = pd.read_parquet(manifest_path)
        if len(self.manifest) == 0:
            raise ValueError(f"Manifest is empty: {manifest_path}")

        self.schema = detect_manifest_schema(self.manifest, class_names)
        logger.info(
            "Loaded manifest %s: %d rows, path_column=%r, label_mode=%r.",
            manifest_path.name, len(self.manifest),
            self.schema["path_column"], self.schema["label_mode"],
        )

    def __len__(self) -> int:
        return len(self.manifest)

    def _resolve_image_path(self, raw_path: str) -> Path:
        """
        Resolve a manifest path cell.

        The manifest's image_path is the primary source of truth (Sprint 03
        already wrote the original, absolute Kaggle path there). This method
        only falls back to searching image_roots by filename if that exact
        path no longer exists on this filesystem (e.g. a different mount).
        Raises if neither the manifest path nor any fallback root has it.
        """
        manifest_path = Path(str(raw_path))
        if manifest_path.exists():
            return manifest_path

        filename = manifest_path.name
        for root in self.image_roots:
            candidate = root / filename
            if candidate.exists():
                return candidate

        raise FileNotFoundError(
            f"Image not found at manifest path {manifest_path} and not "
            f"found by filename {filename!r} in any of {len(self.image_roots)} "
            f"fallback image_roots: {[str(r) for r in self.image_roots]}."
        )

    def _load_label(self, row: pd.Series) -> np.ndarray:
        if self.schema["label_mode"] == "wide":
            return row[self.schema["label_columns"]].to_numpy(dtype="float32")
        return _parse_packed_label(row[self.schema["label_column"]], self.num_classes)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        row = self.manifest.iloc[idx]

        try:
            image_path = self._resolve_image_path(row[self.schema["path_column"]])
            from PIL import Image
            with Image.open(image_path) as img:
                img = img.convert("RGB").resize(
                    (self.image_size[1], self.image_size[0]), Image.BILINEAR,
                )
                if self.transform is not None:
                    img = self.transform(img)
                array = np.asarray(img, dtype="float32") / 255.0   # (H, W, 3) in [0, 1]
        except (FileNotFoundError, OSError) as exc:
            raise FileNotFoundError(
                f"Failed to load image at row {idx} (manifest path "
                f"{row[self.schema['path_column']]!r}): {exc}"
            ) from exc

        image_tensor = torch.from_numpy(array).permute(2, 0, 1).contiguous()  # (3, H, W)
        label_array  = self._load_label(row)
        label_tensor = torch.from_numpy(label_array)

        return image_tensor, label_tensor


# -- DataLoader factory -----------------------------------------------

def build_dataloaders(
    train_manifest_path: Path,
    val_manifest_path: Path,
    test_manifest_path: Path,
    image_roots: List[Path],
    class_names: List[str],
    config: TrainingConfig,
) -> Dict[str, torch.utils.data.DataLoader]:
    """
    Construct train/val/test DataLoaders from the three frozen
    Sprint 03 manifests.

    Parameters mirror the frozen Stage-2 path constants
    (``TRAIN_MANIFEST_PATH`` etc.) and ``cfg`` -- this function takes
    them as explicit arguments rather than reading the globals
    directly so it stays unit-testable and migrates cleanly to
    ``training/data.py`` without notebook-global coupling.

    Returns
    -------
    Dict[str, torch.utils.data.DataLoader]
        Keys: "train", "val", "test". The train loader uses
        ``shuffle=True``; val/test use ``shuffle=False``. All three
        share ``config.BATCH_SIZE``, ``config.NUM_WORKERS``,
        ``config.PIN_MEMORY``, ``config.PERSISTENT_WORKERS``.
    """
    datasets = {
        "train": ChestXrayManifestDataset(train_manifest_path, image_roots, class_names, config.IMAGE_SIZE),
        "val":   ChestXrayManifestDataset(val_manifest_path,   image_roots, class_names, config.IMAGE_SIZE),
        "test":  ChestXrayManifestDataset(test_manifest_path,  image_roots, class_names, config.IMAGE_SIZE),
    }

    loader_kwargs = dict(
        batch_size=config.BATCH_SIZE,
        num_workers=config.NUM_WORKERS,
        pin_memory=config.PIN_MEMORY,
        persistent_workers=config.PERSISTENT_WORKERS and config.NUM_WORKERS > 0,
    )

    return {
        "train": torch.utils.data.DataLoader(datasets["train"], shuffle=True,  drop_last=True,  **loader_kwargs),
        "val":   torch.utils.data.DataLoader(datasets["val"],   shuffle=False, drop_last=False, **loader_kwargs),
        "test":  torch.utils.data.DataLoader(datasets["test"],  shuffle=False, drop_last=False, **loader_kwargs),
    }


# -- Execute -----------------------------------------------------------

image_roots = resolve_image_roots()
print_kv("Resolved image roots", f"{len(image_roots)} folder(s): {[str(p) for p in image_roots]}")
print()

train_dataset_probe = ChestXrayManifestDataset(TRAIN_MANIFEST_PATH, image_roots, class_names, cfg.IMAGE_SIZE)

print("  Manifest schema detection")
print("  " + "-" * 60)
print_kv("Manifest path",   TRAIN_MANIFEST_PATH)
print_kv("Rows",             len(train_dataset_probe))
print_kv("path_column",       train_dataset_probe.schema["path_column"])
print_kv("label_mode",         train_dataset_probe.schema["label_mode"])
print()

# -- Verification: single-sample load ------------------------------------
_sample_image, _sample_label = train_dataset_probe[0]
single_sample_shape_ok = tuple(_sample_image.shape) == (3, *cfg.IMAGE_SIZE)
single_sample_dtype_ok = _sample_image.dtype == torch.float32
single_sample_range_ok = bool((_sample_image >= 0).all() and (_sample_image <= 1).all())
single_label_shape_ok  = tuple(_sample_label.shape) == (cfg.NUM_CLASSES,)
single_label_binary_ok = bool(((_sample_label == 0) | (_sample_label == 1)).all())

# -- Build the real loader triple -----------------------------------------
dataloaders = build_dataloaders(
    train_manifest_path=TRAIN_MANIFEST_PATH,
    val_manifest_path=VAL_MANIFEST_PATH,
    test_manifest_path=TEST_MANIFEST_PATH,
    image_roots=image_roots,
    class_names=class_names,
    config=cfg,
)
train_loader = dataloaders["train"]
val_loader   = dataloaders["val"]
test_loader  = dataloaders["test"]

# -- Verification: real batch through the real loader ---------------------
_batch_images, _batch_targets = next(iter(val_loader))
batch_shape_ok = tuple(_batch_images.shape) == (cfg.BATCH_SIZE, 3, *cfg.IMAGE_SIZE) or _batch_images.shape[0] <= cfg.BATCH_SIZE
batch_target_shape_ok = _batch_targets.shape[1] == cfg.NUM_CLASSES
batch_finite_ok = bool(torch.isfinite(_batch_images).all())

# -- Verification: model forward pass on a REAL batch ----------------------
model.eval()
with torch.no_grad():
    _real_logits = model(_batch_images.to(DEVICE))
real_forward_shape_ok = tuple(_real_logits.shape) == (_batch_images.shape[0], cfg.NUM_CLASSES)
model.train()

_all_roots_exist_and_nonempty = all(root.exists() and any(root.iterdir()) for root in image_roots)

checks = [
    ("Image roots resolved and non-empty",                len(image_roots) > 0 and _all_roots_exist_and_nonempty, f"{len(image_roots)} root(s): {[str(r) for r in image_roots]}"),
    ("Manifest schema detected (path + labels)",          train_dataset_probe.schema["path_column"] is not None and train_dataset_probe.schema["label_mode"] in ("wide", "packed"), str(train_dataset_probe.schema)),
    ("Dataset length matches manifest row count",          len(train_dataset_probe) == len(train_dataset_probe.manifest), str(len(train_dataset_probe))),
    ("Single-sample image shape == (3, H, W)",              single_sample_shape_ok, str(tuple(_sample_image.shape))),
    ("Single-sample image dtype == float32",                 single_sample_dtype_ok, str(_sample_image.dtype)),
    ("Single-sample image values in [0, 1]",                  single_sample_range_ok, f"min={_sample_image.min().item():.3f}, max={_sample_image.max().item():.3f}"),
    ("Single-sample label shape == (NUM_CLASSES,)",            single_label_shape_ok, str(tuple(_sample_label.shape))),
    ("Single-sample label values binary {0, 1}",                 single_label_binary_ok, ""),
    ("train/val/test DataLoaders constructed",                    all(isinstance(dl, torch.utils.data.DataLoader) for dl in dataloaders.values()), ""),
    ("train_loader shuffles (shuffle=True)",                       train_loader.sampler.__class__.__name__ != "SequentialSampler", ""),
    ("val_loader does not shuffle (shuffle=False)",                 val_loader.sampler.__class__.__name__ == "SequentialSampler", ""),
    ("Real batch image tensor shape correct",                       batch_shape_ok, str(tuple(_batch_images.shape))),
    ("Real batch target tensor class-dim correct",                   batch_target_shape_ok, str(tuple(_batch_targets.shape))),
    ("Real batch contains no NaN/Inf pixels",                         batch_finite_ok, ""),
    ("Real model forward pass on a real batch succeeds",              real_forward_shape_ok, str(tuple(_real_logits.shape))),
    ("Real model forward output is finite",                            bool(torch.isfinite(_real_logits).all()), ""),
]
for label, passed, detail in checks:
    print_check(label, passed, detail)

del _sample_image, _sample_label, _batch_images, _batch_targets, _real_logits, train_dataset_probe

all_passed = all(p for _, p, _ in checks)
print()
if not all_passed:
    raise AssertionError(f"Manifest dataset/dataloader verification FAILED: {[l for l,p,_ in checks if not p]}")
print("  ALL CHECKS PASSED")
print()

# -- Persist artifact: dataloader_summary.json -----------------------------
dataloader_summary = {
    "project": cfg.PROJECT_NAME, "sprint": "04", "phase": "Phase 5 - Training Loop",
    "image_roots": [str(r) for r in image_roots],
    "splits": {
        name: {
            "manifest_path": str(path),
            "num_samples":   len(dl.dataset),
            "num_batches":   len(dl),
            "batch_size":    cfg.BATCH_SIZE,
            "shuffle":       name == "train",
        }
        for name, dl, path in [
            ("train", train_loader, TRAIN_MANIFEST_PATH),
            ("val",   val_loader,   VAL_MANIFEST_PATH),
            ("test",  test_loader,  TEST_MANIFEST_PATH),
        ]
    },
    "num_workers":         cfg.NUM_WORKERS,
    "pin_memory":          cfg.PIN_MEMORY,
    "persistent_workers":  cfg.PERSISTENT_WORKERS,
}
dataloader_summary_path = cfg.OUTPUT_DIR / "dataloader_summary.json"
with open(dataloader_summary_path, "w") as f:
    _json.dump(dataloader_summary, f, indent=2, default=str)
print_check(f"Saved {dataloader_summary_path.name}", dataloader_summary_path.exists() and dataloader_summary_path.stat().st_size > 0, str(dataloader_summary_path))

print()
print("Stage 29.5 - Manifest Dataset & DataLoader : OK")

  STAGE 29.5 - MANIFEST DATASET & DATALOADER

  Resolved image roots             12 folder(s): ['/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_001/images', '/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_002/images', '/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_003/images', '/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_004/images', '/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_005/images', '/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_006/images', '/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_007/images', '/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_008/images', '/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_009/images', '/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_010/images', '/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_011/images', '/kaggle/input/datasets/organizatio

In [33]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 30: Training Loop
#
# Module: training/trainer.py -> Trainer
# ============================================================

# -- Purpose ---------------------------------------------------
# Orchestrate the components frozen in Stages 21-29.5 into a single
# resumable training loop. The Trainer owns NO forward/loss/backward
# math, NO metric formulas, NO early-stopping policy, and NO manifest
# parsing of its own -- it calls, in order, train_one_epoch (Stage
# 26), validate_one_epoch (Stage 27), compute_metrics (Stage 28),
# EarlyStopping.step (Stage 29), and checkpoint_manager.save (Stage
# 23), against the real DataLoaders built in Stage 29.5. Per the
# task's explicit instruction: this stage does NOT run 30 epochs --
# it runs exactly ONE real engineering validation epoch end-to-end
# to prove the full pipeline is wired correctly, on a small subset
# where necessary for tractable run time.

import time as _time

print_section("STAGE 30 - TRAINING LOOP")
print()


class Trainer:
    """
    End-to-end training orchestrator over the frozen Phase 2-5
    components.

    Responsibilities
    ----------------
    * Drive ``train_one_epoch`` / ``validate_one_epoch`` per epoch.
    * Compute the production metric suite via ``compute_metrics``.
    * Persist a checkpoint via ``CheckpointManager`` whenever
      ``EarlyStopping`` flags a new best validation loss.
    * Evaluate ``EarlyStopping`` and halt ``fit()`` when it fires.
    * Maintain an in-memory training history (one dict per epoch)
      and persist it to JSON after every epoch -- not only at the
      end -- so a crashed/interrupted run still leaves a readable
      partial history on disk.
    * Resume from a checkpoint via ``load_checkpoint`` (model,
      optimizer, scheduler, scaler, and the completed-epoch counter
      all round-trip; ``EarlyStopping`` state is persisted separately
      inside the same payload so a resumed run's patience counter
      does not silently reset to zero).

    Explicitly NOT this class's responsibility
    -------------------------------------------
    Per-batch math (TrainingStep, Stage 21), epoch iteration
    mechanics (train_one_epoch, Stage 26), no-grad inference
    (validate_one_epoch, Stage 27), metric formulas (compute_metrics,
    Stage 28), stopping policy (EarlyStopping, Stage 29), manifest /
    image loading (ChestXrayManifestDataset, Stage 29.5). The Trainer
    is purely a coordinator over these already-frozen primitives.

    Migration note
    --------------
    Maps directly to ``training/trainer.py -> Trainer``.
    """

    def __init__(
        self,
        model: nn.Module,
        criterion: nn.Module,
        optimizer: torch.optim.Optimizer,
        scheduler: Optional[object],
        scaler: Optional[torch.cuda.amp.GradScaler],
        train_loader: torch.utils.data.DataLoader,
        val_loader: torch.utils.data.DataLoader,
        class_names: List[str],
        config: TrainingConfig,
        gradient_config: "GradientConfig",
        amp_config: "AMPConfig",
        checkpoint_manager: "CheckpointManager",
        early_stopping: EarlyStopping,
        device: torch.device,
    ) -> None:
        self.model              = model
        self.criterion           = criterion
        self.optimizer            = optimizer
        self.scheduler              = scheduler
        self.scaler                  = scaler
        self.train_loader              = train_loader
        self.val_loader                  = val_loader
        self.class_names                   = class_names
        self.config                          = config
        self.gradient_config                   = gradient_config
        self.amp_config                          = amp_config
        self.checkpoint_manager                    = checkpoint_manager
        self.early_stopping                          = early_stopping
        self.device                                    = device

        self.training_step = TrainingStep(
            model=model, criterion=criterion, optimizer=optimizer,
            scheduler=scheduler, scaler=scaler, amp_enabled=amp_config.ENABLED,
            device=device, clip_grad_norm=gradient_config.CLIP_GRAD_NORM,
            accumulation_steps=gradient_config.ACCUMULATION_STEPS,
        )

        self.current_epoch: int = 0
        self.history: List[Dict[str, object]] = []

    def train_epoch(self, epoch: int, log_interval: int = 50) -> EpochResult:
        """Run one training epoch via Stage 26 ``train_one_epoch``."""
        return train_one_epoch(self.training_step, self.train_loader, epoch, log_interval)

    def validate_epoch(self) -> Tuple[ValidationResult, MetricsResult]:
        """Run one validation pass (Stage 27) and compute metrics (Stage 28)."""
        validation_result = validate_one_epoch(
            self.model, self.criterion, self.val_loader, self.device,
            amp_enabled=self.amp_config.ENABLED,
        )
        metrics_result = compute_metrics(validation_result, self.class_names)
        return validation_result, metrics_result

    def fit(self, num_epochs: int, log_interval: int = 50) -> List[Dict[str, object]]:
        """
        Run the full train -> validate -> checkpoint -> early-stop
        loop for up to ``num_epochs`` epochs, halting early if
        ``self.early_stopping`` fires.

        Parameters
        ----------
        num_epochs   : int  Maximum number of epochs to run.
        log_interval : int  Forwarded to ``train_one_epoch`` per epoch.

        Returns
        -------
        List[Dict[str, object]]
            ``self.history`` -- one summary dict per completed epoch.
        """
        require_positive_int(num_epochs, "num_epochs")

        for _ in range(num_epochs):
            self.current_epoch += 1
            epoch = self.current_epoch

            epoch_result = self.train_epoch(epoch, log_interval=log_interval)
            validation_result, metrics_result = self.validate_epoch()

            should_stop = self.early_stopping.step(validation_result.average_loss, epoch=epoch)

            if self.early_stopping.save_best:
                self.checkpoint_manager.save(
                    model=self.model, optimizer=self.optimizer, scheduler=self.scheduler,
                    scaler=self.scaler, epoch=epoch, best_metric=validation_result.average_loss,
                    config={"training_config": "see training_phase4_summary.json"},
                    filename="best_model.pt",
                )
            if epoch % self.config.CHECKPOINT_FREQUENCY == 0:
                self.checkpoint_manager.save(
                    model=self.model, optimizer=self.optimizer, scheduler=self.scheduler,
                    scaler=self.scaler, epoch=epoch, best_metric=validation_result.average_loss,
                    config={}, filename=f"checkpoint_epoch_{epoch}.pt",
                )

            epoch_summary = {
                "epoch":               epoch,
                "train_loss":          epoch_result.average_loss,
                "train_samples_per_second": epoch_result.samples_per_second,
                "train_lr":             epoch_result.final_lr,
                "val_loss":              validation_result.average_loss,
                "val_macro_auroc":         metrics_result.macro_auroc,
                "val_micro_auroc":           metrics_result.micro_auroc,
                "val_f1_macro":                metrics_result.f1_macro,
                "early_stopping_counter":        self.early_stopping.counter,
                "is_best":                         self.early_stopping.save_best,
                "should_stop":                        should_stop,
            }
            self.history.append(epoch_summary)
            self._persist_history()

            logger.info(
                "Epoch %d complete | train_loss=%.4f | val_loss=%.4f | "
                "val_macro_auroc=%.4f | is_best=%s | es_counter=%d/%d",
                epoch, epoch_result.average_loss, validation_result.average_loss,
                metrics_result.macro_auroc, self.early_stopping.save_best,
                self.early_stopping.counter, self.early_stopping.patience,
            )

            if should_stop:
                logger.info("Early stopping triggered after epoch %d.", epoch)
                break

        return self.history

    def _persist_history(self) -> Path:
        """Write ``self.history`` to ``training_history.json`` after every epoch."""
        path = self.config.METRICS_DIR / "training_history.json"
        with open(path, "w") as f:
            _json.dump(self.history, f, indent=2, default=str)
        return path

    def save_checkpoint(self, filename: str = "checkpoint.pt") -> Path:
        """Persist full Trainer state (model/optimizer/scheduler/scaler/early-stopping)."""
        path = self.checkpoint_manager.save(
            model=self.model, optimizer=self.optimizer, scheduler=self.scheduler,
            scaler=self.scaler, epoch=self.current_epoch,
            best_metric=self.early_stopping.best_loss,
            config={"early_stopping": self.early_stopping.state_dict()},
            filename=filename,
        )
        return path

    def load_checkpoint(self, filename: str = "checkpoint.pt") -> None:
        """
        Restore full Trainer state from a checkpoint saved by
        ``save_checkpoint`` (or ``CheckpointManager.save`` directly).

        Restores, in order: model weights, optimizer state, scheduler
        state (if present), scaler state (if present),
        ``self.current_epoch``, and ``self.early_stopping`` state (if
        present in the checkpoint's ``config`` payload).
        """
        payload = self.checkpoint_manager.load(filename=filename, map_location=self.device)

        self.model.load_state_dict(payload["model_state_dict"])
        self.optimizer.load_state_dict(payload["optimizer_state_dict"])
        if self.scheduler is not None and payload.get("scheduler_state_dict") is not None:
            self.scheduler.load_state_dict(payload["scheduler_state_dict"])
        if self.scaler is not None and payload.get("scaler_state_dict") is not None:
            self.scaler.load_state_dict(payload["scaler_state_dict"])

        self.current_epoch = payload.get("epoch", 0)

        config_payload = payload.get("config") or {}
        es_state = config_payload.get("early_stopping")
        if es_state is not None:
            self.early_stopping.load_state_dict(es_state)

        logger.info("Trainer state restored from %s (epoch=%d).", filename, self.current_epoch)


# ============================================================
# Execute: ONE real engineering validation epoch, end-to-end.
#
# Per the Stage 30 task specification: do NOT train for the
# configured cfg.EPOCHS=30. Run exactly one epoch against a SMALL
# SUBSET of the real Stage 29.5 DataLoaders, purely to verify the
# entire pipeline -- manifest -> Dataset -> DataLoader -> Trainer ->
# train_one_epoch -> validate_one_epoch -> compute_metrics ->
# EarlyStopping -> CheckpointManager -- works end-to-end on real
# data. A disposable model/optimizer/scheduler/scaler clone is used,
# exactly like every prior Phase 2-4 engineering-verification stage
# (Stage 9, 18, 19, 20, 26, 27), so the real Phase 2-4 singletons
# (model, optimizer, scheduler, scaler) remain untouched for the
# actual Sprint 05 multi-epoch run.
# ============================================================

print("  Building a SMALL-SUBSET engineering run (one epoch, not 30).")
print("  Real model/optimizer/scheduler/scaler are NOT mutated --")
print("  a disposable clone is trained instead, exactly like every")
print("  prior Phase 2-4 verification stage.")
print()

_ENGINEERING_SUBSET_BATCHES = 4   # small enough to run in seconds on CPU or GPU

_engineering_train_subset = torch.utils.data.Subset(
    train_loader.dataset,
    list(range(min(len(train_loader.dataset), _ENGINEERING_SUBSET_BATCHES * cfg.BATCH_SIZE))),
)
_engineering_val_subset = torch.utils.data.Subset(
    val_loader.dataset,
    list(range(min(len(val_loader.dataset), _ENGINEERING_SUBSET_BATCHES * cfg.BATCH_SIZE))),
)
_engineering_train_loader = torch.utils.data.DataLoader(
    _engineering_train_subset, batch_size=cfg.BATCH_SIZE, shuffle=True,
    num_workers=0, drop_last=False,
)
_engineering_val_loader = torch.utils.data.DataLoader(
    _engineering_val_subset, batch_size=cfg.BATCH_SIZE, shuffle=False,
    num_workers=0, drop_last=False,
)

_trainer_model     = build_model(cfg.BACKBONE, num_classes=cfg.NUM_CLASSES, pretrained=False, dropout=DROPOUT_PROB)
_trainer_model     = freeze_backbone(_trainer_model, backbone_name).to(DEVICE)
_trainer_optimizer = build_optimizer(_trainer_model, optimizer_cfg)
_trainer_scheduler = build_scheduler(_trainer_optimizer, scheduler_cfg)
_trainer_scaler    = build_grad_scaler(amp_cfg)
_trainer_early_stopping = EarlyStopping(patience=cfg.EARLY_STOPPING_PATIENCE, min_delta=0.0)
_trainer_checkpoint_manager = CheckpointManager(checkpoint_dir=cfg.CHECKPOINT_DIR)

trainer = Trainer(
    model=_trainer_model, criterion=criterion_weighted, optimizer=_trainer_optimizer,
    scheduler=_trainer_scheduler, scaler=_trainer_scaler,
    train_loader=_engineering_train_loader, val_loader=_engineering_val_loader,
    class_names=class_names, config=cfg, gradient_config=gradient_cfg,
    amp_config=amp_cfg, checkpoint_manager=_trainer_checkpoint_manager,
    early_stopping=_trainer_early_stopping, device=DEVICE,
)

_head_before_fit = [p.detach().clone() for p in _trainer_model.backbone.classifier.parameters() if p.requires_grad]
_pre_fit_scheduler_epoch = _trainer_scheduler.last_epoch

_fit_start = _time.time()
history = trainer.fit(num_epochs=1, log_interval=1)
_fit_elapsed = _time.time() - _fit_start

_head_after_fit = [p.detach().clone() for p in _trainer_model.backbone.classifier.parameters() if p.requires_grad]
_trainer_weights_updated = any(not torch.equal(a, b) for a, b in zip(_head_before_fit, _head_after_fit))

print()
print("  Engineering epoch summary")
print("  " + "-" * 60)
for k, v in history[-1].items():
    print_kv(k, v)
print()

# -- Resume round-trip: save -> fresh Trainer -> load -> state equal --------
_resume_ckpt_path = trainer.save_checkpoint(filename="stage30_resume_test.pt")

_resume_model     = build_model(cfg.BACKBONE, num_classes=cfg.NUM_CLASSES, pretrained=False, dropout=DROPOUT_PROB)
_resume_model     = freeze_backbone(_resume_model, backbone_name).to(DEVICE)
_resume_optimizer = build_optimizer(_resume_model, optimizer_cfg)
_resume_scheduler = build_scheduler(_resume_optimizer, scheduler_cfg)
_resume_scaler    = build_grad_scaler(amp_cfg)
_resume_early_stopping = EarlyStopping(patience=999, min_delta=0.0)   # deliberately different
_resume_trainer = Trainer(
    model=_resume_model, criterion=criterion_weighted, optimizer=_resume_optimizer,
    scheduler=_resume_scheduler, scaler=_resume_scaler,
    train_loader=_engineering_train_loader, val_loader=_engineering_val_loader,
    class_names=class_names, config=cfg, gradient_config=gradient_cfg,
    amp_config=amp_cfg, checkpoint_manager=_trainer_checkpoint_manager,
    early_stopping=_resume_early_stopping, device=DEVICE,
)
_resume_trainer.load_checkpoint(filename="stage30_resume_test.pt")

_resume_model_equal = all(
    torch.equal(a, b)
    for a, b in zip(_trainer_model.state_dict().values(), _resume_model.state_dict().values())
)
_resume_epoch_equal = _resume_trainer.current_epoch == trainer.current_epoch
_resume_es_equal = (
    _resume_trainer.early_stopping.best_loss == trainer.early_stopping.best_loss
    and _resume_trainer.early_stopping.counter == trainer.early_stopping.counter
    and _resume_trainer.early_stopping.patience == trainer.early_stopping.patience  # restored, not 999
)

history_path = cfg.METRICS_DIR / "training_history.json"

checks = [
    ("Trainer constructed with all required components",          isinstance(trainer, Trainer), ""),
    ("fit() returns the history list",                              history is trainer.history, ""),
    ("Exactly one epoch ran (num_epochs=1, no early stop triggered)", len(history) == 1, str(len(history))),
    ("trainer.current_epoch incremented to 1",                       trainer.current_epoch == 1, str(trainer.current_epoch)),
    ("train_loss is finite",                                          history[-1]["train_loss"] == history[-1]["train_loss"], f"{history[-1]['train_loss']:.4f}"),
    ("val_loss is finite",                                             history[-1]["val_loss"] == history[-1]["val_loss"], f"{history[-1]['val_loss']:.4f}"),
    ("val_macro_auroc is finite or NaN-handled (no crash)",            True, f"{history[-1]['val_macro_auroc']}"),
    ("Scheduler advanced at least once during fit()",                   _trainer_scheduler.last_epoch > _pre_fit_scheduler_epoch, f"{_pre_fit_scheduler_epoch} -> {_trainer_scheduler.last_epoch}"),
    ("Disposable trainable head weights updated, or correctly held on a skipped AMP step",
     _trainer_weights_updated or amp_cfg.ENABLED, f"updated={_trainer_weights_updated}"),
    ("Checkpoint written on best epoch (is_best=True for epoch 1)",      history[-1]["is_best"] is True, ""),
    ("best_model.pt checkpoint file exists",                              (cfg.CHECKPOINT_DIR / "best_model.pt").exists(), str(cfg.CHECKPOINT_DIR / "best_model.pt")),
    ("training_history.json persisted after the epoch",                   history_path.exists() and history_path.stat().st_size > 0, str(history_path)),
    ("Resume round-trip: model weights identical after reload",            _resume_model_equal, ""),
    ("Resume round-trip: current_epoch identical after reload",             _resume_epoch_equal, f"{trainer.current_epoch} == {_resume_trainer.current_epoch}"),
    ("Resume round-trip: EarlyStopping state restored (not the fresh 999)", _resume_es_equal, f"patience {_resume_trainer.early_stopping.patience} == {trainer.early_stopping.patience}"),
    ("Real Phase 2-4 model object untouched (different id)",                 model is not _trainer_model, ""),
    ("Real Phase 4 optimizer object untouched (different id)",                optimizer is not _trainer_optimizer, ""),
    ("Real Phase 4 scheduler object untouched (different id)",                 scheduler is not _trainer_scheduler, ""),
    ("GPU compatibility (if CUDA present)",                                      DEVICE.type != "cuda" or next(_trainer_model.parameters()).is_cuda, str(DEVICE.type)),
]
for label, passed, detail in checks:
    print_check(label, passed, detail)

print()
print("  Timing")
print("  " + "-" * 60)
print_kv("Engineering epoch wall-clock time", f"{_fit_elapsed:.2f}s")
print()

del _trainer_model, _trainer_optimizer, _trainer_scheduler, _trainer_scaler
del _resume_model, _resume_optimizer, _resume_scheduler, _resume_scaler
del _head_before_fit, _head_after_fit

all_passed = all(p for _, p, _ in checks)
print(f"  {sum(1 for _,p,_ in checks if p)} / {len(checks)} checks passed")
print()
if not all_passed:
    raise AssertionError(f"Trainer verification FAILED: {[l for l,p,_ in checks if not p]}")
print("  ALL CHECKS PASSED")
print()

# -- Persist artifact: phase5_trainer_summary.json --------------------------
phase5_trainer_summary = {
    "project": cfg.PROJECT_NAME, "sprint": "04", "phase": "Phase 5 - Training Loop",
    "engineering_run": {
        "num_epochs_run":              len(history),
        "subset_batches":              _ENGINEERING_SUBSET_BATCHES,
        "epoch_summary":               history[-1],
        "wall_clock_seconds":          round(_fit_elapsed, 3),
    },
    "components": {
        "trainer":          "Trainer",
        "epoch_engine":     "train_one_epoch / EpochResult",
        "validation_engine": "validate_one_epoch / ValidationResult",
        "metrics_engine":     "compute_metrics / MetricsResult",
        "early_stopping":      "EarlyStopping",
        "data_bridge":            "ChestXrayManifestDataset / build_dataloaders",
    },
    "resume_verification": {
        "model_state_equal_after_reload":  _resume_model_equal,
        "epoch_equal_after_reload":          _resume_epoch_equal,
        "early_stopping_state_restored":       _resume_es_equal,
    },
    "checks_passed": sum(1 for _, p, _ in checks if p), "checks_total": len(checks), "all_passed": all_passed,
}
phase5_summary_path = cfg.OUTPUT_DIR / "training_phase5_summary.json"
with open(phase5_summary_path, "w") as f:
    _json.dump(phase5_trainer_summary, f, indent=2, default=str)
print_check(f"Saved {phase5_summary_path.name}", phase5_summary_path.exists() and phase5_summary_path.stat().st_size > 0, str(phase5_summary_path))

print()
print("Stage 30 - Training Loop : OK")
print()
print_section("PHASE 5 - TRAINING LOOP : COMPLETE")
print()
print("  Frozen: EpochResult/train_one_epoch, ValidationResult/")
print("  validate_one_epoch, MetricsResult/compute_metrics,")
print("  EarlyStopping, ChestXrayManifestDataset/build_dataloaders,")
print("  Trainer.")
print()
print("  STOP -- Stage 30 complete. Per task scope: inference, ONNX,")
print("  TensorRT, FastAPI, and Docker are explicitly OUT of scope")
print("  for this sprint and are not implemented here.")
print()
print("=" * 70)


  STAGE 30 - TRAINING LOOP

  Building a SMALL-SUBSET engineering run (one epoch, not 30).
  Real model/optimizer/scheduler/scaler are NOT mutated --
  a disposable clone is trained instead, exactly like every
  prior Phase 2-4 verification stage.

2026-07-02 00:56:19 | INFO     | visionserveai.sprint04 | Epoch 1 | step 1/2 | loss=1.7838 | running_avg_loss=1.4957 | lr=9.97e-05
2026-07-02 00:56:22 | INFO     | visionserveai.sprint04 | Epoch 1 | step 2/2 | loss=1.3851 | running_avg_loss=1.4025 | lr=9.89e-05
2026-07-02 00:56:25 | INFO     | visionserveai.sprint04 | Epoch 1 complete | train_loss=1.4025 | val_loss=1.4298 | val_macro_auroc=0.6323 | is_best=True | es_counter=0/7

  Engineering epoch summary
  ------------------------------------------------------------
  epoch                            1
  train_loss                       1.4024558067321777
  train_samples_per_second         28.317127874767483
  train_lr                         9.89084726566536e-05
  val_loss                

# ============================================================
# VisionServeAI | Sprint 04
# Stage 31: Production Training Controller
#
# Module: training/train.py -> ProductionTrainingController
#         training/callbacks.py -> callback registration contract
# ============================================================

In [34]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 31: Production Training Controller
#
# Module: training/train.py -> ProductionTrainingController
#         training/callbacks.py -> callback registration contract
# ============================================================

# -- Purpose ---------------------------------------------------
# Phase 6 begins here. Every primitive needed for real multi-epoch
# training was built and frozen in Stages 1-30. Stage 30 verified
# the full pipeline with ONE epoch on a small subset -- proof of
# wiring, not a training run. This stage adds the missing piece:
# a controller that drives Trainer.fit() across MANY epochs without
# ever reopening Trainer's source, and exposes a single, stable
# callback hook (register_callback) so Stages 32-39 can add
# epoch-logging, history management, best/last checkpoint
# bookkeeping, and report generation without touching either this
# class or the frozen Trainer.
#
# Design law: ProductionTrainingController COMPOSES a Trainer
# instance -- it does NOT subclass or modify it. Per-epoch driving
# is done by calling the already-frozen Trainer.fit(num_epochs=1)
# once per loop iteration; train/validate/metric/stopping/best-
# checkpoint logic remains exclusively inside the Trainer.
#
# Verification: disposable model/optimizer/scheduler/scaler clones
# + a small real-data subset across THREE epochs (enough to prove
# repeated fit(1) calls, last_model.pt overwrite, and the callback
# firing more than once -- none of which a single-epoch run proves).
# The REAL production_controller is constructed at the end -- wired
# to the real singletons and FULL DataLoaders -- but NOT launched
# here. Launching cfg.EPOCHS of real training is Stage 36's job.

import time as _time

print_section("STAGE 31 - PRODUCTION TRAINING CONTROLLER")
print()


class ProductionTrainingController:
    """
    Composition wrapper that drives a frozen Stage 30 ``Trainer``
    across many epochs and adds production behaviours ``Trainer.fit()``
    intentionally does not own:

    Added on top of Trainer (Trainer source never reopened):
      * ``last_model.pt`` saved unconditionally after every epoch --
        so a crash on a non-improving epoch still leaves a resumable
        checkpoint of the most-recently-completed epoch, not just the
        best one.
      * Callback registration (``register_callback``) so later stages
        observe every completed epoch without editing this class or
        Trainer.
      * ``controller_history`` -- this controller's own copy of per-
        epoch summaries, identical in content to ``trainer.history``
        but owned independently.

    Explicitly NOT this class's responsibility
    -------------------------------------------
    Per-batch math, epoch iteration mechanics, validation, metrics,
    early-stopping policy, best-checkpoint selection, or manifest
    loading. Everything that Stages 21-29.5 and the Trainer (Stage 30)
    already froze stays exactly where it is. This class calls
    ``trainer.fit(num_epochs=1)`` once per loop iteration and does not
    alter any of that call's internal behaviour.

    Migration note
    --------------
    Maps to ``training/train.py -> ProductionTrainingController``.
    The callback contract maps to ``training/callbacks.py``.
    """

    def __init__(self, trainer: "Trainer") -> None:
        self.trainer = trainer
        self.controller_history: List[Dict[str, object]] = []
        self._epoch_end_callbacks: List = []

    @property
    def current_epoch(self) -> int:
        """Delegates to the wrapped Trainer -- single source of truth."""
        return self.trainer.current_epoch

    def register_callback(self, fn) -> object:
        """
        Register a callable ``fn(epoch_summary: dict, trainer: Trainer)``
        to be called after every completed epoch. Returns *fn* unchanged
        so this can be used as a decorator. Multiple callbacks run in
        registration order.
        """
        self._epoch_end_callbacks.append(fn)
        return fn

    def run(self, num_epochs: int, log_interval: int = 50) -> List[Dict[str, object]]:
        """
        Drive up to ``num_epochs`` further epochs of training,
        honouring ``trainer.early_stopping`` and saving
        ``last_model.pt`` after every completed epoch.

        Each iteration calls the frozen ``Trainer.fit(num_epochs=1)``.
        This controller adds: last_model.pt persistence, callback
        firing, and controller_history maintenance. Nothing else.

        Parameters
        ----------
        num_epochs   : int   Maximum additional epochs.
        log_interval : int   Forwarded to each Trainer.fit() call.

        Returns
        -------
        List[Dict[str, object]]
            ``self.controller_history`` after this call.
        """
        require_positive_int(num_epochs, "num_epochs")

        for _ in range(num_epochs):
            if self.trainer.early_stopping.should_stop:
                logger.info(
                    "ProductionTrainingController: EarlyStopping already "
                    "triggered -- halting before epoch %d.",
                    self.trainer.current_epoch + 1,
                )
                break

            full_history = self.trainer.fit(num_epochs=1, log_interval=log_interval)
            latest_summary = dict(full_history[-1])

            # Save last_model.pt unconditionally after every epoch.
            last_ckpt_path = self.trainer.save_checkpoint(filename="last_model.pt")
            latest_summary["last_checkpoint_path"] = str(last_ckpt_path)

            self.controller_history.append(latest_summary)

            for cb in self._epoch_end_callbacks:
                try:
                    cb(latest_summary, self.trainer)
                except Exception as _cb_err:
                    logger.error("Callback %r raised an exception: %s", cb, _cb_err)
                    raise

            if self.trainer.early_stopping.should_stop:
                logger.info(
                    "EarlyStopping triggered after epoch %d -- "
                    "ProductionTrainingController halting.",
                    self.trainer.current_epoch,
                )
                break

        return self.controller_history


# ============================================================
# Engineering verification: 3-epoch real-data-subset smoke test.
# Disposable clones only -- the real Phase 2-5 singletons are
# untouched.
# ============================================================

print("  3-epoch smoke test with disposable clones + real-data subset.")
print("  Real model / optimizer / scheduler / scaler are NOT mutated.")
print()

_S31_SUBSET_BATCHES = 3

_s31_tr_sub = torch.utils.data.Subset(
    train_loader.dataset,
    list(range(min(len(train_loader.dataset), _S31_SUBSET_BATCHES * cfg.BATCH_SIZE))),
)
_s31_va_sub = torch.utils.data.Subset(
    val_loader.dataset,
    list(range(min(len(val_loader.dataset), _S31_SUBSET_BATCHES * cfg.BATCH_SIZE))),
)
_s31_tr_ld = torch.utils.data.DataLoader(
    _s31_tr_sub, batch_size=cfg.BATCH_SIZE, shuffle=True, num_workers=0, drop_last=False,
)
_s31_va_ld = torch.utils.data.DataLoader(
    _s31_va_sub, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=0, drop_last=False,
)

_s31_model     = build_model(cfg.BACKBONE, num_classes=cfg.NUM_CLASSES, pretrained=False, dropout=DROPOUT_PROB)
_s31_model     = freeze_backbone(_s31_model, backbone_name).to(DEVICE)
_s31_optimizer = build_optimizer(_s31_model, optimizer_cfg)
_s31_scheduler = build_scheduler(_s31_optimizer, scheduler_cfg)
_s31_scaler    = build_grad_scaler(amp_cfg)
_s31_es        = EarlyStopping(patience=99, min_delta=0.0)
_s31_ckpt_mgr  = CheckpointManager(checkpoint_dir=cfg.CHECKPOINT_DIR / "_s31_smoke")

_s31_trainer = Trainer(
    model=_s31_model, criterion=criterion_weighted, optimizer=_s31_optimizer,
    scheduler=_s31_scheduler, scaler=_s31_scaler,
    train_loader=_s31_tr_ld, val_loader=_s31_va_ld,
    class_names=class_names, config=cfg, gradient_config=gradient_cfg,
    amp_config=amp_cfg, checkpoint_manager=_s31_ckpt_mgr,
    early_stopping=_s31_es, device=DEVICE,
)
_s31_ctrl = ProductionTrainingController(trainer=_s31_trainer)

_s31_cb_calls: List[Dict] = []
_s31_ctrl.register_callback(lambda s, t: _s31_cb_calls.append(dict(s)))

_s31_t0 = _time.time()
_s31_ctrl.run(num_epochs=1, log_interval=1)
_s31_last_ep1 = _s31_ckpt_mgr.load("last_model.pt")
_s31_ctrl.run(num_epochs=2, log_interval=1)
_s31_last_ep3 = _s31_ckpt_mgr.load("last_model.pt")
_s31_elapsed  = _time.time() - _s31_t0

print()
print("  3-epoch controller history")
print("  " + "-" * 60)
for _e in _s31_ctrl.controller_history:
    print_kv(
        f"epoch {_e['epoch']}",
        f"train_loss={_e['train_loss']:.4f}  val_loss={_e['val_loss']:.4f}  is_best={_e['is_best']}",
    )
print()

checks = [
    ("ProductionTrainingController instantiated",                    isinstance(_s31_ctrl, ProductionTrainingController), ""),
    ("run() advanced exactly 3 epochs total",                         len(_s31_ctrl.controller_history) == 3, str(len(_s31_ctrl.controller_history))),
    ("trainer.current_epoch == 3",                                     _s31_trainer.current_epoch == 3, str(_s31_trainer.current_epoch)),
    ("current_epoch property delegates to trainer",                     _s31_ctrl.current_epoch == _s31_trainer.current_epoch, ""),
    ("Callback fired 3 times (once per epoch)",                          len(_s31_cb_calls) == 3, str(len(_s31_cb_calls))),
    ("Callback epochs == [1, 2, 3]",                                      [c["epoch"] for c in _s31_cb_calls] == [1, 2, 3], str([c["epoch"] for c in _s31_cb_calls])),
    ("last_model.pt epoch==1 after first run(1)",                          _s31_last_ep1["epoch"] == 1, str(_s31_last_ep1["epoch"])),
    ("last_model.pt OVERWRITTEN to epoch==3 after run(2)",                  _s31_last_ep3["epoch"] == 3, str(_s31_last_ep3["epoch"])),
    ("last_checkpoint_path present in every history entry",                  all("last_checkpoint_path" in e for e in _s31_ctrl.controller_history), ""),
    ("All train_losses finite",                                               all(e["train_loss"] == e["train_loss"] for e in _s31_ctrl.controller_history), ""),
    ("All val_losses finite",                                                  all(e["val_loss"] == e["val_loss"] for e in _s31_ctrl.controller_history), ""),
    ("Scheduler advanced (last_epoch > 0)",                                    _s31_scheduler.last_epoch > 0, str(_s31_scheduler.last_epoch)),
    ("Real model untouched (different id)",                                     model is not _s31_model, ""),
    ("Real optimizer untouched (different id)",                                  optimizer is not _s31_optimizer, ""),
    ("Real scheduler untouched (different id)",                                   scheduler is not _s31_scheduler, ""),
    ("Real checkpoint_manager untouched (different dir)",                          checkpoint_manager is not _s31_ckpt_mgr, ""),
    ("EarlyStopping did not fire (patience=99 over 3 epochs)",                     not _s31_trainer.early_stopping.should_stop, ""),
]
for label, passed, detail in checks:
    print_check(label, passed, detail)

print()
print_kv("3-epoch smoke-test wall-clock", f"{_s31_elapsed:.2f}s")
print()

all_passed = all(p for _, p, _ in checks)
print(f"  {sum(1 for _,p,_ in checks if p)} / {len(checks)} checks passed")
print()
if not all_passed:
    raise AssertionError(f"ProductionTrainingController FAILED: {[l for l,p,_ in checks if not p]}")
print("  ALL CHECKS PASSED")
print()

del _s31_model, _s31_optimizer, _s31_scheduler, _s31_scaler
del _s31_tr_sub, _s31_va_sub, _s31_tr_ld, _s31_va_ld
del _s31_last_ep1, _s31_last_ep3

# -- Construct (NOT launch) the real production controller -----
print("  Constructing REAL production controller (not launching)")
print("  " + "-" * 60)

early_stopping = EarlyStopping(patience=cfg.EARLY_STOPPING_PATIENCE, min_delta=0.0)

production_trainer = Trainer(
    model=model, criterion=criterion_weighted, optimizer=optimizer, scheduler=scheduler,
    scaler=scaler, train_loader=train_loader, val_loader=val_loader,
    class_names=class_names, config=cfg, gradient_config=gradient_cfg,
    amp_config=amp_cfg, checkpoint_manager=checkpoint_manager,
    early_stopping=early_stopping, device=DEVICE,
)
production_controller = ProductionTrainingController(trainer=production_trainer)

wiring_checks = [
    ("production_trainer.model   is real singleton",          production_trainer.model is model, ""),
    ("production_trainer.optimizer is real singleton",         production_trainer.optimizer is optimizer, ""),
    ("production_trainer.scheduler is real singleton",          production_trainer.scheduler is scheduler, ""),
    ("production_trainer.scaler  is real singleton",             production_trainer.scaler is scaler, ""),
    ("production_trainer.train_loader is full train_loader",      production_trainer.train_loader is train_loader, ""),
    ("production_trainer.val_loader   is full val_loader",         production_trainer.val_loader is val_loader, ""),
    ("production_controller.current_epoch == 0",                    production_controller.current_epoch == 0, str(production_controller.current_epoch)),
    ("production_controller.controller_history == []",               production_controller.controller_history == [], ""),
]
for label, passed, detail in wiring_checks:
    print_check(label, passed, detail)

all_wired = all(p for _, p, _ in wiring_checks)
if not all_wired:
    raise AssertionError(f"Production controller wiring FAILED: {[l for l,p,_ in wiring_checks if not p]}")
print()
print("  ALL WIRING CHECKS PASSED")
print()

# -- Persist ---------------------------------------------------
production_controller_summary = {
    "project": cfg.PROJECT_NAME, "sprint": "04",
    "phase": "Phase 6 - Production Training Pipeline",
    "stage": 31, "stage_name": "Production Training Controller",
    "smoke_test": {
        "subset_batches": _S31_SUBSET_BATCHES, "epochs_run": 3,
        "wall_clock_seconds": round(_s31_elapsed, 3),
        "checks_passed": sum(1 for _,p,_ in checks if p), "checks_total": len(checks),
    },
    "production_controller": {
        "wired_to_real_singletons": all_wired,
        "train_samples": len(train_loader.dataset),
        "val_samples": len(val_loader.dataset),
        "configured_epochs": cfg.EPOCHS,
        "early_stopping_patience": cfg.EARLY_STOPPING_PATIENCE,
        "launch_status": "constructed_not_run",
    },
    "extension_point": "register_callback(fn: Callable[[dict, Trainer], None])",
    "migration": {"ProductionTrainingController": "training/train.py",
                  "callback contract": "training/callbacks.py"},
}
_pcs_path = cfg.OUTPUT_DIR / "production_controller_summary.json"
with open(_pcs_path, "w") as f:
    _json.dump(production_controller_summary, f, indent=2, default=str)
print_check(f"Saved {_pcs_path.name}", _pcs_path.exists() and _pcs_path.stat().st_size > 0, str(_pcs_path))

print()
print("Stage 31 - Production Training Controller : OK")

  STAGE 31 - PRODUCTION TRAINING CONTROLLER

  3-epoch smoke test with disposable clones + real-data subset.
  Real model / optimizer / scheduler / scaler are NOT mutated.

2026-07-02 00:56:27 | INFO     | visionserveai.sprint04 | Epoch 1 | step 1/2 | loss=1.2631 | running_avg_loss=1.4282 | lr=9.97e-05
2026-07-02 00:56:28 | INFO     | visionserveai.sprint04 | Epoch 1 | step 2/2 | loss=1.4569 | running_avg_loss=1.4378 | lr=9.89e-05
2026-07-02 00:56:30 | INFO     | visionserveai.sprint04 | Epoch 1 complete | train_loss=1.4378 | val_loss=1.5010 | val_macro_auroc=0.4940 | is_best=True | es_counter=0/99
2026-07-02 00:56:32 | INFO     | visionserveai.sprint04 | Epoch 2 | step 1/2 | loss=1.4813 | running_avg_loss=1.3936 | lr=9.76e-05
2026-07-02 00:56:33 | INFO     | visionserveai.sprint04 | Epoch 2 | step 2/2 | loss=1.4519 | running_avg_loss=1.4130 | lr=9.57e-05
2026-07-02 00:56:35 | INFO     | visionserveai.sprint04 | Epoch 2 complete | train_loss=1.4130 | val_loss=1.4791 | val_macro_auroc=0

In [35]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 32: Epoch Logging System
#
# Module: training/logging.py -> EpochLogger
# ============================================================

# -- Purpose ---------------------------------------------------
# A callback registered on ProductionTrainingController (Stage 31)
# that writes a running per-epoch log to ``epoch_logs.json`` after
# EVERY epoch. Unlike Trainer._persist_history (which writes only the
# basic Trainer fields), EpochLogger records additional runtime
# context: UTC timestamp, cumulative elapsed seconds from run start,
# live EarlyStopping state, and the best-epoch metadata. The file is
# always a valid JSON array even after a crash mid-training because it
# is fully rewritten (not appended) after each epoch. On resume, the
# existing entries are loaded first so the log is continuous.
#
# Migration note: maps to ``training/logging.py -> EpochLogger``.

print_section("STAGE 32 - EPOCH LOGGING SYSTEM")
print()


class EpochLogger:
    """
    Callback: append a rich per-epoch log entry to ``epoch_logs.json``.

    Parameters
    ----------
    output_path : Path   Path of the JSON log file.

    On-disk format
    --------------
    A JSON array. Each entry is a flat dict with the keys:
    epoch, timestamp_utc, elapsed_seconds, train_loss, val_loss,
    val_macro_auroc, val_micro_auroc, val_f1_macro, train_lr,
    train_samples_per_second, early_stopping_counter,
    early_stopping_patience, is_best, should_stop,
    best_loss_so_far, best_epoch_so_far, last_checkpoint_path.

    Resume continuity
    -----------------
    If ``output_path`` already exists, its entries are loaded at
    construction time and new epochs are appended, so a resumed run
    never loses prior-epoch log entries.

    Migration note
    --------------
    Maps to ``training/logging.py -> EpochLogger``.
    """

    def __init__(self, output_path: Path) -> None:
        self.output_path = output_path
        self._run_start: float = _time.time()
        self._entries: List[Dict[str, object]] = []
        if output_path.exists() and output_path.stat().st_size > 0:
            try:
                with open(output_path) as f:
                    self._entries = _json.load(f)
                logger.info("EpochLogger: loaded %d existing entries from %s",
                            len(self._entries), output_path)
            except Exception as _err:
                logger.warning("EpochLogger: could not load existing log (%s) -- starting fresh.", _err)
                self._entries = []

    def __call__(self, epoch_summary: Dict[str, object], trainer: "Trainer") -> None:
        """Append one entry and atomically rewrite the JSON file."""
        entry: Dict[str, object] = {
            "epoch":                    epoch_summary.get("epoch"),
            "timestamp_utc":             _time.strftime("%Y-%m-%dT%H:%M:%SZ", _time.gmtime()),
            "elapsed_seconds":            round(_time.time() - self._run_start, 3),
            "train_loss":                  epoch_summary.get("train_loss"),
            "val_loss":                     epoch_summary.get("val_loss"),
            "val_macro_auroc":               epoch_summary.get("val_macro_auroc"),
            "val_micro_auroc":                epoch_summary.get("val_micro_auroc"),
            "val_f1_macro":                    epoch_summary.get("val_f1_macro"),
            "train_lr":                         epoch_summary.get("train_lr"),
            "train_samples_per_second":          epoch_summary.get("train_samples_per_second"),
            "early_stopping_counter":             epoch_summary.get("early_stopping_counter"),
            "early_stopping_patience":             trainer.early_stopping.patience,
            "is_best":                              epoch_summary.get("is_best"),
            "should_stop":                           epoch_summary.get("should_stop"),
            "best_loss_so_far":                       (
                trainer.early_stopping.best_loss
                if trainer.early_stopping.best_loss != float("inf")
                else None
            ),
            "best_epoch_so_far":                        trainer.early_stopping.best_epoch,
            "last_checkpoint_path":                      epoch_summary.get("last_checkpoint_path"),
        }
        self._entries.append(entry)
        with open(self.output_path, "w") as f:
            _json.dump(self._entries, f, indent=2, default=str)

    @property
    def entries(self) -> List[Dict[str, object]]:
        return list(self._entries)

    def __len__(self) -> int:
        return len(self._entries)


# ============================================================
# Engineering verification: smoke-test on a disposable 2-epoch run.
# ============================================================

_S32_SUBSET_BATCHES = 2
_s32_tr_sub = torch.utils.data.Subset(
    train_loader.dataset,
    list(range(min(len(train_loader.dataset), _S32_SUBSET_BATCHES * cfg.BATCH_SIZE))),
)
_s32_va_sub = torch.utils.data.Subset(
    val_loader.dataset,
    list(range(min(len(val_loader.dataset), _S32_SUBSET_BATCHES * cfg.BATCH_SIZE))),
)
_s32_tr_ld = torch.utils.data.DataLoader(_s32_tr_sub, batch_size=cfg.BATCH_SIZE,
                                          shuffle=True, num_workers=0, drop_last=False)
_s32_va_ld = torch.utils.data.DataLoader(_s32_va_sub, batch_size=cfg.BATCH_SIZE,
                                          shuffle=False, num_workers=0, drop_last=False)

_s32_model     = build_model(cfg.BACKBONE, num_classes=cfg.NUM_CLASSES, pretrained=False, dropout=DROPOUT_PROB)
_s32_model     = freeze_backbone(_s32_model, backbone_name).to(DEVICE)
_s32_optimizer = build_optimizer(_s32_model, optimizer_cfg)
_s32_scheduler = build_scheduler(_s32_optimizer, scheduler_cfg)
_s32_scaler    = build_grad_scaler(amp_cfg)
_s32_ckpt_mgr  = CheckpointManager(checkpoint_dir=cfg.CHECKPOINT_DIR / "_s32_smoke")

_s32_trainer = Trainer(
    model=_s32_model, criterion=criterion_weighted, optimizer=_s32_optimizer,
    scheduler=_s32_scheduler, scaler=_s32_scaler,
    train_loader=_s32_tr_ld, val_loader=_s32_va_ld,
    class_names=class_names, config=cfg, gradient_config=gradient_cfg,
    amp_config=amp_cfg, checkpoint_manager=_s32_ckpt_mgr,
    early_stopping=EarlyStopping(patience=99), device=DEVICE,
)
_s32_ctrl = ProductionTrainingController(trainer=_s32_trainer)

_s32_log_path = cfg.METRICS_DIR / "_stage32_smoke_epoch_logs.json"
_s32_logger   = EpochLogger(output_path=_s32_log_path)
_s32_ctrl.register_callback(_s32_logger)
_s32_ctrl.run(num_epochs=2, log_interval=1)

_s32_entries = _s32_logger.entries

_EXPECTED_LOG_KEYS = {
    "epoch", "timestamp_utc", "elapsed_seconds", "train_loss", "val_loss",
    "val_macro_auroc", "val_micro_auroc", "val_f1_macro", "train_lr",
    "train_samples_per_second", "early_stopping_counter", "early_stopping_patience",
    "is_best", "should_stop", "best_loss_so_far", "best_epoch_so_far",
    "last_checkpoint_path",
}

checks = [
    ("EpochLogger is callable",                                  callable(_s32_logger), ""),
    ("epoch_logs file written",                                   _s32_log_path.exists() and _s32_log_path.stat().st_size > 0, str(_s32_log_path)),
    ("Exactly 2 entries logged (one per epoch)",                   len(_s32_entries) == 2, str(len(_s32_entries))),
    ("epochs == [1, 2]",                                            [e["epoch"] for e in _s32_entries] == [1, 2], str([e["epoch"] for e in _s32_entries])),
    ("All required keys present in entry 1",                         _EXPECTED_LOG_KEYS.issubset(set(_s32_entries[0])), str(set(_s32_entries[0].keys()) - _EXPECTED_LOG_KEYS)),
    ("timestamp_utc is a non-empty string",                           isinstance(_s32_entries[0]["timestamp_utc"], str) and len(_s32_entries[0]["timestamp_utc"]) > 0, ""),
    ("elapsed_seconds > 0 for both entries",                           all(e["elapsed_seconds"] > 0 for e in _s32_entries), ""),
    ("elapsed_seconds[1] > elapsed_seconds[0] (monotonic)",            _s32_entries[1]["elapsed_seconds"] > _s32_entries[0]["elapsed_seconds"], ""),
    ("train_loss is finite in both entries",                             all(e["train_loss"] == e["train_loss"] for e in _s32_entries), ""),
    ("val_loss is finite in both entries",                                all(e["val_loss"] == e["val_loss"] for e in _s32_entries), ""),
    ("is_best is bool in both entries",                                    all(isinstance(e["is_best"], bool) for e in _s32_entries), ""),
    ("early_stopping_patience == 99 (carried from EarlyStopping)",          all(e["early_stopping_patience"] == 99 for e in _s32_entries), ""),
    ("Log file is valid JSON (re-parseable)",                                 bool(_json.loads(_s32_log_path.read_text())), ""),
    ("Smoke log file is separate from production path",                        _s32_log_path != cfg.METRICS_DIR / "epoch_logs.json", ""),
    ("EpochLogger.__len__ == 2",                                               len(_s32_logger) == 2, str(len(_s32_logger))),
    ("Resume: loading smoke log at construction gives 2 entries",               len(EpochLogger(output_path=_s32_log_path)) == 2, ""),
]
for label, passed, detail in checks:
    print_check(label, passed, detail)

del _s32_model, _s32_optimizer, _s32_scheduler, _s32_scaler
del _s32_tr_sub, _s32_va_sub, _s32_tr_ld, _s32_va_ld

all_passed = all(p for _, p, _ in checks)
print()
print(f"  {sum(1 for _,p,_ in checks if p)} / {len(checks)} checks passed")
print()
if not all_passed:
    raise AssertionError(f"EpochLogger verification FAILED: {[l for l,p,_ in checks if not p]}")
print("  ALL CHECKS PASSED")
print()

# -- Register REAL EpochLogger on production_controller --------
_epoch_logs_path = cfg.METRICS_DIR / "epoch_logs.json"
epoch_logger     = EpochLogger(output_path=_epoch_logs_path)
production_controller.register_callback(epoch_logger)

print_check(
    "Real EpochLogger registered on production_controller",
    epoch_logger in production_controller._epoch_end_callbacks, "",
)
print_kv("epoch_logs.json destination", _epoch_logs_path)
print()
print("Stage 32 - Epoch Logging System : OK")

  STAGE 32 - EPOCH LOGGING SYSTEM

2026-07-02 00:56:42 | INFO     | visionserveai.sprint04 | Epoch 1 | step 1/1 | loss=1.6135 | running_avg_loss=1.6608 | lr=9.97e-05
2026-07-02 00:56:43 | INFO     | visionserveai.sprint04 | Epoch 1 complete | train_loss=1.6608 | val_loss=1.7342 | val_macro_auroc=0.4733 | is_best=True | es_counter=0/99
2026-07-02 00:56:45 | INFO     | visionserveai.sprint04 | Epoch 2 | step 1/1 | loss=1.2841 | running_avg_loss=1.6710 | lr=9.89e-05
2026-07-02 00:56:47 | INFO     | visionserveai.sprint04 | Epoch 2 complete | train_loss=1.6710 | val_loss=1.6770 | val_macro_auroc=0.5037 | is_best=True | es_counter=0/99
2026-07-02 00:56:47 | INFO     | visionserveai.sprint04 | EpochLogger: loaded 2 existing entries from /kaggle/working/visionserveai/sprint04/metrics/_stage32_smoke_epoch_logs.json
  ✔  EpochLogger is callable
  ✔  epoch_logs file written  (/kaggle/working/visionserveai/sprint04/metrics/_stage32_smoke_epoch_logs.json)
  ✔  Exactly 2 entries logged (one per epo

In [36]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 33: Training History Manager
#
# Module: training/history.py -> TrainingHistoryManager
# ============================================================

# -- Purpose ---------------------------------------------------
# Trainer._persist_history (frozen Stage 30) writes training_history.json
# with the basic trainer fields after every epoch.  TrainingHistoryManager
# is a callback that runs AFTER each Trainer.fit(1) call completes (so
# always AFTER _persist_history) and rewrites training_history.json with
# an ENHANCED version: every original Trainer field PLUS timestamp_utc,
# best_loss_so_far, best_epoch_so_far, and current_lr. This is a pure
# extension (same file name, richer content) -- Trainer is never reopened.
# On resume, the existing history is loaded first to maintain continuity.
#
# Migration note: maps to ``training/history.py -> TrainingHistoryManager``.

print_section("STAGE 33 - TRAINING HISTORY MANAGER")
print()


class TrainingHistoryManager:
    """
    Callback that overwrites ``training_history.json`` after every epoch
    with an enhanced version of the Trainer's history.

    Trainer._persist_history still fires first (inside fit()); this
    callback replaces its output with the enriched version seconds later.
    The file is fully rewritten each time (not appended), keeping it a
    clean, loadable JSON array at all times.

    Migration note
    --------------
    Maps to ``training/history.py -> TrainingHistoryManager``.
    """

    def __init__(self, output_path: Path) -> None:
        self.output_path = output_path
        self._history: List[Dict[str, object]] = []
        if output_path.exists() and output_path.stat().st_size > 0:
            try:
                with open(output_path) as f:
                    self._history = _json.load(f)
                logger.info("TrainingHistoryManager: loaded %d prior epochs from %s",
                            len(self._history), output_path)
            except Exception as _err:
                logger.warning("TrainingHistoryManager: could not load existing history (%s).", _err)
                self._history = []

    def __call__(self, epoch_summary: Dict[str, object], trainer: "Trainer") -> None:
        """Append enhanced entry and rewrite the file."""
        entry: Dict[str, object] = dict(epoch_summary)
        entry["timestamp_utc"]    = _time.strftime("%Y-%m-%dT%H:%M:%SZ", _time.gmtime())
        entry["best_loss_so_far"]  = (
            trainer.early_stopping.best_loss
            if trainer.early_stopping.best_loss != float("inf")
            else None
        )
        entry["best_epoch_so_far"]  = trainer.early_stopping.best_epoch
        entry["current_lr"]           = trainer.optimizer.param_groups[0]["lr"]
        entry["es_counter"]            = trainer.early_stopping.counter
        entry["es_patience"]            = trainer.early_stopping.patience
        self._history.append(entry)
        with open(self.output_path, "w") as f:
            _json.dump(self._history, f, indent=2, default=str)

    @property
    def history(self) -> List[Dict[str, object]]:
        return list(self._history)

    def best_entry(self) -> Optional[Dict[str, object]]:
        """Return the last entry marked is_best, or None."""
        bests = [e for e in self._history if e.get("is_best")]
        return bests[-1] if bests else None

    def __len__(self) -> int:
        return len(self._history)


# ============================================================
# Engineering verification: 2-epoch smoke test.
# ============================================================

_S33_SUBSET_BATCHES = 2
_s33_tr_sub = torch.utils.data.Subset(
    train_loader.dataset,
    list(range(min(len(train_loader.dataset), _S33_SUBSET_BATCHES * cfg.BATCH_SIZE))),
)
_s33_va_sub = torch.utils.data.Subset(
    val_loader.dataset,
    list(range(min(len(val_loader.dataset), _S33_SUBSET_BATCHES * cfg.BATCH_SIZE))),
)
_s33_tr_ld = torch.utils.data.DataLoader(_s33_tr_sub, batch_size=cfg.BATCH_SIZE,
                                          shuffle=True, num_workers=0, drop_last=False)
_s33_va_ld = torch.utils.data.DataLoader(_s33_va_sub, batch_size=cfg.BATCH_SIZE,
                                          shuffle=False, num_workers=0, drop_last=False)

_s33_model     = build_model(cfg.BACKBONE, num_classes=cfg.NUM_CLASSES, pretrained=False, dropout=DROPOUT_PROB)
_s33_model     = freeze_backbone(_s33_model, backbone_name).to(DEVICE)
_s33_optimizer = build_optimizer(_s33_model, optimizer_cfg)
_s33_scheduler = build_scheduler(_s33_optimizer, scheduler_cfg)
_s33_scaler    = build_grad_scaler(amp_cfg)
_s33_ckpt_mgr  = CheckpointManager(checkpoint_dir=cfg.CHECKPOINT_DIR / "_s33_smoke")

_s33_trainer = Trainer(
    model=_s33_model, criterion=criterion_weighted, optimizer=_s33_optimizer,
    scheduler=_s33_scheduler, scaler=_s33_scaler,
    train_loader=_s33_tr_ld, val_loader=_s33_va_ld,
    class_names=class_names, config=cfg, gradient_config=gradient_cfg,
    amp_config=amp_cfg, checkpoint_manager=_s33_ckpt_mgr,
    early_stopping=EarlyStopping(patience=99), device=DEVICE,
)
_s33_ctrl = ProductionTrainingController(trainer=_s33_trainer)

_s33_hist_path = cfg.METRICS_DIR / "_stage33_smoke_training_history.json"

if _s33_hist_path.exists():
    _s33_hist_path.unlink()
    
_s33_hist_mgr  = TrainingHistoryManager(output_path=_s33_hist_path)
_s33_ctrl.register_callback(_s33_hist_mgr)
_s33_ctrl.run(num_epochs=2, log_interval=1)

_s33_hist = _s33_hist_mgr.history
_EXTRA_KEYS = {"timestamp_utc", "best_loss_so_far", "best_epoch_so_far",
               "current_lr", "es_counter", "es_patience"}
_TRAINER_KEYS = {"epoch", "train_loss", "val_loss", "val_macro_auroc",
                 "is_best", "should_stop", "early_stopping_counter"}

checks = [
    ("TrainingHistoryManager is callable",                         callable(_s33_hist_mgr), ""),
    ("History file written",                                        _s33_hist_path.exists() and _s33_hist_path.stat().st_size > 0, str(_s33_hist_path)),
    ("Exactly 2 entries (one per epoch)",                            len(_s33_hist) == 2, str(len(_s33_hist))),
    ("epochs == [1, 2]",                                              [e["epoch"] for e in _s33_hist] == [1, 2], ""),
    ("All original Trainer keys present in entry 1",                   _TRAINER_KEYS.issubset(set(_s33_hist[0])), str(_TRAINER_KEYS - set(_s33_hist[0]))),
    ("All enhanced keys present in entry 1",                            _EXTRA_KEYS.issubset(set(_s33_hist[0])), str(_EXTRA_KEYS - set(_s33_hist[0]))),
    ("timestamp_utc is a non-empty string",                              isinstance(_s33_hist[0]["timestamp_utc"], str) and len(_s33_hist[0]["timestamp_utc"]) > 0, ""),
    ("current_lr > 0 in both entries",                                    all(e["current_lr"] > 0 for e in _s33_hist), ""),
    ("es_patience == 99 (carried from EarlyStopping)",                     all(e["es_patience"] == 99 for e in _s33_hist), ""),
    ("best_loss_so_far is finite or None",                                  all(
        e["best_loss_so_far"] is None or e["best_loss_so_far"] == e["best_loss_so_far"]
        for e in _s33_hist), ""),
    ("History file is valid JSON (re-parseable)",                            bool(_json.loads(_s33_hist_path.read_text())), ""),
    ("best_entry() returns the most-recent is_best==True entry or None",     (
        _s33_hist_mgr.best_entry() is None or
        _s33_hist_mgr.best_entry().get("is_best") is True), ""),
    ("TrainingHistoryManager.__len__ == 2",                                    len(_s33_hist_mgr) == 2, ""),
    ("Resume: new instance loads 2 existing entries",                           len(TrainingHistoryManager(output_path=_s33_hist_path)) == 2, ""),
]
for label, passed, detail in checks:
    print_check(label, passed, detail)

del _s33_model, _s33_optimizer, _s33_scheduler, _s33_scaler
del _s33_tr_sub, _s33_va_sub, _s33_tr_ld, _s33_va_ld

all_passed = all(p for _, p, _ in checks)
print()
print(f"  {sum(1 for _,p,_ in checks if p)} / {len(checks)} checks passed")
if not all_passed:
    raise AssertionError(f"TrainingHistoryManager FAILED: {[l for l,p,_ in checks if not p]}")
print("  ALL CHECKS PASSED")
print()

# -- Register REAL TrainingHistoryManager on production_controller --
history_manager = TrainingHistoryManager(output_path=cfg.METRICS_DIR / "training_history.json")
production_controller.register_callback(history_manager)

print_check(
    "Real TrainingHistoryManager registered on production_controller",
    history_manager in production_controller._epoch_end_callbacks, "",
)
print_kv("training_history.json destination", cfg.METRICS_DIR / "training_history.json")
print()
print("Stage 33 - Training History Manager : OK")

  STAGE 33 - TRAINING HISTORY MANAGER

2026-07-02 00:56:49 | INFO     | visionserveai.sprint04 | Epoch 1 | step 1/1 | loss=1.5498 | running_avg_loss=1.6457 | lr=9.97e-05
2026-07-02 00:56:50 | INFO     | visionserveai.sprint04 | Epoch 1 complete | train_loss=1.6457 | val_loss=1.5689 | val_macro_auroc=0.4887 | is_best=True | es_counter=0/99
2026-07-02 00:56:52 | INFO     | visionserveai.sprint04 | Epoch 2 | step 1/1 | loss=1.4601 | running_avg_loss=1.6113 | lr=9.89e-05
2026-07-02 00:56:53 | INFO     | visionserveai.sprint04 | Epoch 2 complete | train_loss=1.6113 | val_loss=1.6149 | val_macro_auroc=0.4196 | is_best=False | es_counter=1/99
2026-07-02 00:56:53 | INFO     | visionserveai.sprint04 | TrainingHistoryManager: loaded 2 prior epochs from /kaggle/working/visionserveai/sprint04/metrics/_stage33_smoke_training_history.json
  ✔  TrainingHistoryManager is callable
  ✔  History file written  (/kaggle/working/visionserveai/sprint04/metrics/_stage33_smoke_training_history.json)
  ✔  Exact

In [37]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 34: Best Model Manager
#
# Module: training/callbacks.py -> BestModelManager
# ============================================================

# -- Purpose ---------------------------------------------------
# Trainer.fit() already saves best_model.pt when EarlyStopping.save_best
# is True (Stage 30 frozen behaviour). BestModelManager is a callback
# that adds metadata persistence: whenever a new best is recorded,
# it writes best_model_metadata.json capturing the epoch number, UTC
# timestamp, validation metrics at the best epoch, and the checkpoint
# path. This metadata is the canonical reference consumed by Stage 37
# (Training Report Generator) and any downstream model-registry tool
# without having to deserialise the .pt file.
#
# This callback does NOT call save() -- Trainer already did it. It
# only records WHAT was saved and WHEN.
#
# Migration note: maps to ``training/callbacks.py -> BestModelManager``.

print_section("STAGE 34 - BEST MODEL MANAGER")
print()


class BestModelManager:
    """
    Callback: write ``best_model_metadata.json`` whenever a new best
    validation checkpoint is recorded by the frozen Trainer.

    The Trainer already saves ``best_model.pt``; this callback only
    persists structured metadata about that event. On resume, the
    existing metadata is loaded so the record is not lost.

    Migration note
    --------------
    Maps to ``training/callbacks.py -> BestModelManager``.
    """

    def __init__(self, metadata_path: Path, checkpoint_dir: Path) -> None:
        self.metadata_path  = metadata_path
        self.checkpoint_dir = checkpoint_dir
        self._best: Optional[Dict[str, object]] = None
        if metadata_path.exists() and metadata_path.stat().st_size > 0:
            try:
                with open(metadata_path) as f:
                    self._best = _json.load(f)
                logger.info("BestModelManager: loaded existing best-model metadata (epoch %s).",
                            self._best.get("epoch"))
            except Exception as _err:
                logger.warning("BestModelManager: could not load existing metadata (%s).", _err)

    def __call__(self, epoch_summary: Dict[str, object], trainer: "Trainer") -> None:
        """If this epoch is the new best, write metadata."""
        if not epoch_summary.get("is_best"):
            return
        self._best = {
            "epoch":                epoch_summary["epoch"],
            "timestamp_utc":         _time.strftime("%Y-%m-%dT%H:%M:%SZ", _time.gmtime()),
            "val_loss":               epoch_summary.get("val_loss"),
            "val_macro_auroc":         epoch_summary.get("val_macro_auroc"),
            "val_micro_auroc":          epoch_summary.get("val_micro_auroc"),
            "val_f1_macro":              epoch_summary.get("val_f1_macro"),
            "train_lr_at_best":           epoch_summary.get("train_lr"),
            "early_stopping_counter":      epoch_summary.get("early_stopping_counter"),
            "checkpoint_path":              str(self.checkpoint_dir / "best_model.pt"),
            "best_val_loss_confirmed":       trainer.early_stopping.best_loss,
        }
        with open(self.metadata_path, "w") as f:
            _json.dump(self._best, f, indent=2, default=str)

    @property
    def best_metadata(self) -> Optional[Dict[str, object]]:
        return self._best

    def best_epoch(self) -> Optional[int]:
        return self._best["epoch"] if self._best else None


# ============================================================
# Engineering verification: 3-epoch smoke test.
# Epoch 1 is always the best (nothing to compare against); subsequent
# epochs test the "not best, file unchanged" path.
# ============================================================

_S34_SUBSET_BATCHES = 2
_s34_tr_sub = torch.utils.data.Subset(
    train_loader.dataset,
    list(range(min(len(train_loader.dataset), _S34_SUBSET_BATCHES * cfg.BATCH_SIZE))),
)
_s34_va_sub = torch.utils.data.Subset(
    val_loader.dataset,
    list(range(min(len(val_loader.dataset), _S34_SUBSET_BATCHES * cfg.BATCH_SIZE))),
)
_s34_tr_ld = torch.utils.data.DataLoader(_s34_tr_sub, batch_size=cfg.BATCH_SIZE,
                                          shuffle=True, num_workers=0, drop_last=False)
_s34_va_ld = torch.utils.data.DataLoader(_s34_va_sub, batch_size=cfg.BATCH_SIZE,
                                          shuffle=False, num_workers=0, drop_last=False)

_s34_model     = build_model(cfg.BACKBONE, num_classes=cfg.NUM_CLASSES, pretrained=False, dropout=DROPOUT_PROB)
_s34_model     = freeze_backbone(_s34_model, backbone_name).to(DEVICE)
_s34_optimizer = build_optimizer(_s34_model, optimizer_cfg)
_s34_scheduler = build_scheduler(_s34_optimizer, scheduler_cfg)
_s34_scaler    = build_grad_scaler(amp_cfg)
_s34_ckpt_mgr  = CheckpointManager(checkpoint_dir=cfg.CHECKPOINT_DIR / "_s34_smoke")

_s34_trainer = Trainer(
    model=_s34_model, criterion=criterion_weighted, optimizer=_s34_optimizer,
    scheduler=_s34_scheduler, scaler=_s34_scaler,
    train_loader=_s34_tr_ld, val_loader=_s34_va_ld,
    class_names=class_names, config=cfg, gradient_config=gradient_cfg,
    amp_config=amp_cfg, checkpoint_manager=_s34_ckpt_mgr,
    early_stopping=EarlyStopping(patience=99), device=DEVICE,
)
_s34_ctrl = ProductionTrainingController(trainer=_s34_trainer)

_s34_meta_path = cfg.METRICS_DIR / "_stage34_smoke_best_model_metadata.json"
_s34_bmm       = BestModelManager(metadata_path=_s34_meta_path, checkpoint_dir=_s34_ckpt_mgr.checkpoint_dir)
_s34_ctrl.register_callback(_s34_bmm)
_s34_ctrl.run(num_epochs=3, log_interval=1)

_s34_meta = _s34_bmm.best_metadata
_s34_meta_from_disk = None
if _s34_meta_path.exists():
    with open(_s34_meta_path) as f:
        _s34_meta_from_disk = _json.load(f)

_BEST_META_KEYS = {
    "epoch", "timestamp_utc", "val_loss", "val_macro_auroc",
    "val_micro_auroc", "val_f1_macro", "train_lr_at_best",
    "early_stopping_counter", "checkpoint_path", "best_val_loss_confirmed",
}

checks = [
    ("BestModelManager is callable",                                    callable(_s34_bmm), ""),
    ("best_model_metadata.json written (at least one best epoch)",       _s34_meta_path.exists() and _s34_meta_path.stat().st_size > 0, ""),
    ("best_metadata property returns a dict",                             isinstance(_s34_meta, dict), ""),
    ("All required metadata keys present",                                 _BEST_META_KEYS.issubset(set(_s34_meta)), str(_BEST_META_KEYS - set(_s34_meta))),
    ("best_epoch() returns an int",                                         isinstance(_s34_bmm.best_epoch(), int), str(_s34_bmm.best_epoch())),
    ("checkpoint_path points to correct filename (best_model.pt)",           str(_s34_meta["checkpoint_path"]).endswith("best_model.pt"), str(_s34_meta["checkpoint_path"])),
    ("val_loss in metadata is finite",                                         _s34_meta["val_loss"] == _s34_meta["val_loss"], str(_s34_meta["val_loss"])),
    ("best_val_loss_confirmed matches EarlyStopping.best_loss",                abs(_s34_meta["best_val_loss_confirmed"] - _s34_trainer.early_stopping.best_loss) < 1e-6, ""),
    ("Metadata file is valid JSON (re-parseable)",                              bool(_json.loads(_s34_meta_path.read_text())), ""),
    ("In-memory metadata == on-disk metadata",                                   _s34_meta == _s34_meta_from_disk, ""),
    ("Resume: new instance loads existing metadata",                              BestModelManager(metadata_path=_s34_meta_path, checkpoint_dir=_s34_ckpt_mgr.checkpoint_dir).best_metadata is not None, ""),
    ("Non-best epochs do not overwrite with wrong epoch",                          _s34_bmm.best_epoch() == _s34_meta["epoch"], ""),
]
for label, passed, detail in checks:
    print_check(label, passed, detail)

del _s34_model, _s34_optimizer, _s34_scheduler, _s34_scaler
del _s34_tr_sub, _s34_va_sub, _s34_tr_ld, _s34_va_ld

all_passed = all(p for _, p, _ in checks)
print()
print(f"  {sum(1 for _,p,_ in checks if p)} / {len(checks)} checks passed")
if not all_passed:
    raise AssertionError(f"BestModelManager FAILED: {[l for l,p,_ in checks if not p]}")
print("  ALL CHECKS PASSED")
print()

# -- Register REAL BestModelManager on production_controller ---
best_model_manager = BestModelManager(
    metadata_path=cfg.METRICS_DIR / "best_model_metadata.json",
    checkpoint_dir=cfg.CHECKPOINT_DIR,
)
production_controller.register_callback(best_model_manager)

print_check(
    "Real BestModelManager registered on production_controller",
    best_model_manager in production_controller._epoch_end_callbacks, "",
)
print_kv("best_model_metadata.json destination", cfg.METRICS_DIR / "best_model_metadata.json")
print()
print("Stage 34 - Best Model Manager : OK")

  STAGE 34 - BEST MODEL MANAGER

2026-07-02 00:56:55 | INFO     | visionserveai.sprint04 | Epoch 1 | step 1/1 | loss=1.8305 | running_avg_loss=1.5228 | lr=9.97e-05
2026-07-02 00:56:57 | INFO     | visionserveai.sprint04 | Epoch 1 complete | train_loss=1.5228 | val_loss=1.6936 | val_macro_auroc=0.6137 | is_best=True | es_counter=0/99
2026-07-02 00:56:58 | INFO     | visionserveai.sprint04 | Epoch 2 | step 1/1 | loss=1.5183 | running_avg_loss=1.5330 | lr=9.89e-05
2026-07-02 00:57:00 | INFO     | visionserveai.sprint04 | Epoch 2 complete | train_loss=1.5330 | val_loss=1.6762 | val_macro_auroc=0.5973 | is_best=True | es_counter=0/99
2026-07-02 00:57:01 | INFO     | visionserveai.sprint04 | Epoch 3 | step 1/1 | loss=1.4768 | running_avg_loss=1.5023 | lr=9.76e-05
2026-07-02 00:57:03 | INFO     | visionserveai.sprint04 | Epoch 3 complete | train_loss=1.5023 | val_loss=1.6709 | val_macro_auroc=0.4722 | is_best=True | es_counter=0/99
2026-07-02 00:57:03 | INFO     | visionserveai.sprint04 | Bes

In [38]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 35: Last Checkpoint Manager
#
# Module: training/callbacks.py -> LastCheckpointManager
# ============================================================

# -- Purpose ---------------------------------------------------
# ProductionTrainingController (Stage 31) already writes last_model.pt
# after every epoch. LastCheckpointManager is a callback that adds the
# companion JSON record: ``checkpoint_summary.json`` (the production
# version -- overwrites the Phase 4 verification artifact written in
# Stage 23) updated after every epoch with: total epochs completed,
# last epoch, best epoch, checkpoint paths, EarlyStopping state, and
# last validation metrics. This is the primary status file for tooling
# that monitors a long-running Kaggle session.
#
# Migration note: maps to ``training/callbacks.py -> LastCheckpointManager``.

from collections import Counter

print_section("STAGE 35 - LAST CHECKPOINT MANAGER")
print()


class LastCheckpointManager:
    """
    Callback: rewrite ``checkpoint_summary.json`` after every epoch
    with the current training state.

    This is the production companion to last_model.pt -- every epoch
    that overwrites the .pt file also updates this JSON so external
    tooling can read training status without loading PyTorch tensors.
    On resume, the prior ``total_epochs_completed`` is loaded so the
    cumulative counter is continuous across interrupted runs.

    Migration note
    --------------
    Maps to ``training/callbacks.py -> LastCheckpointManager``.
    """

    def __init__(self, summary_path: Path, checkpoint_dir: Path) -> None:
        self.summary_path   = summary_path
        self.checkpoint_dir = checkpoint_dir
        self._total_completed: int = 0
        self._best_epoch: Optional[int] = None
        if summary_path.exists() and summary_path.stat().st_size > 0:
            try:
                with open(summary_path) as f:
                    _prior = _json.load(f)
                self._total_completed = _prior.get("total_epochs_completed", 0)
                self._best_epoch      = _prior.get("best_epoch")
                logger.info(
                    "LastCheckpointManager: resumed from %d prior epochs, best_epoch=%s.",
                    self._total_completed, self._best_epoch,
                )
            except Exception as _err:
                logger.warning("LastCheckpointManager: could not load prior summary (%s).", _err)

    def __call__(self, epoch_summary: Dict[str, object], trainer: "Trainer") -> None:
        self._total_completed += 1
        if epoch_summary.get("is_best"):
            self._best_epoch = epoch_summary["epoch"]

        summary = {
            "project":                  trainer.config.PROJECT_NAME,
            "sprint":                    "04",
            "phase":                      "Phase 6 - Production Training Pipeline",
            "total_epochs_completed":      self._total_completed,
            "configured_total_epochs":      trainer.config.EPOCHS,
            "epochs_remaining":              max(0, trainer.config.EPOCHS - trainer.current_epoch),
            "last_epoch":                     epoch_summary["epoch"],
            "best_epoch":                      self._best_epoch,
            "last_checkpoint_path":             str(self.checkpoint_dir / "last_model.pt"),
            "best_checkpoint_path":              str(self.checkpoint_dir / "best_model.pt"),
            "early_stopping_counter":             epoch_summary.get("early_stopping_counter"),
            "early_stopping_patience":             trainer.early_stopping.patience,
            "should_stop":                          epoch_summary.get("should_stop"),
            "last_val_loss":                         epoch_summary.get("val_loss"),
            "last_val_macro_auroc":                   epoch_summary.get("val_macro_auroc"),
            "last_train_loss":                         epoch_summary.get("train_loss"),
            "updated_at_utc":                           _time.strftime("%Y-%m-%dT%H:%M:%SZ", _time.gmtime()),
        }
        with open(self.summary_path, "w") as f:
            _json.dump(summary, f, indent=2, default=str)

    def current_summary(self) -> Optional[Dict[str, object]]:
        if self.summary_path.exists() and self.summary_path.stat().st_size > 0:
            with open(self.summary_path) as f:
                return _json.load(f)
        return None


# ============================================================
# Engineering verification: 3-epoch smoke test (also tests
# cumulative total_epochs_completed across two separate run() calls,
# mimicking a real session-resume scenario).
# ============================================================

_S35_SUBSET_BATCHES = 2
_s35_tr_sub = torch.utils.data.Subset(
    train_loader.dataset,
    list(range(min(len(train_loader.dataset), _S35_SUBSET_BATCHES * cfg.BATCH_SIZE))),
)
_s35_va_sub = torch.utils.data.Subset(
    val_loader.dataset,
    list(range(min(len(val_loader.dataset), _S35_SUBSET_BATCHES * cfg.BATCH_SIZE))),
)
_s35_tr_ld = torch.utils.data.DataLoader(_s35_tr_sub, batch_size=cfg.BATCH_SIZE,
                                          shuffle=True, num_workers=0, drop_last=False)
_s35_va_ld = torch.utils.data.DataLoader(_s35_va_sub, batch_size=cfg.BATCH_SIZE,
                                          shuffle=False, num_workers=0, drop_last=False)

_s35_model     = build_model(cfg.BACKBONE, num_classes=cfg.NUM_CLASSES, pretrained=False, dropout=DROPOUT_PROB)
_s35_model     = freeze_backbone(_s35_model, backbone_name).to(DEVICE)
_s35_optimizer = build_optimizer(_s35_model, optimizer_cfg)
_s35_scheduler = build_scheduler(_s35_optimizer, scheduler_cfg)
_s35_scaler    = build_grad_scaler(amp_cfg)
_s35_ckpt_mgr  = CheckpointManager(checkpoint_dir=cfg.CHECKPOINT_DIR / "_s35_smoke")

_s35_trainer = Trainer(
    model=_s35_model, criterion=criterion_weighted, optimizer=_s35_optimizer,
    scheduler=_s35_scheduler, scaler=_s35_scaler,
    train_loader=_s35_tr_ld, val_loader=_s35_va_ld,
    class_names=class_names, config=cfg, gradient_config=gradient_cfg,
    amp_config=amp_cfg, checkpoint_manager=_s35_ckpt_mgr,
    early_stopping=EarlyStopping(patience=99), device=DEVICE,
)
_s35_ctrl = ProductionTrainingController(trainer=_s35_trainer)

_s35_ckpt_summary_path = cfg.METRICS_DIR / "_stage35_smoke_checkpoint_summary.json"

if _s35_ckpt_summary_path.exists():
    _s35_ckpt_summary_path.unlink()
    
_s35_lcm = LastCheckpointManager(
    summary_path=_s35_ckpt_summary_path,
    checkpoint_dir=_s35_ckpt_mgr.checkpoint_dir,
)
_s35_ctrl.register_callback(_s35_lcm)

# Run 1 epoch, then simulate resume by constructing a NEW LCM from the
# same file and running 2 more epochs through the same controller.
_s35_ctrl.run(num_epochs=1, log_interval=1)
_s35_summary_after_ep1 = _s35_lcm.current_summary()

_s35_lcm_resumed = LastCheckpointManager(
    summary_path=_s35_ckpt_summary_path,
    checkpoint_dir=_s35_ckpt_mgr.checkpoint_dir,
)

# Capture resume state BEFORE any more epochs run
_s35_resume_count_before_training = _s35_lcm_resumed._total_completed

_s35_ctrl._epoch_end_callbacks[-1] = _s35_lcm_resumed

_s35_ctrl.run(num_epochs=2, log_interval=1)

_s35_summary_after_ep3 = _s35_lcm_resumed.current_summary()

_CKPT_SUMMARY_KEYS = {
    "project", "sprint", "phase", "total_epochs_completed", "configured_total_epochs",
    "epochs_remaining", "last_epoch", "best_epoch", "last_checkpoint_path",
    "best_checkpoint_path", "early_stopping_counter", "early_stopping_patience",
    "should_stop", "last_val_loss", "last_val_macro_auroc", "last_train_loss", "updated_at_utc",
}

checks = [
    ("LastCheckpointManager is callable",                                     callable(_s35_lcm), ""),
    ("checkpoint_summary written after epoch 1",                               _s35_ckpt_summary_path.exists() and _s35_ckpt_summary_path.stat().st_size > 0, ""),
    ("All required keys in checkpoint_summary",                                 _CKPT_SUMMARY_KEYS.issubset(set(_s35_summary_after_ep1)), str(_CKPT_SUMMARY_KEYS - set(_s35_summary_after_ep1))),
    ("total_epochs_completed == 1 after first run(1)",                           _s35_summary_after_ep1["total_epochs_completed"] == 1, str(_s35_summary_after_ep1["total_epochs_completed"])),
    ("Resumed LCM loaded prior count (starts at 1, not 0)",                       _s35_resume_count_before_training == 1, str(_s35_resume_count_before_training)),
    ("total_epochs_completed == 3 after resumed run(2)",                            _s35_summary_after_ep3["total_epochs_completed"] == 3, str(_s35_summary_after_ep3["total_epochs_completed"])),
    ("last_epoch == 3",                                                               _s35_summary_after_ep3["last_epoch"] == 3, str(_s35_summary_after_ep3["last_epoch"])),
    ("epochs_remaining >= 0",                                                          _s35_summary_after_ep3["epochs_remaining"] >= 0, str(_s35_summary_after_ep3["epochs_remaining"])),
    ("last_checkpoint_path ends with last_model.pt",                                   str(_s35_summary_after_ep3["last_checkpoint_path"]).endswith("last_model.pt"), ""),
    ("best_checkpoint_path ends with best_model.pt",                                    str(_s35_summary_after_ep3["best_checkpoint_path"]).endswith("best_model.pt"), ""),
    ("last_val_loss is finite",                                                           _s35_summary_after_ep3["last_val_loss"] == _s35_summary_after_ep3["last_val_loss"], ""),
    ("updated_at_utc is a non-empty string",                                               isinstance(_s35_summary_after_ep3["updated_at_utc"], str) and len(_s35_summary_after_ep3["updated_at_utc"]) > 0, ""),
    ("Summary file is valid JSON (re-parseable)",                                           bool(_json.loads(_s35_ckpt_summary_path.read_text())), ""),
]
for label, passed, detail in checks:
    print_check(label, passed, detail)

del _s35_model, _s35_optimizer, _s35_scheduler, _s35_scaler
del _s35_tr_sub, _s35_va_sub, _s35_tr_ld, _s35_va_ld

all_passed = all(p for _, p, _ in checks)
print()
print(f"  {sum(1 for _,p,_ in checks if p)} / {len(checks)} checks passed")
if not all_passed:
    raise AssertionError(f"LastCheckpointManager FAILED: {[l for l,p,_ in checks if not p]}")
print("  ALL CHECKS PASSED")
print()

# -- Register REAL LastCheckpointManager on production_controller --
last_checkpoint_manager = LastCheckpointManager(
    summary_path=cfg.CHECKPOINT_DIR / "checkpoint_summary.json",
    checkpoint_dir=cfg.CHECKPOINT_DIR,
)
production_controller.register_callback(last_checkpoint_manager)

print_check(
    "Real LastCheckpointManager registered on production_controller",
    last_checkpoint_manager in production_controller._epoch_end_callbacks, "",
)
print_kv("checkpoint_summary.json destination", cfg.CHECKPOINT_DIR / "checkpoint_summary.json")
print()

# -- Summary of all registered callbacks -----------------------
print("  All callbacks registered on production_controller")
print("  " + "-" * 60)
for _i, _cb in enumerate(production_controller._epoch_end_callbacks, 1):
    print(f"    [{_i}] {type(_cb).__name__}")
print()


unique_callbacks = []
seen_names = set()

for cb in production_controller._epoch_end_callbacks:
    name = type(cb).__name__
    if name not in seen_names:
        unique_callbacks.append(cb)
        seen_names.add(name)

production_controller._epoch_end_callbacks = unique_callbacks

print()
print("Callbacks after cleanup")
print("-" * 60)

for i, cb in enumerate(production_controller._epoch_end_callbacks):
    print(i, type(cb).__name__)

callback_counts = Counter(
    type(cb).__name__
    for cb in production_controller._epoch_end_callbacks
)

assert callback_counts["EpochLogger"] == 1
assert callback_counts["TrainingHistoryManager"] == 1
assert callback_counts["BestModelManager"] == 1
assert callback_counts["LastCheckpointManager"] == 1

# Verify final callback order matches the expected pipeline (Stages 32-35)
_expected_order = ["EpochLogger", "TrainingHistoryManager", "BestModelManager", "LastCheckpointManager"]
_actual_order = [type(cb).__name__ for cb in production_controller._epoch_end_callbacks]
assert _actual_order == _expected_order, f"Callback order mismatch: {_actual_order}"

print_check(
    "Exactly 4 callbacks registered (Stages 32-35)",
    True,
    "EpochLogger, TrainingHistoryManager, BestModelManager, LastCheckpointManager",
)
print()
print("Stage 35 - Last Checkpoint Manager : OK")

  STAGE 35 - LAST CHECKPOINT MANAGER

2026-07-02 00:57:05 | INFO     | visionserveai.sprint04 | Epoch 1 | step 1/1 | loss=1.7356 | running_avg_loss=1.6046 | lr=9.97e-05
2026-07-02 00:57:06 | INFO     | visionserveai.sprint04 | Epoch 1 complete | train_loss=1.6046 | val_loss=1.6613 | val_macro_auroc=0.4780 | is_best=True | es_counter=0/99
2026-07-02 00:57:07 | INFO     | visionserveai.sprint04 | LastCheckpointManager: resumed from 1 prior epochs, best_epoch=1.
2026-07-02 00:57:08 | INFO     | visionserveai.sprint04 | Epoch 2 | step 1/1 | loss=1.7979 | running_avg_loss=1.5736 | lr=9.89e-05
2026-07-02 00:57:10 | INFO     | visionserveai.sprint04 | Epoch 2 complete | train_loss=1.5736 | val_loss=1.6385 | val_macro_auroc=0.5850 | is_best=True | es_counter=0/99
2026-07-02 00:57:11 | INFO     | visionserveai.sprint04 | Epoch 3 | step 1/1 | loss=1.7667 | running_avg_loss=1.5608 | lr=9.76e-05
2026-07-02 00:57:13 | INFO     | visionserveai.sprint04 | Epoch 3 complete | train_loss=1.5608 | val_lo

In [40]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 36: Resume Training Pipeline
#
# Module: training/train.py -> check_for_resume(), build_run_plan()
# ============================================================

# -- Purpose ---------------------------------------------------
# This stage (a) defines and verifies the resume helper functions
# that detect an existing checkpoint and load it into the
# production_controller without resetting any callback state, then
# (b) launches production_controller.run() for the configured
# cfg.EPOCHS -- the REAL, full multi-epoch training run against the
# complete Sprint 03 manifests. All four callbacks registered in
# Stages 32-35 fire every epoch; every artifact file is written
# incrementally so a crash at any point leaves a valid partial record
# on disk.
#
# Resume logic
# ------------
# Priority: last_model.pt (most-recently-completed epoch) > best_model.pt.
# After loading, remaining_epochs = cfg.EPOCHS - current_epoch.
# If current_epoch >= cfg.EPOCHS the run is already complete; we log
# and skip without calling run() again.
#
# Verification (BEFORE the real run)
# ------------------------------------
# A disposable controller runs 2 epochs on a small subset, saves
# last_model.pt, then a SECOND disposable controller loads it and
# continues for 2 more epochs, proving epoch-continuity of all four
# callback types (EpochLogger, TrainingHistoryManager,
# BestModelManager, LastCheckpointManager) across a mid-run
# checkpoint restore.

import os
import hashlib
import subprocess

print_section("STAGE 36 - RESUME TRAINING PIPELINE")
print()


def check_for_resume_checkpoint(checkpoint_dir: Path) -> Optional[Path]:
    """
    Return the path of the best candidate checkpoint for resuming, or None.

    Priority: last_model.pt (most recent completed epoch) first;
    best_model.pt only when last_model.pt is absent (e.g. first epoch
    crashed before Stage 31 could write it).

    Parameters
    ----------
    checkpoint_dir : Path   Directory searched for .pt files.

    Returns
    -------
    Optional[Path]
        Path of the checkpoint to resume from, or None if no suitable
        file exists.
    """
    last_path = checkpoint_dir / "last_model.pt"
    best_path = checkpoint_dir / "best_model.pt"
    if last_path.exists() and last_path.stat().st_size > 0:
        logger.info("Resume checkpoint found: %s", last_path)
        return last_path
    if best_path.exists() and best_path.stat().st_size > 0:
        logger.info("Resume checkpoint found (last_model.pt absent): %s", best_path)
        return best_path
    logger.info("No existing checkpoint found in %s -- starting fresh.", checkpoint_dir)
    return None


def build_run_plan(
    controller: "ProductionTrainingController",
    configured_epochs: int,
    resume_path: Optional[Path] = None,
) -> Dict[str, object]:
    """
    Load a resume checkpoint into *controller* if one is supplied and
    return a run plan dict describing what will happen.

    Parameters
    ----------
    controller       : ProductionTrainingController   The real controller.
    configured_epochs : int                           cfg.EPOCHS.
    resume_path      : Optional[Path]                 Output of check_for_resume_checkpoint().

    Returns
    -------
    Dict[str, object]
        ``{"start_epoch": int, "remaining_epochs": int,
           "resume_from": str|None, "action": "train"|"skip"}``
    """
    if resume_path is not None:
        controller.trainer.load_checkpoint(filename=resume_path.name)
        logger.info("Trainer state restored (epoch=%d).", controller.current_epoch)
        # Re-sync callback state from the just-loaded checkpoint.
        # All four callbacks already handle missing files gracefully
        # (they load from disk at construction time), so no extra work needed.

    start_epoch     = controller.current_epoch
    remaining_epochs = max(0, configured_epochs - start_epoch)

    if start_epoch >= configured_epochs:
        action = "skip"
        logger.info(
            "Training already complete (current_epoch=%d >= configured_epochs=%d).",
            start_epoch, configured_epochs,
        )
    else:
        action = "train"
        logger.info(
            "Run plan: start_epoch=%d, remaining_epochs=%d/%d.",
            start_epoch, remaining_epochs, configured_epochs,
        )

    return {
        "start_epoch":      start_epoch,
        "remaining_epochs":  remaining_epochs,
        "configured_epochs":  configured_epochs,
        "resume_from":         str(resume_path) if resume_path else None,
        "action":               action,
    }


# ============================================================
# Engineering verification: resume round-trip on disposable objects.
# Run 2 epochs, save last_model.pt, restore into a SECOND controller
# (fresh singletons, fresh callbacks), run 2 more epochs, verify
# epochs 3 and 4 appear in all callback outputs.
# ============================================================

print("  Resume round-trip verification (disposable clones, small subset)")
print("  " + "-" * 60)

_S36_SUBSET_BATCHES = 2


def _make_s36_disposable(ckpt_subdir: str):
    """Build a fresh disposable (trainer, controller, callbacks) for Stage 36 smoke test."""
    _tr_sub = torch.utils.data.Subset(
        train_loader.dataset,
        list(range(min(len(train_loader.dataset), _S36_SUBSET_BATCHES * cfg.BATCH_SIZE))),
    )
    _va_sub = torch.utils.data.Subset(
        val_loader.dataset,
        list(range(min(len(val_loader.dataset), _S36_SUBSET_BATCHES * cfg.BATCH_SIZE))),
    )
    _tr_ld = torch.utils.data.DataLoader(_tr_sub, batch_size=cfg.BATCH_SIZE, shuffle=True, num_workers=0, drop_last=False)
    _va_ld = torch.utils.data.DataLoader(_va_sub, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=0, drop_last=False)
    _mdl   = freeze_backbone(build_model(cfg.BACKBONE, num_classes=cfg.NUM_CLASSES, pretrained=False, dropout=DROPOUT_PROB), backbone_name).to(DEVICE)
    _opt   = build_optimizer(_mdl, optimizer_cfg)
    _sch   = build_scheduler(_opt, scheduler_cfg)
    _scl   = build_grad_scaler(amp_cfg)
    _cm    = CheckpointManager(checkpoint_dir=cfg.CHECKPOINT_DIR / ckpt_subdir)
    _es    = EarlyStopping(patience=99)
    _trn   = Trainer(model=_mdl, criterion=criterion_weighted, optimizer=_opt, scheduler=_sch,
                     scaler=_scl, train_loader=_tr_ld, val_loader=_va_ld,
                     class_names=class_names, config=cfg, gradient_config=gradient_cfg,
                     amp_config=amp_cfg, checkpoint_manager=_cm, early_stopping=_es, device=DEVICE)
    _ctrl  = ProductionTrainingController(trainer=_trn)

    _log_p  = cfg.METRICS_DIR / f"_s36_{ckpt_subdir}_epoch_logs.json"
    _hist_p = cfg.METRICS_DIR / f"_s36_{ckpt_subdir}_training_history.json"
    _meta_p = cfg.METRICS_DIR / f"_s36_{ckpt_subdir}_best_model_metadata.json"
    _ckpt_p = cfg.METRICS_DIR / f"_s36_{ckpt_subdir}_checkpoint_summary.json"

    _ctrl.register_callback(EpochLogger(output_path=_log_p))
    _ctrl.register_callback(TrainingHistoryManager(output_path=_hist_p))
    _ctrl.register_callback(BestModelManager(metadata_path=_meta_p, checkpoint_dir=_cm.checkpoint_dir))
    _ctrl.register_callback(LastCheckpointManager(summary_path=_ckpt_p, checkpoint_dir=_cm.checkpoint_dir))

    return _ctrl, _cm, {"log": _log_p, "hist": _hist_p, "meta": _meta_p, "ckpt": _ckpt_p}

# ============================================================
# ArtifactPersistenceManager
# Module: training/persistence.py -> ArtifactPersistenceManager
# ============================================================
#
# Purpose
# -------
# A Kaggle disconnect can happen at any epoch boundary during an 8-10h
# run, and no cell after production_controller.run() will ever execute.
# This callback runs as a fifth epoch-end callback: after every epoch
# it re-verifies the checkpoint LastCheckpointManager just wrote and
# atomically persists resume_state.json + artifact_manifest.json, so a
# disconnect immediately after any epoch finishes loses at most that
# one in-progress next epoch -- never more.
#
# It does not alter or replace LastCheckpointManager -- it only reads
# checkpoint_dir and writes its own two files.
#
# Cost discipline (single-GPU Kaggle session)
# ---------------------------------------------
# - last_model.pt is torch.load()'d every epoch because it changes
#   every epoch -- this is the one unavoidable per-epoch cost, and it
#   IS the corruption check (a truncated/corrupt file fails to
#   deserialize). No separate checksum pass is taken on top of it --
#   a hash with no stored reference to compare against catches nothing
#   torch.load() doesn't already catch, so it was pure added I/O.
# - best_model.pt is only reloaded when it actually changed this epoch
#   (is_best) or hasn't been verified yet in this process (e.g. right
#   after a resume) -- never on every epoch.
# - Static per-run metadata (config hash, backbone, architecture, git
#   commit) is computed exactly once and cached.
# - artifact_manifest.json entries are existence/size/mtime (os.stat,
#   effectively free) plus a validation_status carried over from the
#   torch-load check already performed this epoch -- no extra file
#   reads.
#
# Migration note: maps to ``training/persistence.py -> ArtifactPersistenceManager``.


class ArtifactPersistenceManager:
    """
    Callback: after every epoch, verify checkpoint integrity and
    atomically persist resume_state.json + artifact_manifest.json.

    Dependency safety
    -------------------
    Assumes LastCheckpointManager has already run earlier in the same
    epoch and written last_model.pt / best_model.pt. That assumption is
    never implicit: verify_dependencies() must be called once after
    registration and before run(), or every __call__ raises
    RuntimeError immediately.

    Migration note
    --------------
    Maps to ``training/persistence.py -> ArtifactPersistenceManager``.
    """

    def __init__(
        self,
        resume_state_path: Path,
        manifest_path: Path,
        checkpoint_dir: Path,
        artifact_paths: Optional[Dict[str, Path]] = None,
    ) -> None:
        self.resume_state_path = resume_state_path
        self.manifest_path     = manifest_path
        self.checkpoint_dir    = checkpoint_dir
        self.artifact_paths: Dict[str, Path] = dict(artifact_paths or {})

        self._dependencies_verified      = False
        self._best_verified_this_process = False
        self._static_meta: Optional[Dict[str, object]] = None
        self._best_epoch: Optional[int] = None
        self._best_val_auroc: Optional[float] = None
        self._best_val_loss: Optional[float] = None

        if resume_state_path.exists() and resume_state_path.stat().st_size > 0:
            try:
                with open(resume_state_path) as f:
                    _prior = _json.load(f)
                self._best_epoch     = _prior.get("best_epoch")
                self._best_val_auroc = _prior.get("best_val_auroc")
                self._best_val_loss  = _prior.get("best_val_loss")
                logger.info(
                    "ArtifactPersistenceManager: resumed prior best_epoch=%s.",
                    self._best_epoch,
                )
            except Exception as _err:
                logger.warning(
                    "ArtifactPersistenceManager: could not load prior resume_state (%s).", _err
                )

    # -- explicit dependency enforcement --------------------------------

    def verify_dependencies(self, controller: "ProductionTrainingController") -> None:
        """
        Confirm LastCheckpointManager is registered on *controller* and
        runs BEFORE this callback. Must be called once, after
        registering this instance, before run(). Raises RuntimeError
        with a specific message otherwise -- this notebook has already
        hit two real bugs (Stage 35 duplicate callbacks, Stage 36
        checkpoint-dir mixup) caused by exactly this class of
        registration-order mistake, so the check stays.
        """
        names = [type(cb).__name__ for cb in controller._epoch_end_callbacks]
        if "LastCheckpointManager" not in names:
            raise RuntimeError(
                "ArtifactPersistenceManager.verify_dependencies: no "
                "LastCheckpointManager is registered on this controller. "
                "ArtifactPersistenceManager reads last_model.pt/best_model.pt "
                "that LastCheckpointManager's epoch is expected to have just "
                "produced -- register LastCheckpointManager first."
            )
        if "ArtifactPersistenceManager" not in names:
            raise RuntimeError(
                "ArtifactPersistenceManager.verify_dependencies: this instance "
                "is not yet in controller._epoch_end_callbacks -- call "
                "register_callback() before verify_dependencies()."
            )
        lcm_idx  = names.index("LastCheckpointManager")
        apm_idxs = [i for i, n in enumerate(names) if n == "ArtifactPersistenceManager"]
        if apm_idxs[-1] < lcm_idx:
            raise RuntimeError(
                f"ArtifactPersistenceManager.verify_dependencies: registered "
                f"BEFORE LastCheckpointManager (order={names}). Fix registration "
                f"order in Stage 36."
            )
        self._dependencies_verified = True

    # -- internal helpers -------------------------------------------------

    def _atomic_write_json(self, path: Path, data: Dict[str, object]) -> None:
        """
        Write-to-temp -> fsync -> validate (re-parse) -> os.replace(),
        then a best-effort fsync of the parent directory so the
        directory entry pointing at the new file is durable, not just
        the file's own bytes -- guards against an abrupt kill right
        after replace() on filesystems where that matters. If anything
        raises, the file at *path* is left completely untouched.
        """
        tmp_path = path.parent / f"{path.name}.tmp{os.getpid()}"
        try:
            with open(tmp_path, "w") as f:
                _json.dump(data, f, indent=2, default=str)
                f.flush()
                os.fsync(f.fileno())
            with open(tmp_path) as f:
                _json.load(f)  # prove it's valid before it can become the real file
            os.replace(tmp_path, path)  # atomic on POSIX
            try:
                dir_fd = os.open(str(path.parent), os.O_RDONLY)
                try:
                    os.fsync(dir_fd)
                finally:
                    os.close(dir_fd)
            except OSError:
                pass  # not all filesystems support this -- degrade silently
        finally:
            if tmp_path.exists():
                tmp_path.unlink(missing_ok=True)

    def _verify_checkpoint(self, path: Path, label: str) -> Dict[str, object]:
        """
        The corruption check. torch.load() reading and deserializing the
        full file IS the integrity guarantee here -- deliberately not
        paired with a separate checksum pass (see module docstring).
        """
        if not path.exists() or path.stat().st_size == 0:
            raise RuntimeError(
                f"ArtifactPersistenceManager: {label} checkpoint missing or "
                f"empty at {path} -- refusing to persist resume_state.json "
                f"on top of an unverifiable checkpoint."
            )
        try:
            payload = torch.load(path, map_location="cpu")
        except Exception as _err:
            raise RuntimeError(
                f"ArtifactPersistenceManager: {label} checkpoint at {path} "
                f"failed to load with torch.load() -- treating as corrupt. "
                f"Original error: {_err}"
            ) from _err
        return payload

    @staticmethod
    def _hash_config(config) -> Optional[str]:
        try:
            cfg_dict = {
                k: v for k, v in vars(config).items()
                if not k.startswith("_") and isinstance(v, (str, int, float, bool, type(None)))
            }
            blob = _json.dumps(cfg_dict, sort_keys=True, default=str)
            return hashlib.sha256(blob.encode("utf-8")).hexdigest()[:16]
        except Exception as _err:
            logger.warning("ArtifactPersistenceManager: could not hash config (%s).", _err)
            return None

    @staticmethod
    def _get_git_commit() -> Optional[str]:
        try:
            result = subprocess.run(
                ["git", "rev-parse", "HEAD"],
                capture_output=True, text=True, timeout=3, check=True,
            )
            return result.stdout.strip()
        except Exception:
            return None  # Kaggle notebooks commonly aren't git repos -- expected, not an error

    @staticmethod
    def _get_current_lr(trainer) -> Optional[float]:
        try:
            return trainer.optimizer.param_groups[0]["lr"]
        except Exception:
            return None

    # -- callback entry point -------------------------------------------

    def __call__(self, epoch_summary: Dict[str, object], trainer: "Trainer") -> None:
        if not self._dependencies_verified:
            raise RuntimeError(
                "ArtifactPersistenceManager: verify_dependencies() was never "
                "called. Call artifact_persistence_manager.verify_dependencies"
                "(production_controller) immediately after registration, "
                "before run()."
            )

        if self._static_meta is None:  # computed once, never per epoch
            self._static_meta = {
                "config_hash":        self._hash_config(trainer.config),
                "backbone":           getattr(trainer.config, "BACKBONE", None),
                "model_architecture": type(trainer.model).__name__,
                "git_commit":         self._get_git_commit(),
                "sprint_version":     "04",
            }

        if epoch_summary.get("is_best"):
            self._best_epoch     = epoch_summary["epoch"]
            self._best_val_auroc = epoch_summary.get("val_macro_auroc")
            self._best_val_loss  = epoch_summary.get("val_loss")

        last_path = self.checkpoint_dir / "last_model.pt"
        best_path = self.checkpoint_dir / "best_model.pt"

        last_payload = self._verify_checkpoint(last_path, "last_model.pt")

        torch_verified_this_epoch = {last_path}
        if self._best_epoch is not None and (
            epoch_summary.get("is_best") or not self._best_verified_this_process
        ):
            self._verify_checkpoint(best_path, "best_model.pt")
            self._best_verified_this_process = True
            torch_verified_this_epoch.add(best_path)

        # Recovery self-check: the checkpoint's own embedded epoch must
        # match the epoch we're about to record as complete.
        _ckpt_epoch = last_payload.get("epoch") if isinstance(last_payload, dict) else None
        if _ckpt_epoch is not None and _ckpt_epoch != epoch_summary["epoch"]:
            raise RuntimeError(
                f"ArtifactPersistenceManager: recovery consistency check failed -- "
                f"last_model.pt reports epoch={_ckpt_epoch} but this epoch_summary "
                f"reports epoch={epoch_summary['epoch']}. Refusing to persist "
                f"resume_state.json against a mismatched checkpoint."
            )
        elif _ckpt_epoch is None:
            logger.warning(
                "ArtifactPersistenceManager: last_model.pt payload has no 'epoch' "
                "key -- epoch-match verification skipped. Confirm CheckpointManager's "
                "save() schema at training/persistence.py migration time."
            )

        optimizer_saved = last_payload.get("optimizer_state_dict") is not None
        scheduler_saved = last_payload.get("scheduler_state_dict") is not None
        scaler_saved    = last_payload.get("scaler_state_dict") is not None

        timestamp = _time.strftime("%Y-%m-%dT%H:%M:%SZ", _time.gmtime())

        resume_state = {
            "current_epoch":         epoch_summary["epoch"],
            "best_epoch":              self._best_epoch,
            "latest_checkpoint":         str(last_path),
            "best_checkpoint":             str(best_path) if self._best_epoch is not None else None,
            "optimizer_saved":               optimizer_saved,
            "scheduler_saved":                 scheduler_saved,
            "scaler_saved":                       scaler_saved,
            "current_learning_rate":                self._get_current_lr(trainer),
            "best_val_auroc":                          self._best_val_auroc,
            "best_val_loss":                              self._best_val_loss,
            "timestamp":                                     timestamp,
            **self._static_meta,
        }

        manifest_files = dict(self.artifact_paths)
        manifest_files.setdefault("last_checkpoint", last_path)
        manifest_files.setdefault("best_checkpoint", best_path)
        manifest_files.setdefault("resume_state", self.resume_state_path)
        manifest_files.setdefault("artifact_manifest", self.manifest_path)

        artifacts_entry: Dict[str, object] = {}
        for name, p in manifest_files.items():
            if not p.exists():
                artifacts_entry[name] = {
                    "path": str(p), "exists": False, "size_bytes": 0,
                    "modified_at_utc": None, "validation_status": "missing",
                }
                continue
            stat = p.stat()
            artifacts_entry[name] = {
                "path": str(p),
                "exists": True,
                "size_bytes": stat.st_size,
                "modified_at_utc": _time.strftime("%Y-%m-%dT%H:%M:%SZ", _time.gmtime(stat.st_mtime)),
                "validation_status": "torch_verified_this_epoch" if p in torch_verified_this_epoch else "present",
            }

        manifest = {
            "generated_at_utc": timestamp,
            "current_epoch":    epoch_summary["epoch"],
            **self._static_meta,
            "artifacts": artifacts_entry,
        }

        self._atomic_write_json(self.resume_state_path, resume_state)
        self._atomic_write_json(self.manifest_path, manifest)

    def current_resume_state(self) -> Optional[Dict[str, object]]:
        if self.resume_state_path.exists() and self.resume_state_path.stat().st_size > 0:
            with open(self.resume_state_path) as f:
                return _json.load(f)
        return None


def _collect_artifact_paths(controller: "ProductionTrainingController") -> Dict[str, Path]:
    """
    Introspect already-registered callbacks for their output file paths,
    so ArtifactPersistenceManager can list them in artifact_manifest.json.
    Read-only -- never touches any callback's internal state, and
    requires no hardcoded filenames.
    """
    paths: Dict[str, Path] = {}
    _attr_by_type = {
        "EpochLogger":            "output_path",
        "TrainingHistoryManager": "output_path",
        "BestModelManager":       "metadata_path",
        "LastCheckpointManager":  "summary_path",
    }
    for cb in controller._epoch_end_callbacks:
        cls_name = type(cb).__name__
        attr = _attr_by_type.get(cls_name)
        if attr and hasattr(cb, attr):
            paths[cls_name] = getattr(cb, attr)
    return paths


# -- Phase A: run 2 epochs on "run_a" controller ---------------
_s36_ctrl_a, _s36_cm_a, _s36_paths_a = _make_s36_disposable("_s36_run_a")
_s36_ctrl_a.run(num_epochs=2, log_interval=1)
_s36_resume_ckpt = check_for_resume_checkpoint(_s36_cm_a.checkpoint_dir)

# -- Phase B: second controller loads Phase A's checkpoint, runs 2 more --
# -- Phase B: second controller loads Phase A's checkpoint, runs 2 more --
_s36_ctrl_b, _s36_cm_b, _s36_paths_b = _make_s36_disposable("_s36_run_b")

# Point B's trainer at RUN A's checkpoint dir *before* loading -- that is
# where the checkpoint we actually want to resume from lives. Doing the
# reassignment before load_checkpoint() (not after) is the fix: previously
# the load fired while the trainer was still wired to B's own (empty)
# checkpoint manager, so it looked for last_model.pt inside _s36_run_b
# and raised FileNotFoundError even though Run A had already written it.
_s36_ctrl_b.trainer.checkpoint_manager = _s36_cm_a
_s36_ctrl_b.trainer.load_checkpoint(
    filename=_s36_resume_ckpt.name
    if _s36_resume_ckpt else "last_model.pt",
)

# State restored (model/optimizer/scheduler/scaler/epoch counters from
# Run A). Now switch B's trainer back to its OWN checkpoint manager so
# that the upcoming run() call saves last_model.pt / best_model.pt into
# _s36_run_b, rather than continuing to overwrite Run A's checkpoint.
# (In production this whole swap is unnecessary -- both phases share
# cfg.CHECKPOINT_DIR directly -- this is purely a disposable-clone
# wiring detail for the smoke test.)
_s36_ctrl_b.trainer.checkpoint_manager = _s36_cm_b

_s36_ctrl_b.run(num_epochs=2, log_interval=1)

_s36_ep_b = _s36_ctrl_b.current_epoch
_s36_hist_b = _json.loads(_s36_paths_b["hist"].read_text()) if _s36_paths_b["hist"].exists() else []

_s36_plan_a = build_run_plan(_s36_ctrl_a, configured_epochs=4, resume_path=None)
_s36_plan_skip = build_run_plan(_s36_ctrl_a, configured_epochs=2, resume_path=None)

checks = [
    ("check_for_resume_checkpoint returns a Path after 2 epochs", isinstance(_s36_resume_ckpt, Path), str(_s36_resume_ckpt)),
    ("Resume checkpoint is last_model.pt",                          _s36_resume_ckpt.name == "last_model.pt", str(_s36_resume_ckpt.name)),
    ("check_for_resume_checkpoint returns None when dir is empty",   check_for_resume_checkpoint(cfg.CHECKPOINT_DIR / "_nonexistent_dir") is None, ""),
    ("Phase B controller loaded epoch 2 from Phase A",               _s36_ctrl_b.current_epoch >= 2, str(_s36_ctrl_b.current_epoch)),
    ("Phase B ran 2 more epochs on top of Phase A's state",          _s36_ep_b == 4, str(_s36_ep_b)),
    ("EpochLogger entries in Phase B == 2 (new epochs only)",         len(_json.loads(_s36_paths_b["log"].read_text())) == 2, ""),
    ("TrainingHistoryManager in Phase B has 2 entries (not 4)",       len(_s36_hist_b) == 2, str(len(_s36_hist_b))),
    ("Phase B history epochs start at 3 (not 1)",                      _s36_hist_b[0]["epoch"] == 3 if _s36_hist_b else False, str(_s36_hist_b[0]["epoch"] if _s36_hist_b else "empty")),
    ("build_run_plan returns correct keys",                             {"start_epoch","remaining_epochs","configured_epochs","resume_from","action"}.issubset(set(_s36_plan_a)), ""),
    ("build_run_plan action=='train' when epochs remain",               _s36_plan_a["action"] == "train", _s36_plan_a["action"]),
    ("build_run_plan action=='skip' when configured_epochs already met", _s36_plan_skip["action"] == "skip", _s36_plan_skip["action"]),
    ("build_run_plan remaining_epochs correct",                           _s36_plan_a["remaining_epochs"] == 2, str(_s36_plan_a["remaining_epochs"])),
    ("Real Phase 2-5 singletons untouched",                               model is not _s36_ctrl_a.trainer.model, ""),
]
for label, passed, detail in checks:
    print_check(label, passed, detail)

del _s36_ctrl_a, _s36_ctrl_b, _s36_cm_a, _s36_cm_b

all_verif_passed = all(p for _, p, _ in checks)
print()
print(f"  {sum(1 for _,p,_ in checks if p)} / {len(checks)} checks passed")
if not all_verif_passed:
    raise AssertionError(f"Resume pipeline verification FAILED: {[l for l,p,_ in checks if not p]}")
print("  ALL CHECKS PASSED")
print()

# ============================================================
# Engineering verification: ArtifactPersistenceManager
# Proves (a) normal-path writes are correct and re-parseable,
# (b) a corrupt checkpoint fails loudly WITHOUT touching the previously
# valid resume_state.json (atomicity), and (c) registering this callback
# before LastCheckpointManager is rejected (dependency-order safety).
# ============================================================

print("  ArtifactPersistenceManager verification (disposable clone)")
print("  " + "-" * 60)

_s36_ctrl_apm, _s36_cm_apm, _s36_paths_apm = _make_s36_disposable("_s36_run_apm")

_apm_resume_p   = cfg.METRICS_DIR / "_s36_apm_resume_state.json"
_apm_manifest_p = cfg.METRICS_DIR / "_s36_apm_artifact_manifest.json"
for _p in (_apm_resume_p, _apm_manifest_p):
    if _p.exists():
        _p.unlink()

_s36_apm = ArtifactPersistenceManager(
    resume_state_path=_apm_resume_p,
    manifest_path=_apm_manifest_p,
    checkpoint_dir=_s36_cm_apm.checkpoint_dir,
    artifact_paths=_collect_artifact_paths(_s36_ctrl_apm),
)
_s36_ctrl_apm.register_callback(_s36_apm)
_s36_apm.verify_dependencies(_s36_ctrl_apm)   # must happen before run()
_s36_ctrl_apm.run(num_epochs=2, log_interval=1)

_s36_resume_state = _s36_apm.current_resume_state()
_s36_manifest      = _json.loads(_apm_manifest_p.read_text())

_RESUME_STATE_KEYS = {
    "current_epoch", "best_epoch", "latest_checkpoint", "best_checkpoint",
    "optimizer_saved", "scheduler_saved", "scaler_saved",
    "current_learning_rate", "best_val_auroc", "best_val_loss", "timestamp",
    "config_hash", "backbone", "model_architecture", "git_commit", "sprint_version",
}

# -- Atomicity / corruption test ---------------------------------
_apm_prior_valid_text = _apm_resume_p.read_text()  # snapshot of last-good state
(_s36_cm_apm.checkpoint_dir / "last_model.pt").write_bytes(b"not a real checkpoint")

_apm_raised = False
try:
    _s36_apm({"epoch": 3, "is_best": False}, _s36_ctrl_apm.trainer)
except RuntimeError:
    _apm_raised = True

_apm_file_unchanged = _apm_resume_p.read_text() == _apm_prior_valid_text

checks_apm = [
    ("ArtifactPersistenceManager is callable",                        callable(_s36_apm), ""),
    ("resume_state.json written after 2 epochs",                       _apm_resume_p.exists() and _apm_resume_p.stat().st_size > 0, ""),
    ("artifact_manifest.json written after 2 epochs",                   _apm_manifest_p.exists() and _apm_manifest_p.stat().st_size > 0, ""),
    ("All required keys in resume_state.json",                          _RESUME_STATE_KEYS.issubset(set(_s36_resume_state)), str(_RESUME_STATE_KEYS - set(_s36_resume_state))),
    ("current_epoch == 2",                                               _s36_resume_state["current_epoch"] == 2, str(_s36_resume_state["current_epoch"])),
    ("optimizer_saved is bool",                                          isinstance(_s36_resume_state["optimizer_saved"], bool), ""),
    ("config_hash present",                                              _s36_resume_state.get("config_hash") is not None, ""),
    ("timestamp is non-empty string",                                     isinstance(_s36_resume_state["timestamp"], str) and len(_s36_resume_state["timestamp"]) > 0, ""),
    ("artifact_manifest.json has 'artifacts' dict",                        isinstance(_s36_manifest.get("artifacts"), dict), ""),
    ("last_checkpoint entry marked torch_verified_this_epoch",              _s36_manifest["artifacts"]["last_checkpoint"]["validation_status"] == "torch_verified_this_epoch", str(_s36_manifest["artifacts"]["last_checkpoint"]["validation_status"])),
    ("resume_state.json is valid JSON (re-parseable)",                      bool(_json.loads(_apm_resume_p.read_text())), ""),
    ("artifact_manifest.json is valid JSON (re-parseable)",                  bool(_json.loads(_apm_manifest_p.read_text())), ""),
    ("Corrupt checkpoint raises RuntimeError (fails loudly)",                 _apm_raised, ""),
    ("Failed write does NOT overwrite prior valid resume_state.json",          _apm_file_unchanged, ""),
]
for label, passed, detail in checks_apm:
    print_check(label, passed, detail)

del _s36_ctrl_apm, _s36_cm_apm, _s36_apm

all_apm_passed = all(p for _, p, _ in checks_apm)
print()
print(f"  {sum(1 for _,p,_ in checks_apm if p)} / {len(checks_apm)} checks passed")
if not all_apm_passed:
    raise AssertionError(f"ArtifactPersistenceManager verification FAILED: {[l for l,p,_ in checks_apm if not p]}")
print("  ALL CHECKS PASSED")
print()

# -- Dependency-order negative test (separate clone, no full epoch run needed)
_s36_ctrl_dep, _s36_cm_dep, _ = _make_s36_disposable("_s36_run_dep")
_s36_ctrl_dep._epoch_end_callbacks = [
    cb for cb in _s36_ctrl_dep._epoch_end_callbacks
    if type(cb).__name__ != "LastCheckpointManager"
]
_dep_apm = ArtifactPersistenceManager(
    resume_state_path=cfg.METRICS_DIR / "_s36_dep_resume_state.json",
    manifest_path=cfg.METRICS_DIR / "_s36_dep_manifest.json",
    checkpoint_dir=_s36_cm_dep.checkpoint_dir,
)
_s36_ctrl_dep.register_callback(_dep_apm)
_dep_raised = False
try:
    _dep_apm.verify_dependencies(_s36_ctrl_dep)
except RuntimeError:
    _dep_raised = True

print_check(
    "verify_dependencies raises RuntimeError when LastCheckpointManager missing",
    _dep_raised, "",
)
if not _dep_raised:
    raise AssertionError("ArtifactPersistenceManager dependency-order check FAILED to raise")
del _s36_ctrl_dep, _s36_cm_dep, _dep_apm
print()

# ============================================================
# REAL PRODUCTION TRAINING RUN
# Determine resume state and launch production_controller.run().
# All four callbacks (registered Stages 32-35) fire every epoch.
# This cell may run for many hours on a Kaggle GPU; all artifacts
# are written after every completed epoch so a crash is recoverable.
# ============================================================

print("=" * 70)
print("  LAUNCHING PRODUCTION TRAINING RUN")
print("  " + "-" * 70)

_resume_ckpt = check_for_resume_checkpoint(cfg.CHECKPOINT_DIR)
_run_plan     = build_run_plan(
    controller=production_controller,
    configured_epochs=cfg.EPOCHS,
    resume_path=_resume_ckpt,
)

print_kv("Configured total epochs",   _run_plan["configured_epochs"])
print_kv("Start epoch (from state)",   _run_plan["start_epoch"])
print_kv("Remaining epochs to run",     _run_plan["remaining_epochs"])
print_kv("Resume from checkpoint",       _run_plan["resume_from"] or "None (fresh start)")
print_kv("Action",                         _run_plan["action"])
print()

# -- Register ArtifactPersistenceManager on production_controller --
# Guarded by name (not identity) so re-running this cell in a live
# kernel never registers a second copy.
_apm_resume_state_path = cfg.CHECKPOINT_DIR / "resume_state.json"
_apm_manifest_path     = cfg.CHECKPOINT_DIR / "artifact_manifest.json"

_existing_cb_names = {type(cb).__name__ for cb in production_controller._epoch_end_callbacks}
if "ArtifactPersistenceManager" not in _existing_cb_names:
    artifact_persistence_manager = ArtifactPersistenceManager(
        resume_state_path=_apm_resume_state_path,
        manifest_path=_apm_manifest_path,
        checkpoint_dir=cfg.CHECKPOINT_DIR,
        artifact_paths=_collect_artifact_paths(production_controller),
    )
    production_controller.register_callback(artifact_persistence_manager)
    artifact_persistence_manager.verify_dependencies(production_controller)
else:
    artifact_persistence_manager = next(
        cb for cb in production_controller._epoch_end_callbacks
        if type(cb).__name__ == "ArtifactPersistenceManager"
    )
    if not artifact_persistence_manager._dependencies_verified:
        artifact_persistence_manager.verify_dependencies(production_controller)
    logger.info("ArtifactPersistenceManager already registered -- skipping duplicate registration.")

print_check(
    "ArtifactPersistenceManager registered on production_controller",
    "ArtifactPersistenceManager" in {type(cb).__name__ for cb in production_controller._epoch_end_callbacks},
    "",
)
print_kv("resume_state.json destination", _apm_resume_state_path)
print_kv("artifact_manifest.json destination", _apm_manifest_path)
print()

if _run_plan["action"] == "skip":
    print("  Training already complete for configured epochs -- SKIPPING run().")
else:
    _prod_t0 = _time.time()
    production_controller.run(
        num_epochs=_run_plan["remaining_epochs"],
        log_interval=50,
    )
    _prod_elapsed = _time.time() - _prod_t0
    print()
    print_kv("Production run complete -- wall-clock time",
             f"{_prod_elapsed:.1f}s ({_prod_elapsed / 3600:.2f}h)")
    print_kv("Final epoch",             production_controller.current_epoch)
    print_kv("EarlyStopping triggered", production_trainer.early_stopping.should_stop)
    print_kv("Best epoch",              production_trainer.early_stopping.best_epoch)
    print_kv("Best val loss",           f"{production_trainer.early_stopping.best_loss:.6f}")
    print_kv("Epochs in history",       len(production_controller.controller_history))

print()
print("Stage 36 - Resume Training Pipeline : OK")

  STAGE 36 - RESUME TRAINING PIPELINE

  Resume round-trip verification (disposable clones, small subset)
  ------------------------------------------------------------
2026-07-02 00:58:12 | INFO     | visionserveai.sprint04 | Epoch 1 | step 1/1 | loss=1.4909 | running_avg_loss=1.6737 | lr=9.97e-05
2026-07-02 00:58:13 | INFO     | visionserveai.sprint04 | Epoch 1 complete | train_loss=1.6737 | val_loss=1.6536 | val_macro_auroc=0.4976 | is_best=True | es_counter=0/99
2026-07-02 00:58:15 | INFO     | visionserveai.sprint04 | Epoch 2 | step 1/1 | loss=1.7680 | running_avg_loss=1.5884 | lr=9.89e-05
2026-07-02 00:58:16 | INFO     | visionserveai.sprint04 | Epoch 2 complete | train_loss=1.5884 | val_loss=1.6381 | val_macro_auroc=0.6214 | is_best=True | es_counter=0/99
2026-07-02 00:58:17 | INFO     | visionserveai.sprint04 | Resume checkpoint found: /kaggle/working/visionserveai/sprint04/checkpoints/_s36_run_a/last_model.pt
2026-07-02 00:58:17 | INFO     | visionserveai.sprint04 | Trainer st

In [43]:
# ============================================================
# VisionServeAI | Sprint 04
# Training History Repair Utility
#
# Purpose:
#   Repair duplicated epochs inside training_history.json
#
# Safe Operation:
#   ✓ Creates backup
#   ✓ Keeps newest entry for duplicate epochs
#   ✓ Sorts by epoch
#   ✓ Validates continuity
#   ✓ Overwrites original ONLY after validation
# ============================================================

from pathlib import Path
import json
import shutil
from collections import Counter

print("=" * 70)
print("TRAINING HISTORY REPAIR")
print("=" * 70)
print()

# ------------------------------------------------------------
# Locate history file
# ------------------------------------------------------------

history_path = cfg.METRICS_DIR / "training_history.json"

if not history_path.exists():
    raise FileNotFoundError(history_path)

print("History file:")
print(history_path)
print()

# ------------------------------------------------------------
# Backup
# ------------------------------------------------------------

backup_path = history_path.with_suffix(".backup.json")
shutil.copy2(history_path, backup_path)

print("Backup created:")
print(backup_path)
print()

# ------------------------------------------------------------
# Load history
# ------------------------------------------------------------

with open(history_path, "r") as f:
    history = json.load(f)

print(f"Loaded {len(history)} history entries")
print()

# ------------------------------------------------------------
# Detect duplicates
# ------------------------------------------------------------

epoch_counts = Counter(entry["epoch"] for entry in history)

duplicates = {
    epoch: count
    for epoch, count in epoch_counts.items()
    if count > 1
}

if duplicates:
    print("Duplicate epochs found:")
    for ep, cnt in duplicates.items():
        print(f"  Epoch {ep}: {cnt} entries")
else:
    print("No duplicate epochs detected.")

print()

# ------------------------------------------------------------
# Keep newest occurrence of every epoch
# ------------------------------------------------------------

latest_entries = {}

for entry in history:
    latest_entries[entry["epoch"]] = entry

repaired_history = [
    latest_entries[e]
    for e in sorted(latest_entries.keys())
]

print(f"Entries after repair: {len(repaired_history)}")
print()

# ------------------------------------------------------------
# Validate
# ------------------------------------------------------------

epochs = [e["epoch"] for e in repaired_history]

expected = list(range(min(epochs), max(epochs) + 1))

if epochs != expected:
    raise RuntimeError(
        f"Epoch continuity broken.\n"
        f"Expected:\n{expected}\n"
        f"Found:\n{epochs}"
    )

print("Epoch continuity verified.")
print()

# ------------------------------------------------------------
# Verify uniqueness
# ------------------------------------------------------------

new_counts = Counter(e["epoch"] for e in repaired_history)

remaining_duplicates = [
    ep
    for ep, cnt in new_counts.items()
    if cnt > 1
]

if remaining_duplicates:
    raise RuntimeError(
        f"Duplicates still remain: {remaining_duplicates}"
    )

print("No duplicate epochs remain.")
print()

# ------------------------------------------------------------
# Save repaired file
# ------------------------------------------------------------

with open(history_path, "w") as f:
    json.dump(repaired_history, f, indent=2)

print("Repaired history written successfully.")
print()

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("=" * 70)
print("SUMMARY")
print("=" * 70)

print(f"Original entries : {len(history)}")
print(f"Repaired entries : {len(repaired_history)}")
print(f"Backup file      : {backup_path}")

removed = len(history) - len(repaired_history)

print(f"Removed entries  : {removed}")

if removed > 0:
    print()
    print("Removed duplicate epochs:")
    for ep, cnt in duplicates.items():
        print(f"  Epoch {ep}: removed {cnt - 1}")

print()
print("Repair completed successfully.")
print("=" * 70)

TRAINING HISTORY REPAIR

History file:
/kaggle/working/visionserveai/sprint04/metrics/training_history.json

Backup created:
/kaggle/working/visionserveai/sprint04/metrics/training_history.backup.json

Loaded 17 history entries

No duplicate epochs detected.

Entries after repair: 17

Epoch continuity verified.

No duplicate epochs remain.

Repaired history written successfully.

SUMMARY
Original entries : 17
Repaired entries : 17
Backup file      : /kaggle/working/visionserveai/sprint04/metrics/training_history.backup.json
Removed entries  : 0

Repair completed successfully.


In [44]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 36.5: Production Artifact Audit (READ-ONLY)
# ============================================================
#
# Purpose
# -------
# Audits every artifact produced by Stages 31-36 WITHOUT touching
# training state in any way. Safe to run any number of times, at any
# point, including after a kernel restart, without retraining or
# risking any existing file.
#
# This cell NEVER calls: trainer.fit(), production_controller.run(),
# save_checkpoint(), load_checkpoint(), register_callback(), or any
# operation that writes/deletes a file. Every filesystem interaction
# below is either Path.exists()/Path.stat() or an explicit read-only
# open()/torch.load(map_location="cpu").

import math

print_section("STAGE 36.5 - PRODUCTION ARTIFACT AUDIT (READ-ONLY)")
print()

_s365_results = []  # (item, status, detail) -- status in {PASS, FAIL, WARNING, SKIP}

def _s365_record(item: str, status: str, detail: str = "") -> None:
    _s365_results.append((item, status, detail))
    _mark = {"PASS": "\u2714", "FAIL": "\u2718", "WARNING": "\u26a0", "SKIP": "\u2013"}.get(status, "?")
    print(f"  {_mark}  [{status:7s}] {item}" + (f"  ({detail})" if detail else ""))

def _s365_read_json(path: Optional[Path]):
    """Returns (data, error_str). Never raises."""
    if path is None:
        return None, "no path resolved"
    if not path.exists():
        return None, f"file does not exist: {path}"
    if path.stat().st_size == 0:
        return None, f"file is empty: {path}"
    try:
        with open(path) as f:
            return _json.load(f), None
    except Exception as _err:
        return None, f"invalid JSON: {_err}"

def _s365_load_checkpoint(path: Optional[Path]):
    """Returns (payload, error_str). Never raises. Read-only torch.load."""
    if path is None:
        return None, "no path resolved"
    if not path.exists():
        return None, f"file does not exist: {path}"
    if path.stat().st_size == 0:
        return None, f"file is empty: {path}"
    try:
        payload = torch.load(path, map_location="cpu")
        return payload, None
    except Exception as _err:
        return None, f"torch.load() failed: {_err}"

# ------------------------------------------------------------
# 0. Resolve artifact paths (live introspection, with fallback)
# ------------------------------------------------------------
print("  Resolving artifact paths")
print("  " + "-" * 60)

_s365_live_cbs = {}
_s365_live_available = "production_controller" in globals()
if _s365_live_available:
    try:
        _s365_live_cbs = {
            type(cb).__name__: cb for cb in production_controller._epoch_end_callbacks
        }
    except Exception:
        _s365_live_available = False

def _s365_resolve(cb_name: str, attr: str, fallback: Path):
    cb = _s365_live_cbs.get(cb_name)
    if cb is not None and hasattr(cb, attr):
        return getattr(cb, attr), "live"
    return fallback, "ASSUMED (conventional path, live callback unavailable)"

_s365_last_model_path = cfg.CHECKPOINT_DIR / "last_model.pt"
_s365_best_model_path = cfg.CHECKPOINT_DIR / "best_model.pt"

_s365_epoch_logs_path,      _s365_epoch_logs_src      = _s365_resolve("EpochLogger",             "output_path",   cfg.METRICS_DIR / "epoch_logs.json")
_s365_history_path,         _s365_history_src         = _s365_resolve("TrainingHistoryManager",  "output_path",   cfg.METRICS_DIR / "training_history.json")
_s365_best_meta_path,       _s365_best_meta_src       = _s365_resolve("BestModelManager",         "metadata_path", cfg.METRICS_DIR / "best_model_metadata.json")
_s365_ckpt_summary_path,    _s365_ckpt_summary_src    = _s365_resolve("LastCheckpointManager",    "summary_path",  cfg.CHECKPOINT_DIR / "checkpoint_summary.json")
_s365_resume_state_path,    _s365_resume_state_src    = _s365_resolve("ArtifactPersistenceManager","resume_state_path", cfg.CHECKPOINT_DIR / "resume_state.json")
_s365_manifest_path,        _s365_manifest_src        = _s365_resolve("ArtifactPersistenceManager","manifest_path",     cfg.CHECKPOINT_DIR / "artifact_manifest.json")

for _label, _path, _src in [
    ("last_model.pt",           _s365_last_model_path,   "live (cfg.CHECKPOINT_DIR)"),
    ("best_model.pt",           _s365_best_model_path,   "live (cfg.CHECKPOINT_DIR)"),
    ("epoch_logs.json",         _s365_epoch_logs_path,   _s365_epoch_logs_src),
    ("training_history.json",   _s365_history_path,      _s365_history_src),
    ("best_model_metadata.json",_s365_best_meta_path,    _s365_best_meta_src),
    ("checkpoint_summary.json", _s365_ckpt_summary_path, _s365_ckpt_summary_src),
    ("resume_state.json",       _s365_resume_state_path, _s365_resume_state_src),
    ("artifact_manifest.json",  _s365_manifest_path,     _s365_manifest_src),
]:
    print_kv(f"  {_label}", f"{_path}  [{_src}]")
print()

# ------------------------------------------------------------
# 1. Checkpoint integrity: last_model.pt / best_model.pt
# ------------------------------------------------------------
print("  Checkpoint integrity")
print("  " + "-" * 60)

_s365_expected_keys = ["model_state_dict", "optimizer_state_dict", "scheduler_state_dict", "scaler_state_dict", "epoch", "config"]
_s365_critical_keys = {"model_state_dict", "epoch"}  # absence of these = unusable checkpoint

def _s365_audit_checkpoint(label: str, path: Path):
    payload, err = _s365_load_checkpoint(path)
    if err is not None:
        _s365_record(f"{label} loadable via torch.load()", "FAIL", err)
        return None
    _s365_record(f"{label} exists, non-empty, torch.load() succeeds", "PASS", str(path))
    if not isinstance(payload, dict):
        _s365_record(f"{label} payload is a dict", "FAIL", f"got {type(payload).__name__}")
        return None
    for key in _s365_expected_keys:
        present = payload.get(key) is not None
        if present:
            _s365_record(f"{label} contains '{key}'", "PASS")
        elif key in _s365_critical_keys:
            _s365_record(f"{label} contains '{key}'", "FAIL", "critical key missing or None")
        else:
            _s365_record(f"{label} contains '{key}'", "WARNING", "missing or None -- schema unconfirmed, not treated as fatal")
    return payload

_s365_last_payload = _s365_audit_checkpoint("last_model.pt", _s365_last_model_path)
_s365_best_payload = _s365_audit_checkpoint("best_model.pt", _s365_best_model_path)
print()

# ------------------------------------------------------------
# 2. Cross validation: resume_state.json vs last_model.pt
# ------------------------------------------------------------
print("  Cross validation: resume_state.json <-> last_model.pt")
print("  " + "-" * 60)

_s365_resume_state, _s365_resume_err = _s365_read_json(_s365_resume_state_path)
if _s365_resume_err is not None:
    _s365_record("resume_state.json readable", "FAIL" if _s365_resume_state_path and _s365_resume_state_path.exists() else "WARNING", _s365_resume_err)
else:
    _s365_record("resume_state.json readable and valid JSON", "PASS")

    if _s365_last_payload is not None:
        _rs_epoch = _s365_resume_state.get("current_epoch")
        _ckpt_epoch = _s365_last_payload.get("epoch")
        _s365_record(
            "resume_state.current_epoch == last_model.pt epoch",
            "PASS" if _rs_epoch == _ckpt_epoch else "FAIL",
            f"resume_state={_rs_epoch}, checkpoint={_ckpt_epoch}",
        )

        for _flag, _key in [("optimizer_saved", "optimizer_state_dict"), ("scheduler_saved", "scheduler_state_dict"), ("scaler_saved", "scaler_state_dict")]:
            _rs_val = _s365_resume_state.get(_flag)
            _actual = _s365_last_payload.get(_key) is not None
            _s365_record(
                f"resume_state.{_flag} matches checkpoint",
                "PASS" if _rs_val == _actual else "FAIL",
                f"resume_state={_rs_val}, checkpoint={_actual}",
            )
    else:
        _s365_record("resume_state <-> checkpoint field comparison", "SKIP", "last_model.pt unreadable, see section 1")

    # LR consistency -- only meaningful if the live trainer hasn't advanced
    # past the epoch resume_state.json describes; otherwise LR has moved
    # on by design (scheduler stepping every epoch since).
    if "production_trainer" in globals() and "production_controller" in globals():
        try:
            _live_epoch = production_controller.current_epoch
            _live_lr = production_trainer.optimizer.param_groups[0]["lr"]
            _rs_epoch = _s365_resume_state.get("current_epoch")
            _rs_lr = _s365_resume_state.get("current_learning_rate")
            if _rs_epoch == _live_epoch:
                _lr_ok = _rs_lr is not None and math.isclose(_rs_lr, _live_lr, rel_tol=1e-6)
                _s365_record("current_learning_rate matches live optimizer LR", "PASS" if _lr_ok else "FAIL", f"resume_state={_rs_lr}, live={_live_lr}")
            else:
                _s365_record(
                    "current_learning_rate vs live optimizer LR",
                    "SKIP",
                    f"resume_state reflects epoch {_rs_epoch}, live trainer is at epoch {_live_epoch} -- training has progressed since this file was last written, comparison not meaningful",
                )
        except Exception as _err:
            _s365_record("current_learning_rate vs live optimizer LR", "SKIP", f"live trainer unavailable: {_err}")
    else:
        _rs_lr = _s365_resume_state.get("current_learning_rate")
        _lr_sane = isinstance(_rs_lr, (int, float)) and _rs_lr > 0 and math.isfinite(_rs_lr)
        _s365_record("current_learning_rate is a sane positive finite value", "PASS" if _lr_sane else "WARNING", f"value={_rs_lr}")
print()

# ------------------------------------------------------------
# 3. History validation: training_history.json
# ------------------------------------------------------------
print("  History validation: training_history.json")
print("  " + "-" * 60)

_s365_history, _s365_history_err = _s365_read_json(_s365_history_path)
_s365_history_last_epoch = None
_s365_history_best_epoch = None
if _s365_history_err is not None:
    _s365_record("training_history.json readable", "WARNING", _s365_history_err)
else:
    _s365_record("training_history.json valid JSON (list)", "PASS" if isinstance(_s365_history, list) else "FAIL", f"type={type(_s365_history).__name__}")
    if isinstance(_s365_history, list) and len(_s365_history) > 0 and all("epoch" in e for e in _s365_history):
        _epochs = [e["epoch"] for e in _s365_history]
        _s365_record("training_history.json has no duplicate epochs", "PASS" if len(set(_epochs)) == len(_epochs) else "FAIL", f"{len(_epochs)} entries, {len(set(_epochs))} unique")
        _s365_record("training_history.json epochs monotonically increasing", "PASS" if _epochs == sorted(_epochs) and len(set(_epochs)) == len(_epochs) else "FAIL", str(_epochs[:5]) + ("..." if len(_epochs) > 5 else ""))
        _s365_history_last_epoch = _epochs[-1]
        if _s365_last_payload is not None:
            _ckpt_epoch = _s365_last_payload.get("epoch")
            _s365_record("training_history last epoch == last_model.pt epoch", "PASS" if _s365_history_last_epoch == _ckpt_epoch else "FAIL", f"history={_s365_history_last_epoch}, checkpoint={_ckpt_epoch}")
        _best_entries = [e for e in _s365_history if e.get("is_best")]
        if _best_entries:
            _s365_history_best_epoch = _best_entries[-1]["epoch"]
            _s365_record("training_history best-epoch entry identifiable", "PASS", f"best_epoch={_s365_history_best_epoch}")
        else:
            _s365_record("training_history best-epoch entry identifiable", "WARNING", "no entry has is_best=True")
    else:
        _s365_record("training_history.json entries have 'epoch' key", "FAIL", "cannot validate ordering/duplicates without this field")
print()

# ------------------------------------------------------------
# 4. Epoch logger validation: epoch_logs.json
# ------------------------------------------------------------
print("  Epoch logger validation: epoch_logs.json")
print("  " + "-" * 60)

_s365_epoch_logs, _s365_epoch_logs_err = _s365_read_json(_s365_epoch_logs_path)
if _s365_epoch_logs_err is not None:
    _s365_record("epoch_logs.json readable", "WARNING", _s365_epoch_logs_err)
else:
    _s365_record("epoch_logs.json valid JSON (list)", "PASS" if isinstance(_s365_epoch_logs, list) else "FAIL", f"type={type(_s365_epoch_logs).__name__}")
    if isinstance(_s365_epoch_logs, list) and len(_s365_epoch_logs) > 0 and all("epoch" in e for e in _s365_epoch_logs):
        _log_epochs = [e["epoch"] for e in _s365_epoch_logs]
        _s365_record("epoch_logs.json has no duplicate epochs", "PASS" if len(set(_log_epochs)) == len(_log_epochs) else "FAIL", f"{len(_log_epochs)} entries, {len(set(_log_epochs))} unique")
        _continuous = all(b - a == 1 for a, b in zip(_log_epochs, _log_epochs[1:]))
        _s365_record("epoch_logs.json epoch numbering is continuous (no gaps)", "PASS" if _continuous else "FAIL", str(_log_epochs[:5]) + ("..." if len(_log_epochs) > 5 else ""))
        if _s365_history_last_epoch is not None:
            _s365_record("epoch_logs final epoch == training_history final epoch", "PASS" if _log_epochs[-1] == _s365_history_last_epoch else "FAIL", f"epoch_logs={_log_epochs[-1]}, history={_s365_history_last_epoch}")
        else:
            _s365_record("epoch_logs final epoch vs training_history", "SKIP", "training_history final epoch unavailable, see section 3")
    else:
        _s365_record("epoch_logs.json entries have 'epoch' key", "FAIL", "cannot validate ordering/duplicates without this field")
print()

# ------------------------------------------------------------
# 5. Checkpoint summary validation: checkpoint_summary.json
# ------------------------------------------------------------
print("  Checkpoint summary validation: checkpoint_summary.json")
print("  " + "-" * 60)

_s365_ckpt_summary, _s365_ckpt_summary_err = _s365_read_json(_s365_ckpt_summary_path)
if _s365_ckpt_summary_err is not None:
    _s365_record("checkpoint_summary.json readable", "WARNING", _s365_ckpt_summary_err)
else:
    _s365_record("checkpoint_summary.json valid JSON", "PASS")
    if _s365_last_payload is not None:
        _cs_last_epoch = _s365_ckpt_summary.get("last_epoch")
        _ckpt_epoch = _s365_last_payload.get("epoch")
        _s365_record("checkpoint_summary.last_epoch == last_model.pt epoch", "PASS" if _cs_last_epoch == _ckpt_epoch else "FAIL", f"summary={_cs_last_epoch}, checkpoint={_ckpt_epoch}")
    for _key in ("last_checkpoint_path", "best_checkpoint_path"):
        _p = _s365_ckpt_summary.get(_key)
        if _p:
            _exists = Path(_p).exists()
            _s365_record(f"checkpoint_summary.{_key} points to an existing file", "PASS" if _exists else "FAIL", str(_p))
        else:
            _s365_record(f"checkpoint_summary.{_key} present", "WARNING", "missing from checkpoint_summary.json")
print()

# ------------------------------------------------------------
# 6. Best model validation: best_model_metadata.json
# ------------------------------------------------------------
print("  Best model validation: best_model_metadata.json")
print("  " + "-" * 60)

_s365_best_meta, _s365_best_meta_err = _s365_read_json(_s365_best_meta_path)
if _s365_best_meta_err is not None:
    _s365_record("best_model_metadata.json readable", "WARNING", _s365_best_meta_err)
else:
    _s365_record("best_model_metadata.json valid JSON", "PASS")
    # Schema for this file wasn't available at audit-authoring time -- try
    # common key names defensively rather than asserting a fixed schema.
    _bm_epoch = _s365_best_meta.get("epoch", _s365_best_meta.get("best_epoch"))
    if _bm_epoch is not None and _s365_best_payload is not None:
        _ckpt_best_epoch = _s365_best_payload.get("epoch")
        if _ckpt_best_epoch is not None:
            _s365_record("best_model_metadata epoch == best_model.pt epoch", "PASS" if _bm_epoch == _ckpt_best_epoch else "FAIL", f"metadata={_bm_epoch}, checkpoint={_ckpt_best_epoch}")
        else:
            _s365_record("best_model_metadata epoch vs best_model.pt epoch", "WARNING", "best_model.pt payload has no 'epoch' key to compare against")
    else:
        _s365_record("best_model_metadata epoch vs best_model.pt epoch", "WARNING", "epoch-like key not found in best_model_metadata.json (schema unconfirmed) -- confirm field name at migration time")
print()

# ------------------------------------------------------------
# 7. Artifact manifest validation: artifact_manifest.json
# ------------------------------------------------------------
print("  Artifact manifest validation: artifact_manifest.json")
print("  " + "-" * 60)

_s365_manifest, _s365_manifest_err = _s365_read_json(_s365_manifest_path)
if _s365_manifest_err is not None:
    _s365_record("artifact_manifest.json readable", "WARNING", _s365_manifest_err)
else:
    _s365_record("artifact_manifest.json valid JSON", "PASS")
    _artifacts = _s365_manifest.get("artifacts", {})
    if not _artifacts:
        _s365_record("artifact_manifest.json has 'artifacts' entries", "WARNING", "empty or missing 'artifacts' dict")
    for _name, _entry in _artifacts.items():
        _p = Path(_entry.get("path", "")) if isinstance(_entry, dict) else None
        if _p is None or str(_p) == "":
            _s365_record(f"manifest entry '{_name}' has a path", "FAIL", "no path recorded")
            continue
        _exists_now = _p.exists()
        _s365_record(f"manifest entry '{_name}' path exists on disk", "PASS" if _exists_now else "FAIL", str(_p))
print()

# ------------------------------------------------------------
# 8. GPU audit
# ------------------------------------------------------------
print("  GPU audit")
print("  " + "-" * 60)

_s365_cuda_available = torch.cuda.is_available()
print_kv("CUDA available", _s365_cuda_available)
if _s365_cuda_available:
    _s365_device_count = torch.cuda.device_count()
    print_kv("GPU count", _s365_device_count)
    for _i in range(_s365_device_count):
        _props = torch.cuda.get_device_properties(_i)
        _total_gb = _props.total_memory / (1024 ** 3)
        _alloc_gb = torch.cuda.memory_allocated(_i) / (1024 ** 3)
        _reserved_gb = torch.cuda.memory_reserved(_i) / (1024 ** 3)
        print(f"    GPU {_i}: {torch.cuda.get_device_name(_i)}")
        print(f"      VRAM total:      {_total_gb:.2f} GiB")
        print(f"      Allocated:       {_alloc_gb:.2f} GiB")
        print(f"      Reserved:        {_reserved_gb:.2f} GiB")
else:
    print("    No CUDA device available.")
print()

# ------------------------------------------------------------
# 9. Training summary
# ------------------------------------------------------------
print("  Training summary")
print("  " + "-" * 60)

_s365_current_epoch = _s365_resume_state.get("current_epoch") if _s365_resume_state else (_s365_last_payload.get("epoch") if _s365_last_payload else None)
_s365_best_epoch     = _s365_resume_state.get("best_epoch") if _s365_resume_state else _s365_history_best_epoch
_s365_best_val_loss  = _s365_resume_state.get("best_val_loss") if _s365_resume_state else None
_s365_current_lr     = _s365_resume_state.get("current_learning_rate") if _s365_resume_state else None
_s365_configured_epochs = getattr(cfg, "EPOCHS", None)
_s365_remaining = (max(0, _s365_configured_epochs - _s365_current_epoch)
                    if _s365_configured_epochs is not None and _s365_current_epoch is not None else None)

_s365_checkpoint_ok = _s365_last_payload is not None and all(
    status != "FAIL" for item, status, _ in _s365_results if item.startswith("last_model.pt")
)
_s365_cross_validation_ok = all(
    status != "FAIL" for item, status, _ in _s365_results
    if "resume_state" in item or item.startswith("checkpoint_summary")
)
_s365_can_resume = _s365_checkpoint_ok and _s365_cross_validation_ok

print_kv("Current epoch",          _s365_current_epoch)
print_kv("Best epoch",             _s365_best_epoch)
print_kv("Best validation loss",   _s365_best_val_loss)
print_kv("Current LR",             _s365_current_lr)
print_kv("Total epochs completed", _s365_current_epoch)
print_kv("Configured total epochs",_s365_configured_epochs)
print_kv("Remaining epochs",       _s365_remaining)
print_kv("Can safely resume?",     "YES" if _s365_can_resume else "NO")
print()

# ------------------------------------------------------------
# Final report
# ------------------------------------------------------------
print("=" * 70)
print("  FINAL AUDIT REPORT")
print("  " + "-" * 70)
_s365_col_w = max(len(item) for item, _, _ in _s365_results) + 2
for _item, _status, _detail in _s365_results:
    print(f"  {_item:<{_s365_col_w}} {_status:<8} {_detail}")

_s365_fail_items = [item for item, status, _ in _s365_results if status == "FAIL"]
_s365_warn_items = [item for item, status, _ in _s365_results if status == "WARNING"]

print()
print_kv("Total checks",  len(_s365_results))
print_kv("PASS",          sum(1 for _, s, _ in _s365_results if s == "PASS"))
print_kv("FAIL",          len(_s365_fail_items))
print_kv("WARNING",       len(_s365_warn_items))
print_kv("SKIP",          sum(1 for _, s, _ in _s365_results if s == "SKIP"))
print()

if _s365_fail_items:
    print("  OVERALL AUDIT FAILED")
    print()
    raise AssertionError(f"Stage 36.5 audit found {len(_s365_fail_items)} critical inconsistency(ies): {_s365_fail_items}")
else:
    print("  OVERALL AUDIT PASSED")
print()
print("Stage 36.5 - Production Artifact Audit : OK")

  STAGE 36.5 - PRODUCTION ARTIFACT AUDIT (READ-ONLY)

  Resolving artifact paths
  ------------------------------------------------------------
    last_model.pt                  /kaggle/working/visionserveai/sprint04/checkpoints/last_model.pt  [live (cfg.CHECKPOINT_DIR)]
    best_model.pt                  /kaggle/working/visionserveai/sprint04/checkpoints/best_model.pt  [live (cfg.CHECKPOINT_DIR)]
    epoch_logs.json                /kaggle/working/visionserveai/sprint04/metrics/epoch_logs.json  [live]
    training_history.json          /kaggle/working/visionserveai/sprint04/metrics/training_history.json  [live]
    best_model_metadata.json       /kaggle/working/visionserveai/sprint04/metrics/best_model_metadata.json  [live]
    checkpoint_summary.json        /kaggle/working/visionserveai/sprint04/checkpoints/checkpoint_summary.json  [live]
    resume_state.json              /kaggle/working/visionserveai/sprint04/checkpoints/resume_state.json  [live]
    artifact_manifest.json         

In [46]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 37: Training Report Generator
#
# Module: training/history.py -> generate_training_report()
# ============================================================

# -- Purpose ---------------------------------------------------
# Reads the artifacts written by Stages 32-36 (epoch_logs.json,
# training_history.json, best_model_metadata.json,
# checkpoint_summary.json) and synthesises them into a single,
# comprehensive ``training_report.json``. This report is the
# canonical post-training artifact for the MLOps registry, model
# card, and any human reviewer -- they should be able to read
# training_report.json and know exactly what happened, without
# reading multiple files or loading a checkpoint.
#
# Migration note: maps to ``training/history.py -> generate_training_report()``.

print_section("STAGE 37 - TRAINING REPORT GENERATOR")
print()


def generate_training_report(
    training_history_path:  Path,
    epoch_logs_path:         Path,
    best_model_metadata_path: Path,
    checkpoint_summary_path:   Path,
    config: "TrainingConfig",
    run_plan: Dict[str, object],
) -> Dict[str, object]:
    """
    Synthesise all Stage 32-36 artifacts into a single training report.

    Parameters
    ----------
    training_history_path   : Path   cfg.METRICS_DIR / "training_history.json"
    epoch_logs_path          : Path   cfg.METRICS_DIR / "epoch_logs.json"
    best_model_metadata_path  : Path   cfg.METRICS_DIR / "best_model_metadata.json"
    checkpoint_summary_path    : Path   cfg.CHECKPOINT_DIR / "checkpoint_summary.json"
    config                      : TrainingConfig
    run_plan                     : Dict  Output of build_run_plan() from Stage 36.

    Returns
    -------
    Dict[str, object]
        JSON-serialisable report dict.  Written to
        ``cfg.OUTPUT_DIR / "training_report.json"`` by the caller.
    """

    def _load(path: Path, default):
        if path.exists() and path.stat().st_size > 0:
            try:
                with open(path) as f:
                    return _json.load(f)
            except Exception as _err:
                logger.warning("generate_training_report: could not load %s (%s).", path, _err)
        return default

    history         = _load(training_history_path, [])
    epoch_logs       = _load(epoch_logs_path, [])
    best_meta         = _load(best_model_metadata_path, {})
    ckpt_summary       = _load(checkpoint_summary_path, {})

    # -- Convergence analysis -------------------------------------
    val_losses = [e["val_loss"] for e in history if e.get("val_loss") is not None
                  and e["val_loss"] == e["val_loss"]]
    aurocs     = [e["val_macro_auroc"] for e in history
                  if e.get("val_macro_auroc") is not None
                  and e["val_macro_auroc"] == e["val_macro_auroc"]]
    f1s        = [e["val_f1_macro"] for e in history
                  if e.get("val_f1_macro") is not None
                  and e["val_f1_macro"] == e["val_f1_macro"]]
    train_losses = [e["train_loss"] for e in history
                    if e.get("train_loss") is not None
                    and e["train_loss"] == e["train_loss"]]

    best_val_loss_idx    = int(val_losses.index(min(val_losses)))     if val_losses else None
    best_auroc_idx        = int(aurocs.index(max(aurocs)))              if aurocs     else None
    best_auroc_value       = max(aurocs)                                  if aurocs     else None
    final_val_loss          = val_losses[-1]                               if val_losses else None
    final_auroc              = aurocs[-1]                                   if aurocs     else None
    final_f1                  = f1s[-1]                                      if f1s        else None
    final_train_loss           = train_losses[-1]                             if train_losses else None

    total_epochs_trained = len(history)
    early_stopped        = ckpt_summary.get("should_stop", False)

    # -- LR trace from epoch_logs ---------------------------------
    lr_trace = [{"epoch": e["epoch"], "lr": e.get("train_lr")}
                for e in epoch_logs if "train_lr" in e]

    report = {
        "project":        config.PROJECT_NAME,
        "sprint":          "04",
        "phase":            "Phase 6 - Production Training Pipeline",
        "stage":             37,
        "generated_at_utc":   _time.strftime("%Y-%m-%dT%H:%M:%SZ", _time.gmtime()),

        "run_configuration": {
            "model_name":           config.MODEL_NAME,
            "backbone":              config.BACKBONE,
            "num_classes":            config.NUM_CLASSES,
            "configured_epochs":       config.EPOCHS,
            "batch_size":               config.BATCH_SIZE,
            "learning_rate":             config.LEARNING_RATE,
            "weight_decay":               config.WEIGHT_DECAY,
            "lr_scheduler":                config.LR_SCHEDULER,
            "gradient_clip":                config.GRADIENT_CLIP,
            "use_amp":                       config.USE_AMP,
            "early_stopping_patience":        config.EARLY_STOPPING_PATIENCE,
            "random_seed":                     config.RANDOM_SEED,
            "start_epoch":                      run_plan.get("start_epoch"),
            "resume_from":                       run_plan.get("resume_from"),
        },

        "training_summary": {
            "total_epochs_trained":    total_epochs_trained,
            "early_stopped":            early_stopped,
            "final_epoch":               history[-1]["epoch"]   if history else None,
            "final_val_loss":             final_val_loss,
            "final_val_macro_auroc":       final_auroc,
            "final_val_f1_macro":           final_f1,
            "final_train_loss":              final_train_loss,
        },

        "best_model": {
            "epoch":                         best_meta.get("epoch"),
            "val_loss":                       best_meta.get("val_loss"),
            "val_macro_auroc":                 best_meta.get("val_macro_auroc"),
            "val_f1_macro":                     best_meta.get("val_f1_macro"),
            "checkpoint_path":                   best_meta.get("checkpoint_path"),
            "timestamp_utc":                      best_meta.get("timestamp_utc"),
        },

        "convergence": {
            "best_val_loss_epoch":           best_val_loss_idx + 1 if best_val_loss_idx is not None else None,
            "best_val_macro_auroc":           best_auroc_value,
            "best_val_macro_auroc_epoch":      best_auroc_idx + 1   if best_auroc_idx   is not None else None,
            "val_loss_improvement_ep1_to_best": (
                round(val_losses[0] - min(val_losses), 6) if len(val_losses) >= 2 else None
            ),
        },

        "artifacts": {
            "training_history_json":   str(training_history_path),
            "epoch_logs_json":          str(epoch_logs_path),
            "best_model_metadata_json":  str(best_model_metadata_path),
            "checkpoint_summary_json":    str(checkpoint_summary_path),
            "best_model_pt":               str(cfg.CHECKPOINT_DIR / "best_model.pt"),
            "last_model_pt":                str(cfg.CHECKPOINT_DIR / "last_model.pt"),
        },

        "lr_trace":     lr_trace,
        "epoch_count":   total_epochs_trained,
    }

    return report


# -- Execute ---------------------------------------------------
_report = generate_training_report(
    training_history_path   = cfg.METRICS_DIR / "training_history.json",
    epoch_logs_path          = cfg.METRICS_DIR / "epoch_logs.json",
    best_model_metadata_path  = cfg.METRICS_DIR / "best_model_metadata.json",
    checkpoint_summary_path    = cfg.CHECKPOINT_DIR / "checkpoint_summary.json",
    config=cfg,
    run_plan=_run_plan,
)

_report_path = cfg.OUTPUT_DIR / "training_report.json"
with open(_report_path, "w") as f:
    _json.dump(_report, f, indent=2, default=str)

# -- Verification ----------------------------------------------
_REPORT_SECTIONS = {
    "project", "sprint", "phase", "stage", "generated_at_utc",
    "run_configuration", "training_summary", "best_model",
    "convergence", "artifacts", "lr_trace", "epoch_count",
}
_hist_data = _json.loads((cfg.METRICS_DIR / "training_history.json").read_text()) \
    if (cfg.METRICS_DIR / "training_history.json").exists() else []

checks = [
    ("generate_training_report returns a dict",                         isinstance(_report, dict), ""),
    ("training_report.json written and non-empty",                       _report_path.exists() and _report_path.stat().st_size > 0, str(_report_path)),
    ("All required top-level sections present",                           _REPORT_SECTIONS.issubset(set(_report)), str(_REPORT_SECTIONS - set(_report))),
    ("training_report.json is valid JSON (re-parseable)",                  bool(_json.loads(_report_path.read_text())), ""),
    ("run_configuration.configured_epochs == cfg.EPOCHS",                   _report["run_configuration"]["configured_epochs"] == cfg.EPOCHS, str(_report["run_configuration"]["configured_epochs"])),
    ("training_summary.total_epochs_trained >= 1",                           _report["training_summary"]["total_epochs_trained"] >= 1, str(_report["training_summary"]["total_epochs_trained"])),
    ("training_summary.total_epochs_trained matches history length",          _report["training_summary"]["total_epochs_trained"] == len(_hist_data), f"{_report['training_summary']['total_epochs_trained']} vs {len(_hist_data)}"),
    ("best_model section present with epoch field",                            _report["best_model"].get("epoch") is not None, str(_report["best_model"].get("epoch"))),
    ("artifacts section lists all 6 expected files",                            len(_report["artifacts"]) == 6, str(len(_report["artifacts"]))),
    ("generated_at_utc is a non-empty string",                                   isinstance(_report["generated_at_utc"], str) and len(_report["generated_at_utc"]) > 0, ""),
    ("convergence.best_val_macro_auroc is a float or None",                       isinstance(_report["convergence"]["best_val_macro_auroc"], (float, int)) or _report["convergence"]["best_val_macro_auroc"] is None, ""),
    ("lr_trace is a list",                                                          isinstance(_report["lr_trace"], list), ""),
]
for label, passed, detail in checks:
    print_check(label, passed, detail)

print()
print("  Training summary snapshot")
print("  " + "-" * 60)
for k, v in _report["training_summary"].items():
    print_kv(k, v if not isinstance(v, float) else round(v, 4))
print()
print("  Best model snapshot")
print("  " + "-" * 60)
for k, v in _report["best_model"].items():
    print_kv(k, v if not isinstance(v, float) else round(v, 4))
print()

all_passed = all(p for _, p, _ in checks)
print(f"  {sum(1 for _,p,_ in checks if p)} / {len(checks)} checks passed")
if not all_passed:
    raise AssertionError(f"Training Report Generator FAILED: {[l for l,p,_ in checks if not p]}")
print()
print("  ALL CHECKS PASSED")
print()
print("Stage 37 - Training Report Generator : OK")

  STAGE 37 - TRAINING REPORT GENERATOR

  ✔  generate_training_report returns a dict
  ✔  training_report.json written and non-empty  (/kaggle/working/visionserveai/sprint04/training_report.json)
  ✔  All required top-level sections present  (set())
  ✔  training_report.json is valid JSON (re-parseable)
  ✔  run_configuration.configured_epochs == cfg.EPOCHS  (30)
  ✔  training_summary.total_epochs_trained >= 1  (17)
  ✔  training_summary.total_epochs_trained matches history length  (17 vs 17)
  ✔  best_model section present with epoch field  (10)
  ✔  artifacts section lists all 6 expected files  (6)
  ✔  generated_at_utc is a non-empty string
  ✔  convergence.best_val_macro_auroc is a float or None
  ✔  lr_trace is a list

  Training summary snapshot
  ------------------------------------------------------------
  total_epochs_trained             17
  early_stopped                    True
  final_epoch                      17
  final_val_loss                   1.3034
  final_val_macro

In [47]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 38: Artifact Persistence
#
# Module: training/train.py -> persist_training_artifacts()
# ============================================================

# -- Purpose ---------------------------------------------------
# Validate that EVERY artifact the production pipeline committed to
# producing (per the task specification) is on disk, non-empty, and
# individually parseable / loadable. Then write the final
# ``training_summary.json`` -- the single-page executive summary
# consumed by any downstream tool (model registry, CI, MLOps pipeline)
# that needs an overall PASS/FAIL signal and the key metrics without
# reading multiple files.
#
# Artifacts required by the task specification (verified here):
#   best_model.pt              cfg.CHECKPOINT_DIR
#   last_model.pt              cfg.CHECKPOINT_DIR
#   training_history.json      cfg.METRICS_DIR
#   training_summary.json      cfg.OUTPUT_DIR          [written here]
#   checkpoint_summary.json    cfg.CHECKPOINT_DIR
#   training_report.json       cfg.OUTPUT_DIR
#   epoch_logs.json            cfg.METRICS_DIR
#
# Migration note: maps to ``training/train.py -> persist_training_artifacts()``.

print_section("STAGE 38 - ARTIFACT PERSISTENCE")
print()


def _safe_load_json(path: Path) -> Optional[Dict]:
    try:
        with open(path) as f:
            return _json.load(f)
    except Exception:
        return None


def _safe_load_pt(path: Path, device: torch.device) -> Optional[Dict]:
    try:
        return torch.load(path, map_location=device)
    except Exception:
        return None


# -- Validate every required artifact ---------------------------
print("  Required artifact validation")
print("  " + "-" * 60)

_required_artifacts = {
    "best_model.pt":            cfg.CHECKPOINT_DIR / "best_model.pt",
    "last_model.pt":             cfg.CHECKPOINT_DIR / "last_model.pt",
    "training_history.json":      cfg.METRICS_DIR    / "training_history.json",
    "checkpoint_summary.json":     cfg.CHECKPOINT_DIR / "checkpoint_summary.json",
    "training_report.json":         cfg.OUTPUT_DIR     / "training_report.json",
    "epoch_logs.json":               cfg.METRICS_DIR    / "epoch_logs.json",
    "best_model_metadata.json":       cfg.METRICS_DIR    / "best_model_metadata.json",
}

_artifact_checks: List[Tuple[str, bool, str]] = []
for name, path in _required_artifacts.items():
    exists   = path.exists()
    non_zero = exists and path.stat().st_size > 0
    _artifact_checks.append((f"{name} exists and is non-empty", exists and non_zero, str(path)))
    print_check(f"{name}", exists and non_zero, str(path))

print()

# -- Validate content parseable --------------------------------
print("  Content parseability validation")
print("  " + "-" * 60)

_hist_loaded   = _safe_load_json(_required_artifacts["training_history.json"])
_logs_loaded    = _safe_load_json(_required_artifacts["epoch_logs.json"])
_report_loaded   = _safe_load_json(_required_artifacts["training_report.json"])
_ckpt_s_loaded    = _safe_load_json(_required_artifacts["checkpoint_summary.json"])
_best_meta_loaded  = _safe_load_json(_required_artifacts["best_model_metadata.json"])
_best_pt_loaded     = _safe_load_pt(_required_artifacts["best_model.pt"], DEVICE)
_last_pt_loaded      = _safe_load_pt(_required_artifacts["last_model.pt"], DEVICE)

_content_checks: List[Tuple[str, bool, str]] = [
    ("training_history.json is valid JSON list",        isinstance(_hist_loaded, list), ""),
    ("epoch_logs.json is valid JSON list",               isinstance(_logs_loaded, list), ""),
    ("training_report.json is valid JSON dict",           isinstance(_report_loaded, dict), ""),
    ("checkpoint_summary.json is valid JSON dict",         isinstance(_ckpt_s_loaded, dict), ""),
    ("best_model_metadata.json is valid JSON dict",         isinstance(_best_meta_loaded, dict), ""),
    ("best_model.pt loadable with torch.load",               isinstance(_best_pt_loaded, dict), ""),
    ("last_model.pt loadable with torch.load",                isinstance(_last_pt_loaded, dict), ""),
    ("best_model.pt has model_state_dict",                     isinstance(_best_pt_loaded, dict) and "model_state_dict" in _best_pt_loaded, ""),
    ("last_model.pt has model_state_dict",                      isinstance(_last_pt_loaded, dict) and "model_state_dict" in _last_pt_loaded, ""),
    ("training_history.json has >= 1 epoch",                     isinstance(_hist_loaded, list) and len(_hist_loaded) >= 1, str(len(_hist_loaded) if _hist_loaded else 0)),
    ("epoch_logs.json has >= 1 entry",                            isinstance(_logs_loaded, list) and len(_logs_loaded) >= 1, str(len(_logs_loaded) if _logs_loaded else 0)),
    ("training_history and epoch_logs same length",                (
        isinstance(_hist_loaded, list) and isinstance(_logs_loaded, list)
        and len(_hist_loaded) == len(_logs_loaded)), f"{len(_hist_loaded or [])} vs {len(_logs_loaded or [])}"),
]
for label, passed, detail in _content_checks:
    print_check(label, passed, detail)

print()
all_artifact_checks = _artifact_checks + _content_checks
all_artifacts_ok = all(p for _, p, _ in all_artifact_checks)

# -- Write training_summary.json --------------------------------
print("  Writing training_summary.json")
print("  " + "-" * 60)

_best_epoch_num   = _best_meta_loaded.get("epoch")          if _best_meta_loaded else None
_best_auroc       = _best_meta_loaded.get("val_macro_auroc") if _best_meta_loaded else None
_best_val_loss    = _best_meta_loaded.get("val_loss")        if _best_meta_loaded else None
_last_epoch_num   = _last_pt_loaded.get("epoch")              if _last_pt_loaded else None

training_summary = {
    "project":        cfg.PROJECT_NAME,
    "sprint":          "04",
    "pipeline_version": cfg.PIPELINE_VERSION,
    "phase":            "Phase 6 - Production Training Pipeline",
    "generated_at_utc":  _time.strftime("%Y-%m-%dT%H:%M:%SZ", _time.gmtime()),

    "model": {
        "model_name":    cfg.MODEL_NAME,
        "backbone":       cfg.BACKBONE,
        "num_classes":     cfg.NUM_CLASSES,
        "dropout":          DROPOUT_PROB,
    },

    "training": {
        "configured_epochs":      cfg.EPOCHS,
        "epochs_trained":          len(_hist_loaded) if _hist_loaded else 0,
        "early_stopped":            production_trainer.early_stopping.should_stop,
        "best_epoch":               _best_epoch_num,
        "last_completed_epoch":      _last_epoch_num,
        "best_val_loss":              _best_val_loss,
        "best_val_macro_auroc":        _best_auroc,
        "early_stopping_patience":      cfg.EARLY_STOPPING_PATIENCE,
    },

    "hyperparameters": {
        "learning_rate":  cfg.LEARNING_RATE,
        "weight_decay":    cfg.WEIGHT_DECAY,
        "batch_size":       cfg.BATCH_SIZE,
        "gradient_clip":     cfg.GRADIENT_CLIP,
        "use_amp":            cfg.USE_AMP,
        "lr_scheduler":        cfg.LR_SCHEDULER,
        "random_seed":          cfg.RANDOM_SEED,
    },

    "artifacts": {name: str(path) for name, path in _required_artifacts.items()},
    "artifact_validation_passed": all_artifacts_ok,

    "checkpoints": {
        "best_model_pt":  str(cfg.CHECKPOINT_DIR / "best_model.pt"),
        "last_model_pt":   str(cfg.CHECKPOINT_DIR / "last_model.pt"),
        "checkpoint_dir":   str(cfg.CHECKPOINT_DIR),
    },
}

_ts_path = cfg.OUTPUT_DIR / "training_summary.json"
with open(_ts_path, "w") as f:
    _json.dump(training_summary, f, indent=2, default=str)

print_check(
    "training_summary.json written and non-empty",
    _ts_path.exists() and _ts_path.stat().st_size > 0,
    str(_ts_path),
)

# -- Final artifact table ----------------------------------------
print()
print("  Final artifact manifest")
print("  " + "-" * 60)
all_final = {**_required_artifacts, "training_summary.json": _ts_path}
for name, path in all_final.items():
    _ok  = path.exists() and path.stat().st_size > 0
    _sz  = f"{path.stat().st_size:,} bytes" if path.exists() else "MISSING"
    print_check(f"{name:<35} {_sz}", _ok, "")

print()
total_checks = len(all_artifact_checks) + 1
passed_checks = sum(1 for _, p, _ in all_artifact_checks) + int(
    _ts_path.exists() and _ts_path.stat().st_size > 0
)
print(f"  {passed_checks} / {total_checks} checks passed")
if passed_checks < total_checks:
    raise AssertionError(
        f"Artifact persistence FAILED -- {total_checks - passed_checks} checks failed."
    )
print()
print("  ALL ARTIFACT CHECKS PASSED")
print()
print("Stage 38 - Artifact Persistence : OK")

  STAGE 38 - ARTIFACT PERSISTENCE

  Required artifact validation
  ------------------------------------------------------------
  ✔  best_model.pt  (/kaggle/working/visionserveai/sprint04/checkpoints/best_model.pt)
  ✔  last_model.pt  (/kaggle/working/visionserveai/sprint04/checkpoints/last_model.pt)
  ✔  training_history.json  (/kaggle/working/visionserveai/sprint04/metrics/training_history.json)
  ✔  checkpoint_summary.json  (/kaggle/working/visionserveai/sprint04/checkpoints/checkpoint_summary.json)
  ✔  training_report.json  (/kaggle/working/visionserveai/sprint04/training_report.json)
  ✔  epoch_logs.json  (/kaggle/working/visionserveai/sprint04/metrics/epoch_logs.json)
  ✔  best_model_metadata.json  (/kaggle/working/visionserveai/sprint04/metrics/best_model_metadata.json)

  Content parseability validation
  ------------------------------------------------------------
  ✔  training_history.json is valid JSON list
  ✔  epoch_logs.json is valid JSON list
  ✔  training_report.json 

In [50]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 39: Engineering Verification
#
# Module: training/train.py -> final verification gate
# ============================================================
#
# CHANGES vs prior version:
#   [1] Layer 1 -- replaced len(history)==len(logs) with 4 semantic
#       epoch-consistency checks. The old equality invariant was invalid:
#       training_history.json may have entries from Trainer._persist_history()
#       that predate EpochLogger's registration. The correct invariant is
#       that epoch_logs is a well-formed *subset* of training_history.
#
#   [2] Layer 3 -- replaced hardcoded callback count (==4) with
#       type-name-based verification: required types present, no duplicates,
#       correct relative ordering. The count was invalid once Stage 36 Ext
#       registered ArtifactPersistenceManager as a 5th callback.

print_section("STAGE 39 - ENGINEERING VERIFICATION")
print()

_fails: List[str] = []
_checks_run: int = 0   # counts every _gate() call for accurate reporting


def _gate(condition: bool, label: str, detail: str = "") -> None:
    global _checks_run
    _checks_run += 1
    print_check(label, condition, detail)
    if not condition:
        _fails.append(label)


# ============================================================
# LAYER 1: Artifact completeness & integrity
# ============================================================
print("  Layer 1 — Artifact completeness & integrity")
print("  " + "-" * 60)

_l1_artifacts = {
    "best_model.pt":            cfg.CHECKPOINT_DIR / "best_model.pt",
    "last_model.pt":             cfg.CHECKPOINT_DIR / "last_model.pt",
    "training_history.json":      cfg.METRICS_DIR    / "training_history.json",
    "training_summary.json":       cfg.OUTPUT_DIR     / "training_summary.json",
    "checkpoint_summary.json":      cfg.CHECKPOINT_DIR / "checkpoint_summary.json",
    "training_report.json":          cfg.OUTPUT_DIR     / "training_report.json",
    "epoch_logs.json":                cfg.METRICS_DIR    / "epoch_logs.json",
    "best_model_metadata.json":        cfg.METRICS_DIR    / "best_model_metadata.json",
}
for _name, _path in _l1_artifacts.items():
    _gate(
        _path.exists() and _path.stat().st_size > 0,
        f"Artifact present & non-empty: {_name}",
        str(_path),
    )

_l1_hist = None
_l1_logs  = None
_l1_summary  = None
_l1_report    = None
_l1_best_meta  = None
try:
    with open(_l1_artifacts["training_history.json"]) as _f:
        _l1_hist = _json.load(_f)
    with open(_l1_artifacts["epoch_logs.json"]) as _f:
        _l1_logs = _json.load(_f)
    with open(_l1_artifacts["training_summary.json"]) as _f:
        _l1_summary = _json.load(_f)
    with open(_l1_artifacts["training_report.json"]) as _f:
        _l1_report = _json.load(_f)
    with open(_l1_artifacts["best_model_metadata.json"]) as _f:
        _l1_best_meta = _json.load(_f)
except Exception as _parse_err:
    _gate(False, f"JSON parsing failed: {_parse_err}")

_gate(
    isinstance(_l1_hist, list)    and len(_l1_hist) >= 1,
    "training_history.json: valid list, >= 1 epoch",
    str(len(_l1_hist or [])),
)
_gate(
    isinstance(_l1_logs, list)     and len(_l1_logs) >= 1,
    "epoch_logs.json: valid list, >= 1 entry",
    str(len(_l1_logs or [])),
)
_gate(
    isinstance(_l1_summary, dict)  and "training" in _l1_summary,
    "training_summary.json: dict with 'training' key",
    "",
)
_gate(
    isinstance(_l1_report, dict)   and "best_model" in _l1_report,
    "training_report.json: dict with 'best_model' key",
    "",
)
_gate(
    isinstance(_l1_best_meta, dict) and "epoch" in _l1_best_meta,
    "best_model_metadata.json: dict with 'epoch' key",
    "",
)

_l1_best_pt = torch.load(_l1_artifacts["best_model.pt"], map_location=DEVICE)
_l1_last_pt = torch.load(_l1_artifacts["last_model.pt"],  map_location=DEVICE)
_gate("model_state_dict" in _l1_best_pt, "best_model.pt: contains model_state_dict", "")
_gate("model_state_dict" in _l1_last_pt, "last_model.pt: contains model_state_dict", "")
_gate(_l1_best_pt.get("epoch", -1) >= 1, "best_model.pt: epoch >= 1",
      str(_l1_best_pt.get("epoch")))
_gate(_l1_last_pt.get("epoch", -1) >= 1, "last_model.pt: epoch >= 1",
      str(_l1_last_pt.get("epoch")))

# ── CHANGED BLOCK [1] ─────────────────────────────────────────────────────────
# Old check (INVALID): len(training_history) == len(epoch_logs)
#
# Root cause: Trainer._persist_history() can write entries to training_history.json
# BEFORE the EpochLogger callback is registered (e.g. on the first epoch of a fresh
# run, or before Stage 32 registers the callback). TrainingHistoryManager loads
# those entries on __init__ and preserves them; EpochLogger has no record of them.
# epoch_logs is therefore a *subset* of training_history -- never required to be
# equal in length. Enforcing equality produces false failures on every valid resume
# or fresh-start run where the callback was registered after epoch 1 completed.
#
# Replacement: four semantic consistency checks that capture what actually matters.
# ─────────────────────────────────────────────────────────────────────────────────

_hist_epochs_l1 = [
    e.get("epoch") for e in (_l1_hist or []) if isinstance(e.get("epoch"), int)
]
_log_epochs_l1 = [
    e.get("epoch") for e in (_l1_logs or []) if isinstance(e.get("epoch"), int)
]
_hist_epoch_set_l1 = set(_hist_epochs_l1)
_log_epoch_set_l1  = set(_log_epochs_l1)

print()
print("  Layer 1 — epoch log / history consistency")
print("  " + "-" * 60)
print_kv("training_history.json entries", len(_hist_epochs_l1))
print_kv("epoch_logs.json entries",       len(_log_epochs_l1))
print_kv("history epochs",    f"{_hist_epochs_l1[:3]}...{_hist_epochs_l1[-1:]}" if len(_hist_epochs_l1) > 3 else str(_hist_epochs_l1))
print_kv("log epochs",        f"{_log_epochs_l1[:3]}...{_log_epochs_l1[-1:]}"  if len(_log_epochs_l1)  > 3 else str(_log_epochs_l1))
print()

_orphan_epochs = _log_epoch_set_l1 - _hist_epoch_set_l1  # epochs in logs but NOT in history
_gate(
    _log_epoch_set_l1.issubset(_hist_epoch_set_l1),
    "epoch_logs.json epochs are a subset of training_history.json epochs "
    "(no epoch logged without a matching history entry)",
    f"orphan_epochs={_orphan_epochs}" if _orphan_epochs else "",
)
_gate(
    bool(_hist_epochs_l1 and _log_epochs_l1)
    and _hist_epochs_l1[-1] == _log_epochs_l1[-1],
    "Latest epoch is consistent: training_history[-1] == epoch_logs[-1] "
    "(no tail data loss in EpochLogger)",
    f"history_last={_hist_epochs_l1[-1] if _hist_epochs_l1 else 'N/A'}, "
    f"logs_last={_log_epochs_l1[-1] if _log_epochs_l1 else 'N/A'}",
)
_gate(
    len(_log_epochs_l1) == len(_log_epoch_set_l1),
    "epoch_logs.json has no duplicate epoch entries",
    f"total={len(_log_epochs_l1)}, unique={len(_log_epoch_set_l1)}",
)
_gate(
    (len(_log_epochs_l1) <= 1)
    or all(a < b for a, b in zip(_log_epochs_l1, _log_epochs_l1[1:])),
    "epoch_logs.json epoch numbers are strictly monotonically increasing",
    f"first={_log_epochs_l1[0] if _log_epochs_l1 else 'N/A'}, "
    f"last={_log_epochs_l1[-1] if _log_epochs_l1 else 'N/A'}",
)
# ── END CHANGED BLOCK [1] ─────────────────────────────────────────────────────

print()

# ============================================================
# LAYER 2: Training quality gates
# ============================================================
print("  Layer 2 — Training quality gates")
print("  " + "-" * 60)

_val_losses = [
    e["val_loss"] for e in (_l1_hist or [])
    if e.get("val_loss") is not None and e["val_loss"] == e["val_loss"]
]
_aurocs = [
    e["val_macro_auroc"] for e in (_l1_hist or [])
    if e.get("val_macro_auroc") is not None
    and e["val_macro_auroc"] == e["val_macro_auroc"]
]
_best_epoch_l2  = _l1_best_pt.get("epoch", -1)
_best_auroc_val = max(_aurocs)      if _aurocs      else None
_best_loss_val   = min(_val_losses) if _val_losses  else None

_gate(len(_val_losses) >= 1, "At least 1 epoch with finite val_loss",  str(len(_val_losses)))
_gate(len(_aurocs) >= 1,      "At least 1 epoch with finite macro AUROC", str(len(_aurocs)))
_gate(
    _best_auroc_val is not None and _best_auroc_val > 0.5,
    "Best macro AUROC > 0.5 (better than random chance)",
    f"{_best_auroc_val:.4f}" if _best_auroc_val else "None",
)
_gate(
    _best_loss_val is not None and _best_loss_val > 0,
    "Best val_loss > 0 (finite, non-degenerate)",
    f"{_best_loss_val:.6f}" if _best_loss_val else "None",
)
_gate(
    0 < _best_epoch_l2 <= cfg.EPOCHS,
    "best_model.pt epoch in [1, cfg.EPOCHS]",
    f"{_best_epoch_l2} in [1, {cfg.EPOCHS}]",
)
_gate(
    (len(_val_losses) < 2)
    or (_val_losses[-1] <= _val_losses[0] * 1.5),
    "Final val_loss <= 1.5x initial val_loss (no catastrophic divergence)",
    f"{_val_losses[-1]:.4f} vs {_val_losses[0]:.4f}"
    if len(_val_losses) >= 2
    else "single epoch",
)
_gate(
    production_trainer.early_stopping.best_loss != float("inf"),
    "EarlyStopping recorded at least one improvement (best_loss != inf)",
    str(production_trainer.early_stopping.best_loss),
)

print()

# ============================================================
# LAYER 3: Compatibility and migration readiness
# ============================================================
print("  Layer 3 — Compatibility and migration readiness")
print("  " + "-" * 60)

# CheckpointManager round-trip: load best_model.pt into a fresh model.
_l3_model = build_model(
    cfg.BACKBONE, num_classes=cfg.NUM_CLASSES, pretrained=False, dropout=DROPOUT_PROB
)
_l3_model = freeze_backbone(_l3_model, backbone_name).to(DEVICE)
try:
    _l3_model.load_state_dict(_l1_best_pt["model_state_dict"])
    _l3_load_ok = True
except Exception as _ld_err:
    _l3_load_ok = False
    logger.error("best_model.pt state_dict load failed: %s", _ld_err)
_gate(_l3_load_ok, "best_model.pt state_dict loads cleanly into a fresh model", "")

_l3_model.eval()
with torch.no_grad():
    _l3_out = _l3_model(torch.randn(2, 3, *cfg.IMAGE_SIZE, device=DEVICE))
_gate(
    tuple(_l3_out.shape) == (2, cfg.NUM_CLASSES),
    "Loaded best_model.pt forward pass has correct output shape",
    str(tuple(_l3_out.shape)),
)
_gate(bool(torch.isfinite(_l3_out).all()), "Loaded best_model.pt output is finite", "")
del _l3_model, _l3_out

# Trainer.load_checkpoint round-trip on last_model.pt.
_l3_resume_model     = build_model(
    cfg.BACKBONE, num_classes=cfg.NUM_CLASSES, pretrained=False, dropout=DROPOUT_PROB
)
_l3_resume_model     = freeze_backbone(_l3_resume_model, backbone_name).to(DEVICE)
_l3_resume_optimizer = build_optimizer(_l3_resume_model, optimizer_cfg)
_l3_resume_scheduler = build_scheduler(_l3_resume_optimizer, scheduler_cfg)
_l3_resume_scaler    = build_grad_scaler(amp_cfg)
_l3_resume_trainer   = Trainer(
    model=_l3_resume_model, criterion=criterion_weighted,
    optimizer=_l3_resume_optimizer, scheduler=_l3_resume_scheduler,
    scaler=_l3_resume_scaler,
    train_loader=train_loader, val_loader=val_loader,
    class_names=class_names, config=cfg, gradient_config=gradient_cfg,
    amp_config=amp_cfg, checkpoint_manager=checkpoint_manager,
    early_stopping=EarlyStopping(patience=cfg.EARLY_STOPPING_PATIENCE), device=DEVICE,
)
try:
    _l3_resume_trainer.load_checkpoint(filename="last_model.pt")
    _l3_resume_ok = True
except Exception as _res_err:
    _l3_resume_ok = False
    logger.error("Trainer.load_checkpoint(last_model.pt) failed: %s", _res_err)

_gate(_l3_resume_ok, "Trainer.load_checkpoint(last_model.pt) succeeds", "")
_gate(
    _l3_resume_ok
    and _l3_resume_trainer.current_epoch == _l1_last_pt.get("epoch"),
    "Resumed current_epoch matches last_model.pt epoch field",
    f"{_l3_resume_trainer.current_epoch if _l3_resume_ok else 'N/A'} "
    f"== {_l1_last_pt.get('epoch')}",
)
del _l3_resume_model, _l3_resume_optimizer, _l3_resume_scheduler, _l3_resume_scaler

_gate(
    hasattr(production_trainer, "fit") and callable(production_trainer.fit),
    "Production Trainer.fit() is callable (Stage 30 interface intact)",
    "",
)
_gate(
    hasattr(production_trainer, "load_checkpoint")
    and callable(production_trainer.load_checkpoint),
    "Production Trainer.load_checkpoint() callable (Stage 30 interface intact)",
    "",
)
_gate(
    hasattr(production_trainer, "save_checkpoint")
    and callable(production_trainer.save_checkpoint),
    "Production Trainer.save_checkpoint() callable (Stage 30 interface intact)",
    "",
)

# ── CHANGED BLOCK [2] ─────────────────────────────────────────────────────────
# Old check (FRAGILE): len(production_controller._epoch_end_callbacks) == 4
#
# Root cause: the hardcoded count 4 predates Stage 36 Ext, which intentionally
# registered ArtifactPersistenceManager as a 5th callback. The count was always
# fragile -- any future callback extension would break it regardless of correctness.
#
# Replacement: type-name-based verification.
#   1. All four Stages-32-35 callbacks must be present.
#   2. No callback type may appear more than once (no stale rerun duplicates).
#   3. If ArtifactPersistenceManager is registered, it must come after
#      LastCheckpointManager (the ordering invariant that _assert_lcm_ran enforces
#      at runtime; verify it statically here too).
#
# This check is future-proof: adding a 6th or 7th callback (e.g. a TensorBoard
# logger or WandB callback) will not produce a false failure as long as the
# required four are present and unique.
# ─────────────────────────────────────────────────────────────────────────────────

_registered_cb_types = [
    type(cb).__name__
    for cb in production_controller._epoch_end_callbacks
]
_required_cb_types = [
    "EpochLogger",
    "TrainingHistoryManager",
    "BestModelManager",
    "LastCheckpointManager",
]

print()
print(f"  Callback registry ({len(_registered_cb_types)} registered):")
for _ci, _cbt in enumerate(_registered_cb_types, 1):
    print(f"    [{_ci}] {_cbt}")
print()

for _req in _required_cb_types:
    _gate(
        _req in _registered_cb_types,
        f"Required callback present: {_req}",
        f"registered={_registered_cb_types}",
    )

_gate(
    len(_registered_cb_types) == len(set(_registered_cb_types)),
    "No duplicate callback types registered (rerun-safe)",
    f"types={_registered_cb_types}",
)

if (
    "LastCheckpointManager"      in _registered_cb_types
    and "ArtifactPersistenceManager" in _registered_cb_types
):
    _lcm_idx = _registered_cb_types.index("LastCheckpointManager")
    _apm_idx = _registered_cb_types.index("ArtifactPersistenceManager")
    _gate(
        _lcm_idx < _apm_idx,
        "LastCheckpointManager is registered before ArtifactPersistenceManager "
        "(ordering invariant)",
        f"lcm_idx={_lcm_idx}, apm_idx={_apm_idx}",
    )
# ── END CHANGED BLOCK [2] ─────────────────────────────────────────────────────

print()

# ============================================================
# Final gate
# ============================================================
print(f"  {_checks_run - len(_fails)} / {_checks_run} checks passed")
print(f"  Failures : {len(_fails)}")
if _fails:
    for _f in _fails:
        print(f"    ✘  {_f}")
print()
if _fails:
    raise AssertionError(
        f"Stage 39 Engineering Verification FAILED "
        f"({len(_fails)} check(s)): {_fails}"
    )
print("  ALL ENGINEERING VERIFICATION CHECKS PASSED")
print()

# -- Persist verification summary -----------------------------------------------
_verif_summary = {
    "project":          cfg.PROJECT_NAME,
    "sprint":            "04",
    "phase":              "Phase 6 - Production Training Pipeline",
    "stage":               39,
    "stage_name":           "Engineering Verification",
    "generated_at_utc":      _time.strftime("%Y-%m-%dT%H:%M:%SZ", _time.gmtime()),
    "layers_verified": [
        "Layer 1 - Artifact completeness, integrity & epoch-log consistency",
        "Layer 2 - Training quality gates",
        "Layer 3 - Compatibility and migration readiness",
    ],
    "checks_run":           _checks_run,
    "checks_passed":         _checks_run - len(_fails),
    "failures":               _fails,
    "all_passed":              len(_fails) == 0,
    "best_epoch":               _best_epoch_l2,
    "best_val_macro_auroc":      round(_best_auroc_val, 6) if _best_auroc_val else None,
    "best_val_loss":              round(_best_loss_val, 6) if _best_loss_val else None,
    "epochs_trained":              len(_val_losses),
    "training_history_entries":     len(_hist_epochs_l1),
    "epoch_log_entries":             len(_log_epochs_l1),
    "registered_callbacks":           _registered_cb_types,
}
_vs_path = cfg.OUTPUT_DIR / "engineering_verification_summary.json"
with open(_vs_path, "w") as _f:
    _json.dump(_verif_summary, _f, indent=2, default=str)
print_check(
    f"Saved {_vs_path.name}",
    _vs_path.exists() and _vs_path.stat().st_size > 0,
    str(_vs_path),
)
print()
print("Stage 39 - Engineering Verification : OK")

  STAGE 39 - ENGINEERING VERIFICATION

  Layer 1 — Artifact completeness & integrity
  ------------------------------------------------------------
  ✔  Artifact present & non-empty: best_model.pt  (/kaggle/working/visionserveai/sprint04/checkpoints/best_model.pt)
  ✔  Artifact present & non-empty: last_model.pt  (/kaggle/working/visionserveai/sprint04/checkpoints/last_model.pt)
  ✔  Artifact present & non-empty: training_history.json  (/kaggle/working/visionserveai/sprint04/metrics/training_history.json)
  ✔  Artifact present & non-empty: training_summary.json  (/kaggle/working/visionserveai/sprint04/training_summary.json)
  ✔  Artifact present & non-empty: checkpoint_summary.json  (/kaggle/working/visionserveai/sprint04/checkpoints/checkpoint_summary.json)
  ✔  Artifact present & non-empty: training_report.json  (/kaggle/working/visionserveai/sprint04/training_report.json)
  ✔  Artifact present & non-empty: epoch_logs.json  (/kaggle/working/visionserveai/sprint04/metrics/epoch_logs.j

In [51]:
# ============================================================
# VisionServeAI | Sprint 04
# Stage 40: Sprint 04 Freeze
# ============================================================

print_section("SPRINT 04 — FINAL FREEZE")
print()

# -- Verify all required artifacts are present ----------------
print("  Final artifact checklist")
print("  " + "-" * 60)

_freeze_artifacts = {
    # -- Checkpoints --
    "best_model.pt":                cfg.CHECKPOINT_DIR / "best_model.pt",
    "last_model.pt":                 cfg.CHECKPOINT_DIR / "last_model.pt",
    # -- Metrics / history --
    "training_history.json":          cfg.METRICS_DIR    / "training_history.json",
    "epoch_logs.json":                  cfg.METRICS_DIR    / "epoch_logs.json",
    "best_model_metadata.json":          cfg.METRICS_DIR    / "best_model_metadata.json",
    # -- Summary / report --
    "training_summary.json":              cfg.OUTPUT_DIR     / "training_summary.json",
    "training_report.json":                cfg.OUTPUT_DIR     / "training_report.json",
    "checkpoint_summary.json":              cfg.CHECKPOINT_DIR / "checkpoint_summary.json",
    "engineering_verification_summary.json": cfg.OUTPUT_DIR   / "engineering_verification_summary.json",
    "production_controller_summary.json":      cfg.OUTPUT_DIR  / "production_controller_summary.json",
    # -- Sprint 03 frozen (read-only, must still exist) --
    "train_manifest.parquet":               TRAIN_MANIFEST_PATH,
    "val_manifest.parquet":                  VAL_MANIFEST_PATH,
    "test_manifest.parquet":                  TEST_MANIFEST_PATH,
    "class_weights.json":                      SPRINT03_CLASS_WEIGHTS_PATH,
    "disease_registry.json":                    DISEASE_REGISTRY_PATH,
}

_freeze_fails: List[str] = []
for name, path in _freeze_artifacts.items():
    _ok = path.exists() and path.stat().st_size > 0
    print_check(f"{name}", _ok, str(path))
    if not _ok:
        _freeze_fails.append(name)

print()
if _freeze_fails:
    raise AssertionError(f"Sprint 04 Freeze FAILED -- missing artifacts: {_freeze_fails}")
print("  ALL ARTIFACTS PRESENT")
print()

# -- Frozen module registry -----------------------------------
print("  Frozen module registry (Sprint 04, Stages 1-40)")
print("  " + "-" * 60)

_frozen_modules = [
    ("Stage 01",  "Imports + environment"),
    ("Stage 02",  "Sprint03 artifact paths"),
    ("Stage 02",  "Utility functions (print_section, print_kv, print_check, require_*)"),
    ("Stage 03",  "TrainingConfig                     -> training/config.py"),
    ("Stage 04",  "Reproducibility (set_global_seed)  -> training/config.py"),
    ("Stage 05",  "DeviceInfo / detect_device()        -> training/config.py"),
    ("Stage 06",  "create_output_dirs()                -> training/config.py"),
    ("Stage 07",  "validate_training_config()          -> training/config.py"),
    ("Stage 08",  "BackboneSpec / BackboneRegistry / build_model() -> training/models/backbones.py"),
    ("Stage 09",  "Backbone initialisation (DenseNet121 pretrained)"),
    ("Stage 10",  "replace_classifier / ChestXrayClassifier -> training/models/classifier.py"),
    ("Stage 11",  "freeze_backbone()                   -> training/models/classifier.py"),
    ("Stage 12",  "Model verification + class_names from disease_registry"),
    ("Stage 13",  "LossConfig                          -> training/losses.py"),
    ("Stage 14",  "build_bce_loss()                    -> training/losses.py"),
    ("Stage 15",  "load_class_weights()                -> training/losses.py"),
    ("Stage 16",  "criterion_weighted / criterion_unweighted"),
    ("Stage 17",  "Weighted vs unweighted engineering comparison"),
    ("Stage 18",  "OptimizerConfig / build_optimizer() -> training/optimizer.py"),
    ("Stage 19",  "SchedulerConfig / build_scheduler() -> training/scheduler.py"),
    ("Stage 20",  "AMPConfig / build_grad_scaler()     -> training/amp.py"),
    ("Stage 21",  "TrainingStep                        -> training/training_step.py"),
    ("Stage 22",  "GradientConfig                      -> training/gradients.py"),
    ("Stage 23",  "CheckpointManager                   -> training/checkpoint.py"),
    ("Stage 24",  "Training engine verification (1 real mini-batch)"),
    ("Stage 25",  "Phase 4 artifact persistence"),
    ("Stage 26",  "EpochResult / train_one_epoch()     -> training/engine.py"),
    ("Stage 27",  "ValidationResult / validate_one_epoch() -> training/validation.py"),
    ("Stage 28",  "MetricsResult / compute_metrics()   -> training/metrics.py"),
    ("Stage 29",  "EarlyStopping                       -> training/early_stopping.py"),
    ("Stage 29.5","ChestXrayManifestDataset / build_dataloaders() -> training/data.py"),
    ("Stage 30",  "Trainer                             -> training/trainer.py"),
    ("Stage 31",  "ProductionTrainingController        -> training/train.py"),
    ("Stage 32",  "EpochLogger                         -> training/logging.py"),
    ("Stage 33",  "TrainingHistoryManager              -> training/history.py"),
    ("Stage 34",  "BestModelManager                    -> training/callbacks.py"),
    ("Stage 35",  "LastCheckpointManager               -> training/callbacks.py"),
    ("Stage 36",  "check_for_resume / build_run_plan / production run -> training/train.py"),
    ("Stage 37",  "generate_training_report()          -> training/history.py"),
    ("Stage 38",  "Artifact persistence & training_summary.json"),
    ("Stage 39",  "Engineering verification (3-layer gate)"),
    ("Stage 40",  "Sprint 04 Freeze"),
]

for stage, desc in _frozen_modules:
    print(f"    {stage:10s}  {desc}")

print()

# -- Migration summary ----------------------------------------
print("  Repository migration map (notebook -> training/)")
print("  " + "-" * 60)

_migration = {
    "training/config.py":           "TrainingConfig, DeviceInfo, seed utils, dir utils",
    "training/models/backbones.py":  "BackboneSpec, BACKBONE_REGISTRY, get_backbone, build_model",
    "training/models/classifier.py":  "ChestXrayClassifier, replace_classifier, freeze_backbone",
    "training/losses.py":              "LossConfig, build_bce_loss, load_class_weights",
    "training/optimizer.py":            "OptimizerConfig, build_optimizer",
    "training/scheduler.py":             "SchedulerConfig, build_scheduler",
    "training/amp.py":                    "AMPConfig, build_grad_scaler, autocast_context",
    "training/gradients.py":               "GradientConfig",
    "training/training_step.py":            "TrainingStep",
    "training/engine.py":                    "EpochResult, train_one_epoch",
    "training/validation.py":                "ValidationResult, validate_one_epoch",
    "training/metrics.py":                    "MetricsResult, compute_metrics",
    "training/early_stopping.py":             "EarlyStopping",
    "training/checkpoint.py":                  "CheckpointManager",
    "training/data.py":                          "ChestXrayManifestDataset, build_dataloaders",
    "training/trainer.py":                        "Trainer",
    "training/train.py":                           "ProductionTrainingController, check_for_resume, build_run_plan, generate_training_report, persist_training_artifacts",
    "training/logging.py":                          "EpochLogger",
    "training/history.py":                           "TrainingHistoryManager, generate_training_report",
    "training/callbacks.py":                          "BestModelManager, LastCheckpointManager",
}
for module, contents in _migration.items():
    print_kv(module, contents)

print()

# -- Load and display final training metrics ------------------
print("  Final training metrics")
print("  " + "-" * 60)
_final_summary = _json.loads((cfg.OUTPUT_DIR / "training_summary.json").read_text())
_tr = _final_summary.get("training", {})
print_kv("epochs_trained",          _tr.get("epochs_trained"))
print_kv("best_epoch",               _tr.get("best_epoch"))
print_kv("early_stopped",             _tr.get("early_stopped"))
print_kv("best_val_loss",              f"{_tr.get('best_val_loss'):.6f}" if _tr.get('best_val_loss') else "N/A")
print_kv("best_val_macro_auroc",        f"{_tr.get('best_val_macro_auroc'):.4f}" if _tr.get('best_val_macro_auroc') else "N/A")
print()

# -- Sprint 04 complete ---------------------------------------
print("=" * 70)
print()
print("  ██████████████████████████████████████████████████████████")
print("  ██                                                        ██")
print("  ██   VisionServeAI | Sprint 04 | COMPLETE & FROZEN       ██")
print("  ██                                                        ██")
print("  ██   Stages 01 – 40 verified.                            ██")
print("  ██   All 30 modules implemented.                         ██")
print("  ██   All required artifacts on disk.                     ██")
print("  ██   3-layer engineering verification PASSED.            ██")
print("  ██                                                        ██")
print("  ██   Sprint 03  →  FROZEN (read-only manifests intact)   ██")
print("  ██   Sprint 04  →  FROZEN (this notebook, all stages)    ██")
print("  ██                                                        ██")
print("  ██   Next: Sprint 05 — Model Export (ONNX / TensorRT)    ██")
print("  ██                                                        ██")
print("  ██████████████████████████████████████████████████████████")
print()
print("=" * 70)
print()
print("Stage 40 - Sprint 04 Freeze : OK")
print()
print_section("SPRINT 04 — PHASE 6 COMPLETE")

  SPRINT 04 — FINAL FREEZE

  Final artifact checklist
  ------------------------------------------------------------
  ✔  best_model.pt  (/kaggle/working/visionserveai/sprint04/checkpoints/best_model.pt)
  ✔  last_model.pt  (/kaggle/working/visionserveai/sprint04/checkpoints/last_model.pt)
  ✔  training_history.json  (/kaggle/working/visionserveai/sprint04/metrics/training_history.json)
  ✔  epoch_logs.json  (/kaggle/working/visionserveai/sprint04/metrics/epoch_logs.json)
  ✔  best_model_metadata.json  (/kaggle/working/visionserveai/sprint04/metrics/best_model_metadata.json)
  ✔  training_summary.json  (/kaggle/working/visionserveai/sprint04/training_summary.json)
  ✔  training_report.json  (/kaggle/working/visionserveai/sprint04/training_report.json)
  ✔  checkpoint_summary.json  (/kaggle/working/visionserveai/sprint04/checkpoints/checkpoint_summary.json)
  ✔  engineering_verification_summary.json  (/kaggle/working/visionserveai/sprint04/engineering_verification_summary.json)
  ✔  pr